In [ ]:
# ---------------------------------------------------------
# Precision Agriculture, TUM, Germany
# Time, Start from 20.01.2026
# Author, Qiang Sun, qiang.sun@tum.de
# Discribation, this Code is for Global NUE calculate，Including：
# 1) Dataset Processing
# 2) NUE calculating
# ---------------------------------------------------------

In [ ]:
# ---------------------------------------------------------
# First Step
# Data processing, Input datasets
# ---------------------------------------------------------

In [ ]:
# ========================================
# Input fertilizer
# Dataset 1 GCSN
# ----02GCSN_Fer_Maize/N_application_rate_Maize.h5----
# Show the Information
# ========================================

import h5py
file_path = 'Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/01DownloadDatasets/02Fer_Maize/02GCSN_Fer_Maize/N_application_rate_Maize.h5'

with h5py.File(file_path, 'r') as f:
    print("📂 HDF5 File structure：")

    def print_structure(name, obj):
        if isinstance(obj, h5py.Group):
            print(f"Group   ⤷ {name}/")
        elif isinstance(obj, h5py.Dataset):
            print(f"Dataset ⤷ {name} | shape: {obj.shape}, dtype: {obj.dtype}")
    f.visititems(print_structure)

In [ ]:
# ========================================
# Input fertilizer
# Dataset 1 GCSN
# Processing GCSN Synthetic N, Calculating the Maize synthetic N
# Processing Global_cropspecific_nitrogen_fertilization_dataset_in_1961_2020 dataset https://www.nature.com/articles/s41597-023-02526-z?fromPaywallRec=false
# Some important Parameters of the oriangal dataset, （1）.h5 format (2)5-arc-min grid cell
# ========================================

import h5py
import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.crs import CRS
import os

# === input and output path ===
h5_path = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/01DownloadDatasets/02Fer_Maize/02GCSN_Fer_Maize/N_application_rate_Maize.h5"
CropS = 'Maize' #Barley,Cassava,Cotton,Fruits,Groundnut, Maize,Millet, Oilpalm, Others crops, Potato, Rapeseed, Rice,Rye, Sorghum, Sugarbeet,Sugarcane, Sunflower,Sweetpotato, Vegetables,Wheat
output_dir = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/02StandardizedDatasets/02Fer_Maize_Standardized"

os.makedirs(output_dir, exist_ok=True)

# === Ignored variables ===
#For Synthetic N
exclude_vars = {'CR_deep', 'CR_surface', 'MA_deep', 'MA_surface'} 

# === Set the year range ===
years = list(range(1995, 2021))  
year_indices = [y - 1961 for y in years]  # Assume index 0 corresponds to 1961

# === open HDF5 files ===
with h5py.File(h5_path, 'r') as file:
    # Read any variable and get size information
    sample_dataset = file[list(file.keys())[0]]
    _, height, width = sample_dataset.shape

    pixel_size = 360 / width
    transform = from_origin(-180, 90, pixel_size, pixel_size)

    # === Traverse the year ===
    for i, year in zip(year_indices, years):
        total = np.full((height, width), np.nan, dtype=np.float32)

        # Iterate over variables and accumulate
        for name in file.keys():
            if name in exclude_vars:
                print(f"⏭️ Skipping variable：{name}")
                continue

            arr = file[name][i, :, :]
            valid = ~np.isnan(arr)
            total[valid] = np.nansum([total[valid], arr[valid]], axis=0)

        # === Flip and rotation correction ===
        data_corrected = np.flipud(np.rot90(total, k=1))

        # Update georeference information
        new_height, new_width = data_corrected.shape
        pixel_size_lon = 360 / new_width
        pixel_size_lat = 180 / new_height
        new_transform = from_origin(-180, 90, pixel_size_lon, pixel_size_lat)

        # === Save GeoTIFF ===
        output_path = os.path.join(output_dir, f"GCSN_FerN_{CropS}_{year}.tif")
        with rasterio.open(
            output_path,
            'w',
            driver='GTiff',
            height=new_height,
            width=new_width,
            count=1,
            dtype='float32',
            crs=CRS.from_epsg(4326),
            transform=new_transform,
            nodata=np.nan
        ) as dst:
            dst.write(data_corrected, 1)

        print(f"✅ Finished：{output_path}")

print("🎯All Finished, Please Check Results!")

In [ ]:
# Processing NPKGRIDSv1.08_maize, Calculating the Maize synthetic N
# Show the Information
import xarray as xr
ds = xr.open_dataset("Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/01DownloadDatasets/02Fer_Maize/02NPKGRIDS_Fer_Maize/NPKGRIDSv1.08_maize.nc", decode_times=False)
print(ds)

In [ ]:
# ============================================================
# Input fertilizer
# Processing NPKGRIDSv1.08_maize, Calculating the Maize synthetic N
# Transform from .nc to .tif
# ============================================================

import numpy as np
import xarray as xr
import rasterio
from rasterio.transform import from_origin
import pandas as pd
from pathlib import Path

# ======================================
# Set File path
# ======================================
input_nc = Path("Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/01DownloadDatasets/02Fer_Maize/02NPKGRIDS_Fer_Maize/NPKGRIDSv1.08_maize.nc")
output_dir = Path("Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/03TempData/02NPKGRIDS_Fer_Maize_Temp")
output_dir.mkdir(parents=True, exist_ok=True)

output_tif = output_dir / "NPKGRIDS_FerN_Maize_2020.tif"

# open NetCDF file
ds = xr.open_dataset(input_nc)

# Extract and squeeze out redundant dimensions, such as time.
nrat = ds['Nrate'].squeeze()
lat = ds['lat'].values
lon = ds['lon'].values

# ======================================
# 🚨 Check if the order of the lat (latitude from north to south) and nrat data needs to be reversed.
# ======================================
if lat[0] < lat[-1]:
    lat = lat[::-1]
    nrat = nrat[::-1, :]

# ======================================
# 🚨 Check if the lon (from west to east) needs to be flipped.
# ======================================
if lon[0] > lon[-1]:
    lon = lon[::-1]
    nrat = nrat[:, ::-1]

# Calculate resolution
res_x = abs(lon[1] - lon[0])
res_y = abs(lat[1] - lat[0])

# Create a transform using the top left corner as the origin.
transform = from_origin(lon.min(), lat.max(), res_x, res_y)

# Write GeoTIFF
with rasterio.open(
    output_tif,
    'w',
    driver='GTiff',
    height=nrat.shape[0],
    width=nrat.shape[1],
    count=1,
    dtype=nrat.dtype,
    crs='EPSG:4326',
    transform=transform
) as dst:
    dst.write(nrat.values, 1)

print(f"✅ GeoTIFF finished：{output_tif}")

In [ ]:
# ============================================================
# Input fertilizer
# dataset 2 Processing NPKGRIDSv1.08_maize
# Processing NPKGRIDSv1.08_maize
# Tools，very useful, Unify into global data., 0.083333333
# 3 arcmin → 5 arcmin
# ============================================================
import os
import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.warp import reproject, Resampling

# ===Parameter settings ===
res = 0.083333333
crs = "EPSG:4326"

# path
input_dir = "Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/03TempData/02NPKGRIDS_Fer_Maize_Step1Temp"         
output_dir = "Z:\staff\Qiang_S\09_Datasets\01N_Input\01Global_SyntheticFertilizer\03TempData\02NPKGRIDS_Fer_Maize_Step2Temp" 

# Calculate global grid size
width = int(360 / res)
height = int(180 / res)

#Start constructing the affine transformation from the top left corner.
transform = from_origin(-180, 90, res, res)

# Global output profile (no nodata value)
global_profile = {
    'driver': 'GTiff',
    'height': height,
    'width': width,
    'count': 1,
    'dtype': 'float32',
    'crs': crs,
    'transform': transform,
    'nodata': -9999  # nodata
}

os.makedirs(output_dir, exist_ok=True)

# === Batch processing ===
for root, _, files in os.walk(input_dir):
    for file in files:
        if file.endswith(".tif"):
            input_path = os.path.join(root, file)
            output_path = os.path.join(output_dir, file)

            with rasterio.open(input_path) as src:
                dst_array = np.full((height, width), np.nan, dtype='float32')

                reproject(
                    source=rasterio.band(src, 1),
                    destination=dst_array,
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=crs,
                    resampling=Resampling.average, # this is an important set
                    dst_nodata= -9999
                )

                with rasterio.open(output_path, "w", **global_profile) as dst:
                    dst.write(dst_array, 1)

print("✅ All images have been standardized to a global scale; output complete!")

In [ ]:
# ============================================================
# Input fertilizer
# dataset 2 Processing NPKGRIDSv1.08_maize
# useful tools,🎉🎉🎉🎉🎉，one datasets, first Multiply ratio and Remove specific values
# ---the last step
# ============================================================

from osgeo import gdal
import numpy as np
import os
import glob

# =====================================================
# User Parameters
# =====================================================

# Input / Output folders
input_folder = r"Z:\staff\Qiang_S\09_Datasets\01N_Input\01Global_SyntheticFertilizer\03TempData\02NPKGRIDS_Fer_Maize_Step2Temp"
output_folder = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/02StandardizedDatasets/02NPKGRIDS_Fer_Maize_Standardized"
os.makedirs(output_folder, exist_ok=True)

# ---- Step 1: Multiply ratio ----
ratio = 1     # e.g. 6, 0.001, 10.5

# ---- Step 2: Remove specific values ----
replace_values = [-1]   # values to be treated as invalid

# ---- Step 3: Remove value range ----
range_min = -2
range_max = 0       # large number

# ---- Final NoData value ----
nodata_value = -9999

# =====================================================
# Batch Processing
# =====================================================
tif_files = glob.glob(os.path.join(input_folder, "*.tif"))
print(f"Found {len(tif_files)} GeoTIFF files")

for i, in_file in enumerate(tif_files, 1):
    fname = os.path.basename(in_file)
    out_file = os.path.join(output_folder, fname)

    print(f"[{i}/{len(tif_files)}] Processing: {fname}")

    ds = gdal.Open(in_file)
    band = ds.GetRasterBand(1)

    # Read original nodata
    ori_nodata = band.GetNoDataValue()

    arr = band.ReadAsArray().astype(np.float32)

    # -------------------------------------------------
    # 1) Convert original nodata → NaN
    # -------------------------------------------------
    if ori_nodata is not None:
        arr[arr == ori_nodata] = np.nan

    # -------------------------------------------------
    # 2) Multiply by ratio
    # -------------------------------------------------
    arr = arr * ratio

    # -------------------------------------------------
    # 3) Replace specific values → NaN
    # -------------------------------------------------
    for v in replace_values:
        arr[arr == v] = np.nan

    # -------------------------------------------------
    # 4) Replace values within range → NaN
    #    Condition: range_min < value < range_max
    # -------------------------------------------------
    mask_range = (arr > range_min) & (arr < range_max)
    arr[mask_range] = np.nan

    # -------------------------------------------------
    # 5) Convert all NaN → unified nodata
    # -------------------------------------------------
    arr = np.where(np.isnan(arr), nodata_value, arr)

    # -------------------------------------------------
    # 6) Write output GeoTIFF
    # -------------------------------------------------
    driver = gdal.GetDriverByName("GTiff")
    out_ds = driver.Create(
        out_file,
        ds.RasterXSize,
        ds.RasterYSize,
        1,
        gdal.GDT_Float32,
        options=["COMPRESS=DEFLATE", "PREDICTOR=2", "TILED=YES"]
    )

    out_ds.SetGeoTransform(ds.GetGeoTransform())
    out_ds.SetProjection(ds.GetProjection())

    out_band = out_ds.GetRasterBand(1)
    out_band.SetNoDataValue(nodata_value)
    out_band.WriteArray(arr)

    out_ds.FlushCache()
    out_ds = None
    ds = None

print("\n✅ ALL FILES FINISHED")
print(f"Output directory: {output_folder}")

In [ ]:
# ========================================
# Input fertilizer
# Dataset 3 LUH2v2
# # Show the Information
# ========================================

import xarray as xr
ds = xr.open_dataset("Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/01DownloadDatasets/02Fer_Maize/02LUH2v2_Fer_Maize/fertilizer_application_histsoc_mai_1850-2015.nc", decode_times=False)
print(ds)

In [ ]:
# =====================================================
# ---Input fertilizer---
# ---Dataset 3 LUH2v2---
# ---Step2 transform formate, .nc to .tif
# =====================================================
import os
import numpy as np
import xarray as xr
import rasterio
from rasterio.transform import from_origin

# =========================
# User configuration
# =========================
nc_path = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/01DownloadDatasets/02Fer_Maize/02LUH2v2_Fer_Maize/fertilizer_application_histsoc_mai_1850-2015.nc"
out_dir = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/03TempData/LUH2v2_fer_Step1Temp"

start_year = 1995
end_year   = 2015

var_name = "fertrate"
nodata_value = -9999

os.makedirs(out_dir, exist_ok=True)

# =========================
# Open NetCDF
# =========================
ds = xr.open_dataset(nc_path)

data = ds[var_name]          # (time, lat, lon)
lats = ds["latitude"].values
lons = ds["longitude"].values
years = ds["time"].values

# =========================
# Calculate spatial info
# =========================
res_x = abs(lons[1] - lons[0])
res_y = abs(lats[0] - lats[1])

# Top-left corner
transform = from_origin(
    lons.min() - res_x / 2,
    lats.max() + res_y / 2,
    res_x,
    res_y
)

# =========================
# Loop through years
# =========================
for year in range(start_year, end_year + 1):

    if year not in years:
        print(f"⚠ Year {year} not found, skipped")
        continue

    print(f"Processing year {year}")

    # Extract one year
    arr = data.sel(time=year).values.astype(np.float32)

    # Handle NaN → NoData
    arr = np.where(np.isnan(arr), nodata_value, arr)

    out_tif = os.path.join(out_dir, f"fertrate_{year}.tif")

    profile = {
        "driver": "GTiff",
        "height": arr.shape[0],
        "width": arr.shape[1],
        "count": 1,
        "dtype": "float32",
        "crs": "EPSG:4326",
        "transform": transform,
        "nodata": nodata_value,
        "compress": "LZW"
    }

    with rasterio.open(out_tif, "w", **profile) as dst:
        dst.write(arr, 1)

    print(f"✔ Saved: {out_tif}")

print("\n✅ All years finished")

In [ ]:
# =====================================================
# ---Input fertilizer---
# ---Dataset 3 LUH2v2---
# ---Step3 Resemple---useful tool--🎉🎉🎉🎉🎉🎉🎉🎉🎉
# =====================================================
import os
import glob
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_origin

# =========================
# User configuration
# =========================
input_dir  = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/03TempData/LUH2v2_fer_Step1Temp"
output_dir = r"Z:\staff\Qiang_S\09_Datasets\01N_Input\01Global_SyntheticFertilizer\02StandardizedDatasets\02LUH2v2_Fer_Maize_Standardized"

os.makedirs(output_dir, exist_ok=True)

nodata = -9999

# 5 arcmin resolution
out_res = 5.0 / 60.0  # 0.08333333333333333

# Global extent
xmin, ymin, xmax, ymax = -180, -90, 180, 90
out_width  = int((xmax - xmin) / out_res)   # 4320
out_height = int((ymax - ymin) / out_res)   # 2160

out_transform = from_origin(xmin, ymax, out_res, out_res)

# =========================
# Batch resampling
# =========================
tif_files = sorted(glob.glob(os.path.join(input_dir, "*.tif")))
print(f"Found {len(tif_files)} files")

for tif in tif_files:
    fname = os.path.basename(tif)
    out_tif = os.path.join(output_dir, fname)

    print(f"Resampling: {fname}")

    with rasterio.open(tif) as src:
        src_data = src.read(1)
        src_transform = src.transform
        src_crs = src.crs
        src_nodata = src.nodata

        # Prepare output array
        dst_data = np.full(
            (out_height, out_width),
            nodata,
            dtype=np.float32
        )

        # Reproject / resample
        reproject(
            source=src_data,
            destination=dst_data,
            src_transform=src_transform,
            src_crs=src_crs,
            src_nodata=src_nodata,
            dst_transform=out_transform,
            dst_crs="EPSG:4326",
            dst_nodata=nodata,
            resampling=Resampling.average   # resample, bilinear, nearest, average等
        )

        profile = {
            "driver": "GTiff",
            "height": out_height,
            "width": out_width,
            "count": 1,
            "dtype": "float32",
            "crs": "EPSG:4326",
            "transform": out_transform,
            "nodata": nodata,
            "compress": "LZW"
        }

        with rasterio.open(out_tif, "w", **profile) as dst:
            dst.write(dst_data, 1)

    print(f"✔ Saved: {out_tif}")

print("\n✅ All files resampled to 5 arcmin")

In [ ]:
# ========================================
# Input fertilizer
# Dataset 4 HaNi
# Processing HaNi_Global_CropSpecific_Nitrogen_Fertilization1860_2019, Calculating the Maize synthetic N 🎉🎉🎉🎉
# ========================================
# read the information
import xarray as xr

ds = xr.open_dataset("Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/01DownloadDatasets/09Fer_Total/HaNi_Global_CropSpecific_Nitrogen_Fertilization1860_2019/nfer_crop_nh4/nfer_crop_nh4.nc", decode_times=False)
ds.time.attrs = {}  # 可选：清掉 units/calendar 属性避免警告
print(ds)

In [ ]:
# Processing HaNi_Global_CropSpecific_Nitrogen_Fertilization1860_2019, Calculating the Maize synthetic N 🎉🎉🎉🎉
# This data processing includes steps such as data fusion.
# formate, .nc to .tif

import xarray as xr
import numpy as np
import rasterio
from rasterio.transform import from_bounds
import os

# ==================== 1. Read file + Manually create real year coordinates ====================
nc_path = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/01DownloadDatasets/09Fer_Total/HaNi_Global_CropSpecific_Nitrogen_Fertilization1860_2019/nfer_crop_nh4/nfer_crop_nh4.nc"
ds = xr.open_dataset(nc_path, decode_times=False)

# Key: Manually calculate the actual year (1925 + time value)
reference_year = 1925   # fertilizer data back to 1925    manure data back to 1860    //The base year written in units
years = np.floor(ds.time.values + reference_year).astype(int)   # Rounding down to the nearest integer year

# Replace the time coordinate with the actual year.
ds = ds.assign_coords(time=xr.DataArray(
    data   = np.array([f"{y}-01-01" for y in years], dtype='datetime64[Y]'),
    dims   = 'time',
    name   = 'time'
))

# Select variables (please modify according to the actual print(ds)).
da = ds['nfer_crop_nh4'] 

print(f"time range：{da.time.dt.year.min().values} – {da.time.dt.year.max().values}")

# ==================== 2. 计算地理 transform ====================
lon = da.lon.values
lat = da.lat.values
dx = lon[1] - lon[0]   # 通常是 0.5
dy = lat[0] - lat[1]   # 注意纬度是倒序，dy 是正的

transform = from_bounds(
    west   = lon.min() - dx/2,
    south  = lat.min() - dy/2,
    east   = lon.max() + dx/2,
    north  = lat.max() + dy/2,
    width  = len(lon),
    height = len(lat)
)

# ==================== 3. Create folders and export year by year====================
output_folder = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/03TempData/HaNi_Global_CropSpecific_Nitrogen_Fertilization1860_2019_Temp/nfer_crop_nh4_tif5min"
os.makedirs(output_folder, exist_ok=True)

for year in da.time.dt.year.values:
    print(f"Exporting {year} year...")
    
    data_2d = da.sel(time=da.time.dt.year == year).squeeze()
    
    # # Handling missing values ​​(HaNi commonly uses 1e20)
    array = data_2d.values.astype('float32')
    array[array > 1e10] = np.nan   # Optional: Convert to NaN for better standardization.
    
    out_path = os.path.join(output_folder, f"nfer_crop_nh4_{year}.tif")
    
    with rasterio.open(
        out_path, 'w',
        driver='GTiff',
        height=array.shape[0],
        width=array.shape[1],
        count=1,
        dtype='float32',
        crs='EPSG:4326',
        transform=transform,
        compress='deflate',
        predictor=3,
        nodata=np.nan,          # NaN is recommended as nodata.
        tiled=False
    ) as dst:
        dst.write(array, 1)

print("All Finished")

In [ ]:
# Processing HaNi_Global_CropSpecific_Nitrogen_Fertilization1860_2019, Calculating the Maize synthetic N 🎉🎉🎉🎉
# read the information
import xarray as xr

ds = xr.open_dataset("Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/01DownloadDatasets/09Fer_Total/HaNi_Global_CropSpecific_Nitrogen_Fertilization1860_2019/nfer_crop_no3/nfer_crop_no3.nc", decode_times=False)
ds.time.attrs = {}  # 可选：清掉 units/calendar 属性避免警告
print(ds)

In [ ]:
# Processing HaNi_Global_CropSpecific_Nitrogen_Fertilization1860_2019, Calculating the Maize synthetic N 🎉🎉🎉🎉
# This data processing includes steps such as data fusion.
# formate, .nc to .tif

import xarray as xr
import numpy as np
import rasterio
from rasterio.transform import from_bounds
import os

# ==================== 1. Read file + Manually create real year coordinates ====================
nc_path = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/01DownloadDatasets/09Fer_Total/HaNi_Global_CropSpecific_Nitrogen_Fertilization1860_2019/nfer_crop_no3/nfer_crop_no3.nc"
ds = xr.open_dataset(nc_path, decode_times=False)

# Key: Manually calculate the actual year (1925 + time value)
reference_year = 1925   # fertilizer data back to 1925    manure data back to 1860    //The base year written in units
years = np.floor(ds.time.values + reference_year).astype(int)   # Rounding down to the nearest integer year

# Replace the time coordinate with the actual year.
ds = ds.assign_coords(time=xr.DataArray(
    data   = np.array([f"{y}-01-01" for y in years], dtype='datetime64[Y]'),
    dims   = 'time',
    name   = 'time'
))

# Select variables (please modify according to the actual print(ds)).
da = ds['nfer_crop_no3'] 

print(f"time range：{da.time.dt.year.min().values} – {da.time.dt.year.max().values}")

# ==================== 2. 计算地理 transform ====================
lon = da.lon.values
lat = da.lat.values
dx = lon[1] - lon[0]   # 通常是 0.5
dy = lat[0] - lat[1]   # 注意纬度是倒序，dy 是正的

transform = from_bounds(
    west   = lon.min() - dx/2,
    south  = lat.min() - dy/2,
    east   = lon.max() + dx/2,
    north  = lat.max() + dy/2,
    width  = len(lon),
    height = len(lat)
)

# ==================== 3. Create folders and export year by year====================
output_folder = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/03TempData/HaNi_Global_CropSpecific_Nitrogen_Fertilization1860_2019_Temp/nfer_crop_no3_tif5min"
os.makedirs(output_folder, exist_ok=True)

for year in da.time.dt.year.values:
    print(f"Exporting {year} year...")
    
    data_2d = da.sel(time=da.time.dt.year == year).squeeze()
    
    # # Handling missing values ​​(HaNi commonly uses 1e20)
    array = data_2d.values.astype('float32')
    array[array > 1e10] = np.nan   # Optional: Convert to NaN for better standardization.
    
    out_path = os.path.join(output_folder, f"nfer_crop_no3_{year}.tif")
    
    with rasterio.open(
        out_path, 'w',
        driver='GTiff',
        height=array.shape[0],
        width=array.shape[1],
        count=1,
        dtype='float32',
        crs='EPSG:4326',
        transform=transform,
        compress='deflate',
        predictor=3,
        nodata=np.nan,          # NaN is recommended as nodata.
        tiled=False
    ) as dst:
        dst.write(array, 1)

print("All Finished")

In [ ]:
# useful tools，two datasets plus,🎉🎉🎉🎉🎉
import os
import re
import rasterio
import numpy as np

# ====set file path==========================
folder_A = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/03TempData/HaNi_Global_CropSpecific_Nitrogen_Fertilization1860_2019_Temp/nfer_crop_nh4_tif5min"    # first folder
folder_B = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/03TempData/HaNi_Global_CropSpecific_Nitrogen_Fertilization1860_2019_Temp/nfer_crop_no3_tif5min"    # second folder
output_folder = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/03TempData/HaNi_Global_CropSpecific_Nitrogen_Fertilization1860_2019_Temp/HaNi_Nh4_No3_tif5min"  # output
os.makedirs(output_folder, exist_ok=True)

# ==============================
# 1.Extract years from a folder → Dictionary
# ==============================
def scan_folder(folder):
    tif_dict = {}
    for file in os.listdir(folder):
        if file.lower().endswith(".tif"):
            # Extract the last 4 digits of the year
            match = re.search(r"(\d{4})(?=\.tif$)", file)
            if match:
                year = match.group(1)
                tif_dict[year] = os.path.join(folder, file)
    return tif_dict

dict_A = scan_folder(folder_A)
dict_B = scan_folder(folder_B)

print(f"A Find the year：{list(dict_A.keys())}")
print(f"B Find the year：{list(dict_B.keys())}")

# ==============================
# 2. Find common years
# ==============================
common_years = sorted(set(dict_A.keys()) & set(dict_B.keys()))
print(f"common years：{common_years}")

if not common_years:
    raise ValueError("❌ The two folders do not contain .tif files with the same year！")

# ==============================
# 3. Add each pair and output
# ==============================
for year in common_years:
    tif_A = dict_A[year]
    tif_B = dict_B[year]

    print(f"Processing year {year}:")
    print(f"  A: {tif_A}")
    print(f"  B: {tif_B}")

    # Read two grids
    with rasterio.open(tif_A) as srcA:
        arr_A = srcA.read(1)
        meta = srcA.meta.copy()
    
    with rasterio.open(tif_B) as srcB:
        arr_B = srcB.read(1)

    # Check dimensional consistency
    if arr_A.shape != arr_B.shape:
        raise ValueError(f"❌ {year} the two files have different shapes.：{arr_A.shape} vs {arr_B.shape}")

    # nodata
    nodata = meta.get("nodata", -9999)
    mask = (arr_A == nodata) | (arr_B == nodata)

    # plus
    arr_new = arr_A.astype("float32") + arr_B.astype("float32")
    arr_new[mask] = nodata

    # output
    out_path = os.path.join(output_folder, f"sum_{year}.tif")
    with rasterio.open(out_path, "w", **meta) as dst:
        dst.write(arr_new, 1)

    print(f"✔ Output: {out_path}")

print("\n🎉 All Finished！")

In [ ]:
# ---Step1--successful ------calculate the Mount of N---for specific crop Fer N rate---
import os
import h5py
import numpy as np
import rasterio
from rasterio.transform import from_origin

# ---------------------------Set parameters ---------------------------
# input file path
har_file = "Z:/staff/Qiang_S/09_Datasets/09OtherDataUsed/GCSN_HarvestedArea_1995_2020/01DownloadDatasets/Harvested_area_1961-2020.h5"  # Harvested_area file
fertilizer_folder = "Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/01DownloadDatasets/09Fer_Total/GCSN_GlobalCropSpecificNitrogenFertilizationDataset1961_2020"  # Fertilizater rate file

# output path
output_folder = "Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/03TempData/GCSN_GlobalCropSpecificNitrogenFertilizationDataset_Temp/Step1Fer_Total"
os.makedirs(output_folder, exist_ok=True)

# set year
start_year = 1995
end_year = 2020
years = list(range(start_year, end_year + 1))

# Crops and corresponding fertilization rate documents
crops = [
    "Barley" , "Cassava", "Cotton" , "Fruits", "Groundnut", "Maize", "Millet" , "Oilpalm", "Others crops",
    "Potato", "Rapeseed" , "Rice" , "Rye", "Sorghum" , "Soybean" , "Sugarbeet" , "Sugarcane" , "Sweetpotato" , "Vegetables" , "Wheat", "sunflower"]

# attributes of the fertilization rate（...v1,v2,v3）
# SN
fert_attrs = ['AA_deep', 'AA_surface',  'AN_deep', 'AN_surface',  'AP_deep',  'AP_surface', 'AS_deep', 'AS_surface', 'CAN_deep', 'CAN_surface', 'NK_deep', 'NK_surface', 'NPK_deep', 'NPK_surface', 'NS_deep', 'NS_surface', 'ONP_deep', 'ONP_surface', 'ONS_deep', 'ONS_surface', 'Urea_deep', 'Urea_surface']
# MN
# fert_attrs = ['MA_deep', 'MA_surface']
# CR
# fert_attrs = ['CR_deep', 'CR_surface']

# --------------------------- file information ---------------------------
# Assuming 4320 rows (-90~90) and 2160 columns (-180~180), with a cell size of 0.083333°
nrows, ncols = 4320, 2160
cellsize = 360 / ncols  
x_min, y_max = -180, 90  # Top left corner

transform = from_origin(x_min, y_max, cellsize, cellsize)
crs = "EPSG:4326"

# --------------------------- Year Index---------------------------
year_index = {1961 + i: i for i in range(60)}

# --------------------------- Calculated per crop, per year ---------------------------
with h5py.File(har_file, "r") as har_f:
    for crop in crops:
        print(f"Processing {crop}...")
        fert_file = os.path.join(fertilizer_folder, f"N_application_rate_{crop}.h5")

        with h5py.File(fert_file, "r") as fert_f:
            for year in years:
                idx = year_index[year]

                # Read the planting area of ​​the year
                har_layer = har_f[crop][idx, :, :]

                # Read the fertilization rate of the year and overlay multiple attributes.
                fert_sum = np.zeros_like(har_layer, dtype=np.float32)
                has_valid = np.zeros_like(har_layer, dtype=bool)

                for attr in fert_attrs:
                    if attr in fert_f:
                        data = fert_f[attr][idx, :, :]
                        mask = ~np.isnan(data)
                        fert_sum[mask] += data[mask]
                        has_valid |= mask

                # Raster cells with no valid attributes are set to NaN.
                fert_sum[~has_valid] = np.nan

                # Calculate the amount of fertilizer
                result = np.where(np.isnan(har_layer) | np.isnan(fert_sum),
                                  np.nan, har_layer * fert_sum)

                # output GeoTIFF
                out_tif = os.path.join(output_folder, f"{crop}_{year}.tif")
                profile = {
                    'driver': 'GTiff',
                    'height': result.shape[0],
                    'width': result.shape[1],
                    'count': 1,
                    'dtype': 'float32',
                    'crs': crs,
                    'transform': transform,
                    'compress': 'lzw'
                }
                with rasterio.open(out_tif, 'w', **profile) as dst:
                    dst.write(result.astype(np.float32), 1)

print("All done!")

In [ ]:
# Step2--need use tool flip to process obove results,, flip and translate usable Multiple files
# flip and translate usable Multiple files-----Useful tools---❀❀❀❀❀---

import os
import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.crs import CRS
from glob import glob

def fix_tif_orientation(input_folder, output_folder):
    """
Batch read all .tif files in input_folder,
perform: rotate 90° + flip vertically + rebuild spatial reference,
and save to output_folder
    """
    os.makedirs(output_folder, exist_ok=True)

    tif_list = glob(os.path.join(input_folder, "*.tif"))

    print(f"find {len(tif_list)}  tif file，Start processing...\n")

    for idx, tif_path in enumerate(tif_list, 1):

        fname = os.path.basename(tif_path)
        out_path = os.path.join(output_folder, fname)

        print(f"[{idx}/{len(tif_list)}] processing → {fname}")

        # === Read raw data ===
        with rasterio.open(tif_path) as src:
            data = src.read(1)
            profile = src.profile.copy()

        # === Image processing===
        data_rotated = np.rot90(data, k=1)       # Rotate 90 degrees to the left
        data_corrected = np.flipud(data_rotated) # Vertical flip

        # === Update space information===
        new_height, new_width = data_corrected.shape

        pixel_size_lon = 360 / new_width
        pixel_size_lat = 180 / new_height

        transform = from_origin(
            -180,
            90,
            pixel_size_lon,
            pixel_size_lat
        )

        profile.update({
            'height': new_height,
            'width': new_width,
            'transform': transform,
            'crs': CRS.from_epsg(4326),
            'nodata': np.nan,
            'dtype': 'float32'
        })

        # === Write result ===
        with rasterio.open(out_path, 'w', **profile) as dst:
            dst.write(data_corrected.astype(np.float32), 1)

        print(f"✔ Output → {out_path}\n")

    print("🎉 All Finished！")

# ================================
# Example
# ================================
input_folder = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/03TempData/GCSN_GlobalCropSpecificNitrogenFertilizationDataset_Temp/Step1Fer_Total"
output_folder = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/03TempData/GCSN_GlobalCropSpecificNitrogenFertilizationDataset_Temp/Step2Fer_Total"

fix_tif_orientation(input_folder, output_folder)

In [ ]:
# ---Step3----Useful for calculate the N-ratio ---successful -----for specific crop Fer N rate---
import os
import re
import numpy as np
import rasterio
from rasterio.transform import from_origin

# ============================================================
# 1. Standardized crop names (important)
# ============================================================
def normalize_crop(name):
    return name.lower().replace(" ", "").replace("_", "").replace("-", "")
# ============================================================
# 2. Automatically scan the folder to obtain {year: {crop_norm: file_path}}
# ============================================================
def scan_crop_files(folder):
    pattern = re.compile(r"(.+?)_(\d{4})\.tif$")

    year_crop_files = {}

    for f in os.listdir(folder):
        m = pattern.match(f)
        if not m:
            continue

        crop_raw = m.group(1)
        year = int(m.group(2))
        crop_norm = normalize_crop(crop_raw)

        if year not in year_crop_files:
            year_crop_files[year] = {}

        year_crop_files[year][crop_norm] = os.path.join(folder, f)
    return year_crop_files

# ============================================================
# 3. Read the tif string as a float32 (to avoid memory errors).
# ============================================================
def read_tif(fp):
    with rasterio.open(fp) as src:
        arr = src.read(1).astype(np.float32)
        meta = src.meta.copy()
    return arr, meta

# ============================================================
# 4. Fertilizer application ratio calculation: CropA / TotalCrops
# ============================================================
def compute_crop_ratio(cropA_raw, folder, out_folder, years=None):

    os.makedirs(out_folder, exist_ok=True)

    # Standardized CropA Name
    cropA = normalize_crop(cropA_raw)

    # Scan all tif files
    year_crop_files = scan_crop_files(folder)

    # Get all years
    if years is None:
        years = sorted(year_crop_files.keys())

    for year in years:

        # Determine if CropA exists in this year.
        if year not in year_crop_files or cropA not in year_crop_files[year]:
            print(f"CropA ({cropA_raw}) missing in {year}, skip.")
            continue

        # Reading molecules CropA
        cropA_fp = year_crop_files[year][cropA]
        numerator, meta = read_tif(cropA_fp)

        # If all molecules are empty (rare), output an empty image directly.
        if np.all(np.isnan(numerator)):
            out_fp = os.path.join(out_folder, f"{cropA_raw}_Ratio_{year}.tif")
            with rasterio.open(out_fp, "w", **meta) as dst:
                dst.write(numerator, 1)
            print(f"{cropA_raw} {year} numerator all NaN, saved empty raster.")
            continue

        # Construct the denominator: Total = the sum of all crops
        total = np.zeros_like(numerator, dtype=np.float32)

        for crop_norm, fp in year_crop_files[year].items():

            # Read the crop image
            arr, _ = read_tif(fp)

            # If arr is NaN, it is not considered; that is, it is treated as 0.
            arr_nonan = np.nan_to_num(arr, nan=0.0)

            total += arr_nonan

        # -----------------------------------------------------------
        # Denominator Handling
        # Rules:
        # 1) total == 0 → Output NaN
        # 2) numerator is NaN → Output NaN
        # 3) Normal calculation: ratio = numerator / total
        # 4) ratio > 1 → Set to 1
        # -----------------------------------------------------------
        ratio = np.full_like(total, np.nan, dtype=np.float32)

        valid_mask = (~np.isnan(numerator)) & (total > 0)

        ratio[valid_mask] = numerator[valid_mask] / total[valid_mask]

        # Limit ratio ≤ 1
        ratio[ratio > 1] = 1

        # Output file
        out_fp = os.path.join(out_folder, f"{cropA_raw}_Ratio_{year}.tif")

        with rasterio.open(out_fp, "w", **meta) as dst:
            dst.write(ratio, 1)

        print(f"Saved: {out_fp}")

# ============================================================
# 5. Example
# ============================================================
# Input folder: All 21 crops * 26 years of TIF files
in_folder = "Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/03TempData/GCSN_GlobalCropSpecificNitrogenFertilizationDataset_Temp/Step2Fer_Total"
out_folder = "Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/03TempData/GCSN_GlobalCropSpecificNitrogenFertilizationDataset_Temp/Step3Maize_Fer_Ratio1995_2020"

# Calculate the fertilizer application ratio for maize (1995-2020)
compute_crop_ratio(
    cropA_raw="Maize",  # Maize,rice
    folder=in_folder,
    out_folder=out_folder,
    years=range(1995, 2021)
)

In [ ]:
# useful tools,🎉🎉🎉🎉🎉，two datasets Multiplication/Times
# -Step4-----for specific crop Fer N rate (maize)---

import os
import re
import numpy as np
import rasterio

# ===========================================
# Set file path
# ===========================================
folder1 = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/03TempData/GCSN_GlobalCropSpecificNitrogenFertilizationDataset_Temp/Step3Maize_Fer_Ratio1995_2020"   # input 1
folder2 = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/03TempData/HaNi_Global_CropSpecific_Nitrogen_Fertilization1860_2019_Temp/HaNi_Nh4_No3_tif5min"   # input 2
output_folder = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/03TempData/GCSN_GlobalCropSpecificNitrogenFertilizationDataset_Temp/Step4Maize_FerTotal_1995_2020"  # Output

output_nodata = -9999
os.makedirs(output_folder, exist_ok=True)

print("===Batch match TIFFs by year and multiply pixel by pixel.===")

# ===========================================
# Utility function — Extract the year from a filename (the last 4 digits of the filename).
# ===========================================
def extract_year(fname):
    match = re.search(r"(\d{4})\.tif$", fname)
    return match.group(1) if match else None

# ===========================================
# Utility function — One-to-one TIFF pixel-by-pixel multiplication
# ===========================================
def multiply_two_tif(path1, path2, out_path, out_nodata=-9999):

    with rasterio.open(path1) as src1, rasterio.open(path2) as src2:

        #Basic checks
        if (src1.width != src2.width) or (src1.height != src2.height):
            raise ValueError(f"size Inconsistent:\n{path1}\n{path2}")

        # if src1.transform != src2.transform:
            #  raise ValueError(f"transform Inconsistent:\n{path1}\n{path2}")

        if src1.crs != src2.crs:
            raise ValueError(f"CRS Inconsistent:\n{path1}\n{path2}")

        profile = src1.profile.copy()
        profile.update({
            "dtype": "float32",
            "nodata": out_nodata,
            "count": 1,
            "compress": "lzw"
        })

        nodata1 = src1.nodata
        nodata2 = src2.nodata

        with rasterio.open(out_path, "w", **profile) as dst:

            # Process by block (suitable for large images)
            for (idx, window) in src1.block_windows(1):

                a1 = src1.read(1, window=window).astype("float32")
                a2 = src2.read(1, window=window).astype("float32")

                mask = np.zeros_like(a1, dtype=bool)

                # nodata / nan 
                if nodata1 is not None:
                    mask |= (a1 == nodata1)
                if nodata2 is not None:
                    mask |= (a2 == nodata2)

                mask |= np.isnan(a1) | np.isnan(a2)

                result = (a1 * a2).astype("float32")

                # Invalid position assignment nodata
                result[mask] = out_nodata

                dst.write(result, 1, window=window)

    print(f"✔ Output: {out_path}")

# ===========================================
# Collect files and match them by year
# ===========================================
files1 = {extract_year(f): f for f in os.listdir(folder1) if f.endswith(".tif")}
files2 = {extract_year(f): f for f in os.listdir(folder2) if f.endswith(".tif")}

years = sorted(set(files1.keys()) & set(files2.keys()))

print(f"match year: {years}")

# ===========================================
# Multiply by year in batches
# ===========================================
for year in years:
    f1 = os.path.join(folder1, files1[year])
    f2 = os.path.join(folder2, files2[year])
    out_path = os.path.join(output_folder, f"mul_{year}.tif")

    print(f"\n===Processing Year {year} ===")
    multiply_two_tif(f1, f2, out_path, output_nodata)

print("\n=== All Finished ===")

In [ ]:
# ============================================================
# Step5: Output Harvested Area (1995–2020) for Specific Crops
# ============================================================
import os
import h5py
import numpy as np
import rasterio
from rasterio.transform import from_origin

# ------------------------------------------------------------
# Input & Output Paths
# ------------------------------------------------------------
h5_path = (r'Z:/staff/Qiang_S/09_Datasets/09OtherDataUsed/GCSN_HarvestedArea_1995_2020/01DownloadDatasets/Harvested_area_1961-2020.h5')
output_dir = r'Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/03TempData/GCSN_GlobalCropSpecificNitrogenFertilizationDataset_Temp/Step5Maize_HarArea_1995_2020'
os.makedirs(output_dir, exist_ok=True)

# ------------------------------------------------------------
# Variables to exclude
# ------------------------------------------------------------
# exclude_vars = {'Barley','Cassava','Cotton','Fruits','Groundnut','Maize','Millet','Oilpalm','Others crops','Potato','Rapeseed','Rye','Sorghum','Soybean','Sugarbeet','Sugarcane','Sweetpotato','Vegetables','Wheat','sunflower'} #Rice
exclude_vars = {'Barley','Cassava','Cotton','Fruits','Groundnut','Millet','Oilpalm','Others crops','Potato','Rapeseed','Rice','Rye','Sorghum','Soybean','Sugarbeet','Sugarcane','Sweetpotato','Vegetables','Wheat','sunflower'} # Maize
# exclude_vars = {'Barley','Cassava','Cotton','Fruits','Groundnut','Millet','Oilpalm','Others crops','Potato','Rapeseed','Rice','Rye','Sorghum','Soybean','Sugarbeet','Sugarcane','Sweetpotato','Vegetables','sunflower', 'Maize'} # Wheat
#  exclude_vars = {}

# ------------------------------------------------------------
# Geographic reference information
# ------------------------------------------------------------
pixel_size = 360 / 4320  # 0.0833333 degree (5 arcmin)
transform = from_origin(-180, 90, pixel_size, pixel_size)

# ------------------------------------------------------------
# Function to save GeoTIFF
# ------------------------------------------------------------
def save_tif(data, out_path):
    """Save a 2D NumPy array as a GeoTIFF file."""
    with rasterio.open(
        out_path,
        'w',
        driver='GTiff',
        height=data.shape[0],
        width=data.shape[1],
        count=1,
        dtype='float32',
        crs='EPSG:4326',      # WGS84
        transform=transform,
        nodata=np.nan,
        compress='lzw'
    ) as dst:
        dst.write(data.astype(np.float32), 1)

# ------------------------------------------------------------
# Read HDF5 and export yearly GeoTIFFs
# ------------------------------------------------------------
with h5py.File(h5_path, 'r') as file:

    # Loop over crop variables
    for var_name in file.keys():

        if var_name in exclude_vars:
            continue

        print(f'Processing crop: {var_name}')
        dataset = file[var_name]  # shape: (60, 4320, 2160)

        # Loop over years: index 34–59 → 1995–2020
        for year in range(1995, 2021):
            idx = year - 1961  # Convert year to array index
            data = dataset[idx, :, :]
            out_name = f'HarArea_{var_name}_{year}.tif'
            out_path = os.path.join(output_dir, out_name)

            save_tif(data, out_path)

        print(f'✔ Finished {var_name}')

print("✅ All GeoTIFFs exported successfully")

In [ ]:
# ============================================================
# Step6--need use tool flip to process obove results,, flip and translate usable Multiple files
# flip and translate usable Multiple files-----Useful tools---❀❀❀❀❀---
# ============================================================

import os
import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.crs import CRS
from glob import glob

def fix_tif_orientation(input_folder, output_folder):
    """
Batch read all .tif files in input_folder,
perform: rotate 90° + flip vertically + rebuild spatial reference,
and save to output_folder
    """
    os.makedirs(output_folder, exist_ok=True)

    tif_list = glob(os.path.join(input_folder, "*.tif"))

    print(f"find {len(tif_list)}  tif file，Start processing...\n")

    for idx, tif_path in enumerate(tif_list, 1):

        fname = os.path.basename(tif_path)
        out_path = os.path.join(output_folder, fname)

        print(f"[{idx}/{len(tif_list)}] processing → {fname}")

        # === Read raw data ===
        with rasterio.open(tif_path) as src:
            data = src.read(1)
            profile = src.profile.copy()

        # === Image processing===
        data_rotated = np.rot90(data, k=1)       # Rotate 90 degrees to the left
        data_corrected = np.flipud(data_rotated) # Vertical flip

        # === Update space information===
        new_height, new_width = data_corrected.shape

        pixel_size_lon = 360 / new_width
        pixel_size_lat = 180 / new_height

        transform = from_origin(
            -180,
            90,
            pixel_size_lon,
            pixel_size_lat
        )

        profile.update({
            'height': new_height,
            'width': new_width,
            'transform': transform,
            'crs': CRS.from_epsg(4326),
            'nodata': np.nan,
            'dtype': 'float32'
        })

        # === Write result ===
        with rasterio.open(out_path, 'w', **profile) as dst:
            dst.write(data_corrected.astype(np.float32), 1)

        print(f"✔ Output → {out_path}\n")

    print("🎉 All Finished！")

# ================================
# Example
# ================================
input_folder = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/03TempData/GCSN_GlobalCropSpecificNitrogenFertilizationDataset_Temp/Step5Maize_HarArea_1995_2020"
output_folder = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/03TempData/GCSN_GlobalCropSpecificNitrogenFertilizationDataset_Temp/Step6Maize_HarArea_1995_2020"

fix_tif_orientation(input_folder, output_folder)

In [ ]:
# ============================================================
# useful tools,🎉🎉🎉🎉🎉，two datasets, a divided by b
# ----Step7-----for specific crop Fer N rate---
# ============================================================

import os
import rasterio
import numpy as np
from scipy.ndimage import distance_transform_edt

def round_sig(x, sig=6):
    """
   This version preserves valid numbers in NumPy arrays without generating errors.
    """
    x = np.asarray(x)   # It is guaranteed to be an array

    # Output array
    out = np.zeros_like(x, dtype="float32")

    # mask：Non-zero and non-NaN portions need to be processed.
    mask = (~np.isnan(x)) & (x != 0)

    # Values ​​to be processed
    xv = x[mask].astype("float64")  # Use float64 to avoid log10 errors

    # Calculate exponent
    exp = np.floor(np.log10(np.abs(xv)))

    # Calculate scaling factor
    factor = np.power(10.0, sig - exp - 1)

    out[mask] = (np.round(xv * factor) / factor).astype("float32")

    #Keep 0 and NaN as is
    out[~mask] = x[~mask].astype("float32")

    return out

def fill_nearest_valid(arr, nodata):
    """Fill in the blanks using nearest neighbors."""
    mask = (arr == nodata) | np.isnan(arr)
    if np.all(~mask):
        return arr

    valid_mask = ~mask
    distances, indices = distance_transform_edt(mask, return_indices=True)
    filled = arr[tuple(indices)]
    return filled


def process_tif_pair(A_path, B_path, output_path, nodata=-9999):
    """Ratio calculation for single-file pairs"""

    with rasterio.open(A_path) as srcA, rasterio.open(B_path) as srcB:
        A = srcA.read(1).astype("float32")
        B = srcB.read(1).astype("float32")

        meta = srcA.meta.copy()
        meta.update(nodata=nodata, dtype="float32")

        A_nodata = srcA.nodata if srcA.nodata is not None else nodata
        B_nodata = srcB.nodata if srcB.nodata is not None else nodata

        A_mask = (A == A_nodata) | np.isnan(A)
        B_mask = (B == B_nodata) | np.isnan(B)

        # ---- STEP 1：若 A>0 且 B 空 → 最近邻填补 ----
        B_filled = B.copy()
        cond_fill = (~A_mask) & (A > 0) & (B_mask)
        if np.any(cond_fill):
            print("  → Perform B nearest neighbor fill...")
            B_nn = fill_nearest_valid(B, B_nodata)
            B_filled[cond_fill] = B_nn[cond_fill]

        # ---- STEP 2：Calculation results ----
        result = np.full_like(A, nodata, dtype="float32")

        valid = (~A_mask) & (A > 0) & (~((B_filled == B_nodata) | np.isnan(B_filled)))

        result[valid] = A[valid] / B_filled[valid]

        # ---- STEP 3：Retain 6 significant digits ----
        result[valid] = round_sig(result[valid], sig=6)

        # ---- Write down the results ----
        with rasterio.open(output_path, "w", **meta) as dst:
            dst.write(result, 1)

        print(f"✔Output complete: {output_path}")


def batch_ratio(folder_A, folder_B, folder_out, nodata=-9999):
    """Batch A/B pixel-by-pixel division"""

    os.makedirs(folder_out, exist_ok=True)

    files_A = [f for f in os.listdir(folder_A) if f.lower().endswith(".tif")]
    years_A = {f[-8:-4]: f for f in files_A}

    files_B = [f for f in os.listdir(folder_B) if f.lower().endswith(".tif")]
    years_B = {f[-8:-4]: f for f in files_B}

    common_years = sorted(set(years_A.keys()) & set(years_B.keys()))

    print(f"Find matching year: {common_years}")

    for year in common_years:
        A_file = os.path.join(folder_A, years_A[year])
        B_file = os.path.join(folder_B, years_B[year])
        out_file = os.path.join(folder_out, f"HaNi_FerN_Maize_{year}.tif")

        print(f"\n=== Processing year {year}: {years_A[year]} / {years_B[year]} ===")
        process_tif_pair(A_file, B_file, out_file, nodata=nodata)

    print("\n🎉 All Done！")

batch_ratio(
folder_A = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/03TempData/GCSN_GlobalCropSpecificNitrogenFertilizationDataset_Temp/Step4Maize_FerTotal_1995_2020", 
folder_B = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/03TempData/GCSN_GlobalCropSpecificNitrogenFertilizationDataset_Temp/Step6Maize_HarArea_1995_2020", 
folder_out = r"Z:\staff\Qiang_S\09_Datasets\01N_Input\01Global_SyntheticFertilizer\03TempData\GCSN_GlobalCropSpecificNitrogenFertilizationDataset_Temp\Step7HaNiMaize_FerN_1995_2019",
nodata=-9999
)

In [ ]:
# ============================================================
# useful tools,🎉🎉🎉🎉🎉，one datasets, first Multiply ratio and Remove specific values
# ---the last step-Step8-----for specific crop Fer N rate---
# ============================================================

from osgeo import gdal
import numpy as np
import os
import glob

# =====================================================
# User Parameters
# =====================================================

# Input / Output folders
input_folder = r"Z:\staff\Qiang_S\09_Datasets\01N_Input\01Global_SyntheticFertilizer\03TempData\GCSN_GlobalCropSpecificNitrogenFertilizationDataset_Temp\Step7HaNiMaize_FerN_1995_2019"
output_folder = r"Z:\staff\Qiang_S\09_Datasets\01N_Input\01Global_SyntheticFertilizer\02StandardizedDatasets\02HaNi_Fer_Maize_Standardized"
os.makedirs(output_folder, exist_ok=True)

# ---- Step 1: Multiply ratio ----
ratio = 0.001     # e.g. 6, 0.001, 10.5

# ---- Step 2: Remove specific values ----
replace_values = [0]   # values to be treated as invalid

# ---- Step 3: Remove value range ----
range_min = 700.1
range_max = 1e10       # large number

# ---- Final NoData value ----
nodata_value = -9999

# =====================================================
# Batch Processing
# =====================================================
tif_files = glob.glob(os.path.join(input_folder, "*.tif"))
print(f"Found {len(tif_files)} GeoTIFF files")

for i, in_file in enumerate(tif_files, 1):
    fname = os.path.basename(in_file)
    out_file = os.path.join(output_folder, fname)

    print(f"[{i}/{len(tif_files)}] Processing: {fname}")

    ds = gdal.Open(in_file)
    band = ds.GetRasterBand(1)

    # Read original nodata
    ori_nodata = band.GetNoDataValue()

    arr = band.ReadAsArray().astype(np.float32)

    # -------------------------------------------------
    # 1) Convert original nodata → NaN
    # -------------------------------------------------
    if ori_nodata is not None:
        arr[arr == ori_nodata] = np.nan

    # -------------------------------------------------
    # 2) Multiply by ratio
    # -------------------------------------------------
    arr = arr * ratio

    # -------------------------------------------------
    # 3) Replace specific values → NaN
    # -------------------------------------------------
    for v in replace_values:
        arr[arr == v] = np.nan

    # -------------------------------------------------
    # 4) Replace values within range → NaN
    #    Condition: range_min < value < range_max
    # -------------------------------------------------
    mask_range = (arr > range_min) & (arr < range_max)
    arr[mask_range] = np.nan

    # -------------------------------------------------
    # 5) Convert all NaN → unified nodata
    # -------------------------------------------------
    arr = np.where(np.isnan(arr), nodata_value, arr)

    # -------------------------------------------------
    # 6) Write output GeoTIFF
    # -------------------------------------------------
    driver = gdal.GetDriverByName("GTiff")
    out_ds = driver.Create(
        out_file,
        ds.RasterXSize,
        ds.RasterYSize,
        1,
        gdal.GDT_Float32,
        options=["COMPRESS=DEFLATE", "PREDICTOR=2", "TILED=YES"]
    )

    out_ds.SetGeoTransform(ds.GetGeoTransform())
    out_ds.SetProjection(ds.GetProjection())

    out_band = out_ds.GetRasterBand(1)
    out_band.SetNoDataValue(nodata_value)
    out_band.WriteArray(arr)

    out_ds.FlushCache()
    out_ds = None
    ds = None

print("\n✅ ALL FILES FINISHED")
print(f"Output directory: {output_folder}")

In [ ]:
# ============================================================
# 01Global_SyntheticFertilizer finished
# Second Step
# 02Global_Manure, Data processing, Input datasets
# ============================================================

In [ ]:
# Processing GCSN Synthetic N, Manure N, Residues N
# Processing Global_cropspecific_nitrogen_fertilization_dataset_in_1961_2020 dataset https://www.nature.com/articles/s41597-023-02526-z?fromPaywallRec=false
# Some important Parameters of the oriangal dataset, （1）.h5 format (2)5-arc-min grid cell
# Step 2 calculate the maize synthetic N

import h5py
import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.crs import CRS
import os

# === input and output path ===
h5_path = r"Z:\staff\Qiang_S\09_Datasets\01N_Input\02Global_Manure\01DownloadDatasets\02Man_Maize\02GCSN_Maize/N_application_rate_Maize.h5"
CropS = 'Maize' #Barley,Cassava,Cotton,Fruits,Groundnut, Maize,Millet, Oilpalm, Others crops, Potato, Rapeseed, Rice,Rye, Sorghum, Sugarbeet,Sugarcane, Sunflower,Sweetpotato, Vegetables,Wheat
output_dir = r"Z:\staff\Qiang_S\09_Datasets\01N_Input\02Global_Manure\02StandardizedDatasets\02GCSN_Man_Maize_Standardized" 

os.makedirs(output_dir, exist_ok=True)

# === Ignored variables ===
#For Manure N
exclude_vars = {'AA_deep', 'AA_surface', 'AN_deep', 'AN_surface', 'AP_deep', 'AP_surface', 'AS_deep', 'AS_surface', 'CAN_deep', 'CAN_surface', 'CR_deep', 'CR_surface', 'NK_deep', 'NK_surface', 'NPK_deep', 'NPK_surface', 'NS_deep', 'NS_surface', 'ONP_deep', 'ONP_surface', 'ONS_deep', 'ONS_surface', 'Urea_deep', 'Urea_surface'}  

# === Set the year range ===
years = list(range(1995, 2021)) 
year_indices = [y - 1961 for y in years]

# === open HDF5 files ===
with h5py.File(h5_path, 'r') as file:
    # Read any variable and get size information
    sample_dataset = file[list(file.keys())[0]]
    _, height, width = sample_dataset.shape

    pixel_size = 360 / width
    transform = from_origin(-180, 90, pixel_size, pixel_size)

    # === Traverse the year ===
    for i, year in zip(year_indices, years):
        total = np.full((height, width), np.nan, dtype=np.float32)

        # Iterate over variables and accumulate
        for name in file.keys():
            if name in exclude_vars:
                print(f"⏭️ Skipping variable：{name}")
                continue

            arr = file[name][i, :, :]
            valid = ~np.isnan(arr)
            total[valid] = np.nansum([total[valid], arr[valid]], axis=0)

        # === Flip and rotation correction ===
        data_corrected = np.flipud(np.rot90(total, k=1))

        # Update georeference information
        new_height, new_width = data_corrected.shape
        pixel_size_lon = 360 / new_width
        pixel_size_lat = 180 / new_height
        new_transform = from_origin(-180, 90, pixel_size_lon, pixel_size_lat)

        # === Save GeoTIFF ===
        output_path = os.path.join(output_dir, f"{CropS}_ManN_{year}.tif")
        with rasterio.open(
            output_path,
            'w',
            driver='GTiff',
            height=new_height,
            width=new_width,
            count=1,
            dtype='float32',
            crs=CRS.from_epsg(4326),
            transform=new_transform,
            nodata=np.nan
        ) as dst:
            dst.write(data_corrected, 1)

        print(f"✅ Finished：{output_path}")

print("🎯All Finished, Please Check Results!")

In [ ]:
# ============================================================
# Processing HaNi_Global_CropSpecific_Nitrogen_Fertilization1860_2019, Calculating the Maize Manure N 🎉🎉🎉🎉
# read the information
# ============================================================
import xarray as xr

ds = xr.open_dataset("Z:/staff/Qiang_S/09_Datasets/01N_Input/02Global_Manure/01DownloadDatasets/09Man_Total/HaNi_GlobalCropSpecificNitrogenFertilization1860_2019/nmanure_app_crop/nmanure_app_crop.nc", decode_times=False)
ds.time.attrs = {}  # 可选：清掉 units/calendar 属性避免警告
print(ds)

In [ ]:
# ============================================================
# Processing HaNi_Global_CropSpecific_Nitrogen_Fertilization1860_2019, Calculating the Maize synthetic N 🎉🎉🎉🎉
# This data processing includes steps such as data fusion.
# formate, .nc to .tif
# ============================================================

import xarray as xr
import numpy as np
import rasterio
from rasterio.transform import from_bounds
import os

# ==================== 1. Read file + Manually create real year coordinates ====================
nc_path = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/02Global_Manure/01DownloadDatasets/09Man_Total/HaNi_GlobalCropSpecificNitrogenFertilization1860_2019/nmanure_app_crop/nmanure_app_crop.nc"
ds = xr.open_dataset(nc_path, decode_times=False)

# Key: Manually calculate the actual year (1925 + time value)
reference_year = 1860   # fertilizer data back to 1925    manure data back to 1860    //The base year written in units
years = np.floor(ds.time.values + reference_year).astype(int)   # Rounding down to the nearest integer year

# Replace the time coordinate with the actual year.
ds = ds.assign_coords(time=xr.DataArray(
    data   = np.array([f"{y}-01-01" for y in years], dtype='datetime64[Y]'),
    dims   = 'time',
    name   = 'time'
))

# Select variables (please modify according to the actual print(ds)).
da = ds['nmanure_app_crop'] 

print(f"time range：{da.time.dt.year.min().values} – {da.time.dt.year.max().values}")

# ==================== 2. 计算地理 transform ====================
lon = da.lon.values
lat = da.lat.values
dx = lon[1] - lon[0]   # 通常是 0.5
dy = lat[0] - lat[1]   # 注意纬度是倒序，dy 是正的

transform = from_bounds(
    west   = lon.min() - dx/2,
    south  = lat.min() - dy/2,
    east   = lon.max() + dx/2,
    north  = lat.max() + dy/2,
    width  = len(lon),
    height = len(lat)
)

# ==================== 3. Create folders and export year by year====================
output_folder = r"Z:\staff\Qiang_S\09_Datasets\01N_Input\02Global_Manure\03TempData/HaNi_Global_CropSpecific_Nitrogen_Fertilization1860_2019_Temp/nmanure_app_crop_tif5min"
os.makedirs(output_folder, exist_ok=True)

for year in da.time.dt.year.values:
    print(f"Exporting {year} year...")
    
    data_2d = da.sel(time=da.time.dt.year == year).squeeze()
    
    # # Handling missing values ​​(HaNi commonly uses 1e20)
    array = data_2d.values.astype('float32')
    array[array > 1e10] = np.nan   # Optional: Convert to NaN for better standardization.
    
    out_path = os.path.join(output_folder, f"nmanure_app_crop_{year}.tif")
    
    with rasterio.open(
        out_path, 'w',
        driver='GTiff',
        height=array.shape[0],
        width=array.shape[1],
        count=1,
        dtype='float32',
        crs='EPSG:4326',
        transform=transform,
        compress='deflate',
        predictor=3,
        nodata=np.nan,          # NaN is recommended as nodata.
        tiled=False
    ) as dst:
        dst.write(array, 1)

print("All Finished")

In [ ]:
# ============================================================
# ---Step1--successful ------calculate the Mount of N---for specific crop Man N rate---
# ============================================================
import os
import h5py
import numpy as np
import rasterio
from rasterio.transform import from_origin

# ---------------------------Set parameters ---------------------------
# input file path
har_file = "Z:/staff/Qiang_S/09_Datasets/09OtherDataUsed/GCSN_HarvestedArea_1995_2020/01DownloadDatasets/Harvested_area_1961-2020.h5"  # Harvested_area file
fertilizer_folder = "Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/01DownloadDatasets/09Fer_Total/GCSN_GlobalCropSpecificNitrogenFertilizationDataset1961_2020"  # Fertilizater rate file

# output path
output_folder = "Z:/staff/Qiang_S/09_Datasets/01N_Input/02Global_Manure/03TempData/GCSN_GlobalCropSpecificNitrogenFertilizationDataset_Temp/Step1Man_Total"
os.makedirs(output_folder, exist_ok=True)

# set year
start_year = 1995
end_year = 2020
years = list(range(start_year, end_year + 1))

# Crops and corresponding fertilization rate documents
crops = [
    "Barley" , "Cassava", "Cotton" , "Fruits", "Groundnut", "Maize", "Millet" , "Oilpalm", "Others crops",
    "Potato", "Rapeseed" , "Rice" , "Rye", "Sorghum" , "Soybean" , "Sugarbeet" , "Sugarcane" , "Sweetpotato" , "Vegetables" , "Wheat", "sunflower"]

# attributes of the fertilization rate（...v1,v2,v3）
# MN
fert_attrs = ['MA_deep', 'MA_surface']
# CR
# fert_attrs = ['CR_deep', 'CR_surface']

# --------------------------- file information ---------------------------
# Assuming 4320 rows (-90~90) and 2160 columns (-180~180), with a cell size of 0.083333°
nrows, ncols = 4320, 2160
cellsize = 360 / ncols  
x_min, y_max = -180, 90  # Top left corner

transform = from_origin(x_min, y_max, cellsize, cellsize)
crs = "EPSG:4326"

# --------------------------- Year Index---------------------------
year_index = {1961 + i: i for i in range(60)}

# --------------------------- Calculated per crop, per year ---------------------------
with h5py.File(har_file, "r") as har_f:
    for crop in crops:
        print(f"Processing {crop}...")
        fert_file = os.path.join(fertilizer_folder, f"N_application_rate_{crop}.h5")

        with h5py.File(fert_file, "r") as fert_f:
            for year in years:
                idx = year_index[year]

                # Read the planting area of ​​the year
                har_layer = har_f[crop][idx, :, :]

                # Read the fertilization rate of the year and overlay multiple attributes.
                fert_sum = np.zeros_like(har_layer, dtype=np.float32)
                has_valid = np.zeros_like(har_layer, dtype=bool)

                for attr in fert_attrs:
                    if attr in fert_f:
                        data = fert_f[attr][idx, :, :]
                        mask = ~np.isnan(data)
                        fert_sum[mask] += data[mask]
                        has_valid |= mask

                # Raster cells with no valid attributes are set to NaN.
                fert_sum[~has_valid] = np.nan

                # Calculate the amount of fertilizer
                result = np.where(np.isnan(har_layer) | np.isnan(fert_sum),
                                  np.nan, har_layer * fert_sum)

                # output GeoTIFF
                out_tif = os.path.join(output_folder, f"{crop}_{year}.tif")
                profile = {
                    'driver': 'GTiff',
                    'height': result.shape[0],
                    'width': result.shape[1],
                    'count': 1,
                    'dtype': 'float32',
                    'crs': crs,
                    'transform': transform,
                    'compress': 'lzw'
                }
                with rasterio.open(out_tif, 'w', **profile) as dst:
                    dst.write(result.astype(np.float32), 1)

print("All done!")

In [ ]:
# ============================================================
# Step2--need use tool flip to process obove results,, flip and translate usable Multiple files
# flip and translate usable Multiple files-----Useful tools---❀❀❀❀❀---
# ============================================================

import os
import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.crs import CRS
from glob import glob

def fix_tif_orientation(input_folder, output_folder):
    """
Batch read all .tif files in input_folder,
perform: rotate 90° + flip vertically + rebuild spatial reference,
and save to output_folder
    """
    os.makedirs(output_folder, exist_ok=True)

    tif_list = glob(os.path.join(input_folder, "*.tif"))

    print(f"find {len(tif_list)}  tif file，Start processing...\n")

    for idx, tif_path in enumerate(tif_list, 1):

        fname = os.path.basename(tif_path)
        out_path = os.path.join(output_folder, fname)

        print(f"[{idx}/{len(tif_list)}] processing → {fname}")

        # === Read raw data ===
        with rasterio.open(tif_path) as src:
            data = src.read(1)
            profile = src.profile.copy()

        # === Image processing===
        data_rotated = np.rot90(data, k=1)       # Rotate 90 degrees to the left
        data_corrected = np.flipud(data_rotated) # Vertical flip

        # === Update space information===
        new_height, new_width = data_corrected.shape

        pixel_size_lon = 360 / new_width
        pixel_size_lat = 180 / new_height

        transform = from_origin(
            -180,
            90,
            pixel_size_lon,
            pixel_size_lat
        )

        profile.update({
            'height': new_height,
            'width': new_width,
            'transform': transform,
            'crs': CRS.from_epsg(4326),
            'nodata': np.nan,
            'dtype': 'float32'
        })

        # === Write result ===
        with rasterio.open(out_path, 'w', **profile) as dst:
            dst.write(data_corrected.astype(np.float32), 1)

        print(f"✔ Output → {out_path}\n")

    print("🎉 All Finished！")

# ================================
# Example
# ================================
input_folder = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/02Global_Manure/03TempData/GCSN_GlobalCropSpecificNitrogenFertilizationDataset_Temp/Step1Man_Total"
output_folder = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/02Global_Manure/03TempData/GCSN_GlobalCropSpecificNitrogenFertilizationDataset_Temp/Step2Man_Total"

fix_tif_orientation(input_folder, output_folder)

In [ ]:
# ============================================================
# ---Step3----Useful for calculate the N-ratio ---successful -----for specific crop Fer N rate---
# ============================================================

import os
import re
import numpy as np
import rasterio
from rasterio.transform import from_origin

# ============================================================
# 1. Standardized crop names (important)
# ============================================================
def normalize_crop(name):
    return name.lower().replace(" ", "").replace("_", "").replace("-", "")
# ============================================================
# 2. Automatically scan the folder to obtain {year: {crop_norm: file_path}}
# ============================================================
def scan_crop_files(folder):
    pattern = re.compile(r"(.+?)_(\d{4})\.tif$")

    year_crop_files = {}

    for f in os.listdir(folder):
        m = pattern.match(f)
        if not m:
            continue

        crop_raw = m.group(1)
        year = int(m.group(2))
        crop_norm = normalize_crop(crop_raw)

        if year not in year_crop_files:
            year_crop_files[year] = {}

        year_crop_files[year][crop_norm] = os.path.join(folder, f)
    return year_crop_files

# ============================================================
# 3. Read the tif string as a float32 (to avoid memory errors).
# ============================================================
def read_tif(fp):
    with rasterio.open(fp) as src:
        arr = src.read(1).astype(np.float32)
        meta = src.meta.copy()
    return arr, meta

# ============================================================
# 4. Fertilizer application ratio calculation: CropA / TotalCrops
# ============================================================
def compute_crop_ratio(cropA_raw, folder, out_folder, years=None):

    os.makedirs(out_folder, exist_ok=True)

    # Standardized CropA Name
    cropA = normalize_crop(cropA_raw)

    # Scan all tif files
    year_crop_files = scan_crop_files(folder)

    # Get all years
    if years is None:
        years = sorted(year_crop_files.keys())

    for year in years:

        # Determine if CropA exists in this year.
        if year not in year_crop_files or cropA not in year_crop_files[year]:
            print(f"CropA ({cropA_raw}) missing in {year}, skip.")
            continue

        # Reading molecules CropA
        cropA_fp = year_crop_files[year][cropA]
        numerator, meta = read_tif(cropA_fp)

        # If all molecules are empty (rare), output an empty image directly.
        if np.all(np.isnan(numerator)):
            out_fp = os.path.join(out_folder, f"{cropA_raw}_Ratio_{year}.tif")
            with rasterio.open(out_fp, "w", **meta) as dst:
                dst.write(numerator, 1)
            print(f"{cropA_raw} {year} numerator all NaN, saved empty raster.")
            continue

        # Construct the denominator: Total = the sum of all crops
        total = np.zeros_like(numerator, dtype=np.float32)

        for crop_norm, fp in year_crop_files[year].items():

            # Read the crop image
            arr, _ = read_tif(fp)

            # If arr is NaN, it is not considered; that is, it is treated as 0.
            arr_nonan = np.nan_to_num(arr, nan=0.0)

            total += arr_nonan

        # -----------------------------------------------------------
        # Denominator Handling
        # Rules:
        # 1) total == 0 → Output NaN
        # 2) numerator is NaN → Output NaN
        # 3) Normal calculation: ratio = numerator / total
        # 4) ratio > 1 → Set to 1
        # -----------------------------------------------------------
        ratio = np.full_like(total, np.nan, dtype=np.float32)

        valid_mask = (~np.isnan(numerator)) & (total > 0)

        ratio[valid_mask] = numerator[valid_mask] / total[valid_mask]

        # Limit ratio ≤ 1
        ratio[ratio > 1] = 1

        # Output file
        out_fp = os.path.join(out_folder, f"{cropA_raw}_Ratio_{year}.tif")

        with rasterio.open(out_fp, "w", **meta) as dst:
            dst.write(ratio, 1)

        print(f"Saved: {out_fp}")

# 5. Example
# Input folder: All 21 crops * 26 years of TIF files
in_folder = "Z:/staff/Qiang_S/09_Datasets/01N_Input/02Global_Manure/03TempData/GCSN_GlobalCropSpecificNitrogenFertilizationDataset_Temp/Step2Man_Total"
out_folder = "Z:/staff/Qiang_S/09_Datasets/01N_Input/02Global_Manure/03TempData/GCSN_GlobalCropSpecificNitrogenFertilizationDataset_Temp/Step3Maize_Man_Ratio1995_2020"

# Calculate the fertilizer application ratio for maize (1995-2020)
compute_crop_ratio(
    cropA_raw="Maize",  # Maize,rice
    folder=in_folder,
    out_folder=out_folder,
    years=range(1995, 2021)
)

In [ ]:
# ============================================================
# useful tools,🎉🎉🎉🎉🎉，two datasets Multiplication/Times
# -Step4-----for specific crop Fer N rate (maize)---
# ============================================================

import os
import re
import numpy as np
import rasterio

# ===========================================
# Set file path
# ===========================================
folder1 = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/02Global_Manure/03TempData/GCSN_GlobalCropSpecificNitrogenFertilizationDataset_Temp/Step3Maize_Man_Ratio1995_2020"   # input 1
folder2 = r"Z:\staff\Qiang_S\09_Datasets\01N_Input\02Global_Manure\03TempData\HaNi_Global_CropSpecific_Nitrogen_Fertilization1860_2019_Temp\nmanure_app_crop_tif5min"   # input 2
output_folder = r"Z:\staff\Qiang_S\09_Datasets\01N_Input\02Global_Manure\03TempData\GCSN_GlobalCropSpecificNitrogenFertilizationDataset_Temp\Step4Maize_ManTotal_1995_2020"  # Output

output_nodata = -9999
os.makedirs(output_folder, exist_ok=True)

print("===Batch match TIFFs by year and multiply pixel by pixel.===")

# ===========================================
# Utility function — Extract the year from a filename (the last 4 digits of the filename).
# ===========================================
def extract_year(fname):
    match = re.search(r"(\d{4})\.tif$", fname)
    return match.group(1) if match else None

# ===========================================
# Utility function — One-to-one TIFF pixel-by-pixel multiplication
# ===========================================
def multiply_two_tif(path1, path2, out_path, out_nodata=-9999):

    with rasterio.open(path1) as src1, rasterio.open(path2) as src2:

        #Basic checks
        if (src1.width != src2.width) or (src1.height != src2.height):
            raise ValueError(f"size Inconsistent:\n{path1}\n{path2}")

        # if src1.transform != src2.transform:
            #  raise ValueError(f"transform Inconsistent:\n{path1}\n{path2}")

        if src1.crs != src2.crs:
            raise ValueError(f"CRS Inconsistent:\n{path1}\n{path2}")

        profile = src1.profile.copy()
        profile.update({
            "dtype": "float32",
            "nodata": out_nodata,
            "count": 1,
            "compress": "lzw"
        })

        nodata1 = src1.nodata
        nodata2 = src2.nodata

        with rasterio.open(out_path, "w", **profile) as dst:

            # Process by block (suitable for large images)
            for (idx, window) in src1.block_windows(1):

                a1 = src1.read(1, window=window).astype("float32")
                a2 = src2.read(1, window=window).astype("float32")

                mask = np.zeros_like(a1, dtype=bool)

                # nodata / nan 
                if nodata1 is not None:
                    mask |= (a1 == nodata1)
                if nodata2 is not None:
                    mask |= (a2 == nodata2)

                mask |= np.isnan(a1) | np.isnan(a2)

                result = (a1 * a2).astype("float32")

                # Invalid position assignment nodata
                result[mask] = out_nodata

                dst.write(result, 1, window=window)

    print(f"✔ Output: {out_path}")

# ===========================================
# Collect files and match them by year
# ===========================================
files1 = {extract_year(f): f for f in os.listdir(folder1) if f.endswith(".tif")}
files2 = {extract_year(f): f for f in os.listdir(folder2) if f.endswith(".tif")}

years = sorted(set(files1.keys()) & set(files2.keys()))

print(f"match year: {years}")

# ===========================================
# Multiply by year in batches
# ===========================================
for year in years:
    f1 = os.path.join(folder1, files1[year])
    f2 = os.path.join(folder2, files2[year])
    out_path = os.path.join(output_folder, f"mul_{year}.tif")

    print(f"\n===Processing Year {year} ===")
    multiply_two_tif(f1, f2, out_path, output_nodata)

print("\n=== All Finished ===")

In [ ]:
# ============================================================
# Skip two steps, the Harvested Area is same with 5 6
# Step5: Output Harvested Area (1995–2020) for Specific Crops
# Step6：need use tool flip to process obove results,, flip and translate usable Multiple files
# ============================================================

In [ ]:
# ============================================================
# useful tools,🎉🎉🎉🎉🎉，two datasets, a divided by b
# ----Step7-----for specific crop Man N rate--- -Maize---
# ============================================================

import os
import rasterio
import numpy as np
from scipy.ndimage import distance_transform_edt

def round_sig(x, sig=6):
    """
   This version preserves valid numbers in NumPy arrays without generating errors.
    """
    x = np.asarray(x)   # It is guaranteed to be an array

    # Output array
    out = np.zeros_like(x, dtype="float32")

    # mask：Non-zero and non-NaN portions need to be processed.
    mask = (~np.isnan(x)) & (x != 0)

    # Values ​​to be processed
    xv = x[mask].astype("float64")  # Use float64 to avoid log10 errors

    # Calculate exponent
    exp = np.floor(np.log10(np.abs(xv)))

    # Calculate scaling factor
    factor = np.power(10.0, sig - exp - 1)

    out[mask] = (np.round(xv * factor) / factor).astype("float32")

    #Keep 0 and NaN as is
    out[~mask] = x[~mask].astype("float32")

    return out

def fill_nearest_valid(arr, nodata):
    """Fill in the blanks using nearest neighbors."""
    mask = (arr == nodata) | np.isnan(arr)
    if np.all(~mask):
        return arr

    valid_mask = ~mask
    distances, indices = distance_transform_edt(mask, return_indices=True)
    filled = arr[tuple(indices)]
    return filled


def process_tif_pair(A_path, B_path, output_path, nodata=-9999):
    """Ratio calculation for single-file pairs"""

    with rasterio.open(A_path) as srcA, rasterio.open(B_path) as srcB:
        A = srcA.read(1).astype("float32")
        B = srcB.read(1).astype("float32")

        meta = srcA.meta.copy()
        meta.update(nodata=nodata, dtype="float32")

        A_nodata = srcA.nodata if srcA.nodata is not None else nodata
        B_nodata = srcB.nodata if srcB.nodata is not None else nodata

        A_mask = (A == A_nodata) | np.isnan(A)
        B_mask = (B == B_nodata) | np.isnan(B)

        # ---- STEP 1：若 A>0 且 B 空 → 最近邻填补 ----
        B_filled = B.copy()
        cond_fill = (~A_mask) & (A > 0) & (B_mask)
        if np.any(cond_fill):
            print("  → Perform B nearest neighbor fill...")
            B_nn = fill_nearest_valid(B, B_nodata)
            B_filled[cond_fill] = B_nn[cond_fill]

        # ---- STEP 2：Calculation results ----
        result = np.full_like(A, nodata, dtype="float32")

        valid = (~A_mask) & (A > 0) & (~((B_filled == B_nodata) | np.isnan(B_filled)))

        result[valid] = A[valid] / B_filled[valid]

        # ---- STEP 3：Retain 6 significant digits ----
        result[valid] = round_sig(result[valid], sig=6)

        # ---- Write down the results ----
        with rasterio.open(output_path, "w", **meta) as dst:
            dst.write(result, 1)

        print(f"✔Output complete: {output_path}")


def batch_ratio(folder_A, folder_B, folder_out, nodata=-9999):
    """Batch A/B pixel-by-pixel division"""

    os.makedirs(folder_out, exist_ok=True)

    files_A = [f for f in os.listdir(folder_A) if f.lower().endswith(".tif")]
    years_A = {f[-8:-4]: f for f in files_A}

    files_B = [f for f in os.listdir(folder_B) if f.lower().endswith(".tif")]
    years_B = {f[-8:-4]: f for f in files_B}

    common_years = sorted(set(years_A.keys()) & set(years_B.keys()))

    print(f"Find matching year: {common_years}")

    for year in common_years:
        A_file = os.path.join(folder_A, years_A[year])
        B_file = os.path.join(folder_B, years_B[year])
        out_file = os.path.join(folder_out, f"HaNi_ManN_Maize_{year}.tif")

        print(f"\n=== Processing year {year}: {years_A[year]} / {years_B[year]} ===")
        process_tif_pair(A_file, B_file, out_file, nodata=nodata)

    print("\n🎉 All Done！")

batch_ratio(
folder_A = r"Z:\staff\Qiang_S\09_Datasets\01N_Input\02Global_Manure\03TempData\GCSN_GlobalCropSpecificNitrogenFertilizationDataset_Temp\Step4Maize_ManTotal_1995_2020", 
folder_B = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/01Global_SyntheticFertilizer/03TempData/GCSN_GlobalCropSpecificNitrogenFertilizationDataset_Temp/Step6Maize_HarArea_1995_2020", 
folder_out = r"Z:\staff\Qiang_S\09_Datasets\01N_Input\02Global_Manure\03TempData\GCSN_GlobalCropSpecificNitrogenFertilizationDataset_Temp\Step7HaNiMaize_ManN_1995_2019",
nodata=-9999
)

In [ ]:
# ============================================================
# useful tools,🎉🎉🎉🎉🎉，one datasets, first Multiply ratio and Remove specific values
# ---the last step-Step8-----for specific crop Fer N rate---
# ============================================================

from osgeo import gdal
import numpy as np
import os
import glob

# User Parameters

# Input / Output folders
input_folder = r"Z:\staff\Qiang_S\09_Datasets\01N_Input\02Global_Manure\03TempData\GCSN_GlobalCropSpecificNitrogenFertilizationDataset_Temp\Step7HaNiMaize_ManN_1995_2019"
output_folder = r"Z:\staff\Qiang_S\09_Datasets\01N_Input\02Global_Manure\02StandardizedDatasets\02HaNi_Man_Maize_Standardized2"
os.makedirs(output_folder, exist_ok=True)

# ---- Step 1: Multiply ratio ----
ratio = 0.001     # e.g. 6, 0.001, 10.5

# ---- Step 2: Remove specific values ----
replace_values = [0]   # values to be treated as invalid

# ---- Step 3: Remove value range ----
range_min = 300.1
range_max = 1e10       # large number

# ---- Final NoData value ----
nodata_value = -9999

# =====================================================
# Batch Processing
# =====================================================
tif_files = glob.glob(os.path.join(input_folder, "*.tif"))
print(f"Found {len(tif_files)} GeoTIFF files\n")

for i, in_file in enumerate(tif_files, 1):
    fname = os.path.basename(in_file)
    out_file = os.path.join(output_folder, fname)

    print(f"[{i}/{len(tif_files)}] Processing: {fname}")

    ds = gdal.Open(in_file)
    band = ds.GetRasterBand(1)
    ori_nodata = band.GetNoDataValue()

    arr = band.ReadAsArray().astype(np.float32)

    # -----------------------------
    # Statistics: total pixels
    # -----------------------------
    total_pixels = arr.size

    # -----------------------------
    # Original valid pixels
    # -----------------------------
    if ori_nodata is not None:
        valid_original = np.sum(arr != ori_nodata)
        arr[arr == ori_nodata] = np.nan
    else:
        valid_original = np.sum(~np.isnan(arr))

    # -----------------------------
    # Multiply by ratio
    # -----------------------------
    arr = arr * ratio

    # -----------------------------
    # Remove specific values
    # -----------------------------
    for v in replace_values:
        arr[arr == v] = np.nan

    # -----------------------------
    # Remove values in range
    # -----------------------------
    mask_range = (arr > range_min) & (arr < range_max)
    arr[mask_range] = np.nan

    # -----------------------------
    # Final statistics
    # -----------------------------
    valid_final = np.sum(~np.isnan(arr))
    removed_pixels = valid_original - valid_final
    removed_ratio = (removed_pixels / valid_original * 100) if valid_original > 0 else 0

    # -----------------------------
    # Convert NaN → nodata
    # -----------------------------
    arr = np.where(np.isnan(arr), nodata_value, arr)

    # -----------------------------
    # Write output GeoTIFF
    # -----------------------------
    driver = gdal.GetDriverByName("GTiff")
    out_ds = driver.Create(
        out_file,
        ds.RasterXSize,
        ds.RasterYSize,
        1,
        gdal.GDT_Float32,
        options=["COMPRESS=DEFLATE", "PREDICTOR=2", "TILED=YES"]
    )

    out_ds.SetGeoTransform(ds.GetGeoTransform())
    out_ds.SetProjection(ds.GetProjection())

    out_band = out_ds.GetRasterBand(1)
    out_band.SetNoDataValue(nodata_value)
    out_band.WriteArray(arr)

    out_ds.FlushCache()
    out_ds = None
    ds = None

    # -----------------------------
    # Print statistics
    # -----------------------------
    print(f"  Total pixels        : {total_pixels:,}")
    print(f"  Valid (original)    : {valid_original:,}")
    print(f"  Valid (final)       : {valid_final:,}")
    print(f"  Removed pixels      : {removed_pixels:,}")
    print(f"  Removed percentage  : {removed_ratio:.2f}%\n")

print("✅ ALL FILES FINISHED")
print(f"Output directory: {output_folder}")

In [ ]:
# ============================================================
# --N input, Deposition---
# ============================================================

In [ ]:
# ============================================================
# --N input, Deposition---
# useful,🎉🎉🎉🎉🎉,Format conversion, 4_ML.xlsx, to, 4_ML.tif
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.transform import from_origin
from rasterio.features import rasterize

# =====================================================
# User parameters
# =====================================================

excel_path = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/03Global_Deposition/01DownloadDatasets/4_WL.xlsx"
sheet_name = "Data"

shp_path = r"Z:/staff/Qiang_S/09_Datasets/09OtherDataUsed/GDAM_GlobalCountryLevel_Shp/GlobalCountryLevel_Shp.shp"

output_dir = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/03Global_Deposition/02StandardizedDatasets/4_WL_Global5minTotal1995_2020"
os.makedirs(output_dir, exist_ok=True)

nodata_value = -9999
resolution = 5 / 60.0  # 5 arcmin = 0.083333333°

start_year = 1995
end_year = 2020

# =====================================================
# 1. Read Excel
# =====================================================

df = pd.read_excel(excel_path, sheet_name=sheet_name)

# Force all column names to string (CRITICAL)
df.columns = df.columns.astype(str)

# Ensure m49 is integer
df["m49"] = df["m49"].astype(int)

# Safety check: missing year columns
missing_years = [str(y) for y in range(start_year, end_year + 1) if str(y) not in df.columns]
if missing_years:
    raise ValueError(f"Missing year columns in Excel: {missing_years}")

# =====================================================
# 2. Read Shapefile
# =====================================================

gdf = gpd.read_file(shp_path)

# Ensure CRS is WGS84
if gdf.crs is None or gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)

# Ensure M49_Code is integer
gdf["M49_Code"] = gdf["M49_Code"].astype(int)

# =====================================================
# 3. Define global raster grid
# =====================================================

xmin, ymin, xmax, ymax = -180, -90, 180, 90

width = int((xmax - xmin) / resolution)   # 4320
height = int((ymax - ymin) / resolution)  # 2160

transform = from_origin(xmin, ymax, resolution, resolution)

# =====================================================
# 4. Loop through years and rasterize
# =====================================================

for year in range(start_year, end_year + 1):

    print(f"Processing year {year} ...")

    # Build lookup table: m49 -> yearly value
    value_map = dict(zip(df["m49"], df[str(year)]))

    # Attach values to shapefile
    gdf["value"] = gdf["M49_Code"].map(value_map)

    # Prepare shapes (skip NaN values)
    shapes = (
        (geom, val)
        for geom, val in zip(gdf.geometry, gdf["value"])
        if pd.notna(val)
    )

    # Rasterize
    raster = rasterize(
        shapes=shapes,
        out_shape=(height, width),
        transform=transform,
        fill=nodata_value,
        dtype="float32"
    )

    # Output file
    out_tif = os.path.join(output_dir, f"value_{year}.tif")

    with rasterio.open(
        out_tif,
        "w",
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype="float32",
        crs="EPSG:4326",
        transform=transform,
        nodata=nodata_value,
        compress="lzw"
    ) as dst:
        dst.write(raster, 1)

    valid_pixels = np.sum(raster != nodata_value)
    print(f"  → Saved {out_tif}, valid pixels: {valid_pixels}")

print("✅ All years (1995–2020) finished successfully")


In [ ]:
# =====================================================
# --N input, Deposition
# useful,🎉🎉🎉🎉🎉，two datasets （4_ML, MCD12Q2） for Deposition calculation
# =====================================================

import os
import glob
import numpy as np
import rasterio
from scipy import ndimage

def fill_nodata_nearest(array, mask):
    """
    Fills invalid values (indicated by mask) with the nearest valid values.
    Args:
        array: Input 2D numpy array.
        mask: Boolean mask where True indicates invalid (NoData) pixels.
    Returns:
        Filled array.
    """
    # 1. distance_transform_edt calculates distance to the nearest zero (background)
    # We want to find the nearest VALID pixel. So we define Valid=0, Invalid=1.
    # The 'indices' will tell us the coordinates (row, col) of the nearest 0.
    distances, indices = ndimage.distance_transform_edt(
        mask, return_distances=True, return_indices=True
    )
    
    # 2. Use the indices to map values from the nearest valid pixels
    # indices[0] are row coordinates, indices[1] are column coordinates
    filled_array = array[tuple(indices)]
    
    return filled_array

def process_rasters_with_nearest_fill(path_a, folder_b, folder_out):
    """
    Args:
        path_a: Path to the reference raster A (0, 1, 2, 3)
        folder_b: Folder containing B rasters
        folder_out: Output folder
    """
    
    # 1. Setup output directory
    if not os.path.exists(folder_out):
        os.makedirs(folder_out)

    # 2. Read Raster A (Reference)
    print(f"Reading Reference A: {path_a}")
    with rasterio.open(path_a) as src_a:
        a_data = src_a.read(1)
        a_shape = a_data.shape
        
        # --- Pre-calculate Coefficient Matrix ---
        # Initialize with zeros
        coeffs = np.zeros(a_shape, dtype=np.float32)
        
        # Vectorized assignment of coefficients
        coeffs[a_data == 1] = 1.0
        coeffs[a_data == 2] = 0.5
        coeffs[a_data == 3] = 0.333
        
        # Define the mask where A is 0 (Output should be strictly NoData here)
        mask_a_zero = (a_data == 0)

    # 3. Process B Files
    tif_files = glob.glob(os.path.join(folder_b, "*.tif"))
    print(f"Found {len(tif_files)} files in B folder.")

    for b_path in tif_files:
        filename = os.path.basename(b_path)
        out_path = os.path.join(folder_out, f"dep_{filename}")
        
        try:
            with rasterio.open(b_path) as src_b:
                if src_b.shape != a_shape:
                    print(f"Skipping {filename}: Shape mismatch.")
                    continue
                
                # Read data as Float32 to handle NaNs correctly
                b_data = src_b.read(1).astype(np.float32)
                profile = src_b.profile.copy()
                
                # Get the NoData value of B (e.g., -9999)
                b_nodata_val = src_b.nodata

                # --- Step A: Standardize NoData to np.nan ---
                # This makes it easier to track which pixels are "empty"
                if b_nodata_val is not None:
                    # Mark explicit NoData values as NaN
                    b_data[b_data == b_nodata_val] = np.nan

                # Create a mask where B is currently NaN
                # True = Invalid (Needs Fill), False = Valid
                mask_b_invalid = np.isnan(b_data)

                # --- Step B: Nearest Neighbor Filling ---
                # Condition: "If A is not 0, and B is NoData -> use Nearest Neighbor"
                # To be safe and vectorized, we create a fully filled version of B first.
                
                if np.any(mask_b_invalid):
                    # Check if the image is completely empty to avoid crash
                    if np.all(mask_b_invalid):
                         print(f"Warning: {filename} is completely empty. Result will be empty.")
                         b_filled = b_data # Stays NaN
                    else:
                        # Perform the nearest neighbor filling
                        # This fills ALL gaps in B using their nearest valid neighbors
                        b_filled = fill_nodata_nearest(b_data, mask_b_invalid)
                else:
                    b_filled = b_data

                # --- Step C: Multiplication ---
                # Result = Filled B * Coefficients from A
                result = b_filled * coeffs

                # --- Step D: Post-processing ---
                # 1. Round to 3 decimal places
                result = np.round(result, 3)
                
                # 2. Enforce logic: If A was 0, result MUST be NoData (-9999)
                # regardless of what B was.
                result[mask_a_zero] = -9999
                
                # 3. Handle remaining NaNs (if B was fully empty)
                result[np.isnan(result)] = -9999

            # --- Export ---
            profile.update({
                'dtype': 'float32',
                'nodata': -9999,
                'compress': 'lzw'
            })
            
            with rasterio.open(out_path, 'w', **profile) as dst:
                dst.write(result.astype(np.float32), 1)

            print(f"Processed: {filename}")

        except Exception as e:
            print(f"Error processing {filename}: {e}")

# ================= Configuration =================
if __name__ == "__main__":
    # Modify paths here
    FILE_A = r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\MCD12Q24Deposition\MCD12Q2_5min2020.tif"
    FOLDER_B = r"Z:/staff/Qiang_S/09_Datasets/01N_Input/03Global_Deposition/03TempData/4_WL_Global5minTotal1995_2020"
    FOLDER_OUT =  r"Z:/staff/Qiang_S/09_Datasets/01N_Input/03Global_Deposition/02StandardizedDatasets/4_WL_5min4Crops1995_2020"

    process_rasters_with_nearest_fill(FILE_A, FOLDER_B, FOLDER_OUT)

In [ ]:
# =====================================================
# 🎉🎉🎉🎉🎉N input, Grid datasets have been processed🎉🎉🎉🎉🎉
# =====================================================

In [ ]:
# =====================================================
# --N Output---Processing Now----
# =====================================================

In [ ]:
# =====================================================
# --N Output---Processing Now----
# ---Dataset 1, GlobalCropYield5min
# ---Step1 Unified into a global scope 1982-2015-🎉🎉---
# =====================================================
import os
import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.warp import reproject, Resampling

res = 0.083333333  
crs = "EPSG:4326"  

width = int(360 / res)
height = int(180 / res)

transform = from_origin(-180, 90, res, res)

global_profile = {
    'driver': 'GTiff',
    'height': height,
    'width': width,
    'count': 1,
    'dtype': 'float32',
    'crs': crs,
    'transform': transform,
    'nodata': -9999  
}

# Path
input_dir = "Z:/staff/Qiang_S/09_Datasets/02N_Output/02Global_Yield_Maize/01DownloadDatasets/MaizeYield_GlobalCropYield5min"         
output_dir = "Z:/staff/Qiang_S/09_Datasets/02N_Output/02Global_Yield_Maize/02StandardizedDatasets/MaizeYield_GlobalCropYield5min"

os.makedirs(output_dir, exist_ok=True)

for root, _, files in os.walk(input_dir):
    for file in files:
        if file.endswith(".tif"):
            input_path = os.path.join(root, file)
            output_path = os.path.join(output_dir, file)

            with rasterio.open(input_path) as src:
                dst_array = np.full((height, width), np.nan, dtype='float32')

                reproject(
                    source=rasterio.band(src, 1),
                    destination=dst_array,
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=crs,
                    resampling=Resampling.nearest,
                    dst_nodata= -9999
                )

                with rasterio.open(output_path, "w", **global_profile) as dst:
                    dst.write(dst_array, 1)

print("✅ All Done")

In [ ]:
# =====================================================
# ---N Output---Processing Now----
# ---Dataset 1, GlobalCropYield5min
# ---Step2 data fusion, FAO and GlobalCropYield5min, new data 2016,2017🎉🎉---
# =====================================================
import geopandas as gpd
import rasterio
import rasterio.mask
import numpy as np
import pandas as pd
from tqdm import tqdm
import os
from rasterio.features import geometry_mask

# ============================================================
# Part 1 -- input1 parameters
# ============================================================
shp_path = r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\GDAM_GlobalCountryLevel_Shp/GlobalCountryLevel_Shp.shp"         # M49_Code was used as the ID
excel_path = r"Z:\staff\Qiang_S\09_Datasets\02N_Output\02Global_Yield_Maize\01DownloadDatasets\FAO_Yield1995_2020/FAOSTAT_Yield1995_2020.xls"       # FAO excel, contain yield...

tif_base_path = r"Z:\staff\Qiang_S\09_Datasets\02N_Output\02Global_Yield_Maize\02StandardizedDatasets\MaizeYield_GlobalCropYield5min/Maize2015.tif"        # base year .tif
base_year = 2015                                    # base year

target_years = [2016, 2017]                     # want to generate the following years, multiple years are possible.

selected_item_code = 112   # 112 Maize (corn); 111 Wheat

Ratio1Or1000 = 1  # GlobalCropYield5min, SAPM2010,SPAM2005，SPAM2000，GDHY2016_5min
# Ratio1Or1000 = 1000 # SPAM2020, GAEZ2020, GAEZ2015, GAEZ2010, GAEZ2000,
Pre_fileName = "GlobalCropYield5min_Maize_I" #output file pre name

print_check_year = 2016  # You want to print the target year at a specific scale, such as 2019.
output_folder = r"Z:\staff\Qiang_S\09_Datasets\02N_Output\02Global_Yield_Maize\02StandardizedDatasets\MaizeYield_GlobalCropYield5min"   # output folder
nodata_value = -9999                                   # Set No data 

# ============================================================
# Part 2 -- Read data
# ============================================================

# read the base .tif
base_ds = rasterio.open(tif_base_path)
base_data = base_ds.read(1)
base_data = base_data.astype("float32")

# ---------------------------------------------------------
# Part 3 -- Build a dictionary of country scaling ratios (including printouts for proofreading).
# ---------------------------------------------------------

# Read Excel
df_excel = pd.read_excel(excel_path)

# Keep only the relevant columns
required_cols = ["Area Code (M49)", "Item Code (CPC)", "Year", "Value"]
df_excel = df_excel[required_cols]

# Filter selected crops（Item Code）
df_excel = df_excel[df_excel["Item Code (CPC)"] == selected_item_code]

# Filtering base year
df_base = df_excel[df_excel["Year"] == base_year]
base_dict = dict(zip(df_base["Area Code (M49)"], df_base["Value"]))

scaling_dict = {}

for ty in target_years:

    df_t = df_excel[df_excel["Year"] == ty]
    t_dict = dict(zip(df_t["Area Code (M49)"], df_t["Value"]))

    scaling_dict[ty] = {}
    common_countries = set(base_dict.keys()).intersection(set(t_dict.keys()))

    for c in common_countries:

        base_val = base_dict.get(c, None)
        t_val = t_dict.get(c, None)

        if base_val is None or t_val is None:
            continue
        if np.isnan(base_val) or np.isnan(t_val):
            continue
        if base_val == 0:
            continue

        scaling_dict[ty][c] = t_val / base_val

    print(f"📌 Target year: {ty}, valid countries: {len(scaling_dict[ty])}")

# ---------------------------------------------------------
# Additional print: Check the percentage value of a specific target_year.
# ---------------------------------------------------------
if print_check_year in scaling_dict:
    print("\n==========Check（Year：{}） ==========".format(print_check_year))
    test_dict = scaling_dict[print_check_year]

    if len(test_dict) == 0:
        print(f"⚠️ No countries can be counted. {print_check_year} scaling ratio (data may be missing)")
    else:
        for country_code, ratio in list(test_dict.items())[:20]:  # Print only the first 20
            print(f"M49: {country_code} → ratio: {ratio:.4f}")

        print("\n（Only the first 20 items are displayed）")

else:
    print(f"⚠️ {print_check_year} The data is not in target_years, so the verification data cannot be printed.")
    

# ---------------------------------------------------------
# Part 4-- Pre-generate "Country → Raster Mask" mapping
# ---------------------------------------------------------

print("\n=== Part 5: Country masks ===")

# read shapefile
gdf = gpd.read_file(shp_path)

# check item
if "M49_Code" not in gdf.columns:
    raise ValueError("❌ ERROR: Shapefile did't find M49_Code！")

# Reading the base year raster for spatial reference
with rasterio.open(tif_base_path) as src:
    raster_meta = src.meta
    transform = src.transform
    raster_shape = (src.height, src.width)

# Pre-generate a mask for each country (True = masked, False = valid pixels).
country_masks = {}

for idx, row in gdf.iterrows():
    m49 = row["M49_Code"]
    geom = row.geometry

    if geom is None:
        continue

    mask = geometry_mask(
        geometries=[geom],
        out_shape=raster_shape,
        transform=transform,
        invert=True  # True 表示 geom 内部为 True
    )

    # 保存 mask
    country_masks[m49] = mask

print(f"✔ Total number of country masks generated: {len(country_masks)}")

# ---------------------------------------------------------
# Effective pixel count for the top 10 countries' masks (for inspection)
# ---------------------------------------------------------
print("\n=== Mask check（top 10）===")
for i, (cid, m) in enumerate(country_masks.items()):
    if i >= 10:
        break
    print(f"M49 {cid}: Number of effective pixels = {m.sum()}")

print("=== Part 5 done ===\n")

# ---------------------------------------------------------
# Part 5-- Scaling the raster by country ratio → Outputting a tif file for target_years
# ---------------------------------------------------------
print("\n=== Part 5: Generate target year TIF ===")

with rasterio.open(tif_base_path) as src:
    base_data = src.read(1).astype(float)
    nodata = src.nodata if src.nodata is not None else -9999
    base_data[base_data == nodata] = np.nan

# Iterate through each target year
for ty in target_years:

    print(f"\n--- processing year {ty} ---")
    ratio_dict = scaling_dict[ty]

    # Create output array
    out_arr = np.full_like(base_data, np.nan)

    # country one by one.
    for cid, mask in country_masks.items():

        if cid not in ratio_dict:
            # No proportion → Fill in nan
            continue

        ratio = ratio_dict[cid]

        # Printing the first few countries for inspection
        if np.random.rand() < 0.005:  # Randomly print a small number of countries
            print(f"  M49 {cid}: Ratio={ratio:.4f}, mask pix={mask.sum()}")
        # ----
        # Scaling
        out_arr[mask] = base_data[mask] * ratio  * Ratio1Or1000

    # --- Check the output array ---
    valid_values = out_arr[~np.isnan(out_arr)]
    print(f"  ✔ Number of effective pixels: {len(valid_values)}")

    if len(valid_values) > 0:
        print(f"  ✔ Scaling results statistics: min={valid_values.min():.4f}, max={valid_values.max():.4f}, mean={valid_values.mean():.4f}")

        # The first 20 values ​​are printed for proofreading.
        check_vals = valid_values[:20]
        print("✔ Preview of the first 20 pixel values:", check_vals)
    else:
        print("⚠️ Warning: Output is empty! Please check Part 5 masks or Excel scale.")

    # Write tif and naming
    out_path = os.path.join(output_folder, f"{Pre_fileName}_{ty}.tif")
    meta = raster_meta.copy()
    meta.update(dtype="float32", nodata=-9999)

    with rasterio.open(out_path, "w", **meta) as dst:
        tmp = out_arr.copy()
        tmp[np.isnan(tmp)] = -9999
        dst.write(tmp.astype("float32"), 1)

    print(f"  ✔ All Done: {out_path}")

print("\n=== Part 5 Done ===")

In [ ]:
# =====================================================
# ---N Output---Processing Now----
# ---Dataset 2, SPAM
# ---Step1 Uniform spatial range, resolution, for each file 🎉🎉🎉General tool🎉🎉🎉
# =====================================================
import os
import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.warp import reproject, Resampling

res = 0.083333333  
crs = "EPSG:4326"  

width = int(360 / res)
height = int(180 / res)

transform = from_origin(-180, 90, res, res)

global_profile = {
    'driver': 'GTiff',
    'height': height,
    'width': width,
    'count': 1,
    'dtype': 'float32',
    'crs': crs,
    'transform': transform,
    'nodata': -9999  
}

# Path
input_dir = "Z:/staff/Qiang_S/09_Datasets/02N_Output/02Global_Yield_Maize/01DownloadDatasets/MaizeYield_SPAM5min"         
output_dir = "Z:/staff/Qiang_S/09_Datasets/02N_Output/02Global_Yield_Maize/03TempData/MaizeYield_SPAM5min"

os.makedirs(output_dir, exist_ok=True)

for root, _, files in os.walk(input_dir):
    for file in files:
        if file.endswith(".tif"):
            input_path = os.path.join(root, file)
            output_path = os.path.join(output_dir, file)

            with rasterio.open(input_path) as src:
                dst_array = np.full((height, width), np.nan, dtype='float32')

                reproject(
                    source=rasterio.band(src, 1),
                    destination=dst_array,
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=crs,
                    resampling=Resampling.nearest,
                    dst_nodata= -9999
                )

                with rasterio.open(output_path, "w", **global_profile) as dst:
                    dst.write(dst_array, 1)

print("✅ All Done")

In [ ]:
# =====================================================
# ---N Output---Processing Now----
# ---Dataset 2, SPAM
# ---Step2 Uniform unit, for 2020, 🎉🎉🎉General tool🎉🎉🎉 data multiplied by a ratio
# =====================================================
from osgeo import gdal
import numpy as np
import os
import glob

input_folder = r"Z:/staff/Qiang_S/09_Datasets/02N_Output/02Global_Yield_Maize/03TempData/MaizeYield_SPAM5min/2020"
output_folder = r"Z:/staff/Qiang_S/09_Datasets/02N_Output/02Global_Yield_Maize/03TempData/MaizeYield_SPAM5min"
os.makedirs(output_folder, exist_ok=True)

# ratio # ← 
N = 1000     

# ==========================
tif_files = glob.glob(os.path.join(input_folder, "*.tif"))
print(f"found {len(tif_files)} ")

for i, in_file in enumerate(tif_files, 1):
    fname = os.path.basename(in_file)
    out_file = os.path.join(output_folder, fname)

    print(f"[{i}/{len(tif_files)}] processing: {fname}")

    ds = gdal.Open(in_file)
    band = ds.GetRasterBand(1)
    nodata = band.GetNoDataValue()
    arr = band.ReadAsArray().astype("float32")
    mask_nodata = (arr == nodata) if nodata is not None else None

    # ==========================
    # arr × N
    # ==========================
    arr = arr * N

    if nodata is not None:
        arr[mask_nodata] = nodata

    # ==========================
    # Write GeoTIFF
    # ==========================
    driver = gdal.GetDriverByName("GTiff")

    out_ds = driver.Create(
        out_file,
        ds.RasterXSize,
        ds.RasterYSize,
        1,
        gdal.GDT_Float32,
        options=["COMPRESS=DEFLATE", "PREDICTOR=2", "TILED=YES"]
    )

    out_ds.SetGeoTransform(ds.GetGeoTransform())
    out_ds.SetProjection(ds.GetProjection())

    out_band = out_ds.GetRasterBand(1)
    if nodata is not None:
        out_band.SetNoDataValue(nodata)

    out_band.WriteArray(arr)
    out_ds.FlushCache()

    out_ds = None
    ds = None

print("\Done！")
print(f"Outpath：{output_folder}")

In [ ]:
# =====================================================
# ---N Output---Processing Now----
# ---Dataset 2, SPAM
# ---Step3 data fusion, FAO and SPAM Add, 1998，1999，2001，2002，2003，2004，2006，2007，2008，2009，2011，2012，2018，2019
# =====================================================

import geopandas as gpd
import rasterio
import rasterio.mask
import numpy as np
import pandas as pd
from tqdm import tqdm
import os
from rasterio.features import geometry_mask

# ============================================================
# Part 1 -- input1 parameters
# ============================================================
shp_path = r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\GDAM_GlobalCountryLevel_Shp/GlobalCountryLevel_Shp.shp"         # M49_Code was used as the ID
excel_path = r"Z:\staff\Qiang_S\09_Datasets\02N_Output\02Global_Yield_Maize\01DownloadDatasets\FAO_Yield1995_2020/FAOSTAT_Yield1995_2020.xls"       # FAO excel, contain yield...

tif_base_path = r"Z:\staff\Qiang_S\09_Datasets\02N_Output\02Global_Yield_Maize\02StandardizedDatasets\MaizeYield_SPAM5min/spamMaize2020.tif"        # base year .tif
base_year = 2020                                    # base year 2000，2005，2010，2020
target_years = [2018, 2019]                     # want to generate the following years, multiple years are possible.

selected_item_code = 112   # 112 Maize (corn); 111 Wheat

Ratio1Or1000 = 1  #
Pre_fileName = "spamMaize_I" #output file pre name

print_check_year = 2018  # You want to print the target year at a specific scale, such as 2019.
output_folder = r"Z:\staff\Qiang_S\09_Datasets\02N_Output\02Global_Yield_Maize\02StandardizedDatasets\MaizeYield_SPAM5min"   # output folder
nodata_value = -9999                                   # Set No data 

# ============================================================
# Part 2 -- Read data
# ============================================================

# read the base .tif
base_ds = rasterio.open(tif_base_path)
base_data = base_ds.read(1)
base_data = base_data.astype("float32")

# ---------------------------------------------------------
# Part 3 -- Build a dictionary of country scaling ratios (including printouts for proofreading).
# ---------------------------------------------------------

# Read Excel
df_excel = pd.read_excel(excel_path)

# Keep only the relevant columns
required_cols = ["Area Code (M49)", "Item Code (CPC)", "Year", "Value"]
df_excel = df_excel[required_cols]

# Filter selected crops（Item Code）
df_excel = df_excel[df_excel["Item Code (CPC)"] == selected_item_code]

# Filtering base year
df_base = df_excel[df_excel["Year"] == base_year]
base_dict = dict(zip(df_base["Area Code (M49)"], df_base["Value"]))

scaling_dict = {}

for ty in target_years:

    df_t = df_excel[df_excel["Year"] == ty]
    t_dict = dict(zip(df_t["Area Code (M49)"], df_t["Value"]))

    scaling_dict[ty] = {}
    common_countries = set(base_dict.keys()).intersection(set(t_dict.keys()))

    for c in common_countries:

        base_val = base_dict.get(c, None)
        t_val = t_dict.get(c, None)

        if base_val is None or t_val is None:
            continue
        if np.isnan(base_val) or np.isnan(t_val):
            continue
        if base_val == 0:
            continue

        scaling_dict[ty][c] = t_val / base_val

    print(f"📌 Target year: {ty}, valid countries: {len(scaling_dict[ty])}")

# ---------------------------------------------------------
# Additional print: Check the percentage value of a specific target_year.
# ---------------------------------------------------------
if print_check_year in scaling_dict:
    print("\n==========Check（Year：{}） ==========".format(print_check_year))
    test_dict = scaling_dict[print_check_year]

    if len(test_dict) == 0:
        print(f"⚠️ No countries can be counted. {print_check_year} scaling ratio (data may be missing)")
    else:
        for country_code, ratio in list(test_dict.items())[:20]:  # Print only the first 20
            print(f"M49: {country_code} → ratio: {ratio:.4f}")

        print("\n（Only the first 20 items are displayed）")

else:
    print(f"⚠️ {print_check_year} The data is not in target_years, so the verification data cannot be printed.")
    

# ---------------------------------------------------------
# Part 4-- Pre-generate "Country → Raster Mask" mapping
# ---------------------------------------------------------

print("\n=== Part 5: Country masks ===")

# read shapefile
gdf = gpd.read_file(shp_path)

# check item
if "M49_Code" not in gdf.columns:
    raise ValueError("❌ ERROR: Shapefile did't find M49_Code！")

# Reading the base year raster for spatial reference
with rasterio.open(tif_base_path) as src:
    raster_meta = src.meta
    transform = src.transform
    raster_shape = (src.height, src.width)

# Pre-generate a mask for each country (True = masked, False = valid pixels).
country_masks = {}

for idx, row in gdf.iterrows():
    m49 = row["M49_Code"]
    geom = row.geometry

    if geom is None:
        continue

    mask = geometry_mask(
        geometries=[geom],
        out_shape=raster_shape,
        transform=transform,
        invert=True  # True 表示 geom 内部为 True
    )

    # 保存 mask
    country_masks[m49] = mask

print(f"✔ Total number of country masks generated: {len(country_masks)}")

# ---------------------------------------------------------
# Effective pixel count for the top 10 countries' masks (for inspection)
# ---------------------------------------------------------
print("\n=== Mask check（top 10）===")
for i, (cid, m) in enumerate(country_masks.items()):
    if i >= 10:
        break
    print(f"M49 {cid}: Number of effective pixels = {m.sum()}")

print("=== Part 5 done ===\n")

# ---------------------------------------------------------
# Part 5-- Scaling the raster by country ratio → Outputting a tif file for target_years
# ---------------------------------------------------------
print("\n=== Part 5: Generate target year TIF ===")

with rasterio.open(tif_base_path) as src:
    base_data = src.read(1).astype(float)
    nodata = src.nodata if src.nodata is not None else -9999
    base_data[base_data == nodata] = np.nan

# Iterate through each target year
for ty in target_years:

    print(f"\n--- processing year {ty} ---")
    ratio_dict = scaling_dict[ty]

    # Create output array
    out_arr = np.full_like(base_data, np.nan)

    # country one by one.
    for cid, mask in country_masks.items():

        if cid not in ratio_dict:
            # No proportion → Fill in nan
            continue

        ratio = ratio_dict[cid]

        # Printing the first few countries for inspection
        if np.random.rand() < 0.005:  # Randomly print a small number of countries
            print(f"  M49 {cid}: Ratio={ratio:.4f}, mask pix={mask.sum()}")
        # ----
        # Scaling
        out_arr[mask] = base_data[mask] * ratio  * Ratio1Or1000

    # --- Check the output array ---
    valid_values = out_arr[~np.isnan(out_arr)]
    print(f"  ✔ Number of effective pixels: {len(valid_values)}")

    if len(valid_values) > 0:
        print(f"  ✔ Scaling results statistics: min={valid_values.min():.4f}, max={valid_values.max():.4f}, mean={valid_values.mean():.4f}")

        # The first 20 values ​​are printed for proofreading.
        check_vals = valid_values[:20]
        print("✔ Preview of the first 20 pixel values:", check_vals)
    else:
        print("⚠️ Warning: Output is empty! Please check Part 5 masks or Excel scale.")

    # Write tif and naming
    out_path = os.path.join(output_folder, f"{Pre_fileName}_{ty}.tif")
    meta = raster_meta.copy()
    meta.update(dtype="float32", nodata=-9999)

    with rasterio.open(out_path, "w", **meta) as dst:
        tmp = out_arr.copy()
        tmp[np.isnan(tmp)] = -9999
        dst.write(tmp.astype("float32"), 1)

    print(f"  ✔ All Done: {out_path}")

print("\n===Done！！！===")

In [ ]:
# =====================================================
# ---N Output---Processing Now----
# ---Dataset 3, GAEZ
# ---Step1 Uniform unit, for 2000，2010，2015，2020, t/ha → kg/ha 🎉🎉🎉General tool🎉🎉🎉 data multiplied by a ratio
# =====================================================
from osgeo import gdal
import numpy as np
import os
import glob

input_folder = r"Z:\staff\Qiang_S\09_Datasets\02N_Output\02Global_Yield_Maize\01DownloadDatasets\MaizeYield_GAEZ5min"
output_folder = r"Z:\staff\Qiang_S\09_Datasets\02N_Output\02Global_Yield_Maize\03TempData\MaizeYield_GAEZ5min_Temp"
os.makedirs(output_folder, exist_ok=True)

# ratio # ← 
N = 1000     

# ==========================
tif_files = glob.glob(os.path.join(input_folder, "*.tif"))
print(f"found {len(tif_files)} ")

for i, in_file in enumerate(tif_files, 1):
    fname = os.path.basename(in_file)
    out_file = os.path.join(output_folder, fname)

    print(f"[{i}/{len(tif_files)}] processing: {fname}")

    ds = gdal.Open(in_file)
    band = ds.GetRasterBand(1)
    nodata = band.GetNoDataValue()
    arr = band.ReadAsArray().astype("float32")
    mask_nodata = (arr == nodata) if nodata is not None else None

    # ==========================
    # arr × N
    # ==========================
    arr = arr * N

    if nodata is not None:
        arr[mask_nodata] = nodata

    # ==========================
    # Write GeoTIFF
    # ==========================
    driver = gdal.GetDriverByName("GTiff")

    out_ds = driver.Create(
        out_file,
        ds.RasterXSize,
        ds.RasterYSize,
        1,
        gdal.GDT_Float32,
        options=["COMPRESS=DEFLATE", "PREDICTOR=2", "TILED=YES"]
    )

    out_ds.SetGeoTransform(ds.GetGeoTransform())
    out_ds.SetProjection(ds.GetProjection())

    out_band = out_ds.GetRasterBand(1)
    if nodata is not None:
        out_band.SetNoDataValue(nodata)

    out_band.WriteArray(arr)
    out_ds.FlushCache()

    out_ds = None
    ds = None

print("\Done！")
print(f"Outpath：{output_folder}")

In [ ]:
# =====================================================
# ---N Output---Processing Now----
# ---Dataset 3, GAEZ
# ---Step2 data fusion, FAO and GAEZ Add, 1998，1999，2001，2002，2008，2009，2011，2012，2013，2014，2016，2017，2018，2019
# =====================================================

import geopandas as gpd
import rasterio
import rasterio.mask
import numpy as np
import pandas as pd
from tqdm import tqdm
import os
from rasterio.features import geometry_mask

# ============================================================
# Part 1 -- input1 parameters
# ============================================================
shp_path = r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\GDAM_GlobalCountryLevel_Shp/GlobalCountryLevel_Shp.shp"         # M49_Code was used as the ID
excel_path = r"Z:\staff\Qiang_S\09_Datasets\02N_Output\02Global_Yield_Maize\01DownloadDatasets\FAO_Yield1995_2020/FAOSTAT_Yield1995_2020.xls"       # FAO excel, contain yield...

tif_base_path = r"Z:\staff\Qiang_S\09_Datasets\02N_Output\02Global_Yield_Maize\02StandardizedDatasets\MaizeYield_GAEZ5min/GAEZMaize2020.tif"        # base year .tif
base_year = 2020                                   # base year 2000，2010，2015，2020
target_years = [2018, 2019]                     # want to generate the following years, multiple years are possible.

selected_item_code = 112   # 112 Maize (corn); 111 Wheat

Ratio1Or1000 = 1  #
Pre_fileName = "GAEZMaize_I" #output file pre name

print_check_year = 2019  # You want to print the target year at a specific scale, such as 2019.
output_folder = r"Z:\staff\Qiang_S\09_Datasets\02N_Output\02Global_Yield_Maize\02StandardizedDatasets\MaizeYield_GAEZ5min"   # output folder
nodata_value = -9999                                   # Set No data 

# ============================================================
# Part 2 -- Read data
# ============================================================

# read the base .tif
base_ds = rasterio.open(tif_base_path)
base_data = base_ds.read(1)
base_data = base_data.astype("float32")

# ---------------------------------------------------------
# Part 3 -- Build a dictionary of country scaling ratios (including printouts for proofreading).
# ---------------------------------------------------------

# Read Excel
df_excel = pd.read_excel(excel_path)

# Keep only the relevant columns
required_cols = ["Area Code (M49)", "Item Code (CPC)", "Year", "Value"]
df_excel = df_excel[required_cols]

# Filter selected crops（Item Code）
df_excel = df_excel[df_excel["Item Code (CPC)"] == selected_item_code]

# Filtering base year
df_base = df_excel[df_excel["Year"] == base_year]
base_dict = dict(zip(df_base["Area Code (M49)"], df_base["Value"]))

scaling_dict = {}

for ty in target_years:

    df_t = df_excel[df_excel["Year"] == ty]
    t_dict = dict(zip(df_t["Area Code (M49)"], df_t["Value"]))

    scaling_dict[ty] = {}
    common_countries = set(base_dict.keys()).intersection(set(t_dict.keys()))

    for c in common_countries:

        base_val = base_dict.get(c, None)
        t_val = t_dict.get(c, None)

        if base_val is None or t_val is None:
            continue
        if np.isnan(base_val) or np.isnan(t_val):
            continue
        if base_val == 0:
            continue

        scaling_dict[ty][c] = t_val / base_val

    print(f"📌 Target year: {ty}, valid countries: {len(scaling_dict[ty])}")

# ---------------------------------------------------------
# Additional print: Check the percentage value of a specific target_year.
# ---------------------------------------------------------
if print_check_year in scaling_dict:
    print("\n==========Check（Year：{}） ==========".format(print_check_year))
    test_dict = scaling_dict[print_check_year]

    if len(test_dict) == 0:
        print(f"⚠️ No countries can be counted. {print_check_year} scaling ratio (data may be missing)")
    else:
        for country_code, ratio in list(test_dict.items())[:20]:  # Print only the first 20
            print(f"M49: {country_code} → ratio: {ratio:.4f}")

        print("\n（Only the first 20 items are displayed）")

else:
    print(f"⚠️ {print_check_year} The data is not in target_years, so the verification data cannot be printed.")
    

# ---------------------------------------------------------
# Part 4-- Pre-generate "Country → Raster Mask" mapping
# ---------------------------------------------------------

print("\n=== Part 5: Country masks ===")

# read shapefile
gdf = gpd.read_file(shp_path)

# check item
if "M49_Code" not in gdf.columns:
    raise ValueError("❌ ERROR: Shapefile did't find M49_Code！")

# Reading the base year raster for spatial reference
with rasterio.open(tif_base_path) as src:
    raster_meta = src.meta
    transform = src.transform
    raster_shape = (src.height, src.width)

# Pre-generate a mask for each country (True = masked, False = valid pixels).
country_masks = {}

for idx, row in gdf.iterrows():
    m49 = row["M49_Code"]
    geom = row.geometry

    if geom is None:
        continue

    mask = geometry_mask(
        geometries=[geom],
        out_shape=raster_shape,
        transform=transform,
        invert=True  # True 表示 geom 内部为 True
    )

    # 保存 mask
    country_masks[m49] = mask

print(f"✔ Total number of country masks generated: {len(country_masks)}")

# ---------------------------------------------------------
# Effective pixel count for the top 10 countries' masks (for inspection)
# ---------------------------------------------------------
print("\n=== Mask check（top 10）===")
for i, (cid, m) in enumerate(country_masks.items()):
    if i >= 10:
        break
    print(f"M49 {cid}: Number of effective pixels = {m.sum()}")

print("=== Part 5 done ===\n")

# ---------------------------------------------------------
# Part 5-- Scaling the raster by country ratio → Outputting a tif file for target_years
# ---------------------------------------------------------
print("\n=== Part 5: Generate target year TIF ===")

with rasterio.open(tif_base_path) as src:
    base_data = src.read(1).astype(float)
    nodata = src.nodata if src.nodata is not None else -9999
    base_data[base_data == nodata] = np.nan

# Iterate through each target year
for ty in target_years:

    print(f"\n--- processing year {ty} ---")
    ratio_dict = scaling_dict[ty]

    # Create output array
    out_arr = np.full_like(base_data, np.nan)

    # country one by one.
    for cid, mask in country_masks.items():

        if cid not in ratio_dict:
            # No proportion → Fill in nan
            continue

        ratio = ratio_dict[cid]

        # Printing the first few countries for inspection
        if np.random.rand() < 0.005:  # Randomly print a small number of countries
            print(f"  M49 {cid}: Ratio={ratio:.4f}, mask pix={mask.sum()}")
        # ----
        # Scaling
        out_arr[mask] = base_data[mask] * ratio  * Ratio1Or1000

    # --- Check the output array ---
    valid_values = out_arr[~np.isnan(out_arr)]
    print(f"  ✔ Number of effective pixels: {len(valid_values)}")

    if len(valid_values) > 0:
        print(f"  ✔ Scaling results statistics: min={valid_values.min():.4f}, max={valid_values.max():.4f}, mean={valid_values.mean():.4f}")

        # The first 20 values ​​are printed for proofreading.
        check_vals = valid_values[:20]
        print("✔ Preview of the first 20 pixel values:", check_vals)
    else:
        print("⚠️ Warning: Output is empty! Please check Part 5 masks or Excel scale.")

    # Write tif and naming
    out_path = os.path.join(output_folder, f"{Pre_fileName}_{ty}.tif")
    meta = raster_meta.copy()
    meta.update(dtype="float32", nodata=-9999)

    with rasterio.open(out_path, "w", **meta) as dst:
        tmp = out_arr.copy()
        tmp[np.isnan(tmp)] = -9999
        dst.write(tmp.astype("float32"), 1)

    print(f"  ✔ All Done: {out_path}")

print("\n===Done！！！===")

In [ ]:
# =====================================================
# ---N Output---Processing Now----
# ---Dataset 4, GDHY
# ---Step1 Read Informantion
# =====================================================
import xarray as xr

nc4_file = "Z:/staff/Qiang_S/09_Datasets/02N_Output/02Global_Yield_Maize/01DownloadDatasets/MaizeYield_GDHY30min/GDHY_v1.2_v1.3_20190128/maize/yield_2016.nc4"

ds = xr.open_dataset(nc4_file)

print("📂 File structure information:")
print(ds)

print("\n🌍 global properties:")
for k, v in ds.attrs.items():
    print(f"{k}: {v}")

print("\n📊 Variable details:")
for var in ds.data_vars:
    print(f"\n Variable: {var}")
    print(ds[var])

In [ ]:
# =====================================================
# ---N Output---Processing Now----
# ---Dataset 4, GDHY
# ---Step2 transform formate, .nc to .tif
# =====================================================
import xarray as xr
import rioxarray 
import os

# folder path
input_folder = "Z:/staff/Qiang_S/09_Datasets/02N_Output/02Global_Yield_Maize/01DownloadDatasets/MaizeYield_GDHY30min/GDHY_v1.2_v1.3_20190128/maize"  #
output_folder = "Z:/staff/Qiang_S/09_Datasets/02N_Output/02Global_Yield_Maize/03TempData/MaizeYield_GDHY30min_Temp"
os.makedirs(output_folder, exist_ok=True)

# Iterate through all .nc4 files in the folder.
for file in os.listdir(input_folder):
    if file.endswith(".nc4"):
        nc_path = os.path.join(input_folder, file)
        print(f"processing: {file}")
        
        # open nc4 file
        ds = xr.open_dataset(nc_path)

        # Assuming there is only one data variable, or you want to specify the variable name.
        var_name = list(ds.data_vars.keys())[0]  # Take the first variable
        da = ds[var_name]

        # Confirm the dimension name and rename it.
        # The expression (lat, lon) should be changed to (y, x).
        rename_dict = {}
        if "lat" in da.dims:
            rename_dict["lat"] = "y"
        if "latitude" in da.dims:
            rename_dict["latitude"] = "y"
        if "lon" in da.dims:
            rename_dict["lon"] = "x"
        if "longitude" in da.dims:
            rename_dict["longitude"] = "x"

        da = da.rename(rename_dict)

        # Setting space reference (WGS84 EPSG:4326)
        da = da.rio.write_crs("EPSG:4326")

        # output GeoTIFF
        out_path = os.path.join(output_folder, file.replace(".nc4", ".tif"))
        da.rio.to_raster(out_path)

        print(f"✅ Finished: {out_path}")

In [ ]:
# =====================================================
# ---N Output---Processing Now----
# ---Dataset 4, GDHY
# ---Step3 Uniform unit, t/ha → kg/ha 🎉🎉🎉General tool🎉🎉🎉 data multiplied by a ratio
# =====================================================
from osgeo import gdal
import numpy as np
import os
import glob

input_folder = r"Z:/staff/Qiang_S/09_Datasets/02N_Output/02Global_Yield_Maize/03TempData/MaizeYield_GDHY30min_Temp"
output_folder = r"Z:/staff/Qiang_S/09_Datasets/02N_Output/02Global_Yield_Maize/03TempData/MaizeYield_GDHY30min_Step2Temp"
os.makedirs(output_folder, exist_ok=True)

# ratio # ← 
N = 1000     

# ==========================
tif_files = glob.glob(os.path.join(input_folder, "*.tif"))
print(f"found {len(tif_files)} ")

for i, in_file in enumerate(tif_files, 1):
    fname = os.path.basename(in_file)
    out_file = os.path.join(output_folder, fname)

    print(f"[{i}/{len(tif_files)}] processing: {fname}")

    ds = gdal.Open(in_file)
    band = ds.GetRasterBand(1)
    nodata = band.GetNoDataValue()
    arr = band.ReadAsArray().astype("float32")
    mask_nodata = (arr == nodata) if nodata is not None else None

    # ==========================
    # arr × N
    # ==========================
    arr = arr * N

    if nodata is not None:
        arr[mask_nodata] = nodata

    # ==========================
    # Write GeoTIFF
    # ==========================
    driver = gdal.GetDriverByName("GTiff")

    out_ds = driver.Create(
        out_file,
        ds.RasterXSize,
        ds.RasterYSize,
        1,
        gdal.GDT_Float32,
        options=["COMPRESS=DEFLATE", "PREDICTOR=2", "TILED=YES"]
    )

    out_ds.SetGeoTransform(ds.GetGeoTransform())
    out_ds.SetProjection(ds.GetProjection())

    out_band = out_ds.GetRasterBand(1)
    if nodata is not None:
        out_band.SetNoDataValue(nodata)

    out_band.WriteArray(arr)
    out_ds.FlushCache()

    out_ds = None
    ds = None

print("\Done！")
print(f"Outpath：{output_folder}")

In [ ]:
# =====================================================
# ---N Output---Processing Now----
# ---Dataset 4, GDHY
# ---Step4 resample, 30min to 5min, Processing GDHY dataset,Specially, 🎉🎉🎉General tool🎉🎉🎉 data multiplied by a ratio
# =====================================================

import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_origin
import numpy as np
import os

input_folder = r"Z:/staff/Qiang_S/09_Datasets/02N_Output/02Global_Yield_Maize/03TempData/MaizeYield_GDHY30min_Step2Temp"    
output_folder = r"Z:\staff\Qiang_S\09_Datasets\02N_Output\02Global_Yield_Maize\02StandardizedDatasets\MaizeYield_GDHY5min" 
target_res = 0.083333333                        
nodata_value = -9999                      
resample_method = Resampling.bilinear     # option：Resampling.nearest

os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(input_folder):
    if not filename.lower().endswith(".tif"):
        continue

    input_tif = os.path.join(input_folder, filename)
    output_tif = os.path.join(output_folder, filename)

    print(f"🔹processing: {filename}")

    with rasterio.open(input_tif) as src:
        arr = src.read(1)
        transform = src.transform
        profile = src.profile

        # === Step 1. Correct the inversion (if the Y direction is reversed). ===
        # Determine if the latitude direction is negative from top to bottom; otherwise, it needs to be flipped.
        if transform.e > 0:
            arr = np.flipud(arr)
            transform = from_origin(transform.c, transform.f, transform.a, -abs(transform.e))

        # === Step 2. Corrected longitude range (0–360 → -180–180) ===
        left, bottom, right, top = src.bounds
        print(f"  ↳ Original range: ({left:.2f}, {bottom:.2f}, {right:.2f}, {top:.2f})")

        if left >= 0 and right > 180:
            lon = np.arange(left + transform.a / 2, right, transform.a)
            split_idx = np.searchsorted(lon, 180.0)

            arr_shifted = np.hstack((arr[:, split_idx:], arr[:, :split_idx]))
            new_transform = from_origin(-180, 90, abs(transform.a), abs(transform.e))
        else:
            arr_shifted = arr
            new_transform = transform

        # === Step 3. Resampling to xxx resolution ===
        dst_width = int(360 / target_res)
        dst_height = int(180 / target_res)
        dst_array = np.full((dst_height, dst_width), nodata_value, dtype=np.float32)

        dst_transform = from_origin(-180, 90, target_res, target_res)
        dst_crs = "EPSG:4326"

        reproject(
            source=arr_shifted,
            destination=dst_array,
            src_transform=new_transform,
            src_crs=dst_crs,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            src_nodata=profile.get("nodata", nodata_value),
            dst_nodata=nodata_value,
            resampling=resample_method
        )

        # === Step 4. Output ===
        profile.update({
            "height": dst_height,
            "width": dst_width,
            "transform": dst_transform,
            "crs": dst_crs,
            "dtype": "float32",
            "nodata": nodata_value
        })

        with rasterio.open(output_tif, "w", **profile) as dst:
            dst.write(dst_array, 1)

    print(f"✅ finished: {output_tif}\n")

print("🎯 All Done！")

In [ ]:
# =====================================================
# ---N Output---Processing Now----
# ---Dataset 4, GDHY
# ---Step5 data fusion, FAO and GDHY Add, 2017，2018
# =====================================================

import geopandas as gpd
import rasterio
import rasterio.mask
import numpy as np
import pandas as pd
from tqdm import tqdm
import os
from rasterio.features import geometry_mask

# ============================================================
# Part 1 -- input1 parameters
# ============================================================
shp_path = r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\GDAM_GlobalCountryLevel_Shp/GlobalCountryLevel_Shp.shp"         # M49_Code was used as the ID
excel_path = r"Z:\staff\Qiang_S\09_Datasets\02N_Output\02Global_Yield_Maize\01DownloadDatasets\FAO_Yield1995_2020/FAOSTAT_Yield1995_2020.xls"       # FAO excel, contain yield...

tif_base_path = r"Z:\staff\Qiang_S\09_Datasets\02N_Output\02Global_Yield_Maize\02StandardizedDatasets\MaizeYield_GDHY5min/yield_2016.tif"        # base year .tif
base_year = 2016                                   # base year 2016
target_years = [2017, 2018]                     # want to generate the following years, multiple years are possible.

selected_item_code = 112   # 112 Maize (corn); 111 Wheat

Ratio1Or1000 = 1  #
Pre_fileName = "GDHYMaize_I" #output file pre name

print_check_year = 2018  # You want to print the target year at a specific scale, such as 2019.
output_folder = r"Z:\staff\Qiang_S\09_Datasets\02N_Output\02Global_Yield_Maize\02StandardizedDatasets\MaizeYield_GDHY5min"   # output folder
nodata_value = -9999                                   # Set No data 

# ============================================================
# Part 2 -- Read data
# ============================================================

# read the base .tif
base_ds = rasterio.open(tif_base_path)
base_data = base_ds.read(1)
base_data = base_data.astype("float32")

# ---------------------------------------------------------
# Part 3 -- Build a dictionary of country scaling ratios (including printouts for proofreading).
# ---------------------------------------------------------

# Read Excel
df_excel = pd.read_excel(excel_path)

# Keep only the relevant columns
required_cols = ["Area Code (M49)", "Item Code (CPC)", "Year", "Value"]
df_excel = df_excel[required_cols]

# Filter selected crops（Item Code）
df_excel = df_excel[df_excel["Item Code (CPC)"] == selected_item_code]

# Filtering base year
df_base = df_excel[df_excel["Year"] == base_year]
base_dict = dict(zip(df_base["Area Code (M49)"], df_base["Value"]))

scaling_dict = {}

for ty in target_years:

    df_t = df_excel[df_excel["Year"] == ty]
    t_dict = dict(zip(df_t["Area Code (M49)"], df_t["Value"]))

    scaling_dict[ty] = {}
    common_countries = set(base_dict.keys()).intersection(set(t_dict.keys()))

    for c in common_countries:

        base_val = base_dict.get(c, None)
        t_val = t_dict.get(c, None)

        if base_val is None or t_val is None:
            continue
        if np.isnan(base_val) or np.isnan(t_val):
            continue
        if base_val == 0:
            continue

        scaling_dict[ty][c] = t_val / base_val

    print(f"📌 Target year: {ty}, valid countries: {len(scaling_dict[ty])}")

# ---------------------------------------------------------
# Additional print: Check the percentage value of a specific target_year.
# ---------------------------------------------------------
if print_check_year in scaling_dict:
    print("\n==========Check（Year：{}） ==========".format(print_check_year))
    test_dict = scaling_dict[print_check_year]

    if len(test_dict) == 0:
        print(f"⚠️ No countries can be counted. {print_check_year} scaling ratio (data may be missing)")
    else:
        for country_code, ratio in list(test_dict.items())[:20]:  # Print only the first 20
            print(f"M49: {country_code} → ratio: {ratio:.4f}")

        print("\n（Only the first 20 items are displayed）")

else:
    print(f"⚠️ {print_check_year} The data is not in target_years, so the verification data cannot be printed.")
    

# ---------------------------------------------------------
# Part 4-- Pre-generate "Country → Raster Mask" mapping
# ---------------------------------------------------------

print("\n=== Part 5: Country masks ===")

# read shapefile
gdf = gpd.read_file(shp_path)

# check item
if "M49_Code" not in gdf.columns:
    raise ValueError("❌ ERROR: Shapefile did't find M49_Code！")

# Reading the base year raster for spatial reference
with rasterio.open(tif_base_path) as src:
    raster_meta = src.meta
    transform = src.transform
    raster_shape = (src.height, src.width)

# Pre-generate a mask for each country (True = masked, False = valid pixels).
country_masks = {}

for idx, row in gdf.iterrows():
    m49 = row["M49_Code"]
    geom = row.geometry

    if geom is None:
        continue

    mask = geometry_mask(
        geometries=[geom],
        out_shape=raster_shape,
        transform=transform,
        invert=True  # True 表示 geom 内部为 True
    )

    # Save mask
    country_masks[m49] = mask

print(f"✔ Total number of country masks generated: {len(country_masks)}")

# ---------------------------------------------------------
# Effective pixel count for the top 10 countries' masks (for inspection)
# ---------------------------------------------------------
print("\n=== Mask check（top 10）===")
for i, (cid, m) in enumerate(country_masks.items()):
    if i >= 10:
        break
    print(f"M49 {cid}: Number of effective pixels = {m.sum()}")

print("=== Part 5 done ===\n")

# ---------------------------------------------------------
# Part 5-- Scaling the raster by country ratio → Outputting a tif file for target_years
# ---------------------------------------------------------
print("\n=== Part 5: Generate target year TIF ===")

with rasterio.open(tif_base_path) as src:
    base_data = src.read(1).astype(float)
    nodata = src.nodata if src.nodata is not None else -9999
    base_data[base_data == nodata] = np.nan

# Iterate through each target year
for ty in target_years:

    print(f"\n--- processing year {ty} ---")
    ratio_dict = scaling_dict[ty]

    # Create output array
    out_arr = np.full_like(base_data, np.nan)

    # country one by one.
    for cid, mask in country_masks.items():

        if cid not in ratio_dict:
            # No proportion → Fill in nan
            continue

        ratio = ratio_dict[cid]

        # Printing the first few countries for inspection
        if np.random.rand() < 0.005:  # Randomly print a small number of countries
            print(f"  M49 {cid}: Ratio={ratio:.4f}, mask pix={mask.sum()}")
        # ----
        # Scaling
        out_arr[mask] = base_data[mask] * ratio  * Ratio1Or1000

    # --- Check the output array ---
    valid_values = out_arr[~np.isnan(out_arr)]
    print(f"  ✔ Number of effective pixels: {len(valid_values)}")

    if len(valid_values) > 0:
        print(f"  ✔ Scaling results statistics: min={valid_values.min():.4f}, max={valid_values.max():.4f}, mean={valid_values.mean():.4f}")

        # The first 20 values ​​are printed for proofreading.
        check_vals = valid_values[:20]
        print("✔ Preview of the first 20 pixel values:", check_vals)
    else:
        print("⚠️ Warning: Output is empty! Please check Part 5 masks or Excel scale.")

    # Write tif and naming
    out_path = os.path.join(output_folder, f"{Pre_fileName}_{ty}.tif")
    meta = raster_meta.copy()
    meta.update(dtype="float32", nodata=-9999)

    with rasterio.open(out_path, "w", **meta) as dst:
        tmp = out_arr.copy()
        tmp[np.isnan(tmp)] = -9999
        dst.write(tmp.astype("float32"), 1)

    print(f"  ✔ All Done: {out_path}")

print("\n===Done！！！===")

In [ ]:
# =====================================================
# ---N Output---Processing Now----
# ---Dataset 5, GGCP10 + SPAM   https://doi.org/10.7910/DVN/G1HBNK
# ---Step1 data fusion,FAO + SAPM, Harvested Ared, add SAPM2019, SAPM2018
# =====================================================
import geopandas as gpd
import rasterio
import rasterio.mask
import numpy as np
import pandas as pd
from tqdm import tqdm
import os
from rasterio.features import geometry_mask

# ============================================================
# Part 1 -- input1 parameters
# ============================================================
shp_path = r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\GDAM_GlobalCountryLevel_Shp/GlobalCountryLevel_Shp.shp"         # M49_Code was used as the ID
excel_path = r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\FAO_HarvestedArea/FAOSTAT_HarvestedArea_1995_2020.xls"       # FAO excel, contain yield...

tif_base_path = r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\SPAM_HarvestedArea\spam2020_V2r0_global_H_MAIZ_A\spamMaize2020.tif"        # base year .tif
base_year = 2020                                    # base year

target_years = [2018, 2019]                     # want to generate the following years, multiple years are possible.

selected_item_code = 112   # 112 Maize (corn); 111 Wheat

Ratio1Or1000 = 1  
Pre_fileName = "SPAMHarArea_Maize_I" #output file pre name

print_check_year = 2019  # You want to print the target year at a specific scale, such as 2019.
output_folder = r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\SPAM_HarvestedArea\spam2020_V2r0_global_H_MAIZ_A"   # output folder
nodata_value = -9999                                   # Set No data 

# ============================================================
# Part 2 -- Read data
# ============================================================

# read the base .tif
base_ds = rasterio.open(tif_base_path)
base_data = base_ds.read(1)
base_data = base_data.astype("float32")

# ---------------------------------------------------------
# Part 3 -- Build a dictionary of country scaling ratios (including printouts for proofreading).
# ---------------------------------------------------------

# Read Excel
df_excel = pd.read_excel(excel_path)

# Keep only the relevant columns
required_cols = ["Area Code (M49)", "Item Code (CPC)", "Year", "Value"]
df_excel = df_excel[required_cols]

# Filter selected crops（Item Code）
df_excel = df_excel[df_excel["Item Code (CPC)"] == selected_item_code]

# Filtering base year
df_base = df_excel[df_excel["Year"] == base_year]
base_dict = dict(zip(df_base["Area Code (M49)"], df_base["Value"]))

scaling_dict = {}

for ty in target_years:

    df_t = df_excel[df_excel["Year"] == ty]
    t_dict = dict(zip(df_t["Area Code (M49)"], df_t["Value"]))

    scaling_dict[ty] = {}
    common_countries = set(base_dict.keys()).intersection(set(t_dict.keys()))

    for c in common_countries:

        base_val = base_dict.get(c, None)
        t_val = t_dict.get(c, None)

        if base_val is None or t_val is None:
            continue
        if np.isnan(base_val) or np.isnan(t_val):
            continue
        if base_val == 0:
            continue

        scaling_dict[ty][c] = t_val / base_val

    print(f"📌 Target year: {ty}, valid countries: {len(scaling_dict[ty])}")

# ---------------------------------------------------------
# Additional print: Check the percentage value of a specific target_year.
# ---------------------------------------------------------
if print_check_year in scaling_dict:
    print("\n==========Check（Year：{}） ==========".format(print_check_year))
    test_dict = scaling_dict[print_check_year]

    if len(test_dict) == 0:
        print(f"⚠️ No countries can be counted. {print_check_year} scaling ratio (data may be missing)")
    else:
        for country_code, ratio in list(test_dict.items())[:20]:  # Print only the first 20
            print(f"M49: {country_code} → ratio: {ratio:.4f}")

        print("\n（Only the first 20 items are displayed）")

else:
    print(f"⚠️ {print_check_year} The data is not in target_years, so the verification data cannot be printed.")
    

# ---------------------------------------------------------
# Part 4-- Pre-generate "Country → Raster Mask" mapping
# ---------------------------------------------------------

print("\n=== Part 5: Country masks ===")

# read shapefile
gdf = gpd.read_file(shp_path)

# check item
if "M49_Code" not in gdf.columns:
    raise ValueError("❌ ERROR: Shapefile did't find M49_Code！")

# Reading the base year raster for spatial reference
with rasterio.open(tif_base_path) as src:
    raster_meta = src.meta
    transform = src.transform
    raster_shape = (src.height, src.width)

# Pre-generate a mask for each country (True = masked, False = valid pixels).
country_masks = {}

for idx, row in gdf.iterrows():
    m49 = row["M49_Code"]
    geom = row.geometry

    if geom is None:
        continue

    mask = geometry_mask(
        geometries=[geom],
        out_shape=raster_shape,
        transform=transform,
        invert=True  # True 表示 geom 内部为 True
    )

    # 保存 mask
    country_masks[m49] = mask

print(f"✔ Total number of country masks generated: {len(country_masks)}")

# ---------------------------------------------------------
# Effective pixel count for the top 10 countries' masks (for inspection)
# ---------------------------------------------------------
print("\n=== Mask check（top 10）===")
for i, (cid, m) in enumerate(country_masks.items()):
    if i >= 10:
        break
    print(f"M49 {cid}: Number of effective pixels = {m.sum()}")

print("=== Part 5 done ===\n")

# ---------------------------------------------------------
# Part 5-- Scaling the raster by country ratio → Outputting a tif file for target_years
# ---------------------------------------------------------
print("\n=== Part 5: Generate target year TIF ===")

with rasterio.open(tif_base_path) as src:
    base_data = src.read(1).astype(float)
    nodata = src.nodata if src.nodata is not None else -9999
    base_data[base_data == nodata] = np.nan

# Iterate through each target year
for ty in target_years:

    print(f"\n--- processing year {ty} ---")
    ratio_dict = scaling_dict[ty]

    # Create output array
    out_arr = np.full_like(base_data, np.nan)

    # country one by one.
    for cid, mask in country_masks.items():

        if cid not in ratio_dict:
            # No proportion → Fill in nan
            continue

        ratio = ratio_dict[cid]

        # Printing the first few countries for inspection
        if np.random.rand() < 0.005:  # Randomly print a small number of countries
            print(f"  M49 {cid}: Ratio={ratio:.4f}, mask pix={mask.sum()}")
        # ----
        # Scaling
        out_arr[mask] = base_data[mask] * ratio  * Ratio1Or1000

    # --- Check the output array ---
    valid_values = out_arr[~np.isnan(out_arr)]
    print(f"  ✔ Number of effective pixels: {len(valid_values)}")

    if len(valid_values) > 0:
        print(f"  ✔ Scaling results statistics: min={valid_values.min():.4f}, max={valid_values.max():.4f}, mean={valid_values.mean():.4f}")

        # The first 20 values ​​are printed for proofreading.
        check_vals = valid_values[:20]
        print("✔ Preview of the first 20 pixel values:", check_vals)
    else:
        print("⚠️ Warning: Output is empty! Please check Part 5 masks or Excel scale.")

    # Write tif and naming
    out_path = os.path.join(output_folder, f"{Pre_fileName}_{ty}.tif")
    meta = raster_meta.copy()
    meta.update(dtype="float32", nodata=-9999)

    with rasterio.open(out_path, "w", **meta) as dst:
        tmp = out_arr.copy()
        tmp[np.isnan(tmp)] = -9999
        dst.write(tmp.astype("float32"), 1)

    print(f"  ✔ All Done: {out_path}")

print("\n=== Part 5 Done! ===")

In [ ]:
# =====================================================
# ---N Output---Processing Now----
# ---Dataset 5, GGCP10 + SPAM   https://doi.org/10.7910/DVN/G1HBNK
# ---Step1 data fusion,FAO + SAPM, Harvested Ared, add SAPM2019, SAPM2018
# ---Step2 data fusion, yield 2020，2019, 2018， production / harvested areas
# useful tools,🎉🎉🎉🎉🎉，two datasets, a divided by b
# =====================================================
import os
import rasterio
import numpy as np
from scipy.ndimage import distance_transform_edt

def round_sig(x, sig=6):
    """
   This version preserves valid numbers in NumPy arrays without generating errors.
    """
    x = np.asarray(x)   # It is guaranteed to be an array

    # Output array
    out = np.zeros_like(x, dtype="float32")

    # mask：Non-zero and non-NaN portions need to be processed.
    mask = (~np.isnan(x)) & (x != 0)

    # Values ​​to be processed
    xv = x[mask].astype("float64")  # Use float64 to avoid log10 errors

    # Calculate exponent
    exp = np.floor(np.log10(np.abs(xv)))

    # Calculate scaling factor
    factor = np.power(10.0, sig - exp - 1)

    out[mask] = (np.round(xv * factor) / factor).astype("float32")

    #Keep 0 and NaN as is
    out[~mask] = x[~mask].astype("float32")

    return out

def fill_nearest_valid(arr, nodata):
    """Fill in the blanks using nearest neighbors."""
    mask = (arr == nodata) | np.isnan(arr)
    if np.all(~mask):
        return arr

    valid_mask = ~mask
    distances, indices = distance_transform_edt(mask, return_indices=True)
    filled = arr[tuple(indices)]
    return filled


def process_tif_pair(A_path, B_path, output_path, nodata=-9999):
    """Ratio calculation for single-file pairs"""

    with rasterio.open(A_path) as srcA, rasterio.open(B_path) as srcB:
        A = srcA.read(1).astype("float32")
        B = srcB.read(1).astype("float32")

        meta = srcA.meta.copy()
        meta.update(nodata=nodata, dtype="float32")

        A_nodata = srcA.nodata if srcA.nodata is not None else nodata
        B_nodata = srcB.nodata if srcB.nodata is not None else nodata

        A_mask = (A == A_nodata) | np.isnan(A)
        B_mask = (B == B_nodata) | np.isnan(B)

        # ---- STEP 1：若 A>0 且 B 空 → 最近邻填补 ----
        B_filled = B.copy()
        cond_fill = (~A_mask) & (A > 0) & (B_mask)
        if np.any(cond_fill):
            print("  → Perform B nearest neighbor fill...")
            B_nn = fill_nearest_valid(B, B_nodata)
            B_filled[cond_fill] = B_nn[cond_fill]

        # ---- STEP 2：Calculation results ----
        result = np.full_like(A, nodata, dtype="float32")

        valid = (~A_mask) & (A > 0) & (~((B_filled == B_nodata) | np.isnan(B_filled)))

        result[valid] = (A[valid] / B_filled[valid])*1000000

        # ---- STEP 3：Retain 6 significant digits ----
        result[valid] = round_sig(result[valid], sig=6)

        # ---- Write down the results ----
        with rasterio.open(output_path, "w", **meta) as dst:
            dst.write(result, 1)

        print(f"✔Output complete: {output_path}")


def batch_ratio(folder_A, folder_B, folder_out, nodata=-9999):
    """Batch A/B pixel-by-pixel division"""

    os.makedirs(folder_out, exist_ok=True)

    files_A = [f for f in os.listdir(folder_A) if f.lower().endswith(".tif")]
    years_A = {f[-8:-4]: f for f in files_A}

    files_B = [f for f in os.listdir(folder_B) if f.lower().endswith(".tif")]
    years_B = {f[-8:-4]: f for f in files_B}

    common_years = sorted(set(years_A.keys()) & set(years_B.keys()))

    print(f"Find matching year: {common_years}")

    for year in common_years:
        A_file = os.path.join(folder_A, years_A[year])
        B_file = os.path.join(folder_B, years_B[year])
        out_file = os.path.join(folder_out, f"GGCP10SPAM_MaizeYield_{year}.tif")

        print(f"\n=== Processing year {year}: {years_A[year]} / {years_B[year]} ===")
        process_tif_pair(A_file, B_file, out_file, nodata=nodata)

    print("\n🎉 All Done！")

batch_ratio(
folder_A = r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\GGCP10\GGCP10_Production_Maize", 
folder_B = r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\SPAM_HarvestedArea\spam2020_V2r0_global_H_MAIZ_A", 
folder_out = r"Z:\staff\Qiang_S\09_Datasets\02N_Output\02Global_Yield_Maize\03TempData\MaizeYield_GGCP10_SPAM_1Temp",
nodata=-9999
)

In [ ]:
# =====================================================
# ---N Output---Processing Now----
# ---Dataset 5, GGCP10 + SPAM   https://doi.org/10.7910/DVN/G1HBNK
# ---Step1 data fusion, FAO + SAPM, Harvested Ared, add SAPM2019, SAPM2018
# ---Step2 data fusion, yield 2020，2019, 2018， production / harvested areas
# ---Step3  Remove outliers
# =====================================================
from osgeo import gdal
import numpy as np
import os
import glob

# Input / Output folders
input_folder = r"Z:\staff\Qiang_S\09_Datasets\02N_Output\02Global_Yield_Maize\03TempData\MaizeYield_GGCP10_SPAM_1Temp"
output_folder = r"Z:\staff\Qiang_S\09_Datasets\02N_Output\02Global_Yield_Maize\02StandardizedDatasets\MaizeYield_GGCP10SPAM5min"
os.makedirs(output_folder, exist_ok=True)

# ---- Step 1: Multiply ratio ----
ratio = 1     # e.g. 6, 0.001, 10.5

# ---- Step 2: Remove specific values ----
replace_values = [0]   # values to be treated as invalid

# ---- Step 3: Remove value range ----
range_min = 15822.7  # reference SPAM yield 2020, remove 3 time Standard Deviation 
range_max = 1e10       # large number

# ---- Final NoData value ----
nodata_value = -9999

# =====================================================
# Batch Processing
# =====================================================
tif_files = glob.glob(os.path.join(input_folder, "*.tif"))
print(f"Found {len(tif_files)} GeoTIFF files\n")

for i, in_file in enumerate(tif_files, 1):
    fname = os.path.basename(in_file)
    out_file = os.path.join(output_folder, fname)

    print(f"[{i}/{len(tif_files)}] Processing: {fname}")

    ds = gdal.Open(in_file)
    band = ds.GetRasterBand(1)
    ori_nodata = band.GetNoDataValue()

    arr = band.ReadAsArray().astype(np.float32)

    # -----------------------------
    # Statistics: total pixels
    # -----------------------------
    total_pixels = arr.size

    # -----------------------------
    # Original valid pixels
    # -----------------------------
    if ori_nodata is not None:
        valid_original = np.sum(arr != ori_nodata)
        arr[arr == ori_nodata] = np.nan
    else:
        valid_original = np.sum(~np.isnan(arr))

    # -----------------------------
    # Multiply by ratio
    # -----------------------------
    arr = arr * ratio

    # -----------------------------
    # Remove specific values
    # -----------------------------
    for v in replace_values:
        arr[arr == v] = np.nan

    # -----------------------------
    # Remove values in range
    # -----------------------------
    mask_range = (arr > range_min) & (arr < range_max)
    arr[mask_range] = np.nan

    # -----------------------------
    # Final statistics
    # -----------------------------
    valid_final = np.sum(~np.isnan(arr))
    removed_pixels = valid_original - valid_final
    removed_ratio = (removed_pixels / valid_original * 100) if valid_original > 0 else 0

    # -----------------------------
    # Convert NaN → nodata
    # -----------------------------
    arr = np.where(np.isnan(arr), nodata_value, arr)

    # -----------------------------
    # Write output GeoTIFF
    # -----------------------------
    driver = gdal.GetDriverByName("GTiff")
    out_ds = driver.Create(
        out_file,
        ds.RasterXSize,
        ds.RasterYSize,
        1,
        gdal.GDT_Float32,
        options=["COMPRESS=DEFLATE", "PREDICTOR=2", "TILED=YES"]
    )

    out_ds.SetGeoTransform(ds.GetGeoTransform())
    out_ds.SetProjection(ds.GetProjection())

    out_band = out_ds.GetRasterBand(1)
    out_band.SetNoDataValue(nodata_value)
    out_band.WriteArray(arr)

    out_ds.FlushCache()
    out_ds = None
    ds = None

    # -----------------------------
    # Print statistics
    # -----------------------------
    print(f"  Total pixels        : {total_pixels:,}")
    print(f"  Valid (original)    : {valid_original:,}")
    print(f"  Valid (final)       : {valid_final:,}")
    print(f"  Removed pixels      : {removed_pixels:,}")
    print(f"  Removed percentage  : {removed_ratio:.2f}%\n")

print("✅ ALL FILES FINISHED")
print(f"Output directory: {output_folder}")

In [ ]:
# =====================================================
# 🎉🎉🎉🎉🎉N Output, Grid datasets have been processed🎉🎉🎉🎉🎉
# =====================================================

In [ ]:
# =====================================================
# -------Global Maize NUE---Based on the weighted median------
# ---Step 1----Setting parameters----------
# ---Step 2--- Run ----------------
# =====================================================
# Please this two files
# config_GlobalNUEM_W2_PFPN_Nup.yaml
# 02GlobalNUE4Maize_W2_Nup_PFPN_YFer.py
# python 02GlobalNUE4Maize_W2_Nup_PFPN_YFer.py --config config_GlobalNUEM_W2_PFPN_Nup.yaml

In [ ]:
# =====================================================
# -------Global Maize NUE---Based on the weighted median-------------
# ---Step 1----Setting parameters----------
# ---Step 2--- Run ----------------
# ---Step 3--- simple validation --------------
# =====================================================

In [ ]:
# =====================================================
# -------Global Maize NUE---Based on the weighted median-------------
# ---Step 1----Setting parameters----------
# ---Step 2--- Run ----------------
# ---Step 3--- simple validation --------------
# ---Step 4--- Match with harvested area, mask
# =====================================================
# ============================================================
# Year-by-Year GeoTIFF Masking - Intersection Only used 2match
#
# Dataset A: Reference mask (e.g. Area_1995.tif)
# Dataset B: Target data    (e.g. NUE_1995.tif)
#
# Rules:
#   A valid, B valid   -> Keep original B value
#   A valid, B NoData  -> NoData
#   A invalid, B valid -> NoData
#   A invalid, B NoData -> NoData
#
# No interpolation, no filling, no resampling.
# ============================================================

from pathlib import Path
import re
import csv
import numpy as np
import rasterio


# ============================================================
# 1. USER CONFIGURATION
# ============================================================

# Dataset A: Reference mask
FOLDER_A = Path(
    r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\GCSN_HarvestedArea_1995_2020\02StandardizedDatasets\02MaizeHarvestedArea"
)

# Dataset B: Target data
FOLDER_B = Path(
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\02Global_NUE_Maize\02MaizeNUE_M2_20261029V1_Median"
)

# Output folder
OUTPUT_FOLDER = Path(
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\02Global_NUE_Maize\02MaizeNUE_M2_20261029V1_Median_Intersection"
)

# ------------------------------------------------------------
# Mask definition
# ------------------------------------------------------------

# "positive": A > 0 is valid (recommended for harvested area)
# "non_nodata": Any finite A value, including zero, is valid
MASK_MODE = "positive"

MASK_THRESHOLD = 0.0

# Optional additional NoData values
# GeoTIFF metadata NoData is automatically recognized
ADDITIONAL_NODATA_A = [-9999.0]
ADDITIONAL_NODATA_B = [-9999.0]

# Output settings
OUTPUT_NODATA = np.nan
OVERWRITE = False


# ============================================================
# 2. INITIALIZE
# ============================================================

FOLDER_A = FOLDER_A.resolve()
FOLDER_B = FOLDER_B.resolve()
OUTPUT_FOLDER = OUTPUT_FOLDER.resolve()

if not FOLDER_A.is_dir():
    raise FileNotFoundError(f"Folder A not found: {FOLDER_A}")

if not FOLDER_B.is_dir():
    raise FileNotFoundError(f"Folder B not found: {FOLDER_B}")

if MASK_MODE not in ("positive", "non_nodata"):
    raise ValueError("Invalid MASK_MODE")

if OUTPUT_FOLDER in (FOLDER_A, FOLDER_B):
    raise ValueError("Output folder must differ from input folders")

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)


# ============================================================
# 3. COLLECT FILES BY YEAR
# ============================================================

def extract_year(filename):

    stem = Path(filename).stem

    match = re.search(
        r"_(19\d{2}|20\d{2})$",
        stem
    )

    return int(match.group(1)) if match else None


def collect_files(folder):

    files = {}

    for path in sorted(folder.iterdir()):

        if not path.is_file():
            continue

        if path.suffix.lower() not in (".tif", ".tiff"):
            continue

        year = extract_year(path.name)

        if year is None:
            continue

        if year in files:
            raise ValueError(
                f"Duplicate year {year}: "
                f"{files[year].name}, {path.name}"
            )

        files[year] = path

    return files


files_A = collect_files(FOLDER_A)
files_B = collect_files(FOLDER_B)

common_years = sorted(
    set(files_A) & set(files_B)
)

print("=" * 65)
print("GEOTIFF INTERSECTION MASKING")
print("=" * 65)

print(f"A files       : {len(files_A)}")
print(f"B files       : {len(files_B)}")
print(f"Matched years : {len(common_years)}")
print(f"Years         : {common_years}")

print("Missing A:", sorted(set(files_B) - set(files_A)))
print("Missing B:", sorted(set(files_A) - set(files_B)))

if not common_years:
    raise ValueError("No matched years found")


# ============================================================
# 4. CHECK GRID ALIGNMENT
# ============================================================

def check_grid(src_a, src_b):

    if src_a.crs is None or src_b.crs is None:
        raise ValueError("Missing raster CRS")

    if src_a.crs != src_b.crs:
        raise ValueError("CRS mismatch")

    if (
        src_a.width != src_b.width
        or src_a.height != src_b.height
    ):
        raise ValueError("Raster dimensions mismatch")

    if not src_a.transform.almost_equals(
        src_b.transform,
        precision=1e-10
    ):
        raise ValueError("Raster transform mismatch")

    if src_a.count != 1 or src_b.count != 1:
        raise ValueError("Only single-band GeoTIFF supported")


# ============================================================
# 5. READ DATA AND IDENTIFY VALID PIXELS
# ============================================================

def read_valid_data(src, additional_nodata):

    # Automatically mask metadata-defined NoData pixels
    data = src.read(1, masked=True)

    values = data.astype(np.float32).filled(np.nan)

    valid = np.isfinite(values)

    for nd in additional_nodata:
        valid &= (values != nd)

    return values, valid


# ============================================================
# 6. PROCESS ONE YEAR
# ============================================================

def process_year(year, path_a, path_b):

    output_path = OUTPUT_FOLDER / path_b.name

    if output_path.exists() and not OVERWRITE:

        print(f"{year}: SKIPPED - Already exists")

        return {
            "year": year,
            "status": "SKIPPED"
        }

    with rasterio.open(path_a) as src_a, \
         rasterio.open(path_b) as src_b:

        # Ensure identical grids
        check_grid(src_a, src_b)

        # ----------------------------------------------------
        # Read A
        # ----------------------------------------------------

        a_data, a_valid = read_valid_data(
            src_a,
            ADDITIONAL_NODATA_A
        )

        # Define mask
        if MASK_MODE == "positive":

            mask_a = (
                a_valid
                & (a_data > MASK_THRESHOLD)
            )

        else:
            mask_a = a_valid

        # ----------------------------------------------------
        # Read B
        # ----------------------------------------------------

        b_data, b_valid = read_valid_data(
            src_b,
            ADDITIONAL_NODATA_B
        )

        # ----------------------------------------------------
        # Intersection: A valid AND B valid
        # ----------------------------------------------------

        intersection_mask = mask_a & b_valid

        # Initialize all pixels as NoData
        output = np.full(
            b_data.shape,
            np.nan,
            dtype=np.float32
        )

        # Only copy valid B values inside A
        output[intersection_mask] = b_data[intersection_mask]

        # ----------------------------------------------------
        # Statistics
        # ----------------------------------------------------

        total_pixels = b_data.size

        a_pixels = int(np.count_nonzero(mask_a))

        b_pixels = int(np.count_nonzero(b_valid))

        intersection_pixels = int(
            np.count_nonzero(intersection_mask)
        )

        # A valid, B missing
        a_only = int(
            np.count_nonzero(mask_a & (~b_valid))
        )

        # B valid, outside A
        b_removed = int(
            np.count_nonzero((~mask_a) & b_valid)
        )

        # ----------------------------------------------------
        # Export GeoTIFF
        # ----------------------------------------------------

        profile = src_b.profile.copy()

        profile.update(
            driver="GTiff",
            dtype="float32",
            count=1,
            nodata=OUTPUT_NODATA,
            compress="deflate",
            predictor=3,
            tiled=True,
            blockxsize=256,
            blockysize=256,
            BIGTIFF="IF_SAFER"
        )

        with rasterio.open(
            output_path,
            "w",
            **profile
        ) as dst:

            dst.write(output, 1)

        print(
            f"{year} | "
            f"A={a_pixels:,} | "
            f"B={b_pixels:,} | "
            f"Intersection={intersection_pixels:,} | "
            f"Removed_B={b_removed:,} | "
            f"A_without_B={a_only:,}"
        )

        return {
            "year": year,
            "status": "SUCCESS",
            "total_pixels": total_pixels,
            "A_valid": a_pixels,
            "B_valid": b_pixels,
            "intersection": intersection_pixels,
            "B_removed": b_removed,
            "A_without_B": a_only,
            "output_file": str(output_path)
        }


# ============================================================
# 7. BATCH PROCESS
# ============================================================

results = []

for index, year in enumerate(common_years, start=1):

    print(f"\n[{index}/{len(common_years)}] Processing {year}")

    try:

        result = process_year(
            year,
            files_A[year],
            files_B[year]
        )

        results.append(result)

    except Exception as e:

        print(f"ERROR {year}: {e}")

        results.append({
            "year": year,
            "status": "FAILED",
            "error": str(e)
        })


# ============================================================
# 8. EXPORT PROCESSING LOG
# ============================================================

log_path = OUTPUT_FOLDER / "Intersection_Mask_Log.csv"

fieldnames = [
    "year",
    "status",
    "total_pixels",
    "A_valid",
    "B_valid",
    "intersection",
    "B_removed",
    "A_without_B",
    "output_file",
    "error"
]

with open(
    log_path,
    "w",
    newline="",
    encoding="utf-8-sig"
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=fieldnames
    )

    writer.writeheader()
    writer.writerows(results)


# ============================================================
# 9. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("INTERSECTION MASKING COMPLETED")
print("=" * 65)

for status in ("SUCCESS", "SKIPPED", "FAILED"):

    count = sum(
        r["status"] == status
        for r in results
    )

    print(f"{status}: {count}")

print(f"\nOutput folder: {OUTPUT_FOLDER}")
print(f"Log file: {log_path}")

print("=" * 65)

In [ ]:
# =====================================================
# -------Global Maize NUE---Based on the weighted median-------------
# ---Step 1----Setting parameters----------
# ---Step 2--- Run ----------------
# ---Step 3--- simple validation --------------
# ---Step 4--- Match with harvested area, mask
# ---Step 5--- Remove outliers, Useful tool,Pixels number should larger than 6, NUE more then 150; set non, 
# =====================================================
# Global NUE - Parallel Outlier Processing Options 

# Step 1: N < MIN_COUNT or N missing -> REMOVE
# Step 2:
#   Upper threshold -> REMOVE / NEAREST
#   Lower threshold -> REMOVE / NEAREST
# Step 3:
#   Optional country-level mean + STD_FACTOR * STD
#   -> REMOVE / NEAREST
#   Maximum selected proportion per country: 5%
#
# NaN / NoData safe
# Parallel processing by year
# Output: GeoTIFF + Excel (3 original sheets)
#
# JupyterLab single-cell executable
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

from rasterio.features import geometry_mask
from scipy.ndimage import distance_transform_edt
from joblib import Parallel, delayed

from openpyxl import Workbook
from openpyxl.styles import (
    Font, PatternFill, Alignment, Border, Side
)
from openpyxl.utils import get_column_letter

# ============================================================
# 1. USER CONFIGURATION
# ===========================================================
# ------------------------------------------------------------
# Input / output
# ------------------------------------------------------------

A_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\02Global_NUE_Maize\02MaizeNUE_M2_20261029V1_Median_Intersection"
)


MASK_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\02Global_NUE_Maize\02MaizeNUE_M2_20261029V1"
)

OUT_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\02Global_NUE_Maize\02MaizeNUE_M2_20261029V1_M6_150R"
)

SHP_PATH = (
    r"Z:\staff\Qiang_S\09_Datasets"
    r"\09OtherDataUsed\GDAM_GlobalCountryLevel_Shp"
    r"\GlobalCountryLevel_Shp.shp"
)

M49_FIELD = "M49_Code"
NAME_FIELD = "COUNTRY"

EXCEL_OUT = os.path.join(
    OUT_FOLDER,
    "outlier_removal_summary_V1_M6_150R.xlsx"
)

# ------------------------------------------------------------
# Years and parallel processing
# ------------------------------------------------------------

START_YEAR = 1995
END_YEAR = 2020

# None = all years
# Example: [1995, 1996, 2000, 2020]
YEARS_TO_RUN = None

PARALLEL_YEARS = 10

# ------------------------------------------------------------
# Common settings
# ------------------------------------------------------------

NODATA = -9999.0

# ------------------------------------------------------------
# Step 1: Minimum valid sample count
# ------------------------------------------------------------

MIN_COUNT = 6

# ------------------------------------------------------------
# Step 2: Hard thresholds
# ------------------------------------------------------------
LOWER_HARD = 0.01
UPPER_HARD = 1.5

# Options: "REMOVE" or "NEAREST"
UPPER_OUTLIER_ACTION = "REMOVE"
LOWER_OUTLIER_ACTION = "REMOVE"

# ------------------------------------------------------------
# Step 3: Country standard deviation
# ------------------------------------------------------------

# True = enable country standard deviation processing
# False = skip Step 3

APPLY_STD_REMOVAL = False #True

# Mean + STD_FACTOR * standard deviation
STD_FACTOR = 2.0

# Maximum selected outlier ratio per country
MAX_REMOVE_RATIO = 0.05

# NEW:
# "REMOVE"  = Set selected Step 3 outliers to NoData
# "NEAREST" = Replace with nearest qualified value inside the SAME country

STD_OUTLIER_ACTION = "REMOVE"

# ------------------------------------------------------------
# Nearest-neighbor settings
# ------------------------------------------------------------

# None = No maximum distance
# Example: 12 = maximum 12 raster pixels
MAX_NEAREST_DISTANCE_PIXELS = None

# If no qualified nearest value exists,
# the target becomes NoData.


# ============================================================
# 2. PARAMETER VALIDATION
# ============================================================

VALID_ACTIONS = {"REMOVE", "NEAREST"}

UPPER_OUTLIER_ACTION = UPPER_OUTLIER_ACTION.upper()
LOWER_OUTLIER_ACTION = LOWER_OUTLIER_ACTION.upper()
STD_OUTLIER_ACTION = STD_OUTLIER_ACTION.upper()

for name, action in [
    ("UPPER_OUTLIER_ACTION", UPPER_OUTLIER_ACTION),
    ("LOWER_OUTLIER_ACTION", LOWER_OUTLIER_ACTION),
    ("STD_OUTLIER_ACTION", STD_OUTLIER_ACTION)
]:
    if action not in VALID_ACTIONS:
        raise ValueError(
            f"{name} must be REMOVE or NEAREST"
        )

if LOWER_HARD >= UPPER_HARD:
    raise ValueError(
        "LOWER_HARD must be smaller than UPPER_HARD"
    )

if PARALLEL_YEARS < 1:
    raise ValueError("PARALLEL_YEARS must be >= 1")

if not (0 <= MAX_REMOVE_RATIO <= 1):
    raise ValueError(
        "MAX_REMOVE_RATIO must be between 0 and 1"
    )

if (
    MAX_NEAREST_DISTANCE_PIXELS is not None
    and MAX_NEAREST_DISTANCE_PIXELS < 0
):
    raise ValueError(
        "MAX_NEAREST_DISTANCE_PIXELS must be >= 0"
    )

if not os.path.isfile(SHP_PATH):
    raise FileNotFoundError(SHP_PATH)

os.makedirs(OUT_FOLDER, exist_ok=True)


# ============================================================
# 3. RASTER UTILITY FUNCTIONS
# ============================================================

def read_raster_fixed(src):
    """
    Read raster and standardize:
      Metadata NoData
      NaN / Inf
      -9999

    Valid values remain unchanged.
    """

    data = (
        src.read(1, masked=True)
        .astype(np.float32)
        .filled(np.nan)
    )

    data = np.asarray(
        data,
        dtype=np.float32
    )

    invalid = (
        ~np.isfinite(data)
        | (data == NODATA)
    )

    data[invalid] = NODATA

    return data


def get_valid_mask(data):

    return (
        np.isfinite(data)
        & (data != NODATA)
    )


def empty_country_record(year, m49, name):

    return {
        "Year": year,
        "M49": m49,
        "Country": name,
        "Valid_before_step3": 0,
        "Mean": np.nan,
        "Std": np.nan,
        "Upper_sigma": np.nan,
        "Candidate_removed": 0,
        "Max_allowed_remove": 0,
        "Step3_removed": 0,
        "Step3_removed_%": 0.0
    }


# ============================================================
# 4. NEAREST-NEIGHBOR REPLACEMENT
# ============================================================

def replace_by_nearest(
    raster,
    target_mask,
    donor_mask,
    max_distance=None
):
    """
    Replace selected target pixels with nearest donor values.

    Parameters
    ----------
    raster:
        2D NumPy array, modified in place.

    target_mask:
        True for pixels selected for replacement.

    donor_mask:
        True for qualified source pixels.

    max_distance:
        Maximum distance in pixel units.
        None = unlimited.

    Returns
    -------
    replaced_count:
        Number successfully replaced.

    failed_count:
        Number without an eligible donor.
        These pixels become NoData.

    Notes
    -----
    - Original donor values are copied exactly.
    - No averaging or interpolation.
    - Calculations use row/column Euclidean distance.
    - Processing uses a local bounding window.
    """

    target_rows, target_cols = np.where(target_mask)

    target_count = len(target_rows)

    if target_count == 0:
        return 0, 0

    donor_mask = (
        donor_mask
        & get_valid_mask(raster)
        & ~target_mask
    )

    donor_rows, donor_cols = np.where(donor_mask)

    # No qualified donors
    if len(donor_rows) == 0:

        raster[target_rows, target_cols] = NODATA

        return 0, target_count

    # --------------------------------------------------------
    # Bounding window containing targets and donors
    # --------------------------------------------------------

    row_min = int(min(
        target_rows.min(),
        donor_rows.min()
    ))

    row_max = int(max(
        target_rows.max(),
        donor_rows.max()
    )) + 1

    col_min = int(min(
        target_cols.min(),
        donor_cols.min()
    ))

    col_max = int(max(
        target_cols.max(),
        donor_cols.max()
    )) + 1

    local_donors = donor_mask[
        row_min:row_max,
        col_min:col_max
    ]

    local_rows = target_rows - row_min
    local_cols = target_cols - col_min

    # --------------------------------------------------------
    # Nearest eligible donor
    # --------------------------------------------------------

    if max_distance is None:

        nearest_indices = distance_transform_edt(
            ~local_donors,
            return_distances=False,
            return_indices=True
        )

        nearest_rows = (
            nearest_indices[
                0,
                local_rows,
                local_cols
            ] + row_min
        )

        nearest_cols = (
            nearest_indices[
                1,
                local_rows,
                local_cols
            ] + col_min
        )

        replacement_values = raster[
            nearest_rows,
            nearest_cols
        ].copy()

        raster[
            target_rows,
            target_cols
        ] = replacement_values

        return target_count, 0

    else:

        distances, nearest_indices = distance_transform_edt(
            ~local_donors,
            return_distances=True,
            return_indices=True
        )

        target_distances = distances[
            local_rows,
            local_cols
        ]

        within_distance = (
            target_distances <= max_distance
        )

        # Qualified targets
        valid_rows = target_rows[within_distance]
        valid_cols = target_cols[within_distance]

        nearest_rows = (
            nearest_indices[
                0,
                local_rows[within_distance],
                local_cols[within_distance]
            ] + row_min
        )

        nearest_cols = (
            nearest_indices[
                1,
                local_rows[within_distance],
                local_cols[within_distance]
            ] + col_min
        )

        replacement_values = raster[
            nearest_rows,
            nearest_cols
        ].copy()

        raster[
            valid_rows,
            valid_cols
        ] = replacement_values

        # Targets exceeding maximum distance
        failed_rows = target_rows[~within_distance]
        failed_cols = target_cols[~within_distance]

        raster[
            failed_rows,
            failed_cols
        ] = NODATA

        replaced_count = int(
            np.count_nonzero(within_distance)
        )

        failed_count = target_count - replaced_count

        return replaced_count, failed_count


# ============================================================
# 5. PROCESS ONE YEAR
# ============================================================

def process_one_year(year):

    country_records = []

    A_tif = os.path.join(
        A_FOLDER,
        f"NUE_M2_median_{year}.tif"
    )

    N_tif = os.path.join(
        MASK_FOLDER,
        f"NUE_M2_n_{year}.tif"
    )

    OUT_tif = os.path.join(
        OUT_FOLDER,
        f"NUE_median_Opt_{year}.tif"
    )

    try:

        if not os.path.exists(A_tif):
            return {
                "year": year,
                "status": "MISSING_A",
                "message": A_tif,
                "annual": None,
                "countries": []
            }

        if not os.path.exists(N_tif):
            return {
                "year": year,
                "status": "MISSING_N",
                "message": N_tif,
                "annual": None,
                "countries": []
            }

        # ----------------------------------------------------
        # Read country polygons within each process
        # ----------------------------------------------------

        gdf = gpd.read_file(SHP_PATH)

        if gdf.crs is None:
            raise ValueError("Shapefile CRS missing")

        if gdf.crs.to_epsg() != 4326:
            gdf = gdf.to_crs(epsg=4326)

        gdf = gdf.sort_values(
            by=M49_FIELD
        ).reset_index(drop=True)

        # ----------------------------------------------------
        # Read raster A
        # ----------------------------------------------------

        with rasterio.open(A_tif) as src_a:

            A = read_raster_fixed(src_a)

            profile = src_a.profile.copy()
            transform = src_a.transform
            crs = src_a.crs

            height, width = A.shape

        # ----------------------------------------------------
        # Read raster N
        # ----------------------------------------------------

        with rasterio.open(N_tif) as src_n:

            N = read_raster_fixed(src_n)

            if (
                src_n.crs != crs
                or src_n.transform != transform
            ):
                raise ValueError(
                    "A and N raster grids are not aligned"
                )

        if A.shape != N.shape:
            raise ValueError(
                f"Shape mismatch: A={A.shape}, N={N.shape}"
            )

        if crs is None:
            raise ValueError("Raster CRS missing")

        # ====================================================
        # STEP 1 - N < MIN_COUNT / Missing N
        # ====================================================

        valid_mask = get_valid_mask(A)

        initial_count = int(
            np.count_nonzero(valid_mask)
        )

        n_valid_mask = get_valid_mask(N)

        step1_mask = (
            valid_mask
            & (
                ~n_valid_mask
                | (N < MIN_COUNT)
            )
        )

        step1_removed = int(
            np.count_nonzero(step1_mask)
        )

        A[step1_mask] = NODATA

        valid_mask = get_valid_mask(A)

        # ====================================================
        # STEP 2 - HARD THRESHOLDS
        # ====================================================

        step2_upper = (
            (A > UPPER_HARD)
            & valid_mask
        )

        step2_lower = (
            (A < LOWER_HARD)
            & valid_mask
        )

        upper_count = int(
            np.count_nonzero(step2_upper)
        )

        lower_count = int(
            np.count_nonzero(step2_lower)
        )

        # Same reported field as original script:
        # Number of pixels identified outside hard thresholds
        step2_removed = upper_count + lower_count

        # ----------------------------------------------------
        # Determine remove / nearest targets
        # ----------------------------------------------------

        step2_remove_mask = np.zeros_like(
            valid_mask,
            dtype=bool
        )

        step2_nearest_mask = np.zeros_like(
            valid_mask,
            dtype=bool
        )

        if UPPER_OUTLIER_ACTION == "REMOVE":
            step2_remove_mask |= step2_upper
        else:
            step2_nearest_mask |= step2_upper

        if LOWER_OUTLIER_ACTION == "REMOVE":
            step2_remove_mask |= step2_lower
        else:
            step2_nearest_mask |= step2_lower

        # Qualified donor pixels are those inside
        # the original hard thresholds, after Step 1.

        step2_donors = (
            valid_mask
            & (A >= LOWER_HARD)
            & (A <= UPPER_HARD)
        )

        # ----------------------------------------------------
        # REMOVE
        # ----------------------------------------------------

        step2_direct_removed = int(
            np.count_nonzero(step2_remove_mask)
        )

        A[step2_remove_mask] = NODATA

        # ----------------------------------------------------
        # NEAREST
        # ----------------------------------------------------

        step2_replaced, step2_failed = replace_by_nearest(
            A,
            step2_nearest_mask,
            step2_donors,
            MAX_NEAREST_DISTANCE_PIXELS
        )

        step2_actual_removed = (
            step2_direct_removed
            + step2_failed
        )

        valid_mask = get_valid_mask(A)

        # ====================================================
        # STEP 3 - COUNTRY STANDARD DEVIATION
        # ====================================================

        # Reported identified/selected count
        step3_total_removed = 0

        # Actual deletion count
        step3_actual_removed = 0

        # Successful replacement count
        step3_total_replaced = 0

        if APPLY_STD_REMOVAL:

            if gdf.crs != crs:
                gdf_reproj = gdf.to_crs(crs)
            else:
                gdf_reproj = gdf

            # Preserve M49 country processing order
            for _, country_row in gdf_reproj.iterrows():

                m49 = country_row[M49_FIELD]

                name = (
                    country_row[NAME_FIELD]
                    if NAME_FIELD in country_row
                    else str(m49)
                )

                geom = country_row.geometry

                if geom is None or geom.is_empty:

                    country_records.append(
                        empty_country_record(
                            year, m49, name
                        )
                    )

                    continue

                # --------------------------------------------
                # Country raster mask
                # --------------------------------------------

                try:

                    country_mask_arr = ~geometry_mask(
                        [geom],
                        transform=transform,
                        invert=False,
                        out_shape=(height, width)
                    )

                except Exception:

                    country_records.append(
                        empty_country_record(
                            year, m49, name
                        )
                    )

                    continue

                # --------------------------------------------
                # Country valid pixels
                # --------------------------------------------

                in_country_valid = (
                    country_mask_arr
                    & valid_mask
                )

                pixel_indices = np.where(
                    in_country_valid
                )

                n_valid = len(pixel_indices[0])

                if n_valid == 0:

                    country_records.append(
                        empty_country_record(
                            year, m49, name
                        )
                    )

                    continue

                # --------------------------------------------
                # Mean and standard deviation
                # --------------------------------------------

                vals = A[pixel_indices]

                mean_val = float(
                    np.mean(
                        vals,
                        dtype=np.float64
                    )
                )

                std_val = float(
                    np.std(
                        vals,
                        dtype=np.float64
                    )
                )

                upper_sigma = (
                    mean_val
                    + STD_FACTOR * std_val
                )

                # --------------------------------------------
                # Identify Step 3 candidates
                # --------------------------------------------

                candidate_mask = (
                    in_country_valid
                    & (A > upper_sigma)
                )

                n_candidate = int(
                    np.count_nonzero(candidate_mask)
                )

                max_allowed = int(
                    np.floor(
                        n_valid * MAX_REMOVE_RATIO
                    )
                )

                # --------------------------------------------
                # Select outliers
                #
                # Same 5% cap and descending-value rule
                # as the original script
                # --------------------------------------------

                selected_mask = np.zeros_like(
                    valid_mask,
                    dtype=bool
                )

                if n_candidate <= max_allowed:

                    selected_mask = candidate_mask

                    actual_selected = n_candidate

                else:

                    actual_selected = max_allowed

                    if max_allowed > 0:

                        cand_rows, cand_cols = np.where(
                            candidate_mask
                        )

                        cand_vals = A[
                            cand_rows,
                            cand_cols
                        ]

                        sort_idx = np.argsort(
                            -cand_vals
                        )[:max_allowed]

                        selected_mask[
                            cand_rows[sort_idx],
                            cand_cols[sort_idx]
                        ] = True

                # --------------------------------------------
                # NEW: Step 3 action
                # --------------------------------------------

                if STD_OUTLIER_ACTION == "REMOVE":

                    A[selected_mask] = NODATA

                    country_replaced = 0
                    country_actual_removed = actual_selected

                else:

                    # Nearest-neighbor donors:
                    # 1. Inside the same country
                    # 2. Valid after Step 2
                    # 3. Not above the country's sigma limit
                    #
                    # Exclude ALL values above sigma as donors,
                    # not just selected outliers.

                    country_donors = (
                        in_country_valid
                        & (A <= upper_sigma)
                    )

                    (
                        country_replaced,
                        country_actual_removed
                    ) = replace_by_nearest(
                        A,
                        selected_mask,
                        country_donors,
                        MAX_NEAREST_DISTANCE_PIXELS
                    )

                # --------------------------------------------
                # Update global raster valid mask
                # --------------------------------------------

                valid_mask = get_valid_mask(A)

                # Preserve existing reported meaning:
                # Step3_removed = selected outlier count
                # even when action is NEAREST.

                step3_total_removed += actual_selected

                step3_actual_removed += (
                    country_actual_removed
                )

                step3_total_replaced += (
                    country_replaced
                )

                # --------------------------------------------
                # Original country Excel fields
                # --------------------------------------------

                country_records.append({
                    "Year": year,
                    "M49": m49,
                    "Country": name,
                    "Valid_before_step3": n_valid,
                    "Mean": round(mean_val, 6),
                    "Std": round(std_val, 6),
                    "Upper_sigma": round(
                        upper_sigma, 6
                    ),
                    "Candidate_removed": n_candidate,
                    "Max_allowed_remove": max_allowed,
                    "Step3_removed": actual_selected,
                    "Step3_removed_%": (
                        round(
                            actual_selected / n_valid * 100,
                            4
                        )
                        if n_valid > 0 else 0.0
                    )
                })

        # ====================================================
        # 6. ANNUAL SUMMARY
        # ====================================================

        final_count = int(
            np.count_nonzero(
                get_valid_mask(A)
            )
        )

        total_removed = (
            initial_count - final_count
        )

        removed_ratio = (
            total_removed / initial_count * 100
            if initial_count > 0 else 0
        )

        # True number of deleted pixels:
        # replacement does NOT count as actual deletion.

        expected_actual_removed = (
            step1_removed
            + step2_actual_removed
            + step3_actual_removed
        )

        count_check = (
            total_removed
            - expected_actual_removed
        )

        # Original Annual Summary column structure
        annual_record = {
            "Year": year,
            "Initial_Valid": initial_count,
            "Step1_removed": step1_removed,
            "Step1_%": (
                round(
                    step1_removed / initial_count * 100,
                    4
                )
                if initial_count else 0
            ),
            "Step2_removed": step2_removed,
            "Step2_%": (
                round(
                    step2_removed / initial_count * 100,
                    4
                )
                if initial_count else 0
            ),
            "Step3_removed": step3_total_removed,
            "Step3_%": (
                round(
                    step3_total_removed / initial_count * 100,
                    4
                )
                if initial_count else 0
            ),
            "Final_Valid": final_count,
            "Total_Removed": total_removed,
            "Removed_Ratio_%": round(
                removed_ratio, 4
            )
        }

        # ====================================================
        # 7. EXPORT GEOTIFF
        # ====================================================

        profile.update(
            dtype="float32",
            nodata=NODATA,
            compress="lzw"
        )

        with rasterio.open(
            OUT_tif,
            "w",
            **profile
        ) as dst:

            dst.write(
                A.astype(np.float32),
                1
            )

        # ----------------------------------------------------
        # Return results to main process
        # ----------------------------------------------------

        return {
            "year": year,
            "status": "SUCCESS",
            "message": "",
            "annual": annual_record,
            "countries": country_records,
            "check": count_check,
            "step2_replaced": step2_replaced,
            "step3_replaced": step3_total_replaced,
            "actual_removed": total_removed,
            "output": OUT_tif
        }

    except Exception as e:

        return {
            "year": year,
            "status": "FAILED",
            "message": repr(e),
            "annual": None,
            "countries": []
        }


# ============================================================
# 8. PARALLEL PROCESSING BY YEAR
# ============================================================

YEARS = (
    list(range(START_YEAR, END_YEAR + 1))
    if YEARS_TO_RUN is None
    else sorted(set(map(int, YEARS_TO_RUN)))
)

if not YEARS:
    raise ValueError("No years selected")

print("=" * 70)
print("GLOBAL NUE OUTLIER PROCESSING")
print("=" * 70)

print(f"Years: {YEARS}")
print(f"Parallel workers: {PARALLEL_YEARS}")

print(f"Step 1 MIN_COUNT: {MIN_COUNT}")

print(
    f"Step 2 Upper: {UPPER_HARD} "
    f"({UPPER_OUTLIER_ACTION})"
)

print(
    f"Step 2 Lower: {LOWER_HARD} "
    f"({LOWER_OUTLIER_ACTION})"
)

print(
    f"Step 3 Enabled: {APPLY_STD_REMOVAL}"
)

print(
    f"Step 3 Action: {STD_OUTLIER_ACTION}"
)

print(
    f"Step 3 STD_FACTOR: {STD_FACTOR}"
)

print("=" * 70, flush=True)


if PARALLEL_YEARS == 1 or len(YEARS) == 1:

    results = [
        process_one_year(year)
        for year in YEARS
    ]

else:

    results = Parallel(
        n_jobs=min(
            PARALLEL_YEARS,
            len(YEARS)
        ),
        backend="loky",
        batch_size=1,
        pre_dispatch="n_jobs",
        verbose=10
    )(
        delayed(process_one_year)(year)
        for year in YEARS
    )


# ============================================================
# 9. COLLECT PARALLEL RESULTS
# ============================================================

results.sort(
    key=lambda item: item["year"]
)

records_year = []
records_country = []

for result in results:

    year = result["year"]
    status = result["status"]

    if status == "SUCCESS":

        records_year.append(
            result["annual"]
        )

        records_country.extend(
            result["countries"]
        )

        a = result["annual"]

        print(
            f"[{year}] "
            f"Initial={a['Initial_Valid']:,} | "
            f"Step1={a['Step1_removed']:,} | "
            f"Step2={a['Step2_removed']:,} | "
            f"Step3={a['Step3_removed']:,} | "
            f"Step2_Replaced={result['step2_replaced']:,} | "
            f"Step3_Replaced={result['step3_replaced']:,} | "
            f"Final={a['Final_Valid']:,} | "
            f"Check={result['check']}"
        )

    else:

        print(
            f"[{year}] {status}: "
            f"{result['message']}"
        )


# ============================================================
# 10. EXCEL REPORT
# ============================================================

COUNTRY_COLS = [
    "Year",
    "M49",
    "Country",
    "Valid_before_step3",
    "Mean",
    "Std",
    "Upper_sigma",
    "Candidate_removed",
    "Max_allowed_remove",
    "Step3_removed",
    "Step3_removed_%"
]

if not records_year:

    print(
        "\nNo successful years. Excel not generated."
    )

else:

    df_year = (
        pd.DataFrame(records_year)
        .sort_values("Year")
        .reset_index(drop=True)
    )

    df_country = pd.DataFrame(
        records_country,
        columns=COUNTRY_COLS
    )

    if not df_country.empty:

        df_country = (
            df_country
            .sort_values(["Year", "M49"])
            .reset_index(drop=True)
        )

    wb = Workbook()

    thin = Side(
        style="thin",
        color="AAAAAA"
    )

    border = Border(
        left=thin,
        right=thin,
        top=thin,
        bottom=thin
    )

    # --------------------------------------------------------
    # Excel formatting helpers
    # --------------------------------------------------------

    def hdr_cell(
        ws, row, col, value,
        bg="2F75B6",
        fg="FFFFFF"
    ):

        c = ws.cell(
            row=row,
            column=col,
            value=value
        )

        c.font = Font(
            bold=True,
            color=fg,
            name="Arial",
            size=10
        )

        c.fill = PatternFill(
            "solid",
            start_color=bg
        )

        c.alignment = Alignment(
            horizontal="center",
            wrap_text=True
        )

        c.border = border

        return c


    def data_cell(
        ws, row, col, value,
        bg="FFFFFF",
        fmt=None
    ):

        if isinstance(value, np.generic):
            value = value.item()

        if (
            isinstance(value, float)
            and not np.isfinite(value)
        ):
            value = None

        c = ws.cell(
            row=row,
            column=col,
            value=value
        )

        c.font = Font(
            name="Arial",
            size=10
        )

        c.fill = PatternFill(
            "solid",
            start_color=bg
        )

        c.alignment = Alignment(
            horizontal="center"
        )

        c.border = border

        if fmt:
            c.number_format = fmt

        return c


    # ========================================================
    # SHEET 1: ANNUAL SUMMARY
    # ========================================================

    ws1 = wb.active

    ws1.title = "Annual Summary"

    col_colors_year = {
        "Year": "FFFFFF",
        "Initial_Valid": "FFFFFF",
        "Step1_removed": "D6E4F0",
        "Step1_%": "D6E4F0",
        "Step2_removed": "D5E8D4",
        "Step2_%": "D5E8D4",
        "Step3_removed": "FFF2CC",
        "Step3_%": "FFF2CC",
        "Final_Valid": "FFFFFF",
        "Total_Removed": "F4CCCC",
        "Removed_Ratio_%": "F4CCCC"
    }

    cols_y = list(df_year.columns)

    for ci, col in enumerate(cols_y, 1):
        hdr_cell(ws1, 1, ci, col)

    for ri, row in enumerate(
        df_year.itertuples(index=False),
        2
    ):

        for ci, (col, val) in enumerate(
            zip(cols_y, row),
            1
        ):

            fmt = (
                "0.00"
                if "%" in col
                else (
                    "#,##0"
                    if col != "Year"
                    else "0"
                )
            )

            data_cell(
                ws1,
                ri,
                ci,
                val,
                bg=col_colors_year.get(
                    col,
                    "FFFFFF"
                ),
                fmt=fmt
            )

    # Original TOTAL/AVG row
    sr = len(df_year) + 2

    data_cell(
        ws1,
        sr,
        1,
        "TOTAL/AVG",
        bg="D9D9D9"
    ).font = Font(
        bold=True,
        name="Arial",
        size=10
    )

    for ci, col in enumerate(
        cols_y[1:],
        2
    ):

        cl = get_column_letter(ci)

        rng = (
            f"{cl}2:"
            f"{cl}{len(df_year) + 1}"
        )

        formula = (
            f"=AVERAGE({rng})"
            if "%" in col
            else f"=SUM({rng})"
        )

        c = ws1.cell(
            row=sr,
            column=ci,
            value=formula
        )

        c.font = Font(
            bold=True,
            name="Arial",
            size=10
        )

        c.fill = PatternFill(
            "solid",
            start_color="D9D9D9"
        )

        c.alignment = Alignment(
            horizontal="center"
        )

        c.border = border

        c.number_format = (
            "0.00"
            if "%" in col
            else "#,##0"
        )

    for ci, col in enumerate(cols_y, 1):

        ws1.column_dimensions[
            get_column_letter(ci)
        ].width = max(
            len(col) + 4,
            14
        )

    ws1.row_dimensions[1].height = 36
    ws1.freeze_panes = "B2"

    # ========================================================
    # SHEET 2: COUNTRY DETAIL
    # ========================================================

    ws2 = wb.create_sheet(
        "Country Detail"
    )

    cols_c = COUNTRY_COLS

    col_colors_cty = {
        "Year": "FFFFFF",
        "M49": "FFFFFF",
        "Country": "FFFFFF",
        "Valid_before_step3": "FFFFFF",
        "Mean": "EAD1DC",
        "Std": "EAD1DC",
        "Upper_sigma": "EAD1DC",
        "Candidate_removed": "FFF2CC",
        "Max_allowed_remove": "FFF2CC",
        "Step3_removed": "F4CCCC",
        "Step3_removed_%": "F4CCCC"
    }

    for ci, col in enumerate(cols_c, 1):
        hdr_cell(ws2, 1, ci, col)

    for ri, row in enumerate(
        df_country.itertuples(index=False),
        2
    ):

        for ci, (col, val) in enumerate(
            zip(cols_c, row),
            1
        ):

            if "%" in col:
                fmt = "0.00"

            elif col in (
                "Mean",
                "Std",
                "Upper_sigma"
            ):
                fmt = "0.000000"

            elif col not in (
                "Year",
                "M49",
                "Country"
            ):
                fmt = "#,##0"

            else:
                fmt = None

            data_cell(
                ws2,
                ri,
                ci,
                val,
                bg=col_colors_cty.get(
                    col,
                    "FFFFFF"
                ),
                fmt=fmt
            )

    for ci, col in enumerate(cols_c, 1):

        ws2.column_dimensions[
            get_column_letter(ci)
        ].width = max(
            len(col) + 3,
            12
        )

    ws2.column_dimensions["C"].width = 22
    ws2.row_dimensions[1].height = 36
    ws2.freeze_panes = "D2"

    # ========================================================
    # SHEET 3: PARAMETERS
    # ========================================================

    ws3 = wb.create_sheet(
        "Parameters"
    )

    params = [
        ("Parameter", "Value"),
        ("MIN_COUNT (Step1)", MIN_COUNT),
        ("LOWER_HARD (Step2)", LOWER_HARD),
        ("UPPER_HARD (Step2)", UPPER_HARD),
        ("LOWER_OUTLIER_ACTION", LOWER_OUTLIER_ACTION),
        ("UPPER_OUTLIER_ACTION", UPPER_OUTLIER_ACTION),
        ("APPLY_STD_REMOVAL", APPLY_STD_REMOVAL),
        ("STD_FACTOR (Step3)", STD_FACTOR),
        ("STD_OUTLIER_ACTION", STD_OUTLIER_ACTION),
        ("MAX_REMOVE_RATIO (Step3)", MAX_REMOVE_RATIO),
        (
            "MAX_NEAREST_DISTANCE_PIXELS",
            "None" if MAX_NEAREST_DISTANCE_PIXELS is None
            else MAX_NEAREST_DISTANCE_PIXELS
        ),
        ("M49_FIELD", M49_FIELD),
        ("SHP_PATH", SHP_PATH),
        ("START_YEAR", START_YEAR),
        ("END_YEAR", END_YEAR),
        ("NODATA", NODATA),
        ("PARALLEL_YEARS", PARALLEL_YEARS),
        (
            "YEARS_TO_RUN",
            "All" if YEARS_TO_RUN is None
            else str(YEARS)
        )
    ]

    for ri, (k, v) in enumerate(
        params,
        1
    ):

        for ci, val in enumerate(
            (k, v),
            1
        ):

            c = ws3.cell(
                row=ri,
                column=ci,
                value=val
            )

            c.font = Font(
                bold=(ri == 1),
                name="Arial",
                size=10,
                color=(
                    "FFFFFF"
                    if ri == 1
                    else "000000"
                )
            )

            c.fill = PatternFill(
                "solid",
                start_color=(
                    "2F75B6"
                    if ri == 1
                    else "F5F5F5"
                )
            )

            c.border = border

            c.alignment = Alignment(
                horizontal="left"
            )

    ws3.column_dimensions["A"].width = 38
    ws3.column_dimensions["B"].width = 60

    # --------------------------------------------------------
    # Save Excel
    # --------------------------------------------------------

    wb.save(EXCEL_OUT)

    print(
        f"\nExcel saved: {EXCEL_OUT}"
    )

    # ========================================================
    # 11. FINAL VALIDATION
    # ========================================================

    check_rows = []

    for result in results:

        if result["status"] == "SUCCESS":

            check_rows.append({
                "Year": result["year"],
                "Step2_detected": (
                    result["annual"]["Step2_removed"]
                ),
                "Step3_detected": (
                    result["annual"]["Step3_removed"]
                ),
                "Step2_replaced": (
                    result["step2_replaced"]
                ),
                "Step3_replaced": (
                    result["step3_replaced"]
                ),
                "Actual_removed": (
                    result["actual_removed"]
                ),
                "Check": result["check"]
            })

    df_check = pd.DataFrame(
        check_rows
    )

    print("\n" + "=" * 70)
    print("FINAL VALIDATION SUMMARY")
    print("=" * 70)

    print(
        df_check.to_string(
            index=False
        )
    )

    if (df_check["Check"] == 0).all():

        print(
            "\nPASS: All annual pixel counts are consistent."
        )

    else:

        print(
            "\nWARNING: Pixel count inconsistency detected."
        )

print("\nAll requested years finished.")

In [ ]:
# =====================================================
# -------Global Maize NUE---Based on the weighted median-------------
# ---Step 1----Setting parameters----------
# ---Step 2--- Run ----------------
# ---Step 3--- simple validation --------------
# ---Step 4--- Match with harvested area, mask
# ---Step 5--- Remove outliers, Useful tool,Pixels number should larger than 6, NUE more then 150; set non, 
# ---Step 6--- Check mean value
# ============================================================
# Tool: Global NUE Annual Mean Comparison - 5 Methods mean4GNUPW
# Method 1: Arithmetic Mean
# Method 2: Percentile Trimmed Mean (95%)
# Method 3: Mean within +/- 2 Standard Deviations
# Method 4: Harvested-Area-Weighted Mean
# Method 5: Nup-based NUE - Ratio of Sums
#
#            SUM(Nup_i * HA_i)
# NUE_global = -----------------------------
#              SUM((Nup_i / NUE_i) * HA_i)
#
# IMPORTANT:
# - Match NUE, Nup, HA by year and pixel
# - NUE is the core validity mask
# - Use identical valid pixels in both sums
# - No raster modification or resampling
# - Run directly in ONE JupyterLab Cell
# ============================================================

import os
import re
import time
from pathlib import Path
import numpy as np
import rasterio

# ============================================================
# 1. USER CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# NUE GeoTIFF folder - original path
# ------------------------------------------------------------

INPUT_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\02Global_NUE_Maize\02MaizeNUE_M2_20261029V1_M6_150R"
)
# ------------------------------------------------------------
# Harvested Area GeoTIFF folder - original path
# ------------------------------------------------------------

AREA_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\GCSN_HarvestedArea_1995_2020\02StandardizedDatasets\02MaizeHarvestedArea"
)

# ------------------------------------------------------------
# NEW: Nup GeoTIFF folder
#
# MUST CHANGE to your actual Nup folder.
# ------------------------------------------------------------

NUP_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\02Global_NUE_Maize\02MaizeNUE_M2_20261029V1_Nup"
)

CROP = "Maize"

# ------------------------------------------------------------
# File matching
#
# NUE example:
#   NUE_median_Opt_1995.tif
#
# Harvested Area example:
#   Maize_HarArea_1995.tif
#
# Nup examples:
#   Maize_Nup_1995.tif
#   Nup_Maize_1995.tif
#   Maize_Nuptake_1995.tif
#
# Adjust NUP_FILENAME_REGEX if necessary.
# ------------------------------------------------------------

NUE_FILENAME_REGEX = (
    r"^.*_(?P<year>(?:19|20)\d{2})"
    r"\.(?:tif|tiff)$"
)

AREA_FILENAME_REGEX = (
    rf"^{re.escape(CROP)}_HarArea_"
    rf"(?P<year>(?:19|20)\d{{2}})"
    rf"\.(?:tif|tiff)$"
)

NUP_FILENAME_REGEX = (
    r"^.*(?:nup|nuptake|n_uptake).*_"
    r"(?P<year>(?:19|20)\d{2})"
    r"\.(?:tif|tiff)$"
)

# ------------------------------------------------------------
# Output file
#
# New filename avoids overwriting the existing mean3.txt.
# ------------------------------------------------------------

OUTPUT_TXT = os.path.join(
    INPUT_FOLDER,
    "mean4_NupRatio.txt"
)

# ------------------------------------------------------------
# NoData handling
# ------------------------------------------------------------

ADDITIONAL_NODATA = [-9999.0]

# ------------------------------------------------------------
# Original statistical parameters
# ------------------------------------------------------------

TRIM_PERCENT = 95.0
STD_MULTIPLIER = 2.0

# ------------------------------------------------------------
# Area settings
# ------------------------------------------------------------

AREA_THRESHOLD = 0.0

# ------------------------------------------------------------
# Nup-based ratio settings
# ------------------------------------------------------------

# NUE must be > 0 to calculate Nup / NUE
NUE_MIN_VALUE = 0.0

# Nup >= 0 is physically admissible.
NUP_MIN_VALUE = 0.0

# ------------------------------------------------------------
# File matching requirements
# ------------------------------------------------------------

REQUIRE_AREA_MATCH = True
REQUIRE_NUP_MATCH = True

# ------------------------------------------------------------
# Optional year selection
#
# None = all years
# Example: [1995, 2000, 2020]
# ------------------------------------------------------------

YEARS_TO_RUN = None


# ============================================================
# 2. VALIDATE CONFIGURATION
# ============================================================

start_time = time.perf_counter()

for folder, label in [
    (INPUT_FOLDER, "NUE folder"),
    (AREA_FOLDER, "Harvested Area folder"),
    (NUP_FOLDER, "Nup folder")
]:

    if not os.path.isdir(folder):

        raise FileNotFoundError(
            f"{label} not found:\n{folder}"
        )

if not (0 < TRIM_PERCENT <= 100):

    raise ValueError(
        "TRIM_PERCENT must be within (0, 100]"
    )

if STD_MULTIPLIER <= 0:

    raise ValueError(
        "STD_MULTIPLIER must be positive"
    )


# ============================================================
# 3. UTILITY FUNCTIONS
# ============================================================

def index_yearly_rasters(
    folder,
    filename_regex,
    recursive=False
):
    """
    Return dictionary:
        year -> complete GeoTIFF path

    Stops if multiple files match the same year.
    """

    pattern = re.compile(
        filename_regex,
        flags=re.IGNORECASE
    )

    root = Path(folder)

    iterator = (
        root.rglob("*")
        if recursive
        else root.iterdir()
    )

    year_files = {}

    for path in sorted(iterator):

        if not path.is_file():
            continue

        if path.suffix.lower() not in (
            ".tif",
            ".tiff"
        ):
            continue

        match = pattern.fullmatch(
            path.name
        )

        if match is None:
            continue

        year = int(
            match.group("year")
        )

        if year in year_files:

            raise ValueError(
                f"Duplicate files for year {year}:\n"
                f"{year_files[year]}\n"
                f"{path}"
            )

        year_files[year] = str(path)

    return year_files


def read_valid_raster(src):
    """
    Read raster and return:
       raw values
       valid pixel mask

    Excludes:
       Metadata NoData
       NaN / Inf
       ADDITIONAL_NODATA
    """

    if src.count != 1:

        raise ValueError(
            f"Expected single-band raster: {src.name}"
        )

    data = src.read(
        1,
        masked=True
    )

    values = data.data

    valid = (
        ~np.ma.getmaskarray(data)
    ).copy()

    valid &= np.isfinite(values)

    for nd in ADDITIONAL_NODATA:
        valid &= (values != nd)

    return values, valid


def check_grid_alignment(
    reference,
    other,
    label
):
    """
    Check identical:
      CRS
      dimensions
      affine transform

    No automatic resampling.
    """

    if (
        reference.crs is None
        or other.crs is None
    ):

        raise ValueError(
            f"{label}: Missing CRS"
        )

    if reference.crs != other.crs:

        raise ValueError(
            f"{label}: CRS mismatch"
        )

    if (
        reference.width != other.width
        or reference.height != other.height
    ):

        raise ValueError(
            f"{label}: Raster dimensions mismatch"
        )

    if not reference.transform.almost_equals(
        other.transform,
        precision=1e-10
    ):

        raise ValueError(
            f"{label}: Pixel grid transform mismatch"
        )


def format_value(value):

    if not np.isfinite(value):
        return "NaN"

    return f"{value:.6f}"


# ============================================================
# 4. MATCH THREE RASTER DATASETS BY YEAR
# ============================================================

nue_year_files = index_yearly_rasters(
    INPUT_FOLDER,
    NUE_FILENAME_REGEX,
    recursive=False
)

area_year_files = index_yearly_rasters(
    AREA_FOLDER,
    AREA_FILENAME_REGEX,
    recursive=True
)

nup_year_files = index_yearly_rasters(
    NUP_FOLDER,
    NUP_FILENAME_REGEX,
    recursive=True
)

if YEARS_TO_RUN is None:

    years = sorted(
        nue_year_files.keys()
    )

else:

    years = sorted(
        set(map(int, YEARS_TO_RUN))
    )

    missing_nue = [
        year
        for year in years
        if year not in nue_year_files
    ]

    if missing_nue:

        raise FileNotFoundError(
            f"Missing NUE years: {missing_nue}"
        )

if not years:

    raise RuntimeError(
        "No matching NUE GeoTIFF files found"
    )

# Check missing matched inputs BEFORE computation
missing_area = [
    year
    for year in years
    if year not in area_year_files
]

missing_nup = [
    year
    for year in years
    if year not in nup_year_files
]

if REQUIRE_AREA_MATCH and missing_area:

    raise FileNotFoundError(
        f"Missing Harvested Area years: {missing_area}"
    )

if REQUIRE_NUP_MATCH and missing_nup:

    raise FileNotFoundError(
        f"Missing Nup years: {missing_nup}\n"
        "Check NUP_FOLDER and NUP_FILENAME_REGEX."
    )

print("=" * 75)
print("GLOBAL NUE - ANNUAL MEAN COMPARISON")
print("=" * 75)

print(f"Crop: {CROP}")
print(f"Years: {years}")

print(f"NUE files found: {len(nue_year_files)}")
print(f"Area files found: {len(area_year_files)}")
print(f"Nup files found: {len(nup_year_files)}")

print(f"Trimmed Mean: {TRIM_PERCENT}%")
print(f"STD multiplier: {STD_MULTIPLIER}")

print("Method 5: Nup-based Ratio of Sums")

print("=" * 75)


# ============================================================
# 5. NEW FUNCTION: NUP-BASED GLOBAL NUE
# ============================================================

def calculate_nup_ratio(
    nue_data,
    nue_valid,
    nup_data,
    nup_valid,
    area_data,
    area_valid
):
    """
    NUE_global =

        SUM(Nup_i * HA_i)
        -----------------------------
        SUM((Nup_i/NUE_i) * HA_i)

    Uses EXACTLY the same pixels in both sums.

    Core: NUE must be valid and > 0.

    Returns:
      ratio
      common_pixel_count
      numerator_sum
      denominator_sum
    """

    common_mask = (
        nue_valid
        & (nue_data > NUE_MIN_VALUE)
        & nup_valid
        & (nup_data >= NUP_MIN_VALUE)
        & area_valid
        & (area_data > AREA_THRESHOLD)
    )

    if not np.any(common_mask):

        return np.nan, 0, 0.0, 0.0

    # --------------------------------------------------------
    # Extract exactly corresponding pixels
    # --------------------------------------------------------

    nue_common = np.asarray(
        nue_data[common_mask],
        dtype=np.float64
    )

    nup_common = np.asarray(
        nup_data[common_mask],
        dtype=np.float64
    )

    area_common = np.asarray(
        area_data[common_mask],
        dtype=np.float64
    )

    # --------------------------------------------------------
    # Calculate per-pixel numerator and denominator
    # --------------------------------------------------------

    with np.errstate(
        divide="ignore",
        invalid="ignore",
        over="ignore"
    ):

        numerator_terms = (
            nup_common * area_common
        )

        denominator_terms = (
            (nup_common / nue_common)
            * area_common
        )

    # --------------------------------------------------------
    # Additional numerical safety:
    #
    # If either term is non-finite, exclude the pixel
    # from BOTH numerator and denominator.
    # --------------------------------------------------------

    finite_terms = (
        np.isfinite(numerator_terms)
        & np.isfinite(denominator_terms)
    )

    numerator_terms = numerator_terms[
        finite_terms
    ]

    denominator_terms = denominator_terms[
        finite_terms
    ]

    common_pixel_count = int(
        numerator_terms.size
    )

    if common_pixel_count == 0:

        return np.nan, 0, 0.0, 0.0

    # --------------------------------------------------------
    # Global sums
    # --------------------------------------------------------

    numerator_sum = float(
        np.sum(
            numerator_terms,
            dtype=np.float64
        )
    )

    denominator_sum = float(
        np.sum(
            denominator_terms,
            dtype=np.float64
        )
    )

    if not (
        np.isfinite(numerator_sum)
        and np.isfinite(denominator_sum)
    ):

        raise ValueError(
            "Non-finite global Nup-based sums"
        )

    # A zero denominator means the ratio is undefined.
    if denominator_sum <= 0:

        return (
            np.nan,
            common_pixel_count,
            numerator_sum,
            denominator_sum
        )

    ratio = (
        numerator_sum / denominator_sum
    )

    return (
        float(ratio),
        common_pixel_count,
        numerator_sum,
        denominator_sum
    )


# ============================================================
# 6. PROCESS ANNUAL GLOBAL RASTERS
# ============================================================

results = []

for year in years:

    nue_tif = nue_year_files[year]
    area_tif = area_year_files.get(year)
    nup_tif = nup_year_files.get(year)

    print("\n" + "-" * 75)
    print(f"Processing year: {year}")

    # --------------------------------------------------------
    # Initialize annual results
    # --------------------------------------------------------

    mean_val = np.nan
    trimmed_mean = np.nan
    std_mean = np.nan

    area_weighted_mean = np.nan
    nue_ratio_of_sums = np.nan

    valid_count = 0
    trimmed_count = 0
    std_count = 0

    weighted_pixel_count = 0
    ratio_common_pixels = 0

    total_harvested_area = 0.0

    sum_nup_ha = np.nan
    sum_ninput_ha = np.nan

    # --------------------------------------------------------
    # Read NUE - core raster
    # --------------------------------------------------------

    with rasterio.open(nue_tif) as src_nue:

        nue_data, nue_valid = read_valid_raster(
            src_nue
        )

        valid_values = nue_data[
            nue_valid
        ].astype(
            np.float64,
            copy=False
        )

        valid_count = valid_values.size

        # ====================================================
        # METHOD 1: ARITHMETIC MEAN
        # ====================================================

        if valid_count > 0:

            mean_val = float(
                np.mean(
                    valid_values,
                    dtype=np.float64
                )
            )

        # ====================================================
        # METHOD 2: PERCENTILE TRIMMED MEAN
        # ====================================================

        if valid_count > 0:

            tail_percent = (
                100.0 - TRIM_PERCENT
            ) / 2.0

            p_low, p_high = np.percentile(
                valid_values,
                [
                    tail_percent,
                    100.0 - tail_percent
                ]
            )

            trimmed_mask = (
                (valid_values >= p_low)
                & (valid_values <= p_high)
            )

            trimmed_values = valid_values[
                trimmed_mask
            ]

            trimmed_count = int(
                trimmed_values.size
            )

            if trimmed_count > 0:

                trimmed_mean = float(
                    np.mean(
                        trimmed_values,
                        dtype=np.float64
                    )
                )

        # ====================================================
        # METHOD 3: MEAN WITHIN +/- k STD
        # ====================================================

        if valid_count > 0:

            std_val = float(
                np.std(
                    valid_values,
                    dtype=np.float64,
                    ddof=0
                )
            )

            std_lower = (
                mean_val - STD_MULTIPLIER * std_val
            )

            std_upper = (
                mean_val + STD_MULTIPLIER * std_val
            )

            std_mask = (
                (valid_values >= std_lower)
                & (valid_values <= std_upper)
            )

            std_values = valid_values[
                std_mask
            ]

            std_count = int(
                std_values.size
            )

            if std_count > 0:

                std_mean = float(
                    np.mean(
                        std_values,
                        dtype=np.float64
                    )
                )

        # ====================================================
        # READ HARVESTED AREA
        # ====================================================

        area_data = None
        area_valid = None

        if area_tif is not None:

            with rasterio.open(area_tif) as src_area:

                check_grid_alignment(
                    src_nue,
                    src_area,
                    f"{year}: NUE / Harvested Area"
                )

                area_data, area_valid = (
                    read_valid_raster(src_area)
                )

        else:

            print(
                "WARNING: Missing Harvested Area "
                f"for year {year}"
            )

        # ====================================================
        # METHOD 4: HARVESTED-AREA-WEIGHTED MEAN
        # ====================================================

        if area_data is not None:

            common_area = (
                nue_valid
                & area_valid
                & (area_data > AREA_THRESHOLD)
            )

            weighted_pixel_count = int(
                np.count_nonzero(common_area)
            )

            if weighted_pixel_count > 0:

                nue_common = np.asarray(
                    nue_data[common_area],
                    dtype=np.float64
                )

                area_common = np.asarray(
                    area_data[common_area],
                    dtype=np.float64
                )

                total_harvested_area = float(
                    np.sum(
                        area_common,
                        dtype=np.float64
                    )
                )

                if total_harvested_area > 0:

                    weighted_sum = float(
                        np.sum(
                            nue_common * area_common,
                            dtype=np.float64
                        )
                    )

                    area_weighted_mean = (
                        weighted_sum
                        / total_harvested_area
                    )

        # ====================================================
        # METHOD 5: NEW NUP-BASED NUE RATIO OF SUMS
        # ====================================================

        if (
            area_data is not None
            and nup_tif is not None
        ):

            with rasterio.open(nup_tif) as src_nup:

                check_grid_alignment(
                    src_nue,
                    src_nup,
                    f"{year}: NUE / Nup"
                )

                nup_data, nup_valid = (
                    read_valid_raster(src_nup)
                )

            (
                nue_ratio_of_sums,
                ratio_common_pixels,
                sum_nup_ha,
                sum_ninput_ha
            ) = calculate_nup_ratio(
                nue_data,
                nue_valid,
                nup_data,
                nup_valid,
                area_data,
                area_valid
            )

        elif nup_tif is None:

            print(
                f"WARNING: Missing Nup for {year}"
            )

    # ========================================================
    # PRINT ANNUAL RESULTS
    # ========================================================

    print(
        f"{year} | "
        f"Mean={mean_val:.6f} | "
        f"Trimmed={trimmed_mean:.6f} | "
        f"STD={std_mean:.6f}"
    )

    print(
        f"     AreaWeighted="
        f"{area_weighted_mean:.6f} | "
        f"NupRatio="
        f"{nue_ratio_of_sums:.6f}"
    )

    print(
        f"     Valid NUE pixels={valid_count:,} | "
        f"Ratio common pixels={ratio_common_pixels:,}"
    )

    print(
        f"     Sum Nup*HA={sum_nup_ha:,.4f} | "
        f"Sum (Nup/NUE)*HA={sum_ninput_ha:,.4f}"
    )

    # ========================================================
    # STORE ANNUAL RESULTS
    # ========================================================

    results.append({
        "year": year,

        # Original 4 methods
        "mean": mean_val,
        "trimmed_mean": trimmed_mean,
        "std_mean": std_mean,
        "valid_pixels": int(valid_count),
        "trimmed_pixels": int(trimmed_count),
        "std_pixels": int(std_count),
        "area_weighted_mean": area_weighted_mean,

        # New method
        "nue_ratio_of_sums": nue_ratio_of_sums,
        "ratio_common_pixels": ratio_common_pixels,
        "sum_nup_ha": sum_nup_ha,
        "sum_ninput_ha": sum_ninput_ha
    })


# ============================================================
# 7. SORT RESULTS BY YEAR
# ============================================================

results.sort(
    key=lambda item: item["year"]
)


# ============================================================
# 8. EXPORT TXT - ORIGINAL COLUMNS + NEW COLUMNS
# ============================================================

# Dynamic labels ensure that changing 95% or 2 sigma
# does not leave misleading column names.

trim_tag = f"{TRIM_PERCENT:g}".replace(".", "p")
std_tag = f"{STD_MULTIPLIER:g}".replace(".", "p")

trim_mean_col = f"trimmed_mean_{trim_tag}"
trim_count_col = f"trimmed_pixels_{trim_tag}"

std_mean_col = f"std{std_tag}_mean"
std_count_col = f"std{std_tag}_pixels"

header = [
    "year",
    "mean",
    trim_mean_col,
    std_mean_col,
    "valid_pixels",
    trim_count_col,
    std_count_col,
    "area_weighted_mean",

    # NEW
    "nue_ratio_of_sums",
    "ratio_common_pixels",
    "sum_nup_ha",
    "sum_ninput_ha"
]

with open(
    OUTPUT_TXT,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        ",".join(header) + "\n"
    )

    for result in results:

        row = [
            str(result["year"]),
            format_value(result["mean"]),
            format_value(result["trimmed_mean"]),
            format_value(result["std_mean"]),
            str(result["valid_pixels"]),
            str(result["trimmed_pixels"]),
            str(result["std_pixels"]),
            format_value(result["area_weighted_mean"]),

            # NEW
            format_value(result["nue_ratio_of_sums"]),
            str(result["ratio_common_pixels"]),
            format_value(result["sum_nup_ha"]),
            format_value(result["sum_ninput_ha"])
        ]

        f.write(
            ",".join(row) + "\n"
        )


# ============================================================
# 9. FINAL SUMMARY
# ============================================================

elapsed = (
    time.perf_counter() - start_time
)

print("\n" + "=" * 75)
print("GLOBAL NUE MEAN COMPARISON COMPLETED")
print("=" * 75)

print(f"Crop: {CROP}")
print(f"Processed years: {len(results)}")

print("\nMethods:")
print("  1. Arithmetic Mean")
print(f"  2. {TRIM_PERCENT:g}% Trimmed Mean")
print(f"  3. +/- {STD_MULTIPLIER:g} STD Mean")
print("  4. Harvested-Area-Weighted Mean")
print("  5. Nup-based Ratio of Sums")

print(
    "\nValid Nup-based annual ratios: "
    f"{sum(np.isfinite(r['nue_ratio_of_sums']) for r in results)}"
)

print(f"\nProcessing time: {elapsed:.2f}s")
print(f"Results saved to:\n{OUTPUT_TXT}")

print("=" * 75)

In [ ]:
# =====================================================
# -------Global Maize NUE---Based on the weighted median-------------
# ---Step 1----Setting parameters----------
# ---Step 2--- Run ----------------
# ---Step 3--- simple validation --------------
# ---Step 4--- Match with harvested area, mask
# ---Step 5--- Remove outliers, Useful tool,Pixels number should larger than 6, NUE more then 150; set non, 
# ---Step 6--- Check mean value
# ---Step 7--- Extract country mean value--Useful---multi-process computing----
# =======V 2026.10.27============================================
# Global NUE - Parallel Country Statistics (5 Methods, C4NUPW)
#
# Method 1: Percentile Trimmed Mean
# Method 2: 2 / 3 Standard Deviation Mean
# Method 3: Arithmetic Mean
# Method 4: Harvested-Area-Weighted Mean
# Method 5 (NEW):
#              SUM(Nup_i * HA_i)
# NUE_region = -------------------------
#              SUM((Nup_i/NUE_i) * HA_i)
#
# Core validity rule:
#   NUE must be valid and > 0
#   Nup must be valid and >= 0
#   HA must be valid and > 0
#   All three pixels must overlap spatially.
#
# Parallel years / Windows / JupyterLab
# No resampling, filling or interpolation
# ============================================================

import os
import re
import time
import traceback

from pathlib import Path
from itertools import repeat

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import ipywidgets as widgets

from rasterstats import gen_zonal_stats
from joblib import Parallel, delayed
from IPython.display import display, clear_output


# ============================================================
# 1. USER CONFIGURATION
# ============================================================

SHP_PATH = (
    r"Z:/staff/Qiang_S/09_Datasets/"
    r"09OtherDataUsed/GDAM_GlobalCountryLevel_Shp/"
    r"GlobalCountryLevel_Shp.shp"
)

# ------------------------------------------------------------
# NUE GeoTIFF folder
# ------------------------------------------------------------

TIF_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\02Global_NUE_Maize\02MaizeNUE_M2_20261029V1_M6_150R"
)

# ------------------------------------------------------------
# Harvested Area GeoTIFF folder
# ------------------------------------------------------------

AREA_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\GCSN_HarvestedArea_1995_2020\02StandardizedDatasets\02MaizeHarvestedArea"
)

# ------------------------------------------------------------
# NEW: Nup GeoTIFF folder
# ------------------------------------------------------------

NUP_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\02Global_NUE_Maize\02MaizeNUE_M2_20261029V1_Nup"
)

CROP = "Maize"

# Harvested Area examples:
#   Maize_HarArea_1995.tif

AREA_FILE_PREFIX = f"{CROP}_HarArea_"

# Nup examples supported by default:
#   Maize_Nup_1995.tif
#   Nup_maize_1995.tif
#   maize_Nuptake_1995.tif
#
# Match any file containing "nup"
# and ending with "_YYYY.tif".

NUP_FILENAME_REGEX = (
    r"^.*nup.*_(?P<year>(?:19|20)\d{2})"
    r"\.(?:tif|tiff)$"
)

# ------------------------------------------------------------
# Output Excel
# New name avoids overwriting original results.
# ------------------------------------------------------------

OUT_EXCEL = os.path.join(
    TIF_FOLDER,
    f"{CROP}_NUE_Country_Mean_WithNup.xlsx"
)

# ------------------------------------------------------------
# NoData settings
# ------------------------------------------------------------

NODATA = -9999.0
ADDITIONAL_NODATA = []

# ------------------------------------------------------------
# Parallel years
# ------------------------------------------------------------

N_WORKERS = 15

# None: all years
# Example: [1995, 1996, 2020]
YEARS_TO_RUN = None

# ------------------------------------------------------------
# Country raster inclusion rule
# ------------------------------------------------------------

ALL_TOUCHED = False

# Percentile / STD filtering only if N > 10
MIN_PIXELS_FOR_FILTER = 10

# ------------------------------------------------------------
# Area and Nup parameters
# ------------------------------------------------------------

AREA_THRESHOLD = 0.0

# If Nup is physically nonnegative,
# zero is allowed but contributes zero.
NUP_MIN_VALUE = 0.0

# NUE must be positive to calculate Nup/NUE.
NUE_MIN_VALUE = 0.0

# Require corresponding files when selected
REQUIRE_AREA_MATCH = True
REQUIRE_NUP_MATCH = True


# ============================================================
# 2. READ AND CLEAN RASTER
# ============================================================

def read_clean_raster(src):
    """
    Invalid:
      Metadata NoData
      NaN / Inf
      -9999
      ADDITIONAL_NODATA

    Return float32 raster with invalid pixels
    standardized to NODATA.
    """

    if src.count != 1:
        raise ValueError(
            f"Expected single-band raster: {src.name}"
        )

    masked = src.read(1, masked=True)

    data = np.array(
        masked.data,
        dtype=np.float32,
        copy=True
    )

    invalid = np.ma.getmaskarray(masked).copy()

    invalid |= ~np.isfinite(data)
    invalid |= (data == NODATA)

    for nd in ADDITIONAL_NODATA:
        invalid |= (data == nd)

    data[invalid] = NODATA

    valid_count = int(
        np.count_nonzero(~invalid)
    )

    return data, valid_count


# ============================================================
# 3. VERIFY GLOBAL RASTER ALIGNMENT
# ============================================================

def check_grid_alignment(reference, other, label):
    """
    All three rasters must have the same:
       CRS
       width / height
       affine transform

    No reprojection / resampling is performed.
    """

    if reference.crs is None or other.crs is None:
        raise ValueError(
            f"{label}: Missing CRS"
        )

    if reference.crs != other.crs:
        raise ValueError(
            f"{label}: CRS mismatch"
        )

    if (
        reference.width != other.width
        or reference.height != other.height
    ):
        raise ValueError(
            f"{label}: Raster dimensions mismatch"
        )

    if not reference.transform.almost_equals(
        other.transform,
        precision=1e-10
    ):
        raise ValueError(
            f"{label}: Pixel grid transform mismatch"
        )


def check_mini_alignment(
    reference_stat,
    other_stat,
    year,
    country,
    label
):
    """
    Verify that rasterstats used exactly the same
    cropped window for a given country.
    """

    if reference_stat is None or other_stat is None:
        raise ValueError(
            f"{year} {country}: Missing {label} statistic"
        )

    a = reference_stat.get("mini_raster_affine")
    b = other_stat.get("mini_raster_affine")

    x = reference_stat.get("mini_raster_array")
    y = other_stat.get("mini_raster_array")

    if a is None or b is None or x is None or y is None:
        raise ValueError(
            f"{year} {country}: Missing {label} mini raster"
        )

    if x.shape != y.shape:
        raise ValueError(
            f"{year} {country}: {label} mini shape mismatch"
        )

    if not a.almost_equals(b, precision=1e-10):
        raise ValueError(
            f"{year} {country}: {label} mini grid mismatch"
        )


# ============================================================
# 4. UTILITY: VALID PIXELS FROM A MASKED ARRAY
# ============================================================

def get_masked_values_and_valid(masked_array):
    """
    Return:
       values as float64
       valid boolean mask

    Polygon-outside pixels remain excluded.
    """

    ma = np.ma.asarray(masked_array)

    values = np.asarray(
        np.ma.getdata(ma),
        dtype=np.float64
    )

    valid = ~np.ma.getmaskarray(ma)
    valid = valid.copy()

    valid &= np.isfinite(values)
    valid &= (values != NODATA)

    for nd in ADDITIONAL_NODATA:
        valid &= (values != nd)

    return values, valid


# ============================================================
# 5. ORIGINAL COUNTRY STATISTICS
# ============================================================

def compute_country_metrics(masked_array, config):
    """
    Original algorithms unchanged:
      Arithmetic mean
      Percentile trimmed mean
      +/- k STD mean

    Small regions <=10 pixels:
      Return original mean for selected filters.
    """

    values, valid = get_masked_values_and_valid(
        masked_array
    )

    values = values[valid]

    n = values.size

    tag = config["trim_tag"]

    result = {
        "Valid_Pixels": int(n),
        "value": None,
        f"Mean_Trimmed_{tag}": None,
        f"N_Trimmed_{tag}": None,
        "Mean_STD_2": None,
        "N_STD_2": None,
        "Mean_STD_3": None,
        "N_STD_3": None
    }

    if n == 0:
        return result

    original_mean = float(
        np.mean(values, dtype=np.float64)
    )

    # --------------------------------------------------------
    # Arithmetic mean
    # --------------------------------------------------------

    if config["enable_all"]:
        result["value"] = original_mean

    # --------------------------------------------------------
    # Percentile trimmed mean
    # --------------------------------------------------------

    if config["enable_trimmed"]:

        mean_col = f"Mean_Trimmed_{tag}"
        count_col = f"N_Trimmed_{tag}"

        if n <= MIN_PIXELS_FOR_FILTER:

            result[mean_col] = original_mean
            result[count_col] = int(n)

        else:

            tail = (
                100.0 - config["keep_percent"]
            ) / 2.0

            p_low, p_high = np.percentile(
                values,
                [tail, 100.0 - tail]
            )

            selected = values[
                (values >= p_low)
                & (values <= p_high)
            ]

            result[count_col] = int(selected.size)

            if selected.size > 0:

                result[mean_col] = float(
                    np.mean(
                        selected,
                        dtype=np.float64
                    )
                )

    # --------------------------------------------------------
    # STD mean
    # --------------------------------------------------------

    if config["enable_std"]:

        if n <= MIN_PIXELS_FOR_FILTER:

            for k in config["std_multipliers"]:

                result[f"Mean_STD_{k}"] = original_mean
                result[f"N_STD_{k}"] = int(n)

        else:

            std_val = float(
                np.std(
                    values,
                    dtype=np.float64,
                    ddof=0
                )
            )

            for k in config["std_multipliers"]:

                lower = original_mean - k * std_val
                upper = original_mean + k * std_val

                selected = values[
                    (values >= lower)
                    & (values <= upper)
                ]

                result[f"N_STD_{k}"] = int(selected.size)

                if selected.size > 0:

                    result[f"Mean_STD_{k}"] = float(
                        np.mean(
                            selected,
                            dtype=np.float64
                        )
                    )

    return result


# ============================================================
# 6. ORIGINAL HARVESTED-AREA-WEIGHTED MEAN
# ============================================================

def compute_area_weighted_mean(
    nue_masked,
    area_masked
):
    """
    Mean = SUM(NUE * HA) / SUM(HA)

    Only common valid NUE and HA pixels.
    """

    output = {
        "Mean_Area_Weighted": None,
        "N_Area_Weighted": 0,
        "Harvested_Area_Used": 0.0
    }

    if nue_masked is None or area_masked is None:
        return output

    nue, valid_nue = get_masked_values_and_valid(
        nue_masked
    )

    area, valid_area = get_masked_values_and_valid(
        area_masked
    )

    if nue.shape != area.shape:
        raise ValueError(
            "NUE / Area mini rasters have different shapes"
        )

    common = (
        valid_nue
        & valid_area
        & (area > AREA_THRESHOLD)
    )

    n_common = int(np.count_nonzero(common))

    output["N_Area_Weighted"] = n_common

    if n_common == 0:
        return output

    a = area[common]
    x = nue[common]

    area_sum = float(
        np.sum(a, dtype=np.float64)
    )

    output["Harvested_Area_Used"] = area_sum

    if area_sum <= 0:
        return output

    output["Mean_Area_Weighted"] = float(
        np.sum(x * a, dtype=np.float64) / area_sum
    )

    return output


# ============================================================
# 7. NEW: REGIONAL NUE - RATIO OF SUMS
# ============================================================

def compute_nup_based_nue(
    nue_masked,
    nup_masked,
    area_masked
):
    """
    NUE_region =

        SUM(Nup_i * HA_i)
        -------------------------------
        SUM((Nup_i / NUE_i) * HA_i)

    IMPORTANT:
      1. NUE is the core mask.
      2. NUE must be finite and > 0.
      3. Nup must be finite and >= 0.
      4. Harvested Area must be finite and > 0.
      5. All three rasters must overlap spatially.
      6. Numerator and denominator use EXACTLY
         the same valid pixels.

    No interpolation or filling.

    Returns:
      NUE_Ratio_of_Sums
      N_Ratio_Common_Pixels
      Sum_Nup_x_HA
      Sum_Ninput_x_HA
    """

    output = {
        "NUE_Ratio_of_Sums": None,
        "N_Ratio_Common_Pixels": 0,
        "Sum_Nup_x_HA": 0.0,
        "Sum_Ninput_x_HA": 0.0
    }

    if (
        nue_masked is None
        or nup_masked is None
        or area_masked is None
    ):
        return output

    nue, nue_valid = get_masked_values_and_valid(
        nue_masked
    )

    nup, nup_valid = get_masked_values_and_valid(
        nup_masked
    )

    area, area_valid = get_masked_values_and_valid(
        area_masked
    )

    if not (
        nue.shape == nup.shape == area.shape
    ):
        raise ValueError(
            "NUE, Nup and Area mini rasters "
            "must have the same shape"
        )

    # --------------------------------------------------------
    # Core mask: NUE must exist and be positive
    # --------------------------------------------------------

    common = (
        nue_valid
        & (nue > NUE_MIN_VALUE)
        & nup_valid
        & (nup >= NUP_MIN_VALUE)
        & area_valid
        & (area > AREA_THRESHOLD)
    )

    n_initial = int(
        np.count_nonzero(common)
    )

    if n_initial == 0:
        return output

    # --------------------------------------------------------
    # Get exactly matched pixels
    # --------------------------------------------------------

    nue_common = nue[common]
    nup_common = nup[common]
    area_common = area[common]

    # --------------------------------------------------------
    # Numerator and denominator contributions
    # --------------------------------------------------------

    with np.errstate(
        divide="ignore",
        invalid="ignore",
        over="ignore"
    ):

        numerator_terms = (
            nup_common * area_common
        )

        denominator_terms = (
            (nup_common / nue_common)
            * area_common
        )

    # Extra numerical safety:
    # If an extreme input produces Inf/NaN during
    # calculation, remove that pixel from BOTH sums.

    finite_terms = (
        np.isfinite(numerator_terms)
        & np.isfinite(denominator_terms)
    )

    numerator_terms = numerator_terms[
        finite_terms
    ]

    denominator_terms = denominator_terms[
        finite_terms
    ]

    n_common = int(
        numerator_terms.size
    )

    output["N_Ratio_Common_Pixels"] = n_common

    if n_common == 0:
        return output

    # --------------------------------------------------------
    # Regional sums
    # --------------------------------------------------------

    sum_nup_ha = float(
        np.sum(
            numerator_terms,
            dtype=np.float64
        )
    )

    sum_ninput_ha = float(
        np.sum(
            denominator_terms,
            dtype=np.float64
        )
    )

    if not (
        np.isfinite(sum_nup_ha)
        and np.isfinite(sum_ninput_ha)
    ):
        raise ValueError(
            "Non-finite regional Nup/N-input sums"
        )

    output["Sum_Nup_x_HA"] = sum_nup_ha

    output["Sum_Ninput_x_HA"] = sum_ninput_ha

    # Denominator must be > 0
    if sum_ninput_ha <= 0:
        return output

    result = (
        sum_nup_ha / sum_ninput_ha
    )

    if np.isfinite(result):
        output["NUE_Ratio_of_Sums"] = float(result)

    return output


# ============================================================
# 8. LOAD COUNTRY SHAPEFILE
# ============================================================

def load_countries():

    if not os.path.isfile(SHP_PATH):
        raise FileNotFoundError(SHP_PATH)

    gdf = gpd.read_file(SHP_PATH)

    if gdf.crs is None:
        raise ValueError(
            "Country shapefile CRS is missing"
        )

    gdf = gdf.to_crs("EPSG:4326")

    gdf = gdf.reset_index(drop=True)

    for field in ["M49_Code", "COUNTRY"]:
        if field not in gdf.columns:
            raise ValueError(
                f"Missing shapefile field: {field}"
            )

    return gdf


# ============================================================
# 9. FIND YEARLY RASTERS
# ============================================================

def extract_year(filename):

    match = re.search(
        r"_(19\d{2}|20\d{2})\.(?:tif|tiff)$",
        Path(filename).name,
        flags=re.IGNORECASE
    )

    if match:
        return int(match.group(1))

    return None


def index_rasters(
    folder,
    filename_filter,
    recursive=True
):
    """
    Build year -> file path mapping.

    Raise on duplicate year files to prevent
    accidentally using a wrong crop or raster.
    """

    if not os.path.isdir(folder):
        raise FileNotFoundError(folder)

    root = Path(folder)

    iterator = (
        root.rglob("*")
        if recursive
        else root.iterdir()
    )

    file_index = {}

    for path in sorted(iterator):

        if not path.is_file():
            continue

        if path.suffix.lower() not in (".tif", ".tiff"):
            continue

        if not filename_filter(path.name):
            continue

        year = extract_year(path.name)

        if year is None:
            continue

        if year in file_index:
            raise ValueError(
                f"Duplicate raster for {year}:\n"
                f"{file_index[year]}\n{path}"
            )

        file_index[year] = str(path)

    return file_index


def find_year_files(config):

    # --------------------------------------------------------
    # NUE rasters
    # --------------------------------------------------------

    nue_files = index_rasters(
        TIF_FOLDER,
        filename_filter=lambda name: True,
        recursive=False
    )

    if YEARS_TO_RUN is not None:
        requested_years = set(
            map(int, YEARS_TO_RUN)
        )
    else:
        requested_years = None

    if requested_years is not None:

        nue_files = {
            y: p
            for y, p in nue_files.items()
            if y in requested_years
        }

    if not nue_files:
        raise RuntimeError(
            "No matching NUE GeoTIFF files found"
        )

    # --------------------------------------------------------
    # Harvested Area rasters
    # --------------------------------------------------------

    need_area = (
        config["enable_weighted"]
        or config["enable_ratio"]
    )

    area_files = {}

    if need_area:

        area_files = index_rasters(
            AREA_FOLDER,
            filename_filter=lambda name: (
                name.lower().startswith(
                    AREA_FILE_PREFIX.lower()
                )
            ),
            recursive=True
        )

    # --------------------------------------------------------
    # NEW: Nup rasters
    # --------------------------------------------------------

    nup_files = {}

    if config["enable_ratio"]:

        pattern = re.compile(
            NUP_FILENAME_REGEX,
            flags=re.IGNORECASE
        )

        nup_files = index_rasters(
            NUP_FOLDER,
            filename_filter=lambda name: (
                pattern.fullmatch(name) is not None
            ),
            recursive=True
        )

    # --------------------------------------------------------
    # Match by year
    # --------------------------------------------------------

    year_files = []

    for year in sorted(nue_files):

        area_path = area_files.get(year)
        nup_path = nup_files.get(year)

        if (
            need_area
            and REQUIRE_AREA_MATCH
            and area_path is None
        ):
            raise FileNotFoundError(
                f"No Harvested Area raster for {year}"
            )

        if (
            config["enable_ratio"]
            and REQUIRE_NUP_MATCH
            and nup_path is None
        ):
            raise FileNotFoundError(
                f"No Nup raster for {year}. "
                "Check NUP_FOLDER and NUP_FILENAME_REGEX."
            )

        year_files.append((
            year,
            nue_files[year],
            area_path,
            nup_path
        ))

    return year_files


# ============================================================
# 10. PROCESS ONE YEAR
# ============================================================

def process_one_year(
    year,
    nue_tif,
    area_tif,
    nup_tif,
    countries,
    config
):

    t0 = time.perf_counter()

    # --------------------------------------------------------
    # Read aligned rasters
    # --------------------------------------------------------

    with rasterio.open(nue_tif) as src_nue:

        if src_nue.crs is None:
            raise ValueError(
                f"{year}: NUE CRS is missing"
            )

        crs = src_nue.crs
        affine = src_nue.transform

        nue_data, nue_valid_count = (
            read_clean_raster(src_nue)
        )

        area_data = None
        nup_data = None

        need_area = (
            config["enable_weighted"]
            or config["enable_ratio"]
        )

        if need_area and area_tif is not None:

            with rasterio.open(area_tif) as src_area:

                check_grid_alignment(
                    src_nue,
                    src_area,
                    f"{year}: NUE / Area"
                )

                area_data, _ = read_clean_raster(
                    src_area
                )

        if (
            config["enable_ratio"]
            and nup_tif is not None
        ):

            with rasterio.open(nup_tif) as src_nup:

                check_grid_alignment(
                    src_nue,
                    src_nup,
                    f"{year}: NUE / Nup"
                )

                nup_data, _ = read_clean_raster(
                    src_nup
                )

    # --------------------------------------------------------
    # Country coordinate system
    # --------------------------------------------------------

    if countries.crs != crs:

        gdf_year = countries.to_crs(
            crs
        )

    else:

        gdf_year = countries

    # --------------------------------------------------------
    # Initialize one row per country
    # --------------------------------------------------------

    records = []

    for _, row in gdf_year.iterrows():

        records.append({
            "M49_Code": row["M49_Code"],
            "COUNTRY": row["COUNTRY"],
            "year": year,

            "Valid_Pixels": 0,

            "Mean_Area_Weighted": None,
            "N_Area_Weighted": 0,
            "Harvested_Area_Used": 0.0,

            "NUE_Ratio_of_Sums": None,
            "N_Ratio_Common_Pixels": None,
            "Sum_Nup_x_HA": None,
            "Sum_Ninput_x_HA": None
        })

    valid_positions = []

    for i, geom in enumerate(
        gdf_year.geometry
    ):

        if geom is not None and not geom.is_empty:
            valid_positions.append(i)

    geometries = (
        gdf_year.geometry
        .iloc[valid_positions]
        .tolist()
    )

    # --------------------------------------------------------
    # Zonal statistics for each country
    # --------------------------------------------------------

    if geometries:

        nue_iterator = gen_zonal_stats(
            geometries,
            nue_data,
            affine=affine,
            stats=["mean"],
            nodata=NODATA,
            all_touched=ALL_TOUCHED,
            geojson_out=False,
            raster_out=True
        )

        if area_data is not None:

            area_iterator = gen_zonal_stats(
                geometries,
                area_data,
                affine=affine,
                stats=["count"],
                nodata=NODATA,
                all_touched=ALL_TOUCHED,
                geojson_out=False,
                raster_out=True
            )

        else:

            area_iterator = repeat(None)

        if nup_data is not None:

            nup_iterator = gen_zonal_stats(
                geometries,
                nup_data,
                affine=affine,
                stats=["count"],
                nodata=NODATA,
                all_touched=ALL_TOUCHED,
                geojson_out=False,
                raster_out=True
            )

        else:

            nup_iterator = repeat(None)

        # ----------------------------------------------------
        # Country loop
        # ----------------------------------------------------

        for pos, stat_nue, stat_area, stat_nup in zip(
            valid_positions,
            nue_iterator,
            area_iterator,
            nup_iterator
        ):

            country_name = records[pos]["COUNTRY"]

            nue_mini = stat_nue.get(
                "mini_raster_array"
            )

            area_mini = (
                stat_area.get("mini_raster_array")
                if stat_area is not None
                else None
            )

            nup_mini = (
                stat_nup.get("mini_raster_array")
                if stat_nup is not None
                else None
            )

            # -----------------------------------------------
            # Methods 1-3: Original country statistics
            # -----------------------------------------------

            if nue_mini is not None:

                metrics = compute_country_metrics(
                    nue_mini,
                    config
                )

                # Preserve rasterstats original mean
                if (
                    config["enable_all"]
                    and stat_nue.get("mean") is not None
                ):

                    original_mean = float(
                        stat_nue["mean"]
                    )

                    if np.isfinite(original_mean):
                        metrics["value"] = original_mean

                records[pos].update(metrics)

            # -----------------------------------------------
            # Check NUE / Area mini-raster alignment
            # -----------------------------------------------

            if stat_area is not None:

                check_mini_alignment(
                    stat_nue,
                    stat_area,
                    year,
                    country_name,
                    "NUE / Area"
                )

            # -----------------------------------------------
            # Method 4: Original area-weighted mean
            # -----------------------------------------------

            if (
                config["enable_weighted"]
                and area_mini is not None
            ):

                weighted_metrics = (
                    compute_area_weighted_mean(
                        nue_mini,
                        area_mini
                    )
                )

                records[pos].update(
                    weighted_metrics
                )

            # -----------------------------------------------
            # Method 5: NEW Nup-based regional NUE
            # -----------------------------------------------

            if config["enable_ratio"]:

                if (
                    stat_area is not None
                    and stat_nup is not None
                ):

                    check_mini_alignment(
                        stat_nue,
                        stat_nup,
                        year,
                        country_name,
                        "NUE / Nup"
                    )

                    # Both Nup and Area use same
                    # country geometry / pixel grid.
                    check_mini_alignment(
                        stat_area,
                        stat_nup,
                        year,
                        country_name,
                        "Area / Nup"
                    )

                    ratio_metrics = compute_nup_based_nue(
                        nue_mini,
                        nup_mini,
                        area_mini
                    )

                    records[pos].update(
                        ratio_metrics
                    )

    elapsed = time.perf_counter() - t0

    print(
        f"[{year}] Completed | "
        f"Countries={len(records)} | "
        f"Valid NUE pixels={nue_valid_count:,} | "
        f"Time={elapsed:.1f}s",
        flush=True
    )

    return year, records, nue_valid_count


# ============================================================
# 11. MAIN PROCESSING
# ============================================================

def run_country_mean_comparison(config):

    start_time = time.perf_counter()

    if N_WORKERS < 1:
        raise ValueError(
            "N_WORKERS must be >= 1"
        )

    if not (
        0 < config["keep_percent"] <= 100
    ):
        raise ValueError(
            "Keep percentile must be within (0,100]"
        )

    print("=" * 75)
    print("GLOBAL NUE - COUNTRY STATISTICS")
    print("=" * 75)

    print(
        f"1. Percentile mean: "
        f"{config['enable_trimmed']}"
    )

    print(
        f"2. STD mean: "
        f"{config['enable_std']} "
        f"{config['std_multipliers']}"
    )

    print(
        f"3. Arithmetic mean: "
        f"{config['enable_all']}"
    )

    print(
        f"4. HA-weighted mean: "
        f"{config['enable_weighted']}"
    )

    print(
        f"5. Nup-based ratio of sums: "
        f"{config['enable_ratio']}"
    )

    print(f"Crop: {CROP}")

    # --------------------------------------------------------
    # Load inputs
    # --------------------------------------------------------

    countries = load_countries()

    year_files = find_year_files(
        config
    )

    n_jobs = min(
        N_WORKERS,
        len(year_files)
    )

    print(f"Countries: {len(countries)}")
    print(f"Years: {[item[0] for item in year_files]}")
    print(f"Parallel workers: {n_jobs}")

    # --------------------------------------------------------
    # Parallel years
    # --------------------------------------------------------

    if n_jobs == 1:

        parallel_results = [
            process_one_year(
                year,
                nue_path,
                area_path,
                nup_path,
                countries,
                config
            )
            for year, nue_path, area_path, nup_path
            in year_files
        ]

    else:

        parallel_results = Parallel(
            n_jobs=n_jobs,
            backend="loky",
            verbose=10,
            batch_size=1,
            pre_dispatch="n_jobs"
        )(
            delayed(process_one_year)(
                year,
                nue_path,
                area_path,
                nup_path,
                countries,
                config
            )
            for year, nue_path, area_path, nup_path
            in year_files
        )

    # --------------------------------------------------------
    # Combine all years
    # --------------------------------------------------------

    parallel_results.sort(
        key=lambda item: item[0]
    )

    records = []

    for _, year_records, _ in parallel_results:
        records.extend(year_records)

    df = pd.DataFrame(records)

    # --------------------------------------------------------
    # Original columns + NEW ratio columns
    # --------------------------------------------------------

    trim_tag = config["trim_tag"]

    output_columns = [
        "M49_Code",
        "COUNTRY",
        "year",

        # Original arithmetic mean
        "value",
        "Valid_Pixels",

        # Original percentile method
        f"Mean_Trimmed_{trim_tag}",
        f"N_Trimmed_{trim_tag}",

        # Original standard deviation methods
        "Mean_STD_2",
        "N_STD_2",
        "Mean_STD_3",
        "N_STD_3",

        # Original harvested-area-weighted method
        "Mean_Area_Weighted",
        "N_Area_Weighted",
        "Harvested_Area_Used",

        # NEW Nup-based regional NUE
        "NUE_Ratio_of_Sums",
        "N_Ratio_Common_Pixels",
        "Sum_Nup_x_HA",
        "Sum_Ninput_x_HA"
    ]

    for col in output_columns:

        if col not in df.columns:
            df[col] = np.nan

    df = df[output_columns]

    # --------------------------------------------------------
    # Save Excel
    # --------------------------------------------------------

    os.makedirs(
        os.path.dirname(OUT_EXCEL),
        exist_ok=True
    )

    if len(df) > 1_048_575:
        raise ValueError(
            "Excel worksheet row limit exceeded"
        )

    df.to_excel(
        OUT_EXCEL,
        index=False,
        engine="openpyxl",
        sheet_name="Country Mean Comparison"
    )

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    elapsed = (
        time.perf_counter() - start_time
    )

    print("\n" + "=" * 75)
    print("COUNTRY STATISTICS COMPLETED")
    print("=" * 75)

    print(f"Total records: {len(df):,}")

    for col in [
        "value",
        f"Mean_Trimmed_{trim_tag}",
        "Mean_STD_2",
        "Mean_STD_3",
        "Mean_Area_Weighted",
        "NUE_Ratio_of_Sums"
    ]:

        print(
            f"{col}: "
            f"{df[col].notna().sum():,} "
            "valid country-year records"
        )

    print(f"\nTotal time: {elapsed:.2f}s")
    print(f"Excel saved: {OUT_EXCEL}")

    display(df.head(15))

    return df


# ============================================================
# 12. JUPYTERLAB INTERACTIVE CONTROLS
# ============================================================

# ------------------------------------------------------------
# Option 1: Percentile trimmed mean
# ------------------------------------------------------------

checkbox_trimmed = widgets.Checkbox(
    value=False,
    description="Option 1: Percentile Trimmed Mean",
    indent=False,
    layout=widgets.Layout(width="380px")
)

percent_widget = widgets.BoundedFloatText(
    value=95.0,
    min=1.0,
    max=100.0,
    step=1.0,
    description="Keep (%)",
    layout=widgets.Layout(width="230px")
)

# ------------------------------------------------------------
# Option 2: Standard deviation mean
# ------------------------------------------------------------

checkbox_std = widgets.Checkbox(
    value=False,
    description="Option 2: Standard Deviation Mean",
    indent=False,
    layout=widgets.Layout(width="380px")
)

checkbox_std2 = widgets.Checkbox(
    value=True,
    description="2 sigma",
    indent=False
)

checkbox_std3 = widgets.Checkbox(
    value=False,
    description="3 sigma",
    indent=False
)

# ------------------------------------------------------------
# Option 3: Arithmetic mean
# ------------------------------------------------------------

checkbox_all = widgets.Checkbox(
    value=True,
    description="Option 3: Arithmetic Mean",
    indent=False,
    layout=widgets.Layout(width="380px")
)

# ------------------------------------------------------------
# Option 4: Harvested-area-weighted mean
# ------------------------------------------------------------

checkbox_weighted = widgets.Checkbox(
    value=True,
    description="Option 4: Harvested-Area-Weighted Mean",
    indent=False,
    layout=widgets.Layout(width="450px")
)

# ------------------------------------------------------------
# Option 5: NEW Nup-based NUE
# ------------------------------------------------------------

checkbox_ratio = widgets.Checkbox(
    value=True,
    description="Option 5: Nup-based NUE (Ratio of Sums)",
    indent=False,
    layout=widgets.Layout(width="450px")
)

# ------------------------------------------------------------
# Run button
# ------------------------------------------------------------

run_button = widgets.Button(
    description="Run All Selected Methods",
    button_style="success",
    icon="play",
    layout=widgets.Layout(width="280px")
)

output_widget = widgets.Output()


# ============================================================
# 13. RUN BUTTON CALLBACK
# ============================================================

def on_run_clicked(button):

    enable_trimmed = checkbox_trimmed.value
    enable_std = checkbox_std.value
    enable_all = checkbox_all.value
    enable_weighted = checkbox_weighted.value
    enable_ratio = checkbox_ratio.value

    if not any([
        enable_trimmed,
        enable_std,
        enable_all,
        enable_weighted,
        enable_ratio
    ]):

        with output_widget:
            clear_output(wait=True)
            print("ERROR: Select at least one method.")
        return

    std_multipliers = []

    if checkbox_std2.value:
        std_multipliers.append(2)

    if checkbox_std3.value:
        std_multipliers.append(3)

    if enable_std and not std_multipliers:

        with output_widget:
            clear_output(wait=True)
            print(
                "ERROR: Select 2 sigma and/or 3 sigma."
            )
        return

    keep_percent = float(
        percent_widget.value
    )

    trim_tag = (
        f"{keep_percent:g}"
        .replace(".", "p")
    )

    config = {
        "enable_trimmed": enable_trimmed,
        "keep_percent": keep_percent,
        "trim_tag": trim_tag,

        "enable_std": enable_std,
        "std_multipliers": std_multipliers,

        "enable_all": enable_all,

        "enable_weighted": enable_weighted,

        "enable_ratio": enable_ratio
    }

    run_button.disabled = True

    with output_widget:

        clear_output(wait=True)

        try:

            global country_mean_results

            country_mean_results = (
                run_country_mean_comparison(
                    config
                )
            )

        except Exception:

            traceback.print_exc()

        finally:

            run_button.disabled = False


run_button.on_click(on_run_clicked)


# ============================================================
# 14. DISPLAY JUPYTER CONTROL PANEL
# ============================================================

control_panel = widgets.VBox([

    widgets.HTML(
        "<h3>Global NUE - Country Statistics Comparison</h3>"
    ),

    checkbox_trimmed,
    percent_widget,

    widgets.HTML("<hr>"),

    checkbox_std,

    widgets.HBox([
        checkbox_std2,
        checkbox_std3
    ]),

    widgets.HTML("<hr>"),

    checkbox_all,

    widgets.HTML("<hr>"),

    checkbox_weighted,

    widgets.HTML("<hr>"),

    checkbox_ratio,

    widgets.HTML("<hr>"),

    run_button
])

display(control_panel)
display(output_widget)

print(
    "Select one or more methods and click "
    "'Run All Selected Methods'."

In [ ]:
# =====================================================
# -------Global Maize NUE---Based on the weighted median-------------
# ---Step 1----Setting parameters----------
# ---Step 2--- Run ----------------
# ---Step 3--- simple validation --------------
# ---Step 4--- Match with harvested area, mask
# ---Step 5--- Remove outliers, Useful tool,Pixels number should larger than 6, NUE more then 150; set non, 
# ---Step 6--- Check mean value
# ---Step 7--- Extract country mean value--Useful---
# ---Step 8--- Extract county mean value--China and US---
# =======V 2026.10.27======================================
# ============================================================
# Regional NUE Statistics - 3 Calculation Methods 4usNUPW
# Method 1: Original Regional Arithmetic Mean
# Method 2: Harvested-Area-Weighted Mean
#           SUM(NUE * HA) / SUM(HA)
#
# Method 3 (NEW): Nup-based Regional NUE
#
#                SUM(Nup * HA)
#   NUE_region = ----------------------
#                SUM((Nup/NUE) * HA)
#
# - Arbitrary SHP polygons (USA / China / other)
# - Preserve all original SHP attribute columns
# - Match annual NUE, Nup, and Harvested Area
# - NUE is the core valid-pixel mask
# - Numerator and denominator use the SAME pixels
# - No interpolation, filling, or resampling
# - Excel: original columns + new comparison columns
#
# JupyterLab - ONE CELL
# ============================================================

import os
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

from rasterstats import gen_zonal_stats
from itertools import repeat
from IPython.display import display


# ============================================================
# 1. USER CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# SHAPEFILE
# ------------------------------------------------------------

SHP_FOLDER = (
    # China example:
    # r"Z:\staff\Qiang_S\09_Datasets"
    # r"\09OtherDataUsed\GDAM_CountyLevel\gadm41_CHN_shp"

    r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\GDAM_CountyLevel\gadm41_USA_shp"
)

# SHAPE_NAME = "gadm41_CHN_0.shp"
# SHAPE_NAME = "gadm41_CHN_3.shp"
SHAPE_NAME = "gadm41_USA_2.shp"

SHP_PATH = os.path.join(
    SHP_FOLDER,
    SHAPE_NAME
)

# ------------------------------------------------------------
# NUE GEOTIFF FOLDER
# ------------------------------------------------------------

TIF_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\02Global_NUE_Maize\02MaizeNUE_M2_20261029V1_M6_150R"
)

# ------------------------------------------------------------
# HARVESTED AREA FOLDER
# ------------------------------------------------------------

AREA_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\GCSN_HarvestedArea_1995_2020\02StandardizedDatasets\02MaizeHarvestedArea"
)

# ------------------------------------------------------------
# NEW: NUP FOLDER
#
# IMPORTANT: Change to your actual Nup folder.
#
# This folder should contain Nup GeoTIFF files
# for the selected crop, with yearly filenames.
# ------------------------------------------------------------

NUP_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\02Global_NUE_Maize\02MaizeNUE_M2_20261029V1_Nup"
)

CROP = "Maize"

# ------------------------------------------------------------
# FILE NAME MATCHING
# ------------------------------------------------------------

# NUE examples:
# NUE_median_Opt_1995.tif
# NUE_median_Opt_2020.tif

NUE_FILENAME_REGEX = (
    r"^.*_(?P<year>(?:19|20)\d{2})"
    r"\.(?:tif|tiff)$"
)

# Harvested Area examples:
# Maize_HarArea_1995.tif
# Maize_HarArea_2020.tif

AREA_FILENAME_REGEX = (
    rf"^{re.escape(CROP)}_HarArea_"
    rf"(?P<year>(?:19|20)\d{{2}})"
    rf"\.(?:tif|tiff)$"
)

# Nup examples:
# Maize_Nup_1995.tif
# Maize_Nuptake_1995.tif
# Nup_Maize_1995.tif
#
# The default matches ANY filename ending in _YYYY.tif
# within the dedicated NUP_FOLDER.
#
# If NUP_FOLDER contains other raster types,
# make this regex more specific.

NUP_FILENAME_REGEX = (
    r"^.*_(?P<year>(?:19|20)\d{2})"
    r"\.(?:tif|tiff)$"
)

# ------------------------------------------------------------
# OUTPUT EXCEL
# ------------------------------------------------------------

OUT_EXCEL = os.path.join(
    TIF_FOLDER,
    f"{Path(SHAPE_NAME).stem}_ZonalMean_NupRatio.xlsx"
)

# ------------------------------------------------------------
# NODATA
# ------------------------------------------------------------

NODATA = -9999.0

ADDITIONAL_NODATA = []

# ------------------------------------------------------------
# ZONAL STATISTICS
# ------------------------------------------------------------

# False = original pixel-center rule
ALL_TOUCHED = False

RASTER_BAND = 1

# ------------------------------------------------------------
# YEAR RANGE
# ------------------------------------------------------------

START_YEAR = 1995
END_YEAR = 2019

# ------------------------------------------------------------
# VALID PIXEL CONDITIONS
# ------------------------------------------------------------

# Harvested Area must be > 0
AREA_THRESHOLD = 0.0

# NUE must be > 0 for Nup/NUE to be defined
NUE_MIN_VALUE = 0.0

# Nup must be >= 0
NUP_MIN_VALUE = 0.0

# ------------------------------------------------------------
# MISSING FILE HANDLING
# ------------------------------------------------------------

REQUIRE_AREA_MATCH = True

REQUIRE_NUP_MATCH = True


# ============================================================
# 2. INPUT VALIDATION
# ============================================================

start_time = time.perf_counter()

if not os.path.isfile(SHP_PATH):
    raise FileNotFoundError(
        f"Shapefile not found:\n{SHP_PATH}"
    )

for folder, label in [
    (TIF_FOLDER, "NUE"),
    (AREA_FOLDER, "Harvested Area"),
    (NUP_FOLDER, "Nup")
]:
    if not os.path.isdir(folder):
        raise FileNotFoundError(
            f"{label} folder not found:\n{folder}"
        )

if START_YEAR > END_YEAR:
    raise ValueError(
        "START_YEAR must be <= END_YEAR"
    )

if not np.isfinite(AREA_THRESHOLD):
    raise ValueError(
        "AREA_THRESHOLD must be finite"
    )


# ============================================================
# 3. LOAD SHAPEFILE
# ============================================================

print("=" * 75)
print("REGIONAL NUE STATISTICS")
print("Arithmetic Mean + HA Weighted Mean + Nup Ratio")
print("=" * 75)

gdf = gpd.read_file(SHP_PATH)

if gdf.empty:
    raise ValueError(
        "Shapefile contains no features"
    )

if gdf.crs is None:
    raise ValueError(
        "Shapefile CRS is missing"
    )

geometry_col = gdf.geometry.name

# Preserve ALL original SHP attributes
attribute_columns = [
    c for c in gdf.columns
    if c != geometry_col
]

# Original + NEW output fields
stat_columns = [
    "mean",
    "year",
    "area_weighted_mean",
    "common_pixels",
    "harvested_area_used",

    # NEW
    "nue_ratio_of_sums",
    "ratio_common_pixels",
    "sum_nup_ha",
    "sum_ninput_ha"
]

# Prevent overwriting original attributes
conflicts = [
    c for c in attribute_columns
    if c.lower() in {
        s.lower() for s in stat_columns
    }
]

if conflicts:
    raise ValueError(
        f"Conflicting SHP attributes: {conflicts}"
    )

gdf = gdf.reset_index(drop=True)

attributes_df = (
    gdf[attribute_columns]
    .copy()
    .reset_index(drop=True)
)

print(f"Shapefile: {SHAPE_NAME}")
print(f"Polygons : {len(gdf):,}")
print(f"CRS      : {gdf.crs}")


# ============================================================
# 4. FIND RASTERS BY YEAR
# ============================================================

def index_yearly_rasters(
    folder,
    filename_regex,
    recursive=True
):
    """
    Return {year: file_path}.

    Stop when multiple files match the same year.
    """

    pattern = re.compile(
        filename_regex,
        re.IGNORECASE
    )

    root = Path(folder)

    iterator = (
        root.rglob("*")
        if recursive
        else root.iterdir()
    )

    result = {}

    for path in sorted(iterator):

        if not path.is_file():
            continue

        if path.suffix.lower() not in (
            ".tif", ".tiff"
        ):
            continue

        match = pattern.fullmatch(path.name)

        if match is None:
            continue

        year = int(
            match.group("year")
        )

        if not (
            START_YEAR <= year <= END_YEAR
        ):
            continue

        if year in result:
            raise ValueError(
                f"Duplicate raster for year {year}:\n"
                f"{result[year]}\n{path}"
            )

        result[year] = str(path)

    return result


# Original NUE raster folder:
# search current folder only
nue_files = index_yearly_rasters(
    TIF_FOLDER,
    NUE_FILENAME_REGEX,
    recursive=False
)

# Harvested Area:
# search current folder + subfolders
area_files = index_yearly_rasters(
    AREA_FOLDER,
    AREA_FILENAME_REGEX,
    recursive=True
)

# NEW Nup:
# search current folder + subfolders
nup_files = index_yearly_rasters(
    NUP_FOLDER,
    NUP_FILENAME_REGEX,
    recursive=True
)

if not nue_files:
    raise RuntimeError(
        "No NUE GeoTIFF files found"
    )

year_files = []
missing_area = []
missing_nup = []

for year in sorted(nue_files):

    area_path = area_files.get(year)
    nup_path = nup_files.get(year)

    if area_path is None:
        missing_area.append(year)

    if nup_path is None:
        missing_nup.append(year)

    year_files.append(
        (
            year,
            nue_files[year],
            area_path,
            nup_path
        )
    )

print(f"\nNUE years : {len(nue_files)}")
print(f"Area years: {len(area_files)}")
print(f"Nup years : {len(nup_files)}")

if missing_area:
    print("Missing Area years:", missing_area)

    if REQUIRE_AREA_MATCH:
        raise FileNotFoundError(
            f"Missing Harvested Area years: {missing_area}"
        )

if missing_nup:
    print("Missing Nup years:", missing_nup)

    if REQUIRE_NUP_MATCH:
        raise FileNotFoundError(
            f"Missing Nup years: {missing_nup}"
        )

print(
    "Years to process:",
    [x[0] for x in year_files]
)


# ============================================================
# 5. READ AND CLEAN RASTER
# ============================================================

def read_clean_raster(src):
    """
    Read raster and standardize invalid pixels.

    Invalid:
      - Metadata NoData
      - NaN / Inf
      - Explicit -9999
      - ADDITIONAL_NODATA

    Returns:
      float32 data
      valid pixel count
    """

    if src.count < RASTER_BAND:
        raise ValueError(
            f"Band {RASTER_BAND} missing: {src.name}"
        )

    masked = src.read(
        RASTER_BAND,
        masked=True
    )

    data = np.array(
        masked.data,
        dtype=np.float32,
        copy=True
    )

    invalid = (
        np.ma.getmaskarray(masked)
        .copy()
    )

    invalid |= ~np.isfinite(data)
    invalid |= (data == NODATA)

    for nd in ADDITIONAL_NODATA:
        invalid |= (data == nd)

    data[invalid] = NODATA

    valid_count = int(
        np.count_nonzero(~invalid)
    )

    return data, valid_count


# ============================================================
# 6. GRID ALIGNMENT CHECK
# ============================================================

def check_grid_alignment(
    src_ref,
    src_other,
    label
):
    """
    NUE, HA and Nup must have identical:
      - CRS
      - dimensions
      - affine transform

    No automatic resampling.
    """

    if (
        src_ref.crs is None
        or src_other.crs is None
    ):
        raise ValueError(
            f"{label}: Missing CRS"
        )

    if src_ref.crs != src_other.crs:
        raise ValueError(
            f"{label}: CRS mismatch"
        )

    if (
        src_ref.width != src_other.width
        or src_ref.height != src_other.height
    ):
        raise ValueError(
            f"{label}: Raster dimensions mismatch"
        )

    if not src_ref.transform.almost_equals(
        src_other.transform,
        precision=1e-10
    ):
        raise ValueError(
            f"{label}: Pixel grid mismatch"
        )


def check_mini_alignment(
    stat_ref,
    stat_other,
    year,
    region_index,
    label
):
    """
    Verify matching zonal mini-raster windows.
    """

    a = stat_ref.get("mini_raster_affine")
    b = stat_other.get("mini_raster_affine")

    x = stat_ref.get("mini_raster_array")
    y = stat_other.get("mini_raster_array")

    if (
        a is None
        or b is None
        or x is None
        or y is None
    ):
        raise ValueError(
            f"{year}, polygon {region_index}: "
            f"{label} mini-raster missing"
        )

    if x.shape != y.shape:
        raise ValueError(
            f"{year}, polygon {region_index}: "
            f"{label} mini-raster shape mismatch"
        )

    if not a.almost_equals(
        b,
        precision=1e-10
    ):
        raise ValueError(
            f"{year}, polygon {region_index}: "
            f"{label} mini-raster grid mismatch"
        )


# ============================================================
# 7. VALID PIXELS FROM ZONAL MINI-RASTER
# ============================================================

def get_values_and_valid(masked_array):

    ma = np.ma.asarray(masked_array)

    values = np.asarray(
        np.ma.getdata(ma),
        dtype=np.float64
    )

    valid = (
        ~np.ma.getmaskarray(ma)
    ).copy()

    valid &= np.isfinite(values)
    valid &= (values != NODATA)

    for nd in ADDITIONAL_NODATA:
        valid &= (values != nd)

    return values, valid


# ============================================================
# 8. ORIGINAL HA-WEIGHTED NUE
# ============================================================

def calculate_area_weighted_mean(
    nue_mini,
    area_mini
):
    """
    Original:
        SUM(NUE * HA) / SUM(HA)

    Uses common NUE and HA valid pixels only.
    """

    if nue_mini is None or area_mini is None:
        return None, 0, 0.0

    nue, nue_valid = get_values_and_valid(
        nue_mini
    )

    area, area_valid = get_values_and_valid(
        area_mini
    )

    if nue.shape != area.shape:
        raise ValueError(
            "NUE / HA mini-raster shape mismatch"
        )

    common = (
        nue_valid
        & area_valid
        & (area > AREA_THRESHOLD)
    )

    count = int(
        np.count_nonzero(common)
    )

    if count == 0:
        return None, 0, 0.0

    nue_values = nue[common]
    ha_values = area[common]

    area_sum = float(
        np.sum(
            ha_values,
            dtype=np.float64
        )
    )

    if not np.isfinite(area_sum) or area_sum <= 0:
        return None, count, area_sum

    weighted_sum = float(
        np.sum(
            nue_values * ha_values,
            dtype=np.float64
        )
    )

    mean_value = weighted_sum / area_sum

    if not np.isfinite(mean_value):
        mean_value = None

    return mean_value, count, area_sum


# ============================================================
# 9. NEW: NUP-BASED REGIONAL NUE
# ============================================================

def calculate_nup_ratio(
    nue_mini,
    nup_mini,
    area_mini
):
    """
    Regional NUE (Ratio of Sums):

             SUM(Nup_i * HA_i)
       ---------------------------------
       SUM((Nup_i / NUE_i) * HA_i)

    RULES:
      1. NUE is the core raster
      2. NUE valid and > 0
      3. Nup valid and >= 0
      4. HA valid and > 0
      5. Same pixels for numerator/denominator
      6. Invalid arithmetic terms excluded from both
      7. No interpolation or resampling

    Returns:
      ratio
      number of common valid pixels
      numerator sum
      denominator sum
    """

    if (
        nue_mini is None
        or nup_mini is None
        or area_mini is None
    ):
        return None, 0, 0.0, 0.0

    nue, nue_valid = get_values_and_valid(
        nue_mini
    )

    nup, nup_valid = get_values_and_valid(
        nup_mini
    )

    area, area_valid = get_values_and_valid(
        area_mini
    )

    if not (
        nue.shape == nup.shape == area.shape
    ):
        raise ValueError(
            "NUE / Nup / HA mini-raster shape mismatch"
        )

    # --------------------------------------------------------
    # CORE VALID MASK
    #
    # NUE missing -> exclude pixel from BOTH sums.
    # --------------------------------------------------------

    common = (
        nue_valid
        & (nue > NUE_MIN_VALUE)
        & nup_valid
        & (nup >= NUP_MIN_VALUE)
        & area_valid
        & (area > AREA_THRESHOLD)
    )

    initial_count = int(
        np.count_nonzero(common)
    )

    if initial_count == 0:
        return None, 0, 0.0, 0.0

    nue_values = nue[common]
    nup_values = nup[common]
    area_values = area[common]

    # --------------------------------------------------------
    # PIXEL CONTRIBUTIONS
    # --------------------------------------------------------

    with np.errstate(
        divide="ignore",
        invalid="ignore",
        over="ignore"
    ):

        numerator_terms = (
            nup_values * area_values
        )

        denominator_terms = (
            (nup_values / nue_values)
            * area_values
        )

    # If numerical overflow creates NaN / Inf,
    # remove from BOTH numerator and denominator.
    finite_terms = (
        np.isfinite(numerator_terms)
        & np.isfinite(denominator_terms)
    )

    numerator_terms = numerator_terms[
        finite_terms
    ]

    denominator_terms = denominator_terms[
        finite_terms
    ]

    common_count = int(
        numerator_terms.size
    )

    if common_count == 0:
        return None, 0, 0.0, 0.0

    # --------------------------------------------------------
    # REGIONAL TOTALS
    # --------------------------------------------------------

    sum_nup_ha = float(
        np.sum(
            numerator_terms,
            dtype=np.float64
        )
    )

    sum_ninput_ha = float(
        np.sum(
            denominator_terms,
            dtype=np.float64
        )
    )

    if not (
        np.isfinite(sum_nup_ha)
        and np.isfinite(sum_ninput_ha)
    ):
        raise ValueError(
            "Non-finite Nup ratio totals"
        )

    # Denominator = 0 -> undefined NUE
    if sum_ninput_ha <= 0:
        return (
            None,
            common_count,
            sum_nup_ha,
            sum_ninput_ha
        )

    nue_ratio = (
        sum_nup_ha / sum_ninput_ha
    )

    if not np.isfinite(nue_ratio):
        nue_ratio = None

    return (
        nue_ratio,
        common_count,
        sum_nup_ha,
        sum_ninput_ha
    )


# ============================================================
# 10. PROCESS ANNUAL REGIONAL STATISTICS
# ============================================================

polygon_cache = {}

result_frames = []

print("\n" + "=" * 75)
print("START REGIONAL NUE CALCULATION")
print("=" * 75)

for index, (
    year,
    nue_tif,
    area_tif,
    nup_tif
) in enumerate(year_files, start=1):

    t0 = time.perf_counter()

    print("\n" + "-" * 75)

    print(
        f"[{index}/{len(year_files)}] "
        f"Processing {year}"
    )

    # --------------------------------------------------------
    # 10.1 READ THREE RASTERS
    # --------------------------------------------------------

    with rasterio.open(nue_tif) as src_nue:

        if src_nue.crs is None:
            raise ValueError(
                f"{year}: NUE CRS missing"
            )

        raster_crs = src_nue.crs
        affine = src_nue.transform

        nue_data, nue_valid_count = (
            read_clean_raster(src_nue)
        )

        area_data = None
        nup_data = None

        # Harvested Area
        if area_tif is not None:

            with rasterio.open(
                area_tif
            ) as src_area:

                check_grid_alignment(
                    src_nue,
                    src_area,
                    f"{year}: NUE / HA"
                )

                area_data, _ = read_clean_raster(
                    src_area
                )

        # NEW: Nup
        if nup_tif is not None:

            with rasterio.open(
                nup_tif
            ) as src_nup:

                check_grid_alignment(
                    src_nue,
                    src_nup,
                    f"{year}: NUE / Nup"
                )

                nup_data, _ = read_clean_raster(
                    src_nup
                )

    print(
        f"Valid NUE pixels: {nue_valid_count:,}"
    )

    # --------------------------------------------------------
    # 10.2 REPROJECT POLYGONS IF NECESSARY
    # --------------------------------------------------------

    crs_key = raster_crs.to_wkt()

    if crs_key not in polygon_cache:

        polygon_cache[crs_key] = (
            gdf
            if gdf.crs == raster_crs
            else gdf.to_crs(raster_crs)
        )

    gdf_projected = polygon_cache[crs_key]

    geometry_valid = (
        gdf_projected.geometry.notna()
        & ~gdf_projected.geometry.is_empty
    )

    valid_indices = np.flatnonzero(
        geometry_valid.to_numpy()
    )

    geometries = (
        gdf_projected.geometry
        .iloc[valid_indices]
        .tolist()
    )

    # --------------------------------------------------------
    # 10.3 INITIALIZE ALL REGION OUTPUTS
    # --------------------------------------------------------

    mean_values = [None] * len(gdf)

    weighted_values = [None] * len(gdf)
    weighted_counts = [0] * len(gdf)
    weighted_area_sums = [0.0] * len(gdf)

    # NEW algorithm
    ratio_values = [None] * len(gdf)
    ratio_counts = [0] * len(gdf)
    numerator_sums = [0.0] * len(gdf)
    denominator_sums = [0.0] * len(gdf)

    # --------------------------------------------------------
    # 10.4 REGIONAL ZONAL STATISTICS
    # --------------------------------------------------------

    if nue_valid_count > 0 and len(geometries) > 0:

        # NUE uses original zonal_stats mean.
        nue_iterator = gen_zonal_stats(
            geometries,
            nue_data,
            affine=affine,
            stats=["mean"],
            nodata=NODATA,
            all_touched=ALL_TOUCHED,
            geojson_out=False,
            raster_out=True
        )

        # HA zonal pixel data
        if area_data is not None:

            area_iterator = gen_zonal_stats(
                geometries,
                area_data,
                affine=affine,
                stats=["count"],
                nodata=NODATA,
                all_touched=ALL_TOUCHED,
                geojson_out=False,
                raster_out=True
            )

        else:
            area_iterator = repeat(None)

        # Nup zonal pixel data
        if nup_data is not None:

            nup_iterator = gen_zonal_stats(
                geometries,
                nup_data,
                affine=affine,
                stats=["count"],
                nodata=NODATA,
                all_touched=ALL_TOUCHED,
                geojson_out=False,
                raster_out=True
            )

        else:
            nup_iterator = repeat(None)

        # ----------------------------------------------------
        # 10.5 PROCESS EACH SHP POLYGON
        # ----------------------------------------------------

        for (
            original_idx,
            nue_stat,
            area_stat,
            nup_stat
        ) in zip(
            valid_indices,
            nue_iterator,
            area_iterator,
            nup_iterator
        ):

            # -----------------------------------------------
            # METHOD 1: ORIGINAL MEAN
            # -----------------------------------------------

            mean_val = nue_stat.get("mean")

            if (
                mean_val is not None
                and np.isfinite(mean_val)
            ):
                mean_values[original_idx] = float(
                    mean_val
                )

            nue_mini = nue_stat.get(
                "mini_raster_array"
            )

            # -----------------------------------------------
            # METHOD 2: HA-WEIGHTED MEAN
            # -----------------------------------------------

            if area_stat is not None:

                check_mini_alignment(
                    nue_stat,
                    area_stat,
                    year,
                    original_idx,
                    "NUE / HA"
                )

                area_mini = area_stat.get(
                    "mini_raster_array"
                )

                (
                    weighted_mean,
                    weighted_count,
                    weighted_area_sum
                ) = calculate_area_weighted_mean(
                    nue_mini,
                    area_mini
                )

                weighted_values[original_idx] = (
                    weighted_mean
                )

                weighted_counts[original_idx] = (
                    weighted_count
                )

                weighted_area_sums[original_idx] = (
                    weighted_area_sum
                )

            else:
                area_mini = None

            # -----------------------------------------------
            # METHOD 3: NEW NUP RATIO OF SUMS
            # -----------------------------------------------

            if (
                area_stat is not None
                and nup_stat is not None
            ):

                check_mini_alignment(
                    nue_stat,
                    nup_stat,
                    year,
                    original_idx,
                    "NUE / Nup"
                )

                check_mini_alignment(
                    area_stat,
                    nup_stat,
                    year,
                    original_idx,
                    "HA / Nup"
                )

                nup_mini = nup_stat.get(
                    "mini_raster_array"
                )

                (
                    ratio,
                    ratio_count,
                    sum_nup,
                    sum_ninput
                ) = calculate_nup_ratio(
                    nue_mini,
                    nup_mini,
                    area_mini
                )

                ratio_values[original_idx] = ratio

                ratio_counts[original_idx] = (
                    ratio_count
                )

                numerator_sums[original_idx] = (
                    sum_nup
                )

                denominator_sums[original_idx] = (
                    sum_ninput
                )

    # --------------------------------------------------------
    # 10.6 CREATE YEARLY DATAFRAME
    # --------------------------------------------------------

    year_df = attributes_df.copy()

    # Original results
    year_df["mean"] = mean_values
    year_df["year"] = year

    year_df["area_weighted_mean"] = (
        weighted_values
    )

    year_df["common_pixels"] = (
        weighted_counts
    )

    year_df["harvested_area_used"] = (
        weighted_area_sums
    )

    # NEW results
    year_df["nue_ratio_of_sums"] = (
        ratio_values
    )

    year_df["ratio_common_pixels"] = (
        ratio_counts
    )

    year_df["sum_nup_ha"] = (
        numerator_sums
    )

    year_df["sum_ninput_ha"] = (
        denominator_sums
    )

    result_frames.append(year_df)

    # --------------------------------------------------------
    # 10.7 YEAR SUMMARY
    # --------------------------------------------------------

    print(
        "Original mean regions :",
        sum(v is not None for v in mean_values)
    )

    print(
        "HA-weighted regions   :",
        sum(v is not None for v in weighted_values)
    )

    print(
        "Nup-ratio regions     :",
        sum(v is not None for v in ratio_values)
    )

    print(
        "Nup common pixels     :",
        f"{sum(ratio_counts):,}"
    )

    print(
        f"Year {year} completed in "
        f"{time.perf_counter() - t0:.1f}s"
    )


# ============================================================
# 11. COMBINE RESULTS
# ============================================================

df_result = pd.concat(
    result_frames,
    ignore_index=True
)

output_columns = (
    attribute_columns
    + stat_columns
)

df_result = df_result[output_columns]

# Standardize invalid numerical outputs
for col in [
    "mean",
    "area_weighted_mean",
    "harvested_area_used",
    "nue_ratio_of_sums",
    "sum_nup_ha",
    "sum_ninput_ha"
]:

    df_result[col] = pd.to_numeric(
        df_result[col],
        errors="coerce"
    )

    df_result[col] = df_result[col].replace(
        [np.inf, -np.inf],
        np.nan
    )

# Convert complex attributes for Excel
for col in attribute_columns:

    df_result[col] = df_result[col].map(
        lambda v: (
            str(v)
            if isinstance(
                v,
                (list, tuple, dict, np.ndarray)
            )
            else v
        )
    )


# ============================================================
# 12. EXPORT EXCEL
# ============================================================

if len(df_result) > 1_048_575:
    raise ValueError(
        "Excel row limit exceeded. "
        "Split results into multiple worksheets."
    )

df_result.to_excel(
    OUT_EXCEL,
    index=False,
    sheet_name="Zonal Mean",
    engine="openpyxl",
    na_rep=""
)


# ============================================================
# 13. FINAL SUMMARY
# ============================================================

elapsed = time.perf_counter() - start_time

print("\n" + "=" * 75)
print("REGIONAL NUE STATISTICS COMPLETED")
print("=" * 75)

print(f"Shapefile          : {SHAPE_NAME}")
print(f"Polygons           : {len(gdf):,}")
print(f"Years              : {len(year_files)}")
print(f"Total records      : {len(df_result):,}")

print(
    "Original means     : "
    f"{df_result['mean'].notna().sum():,}"
)

print(
    "HA-weighted means  : "
    f"{df_result['area_weighted_mean'].notna().sum():,}"
)

print(
    "Nup-based NUE      : "
    f"{df_result['nue_ratio_of_sums'].notna().sum():,}"
)

print(
    f"Total time         : {elapsed:.2f}s"
)

print("\nOutput columns:")
print(df_result.columns.tolist())

print(f"\nExcel saved to:\n{OUT_EXCEL}")

print("=" * 75)

display(df_result.head(10))

In [ ]:
# =====================================================
# -------Global Maize NUE---Based on the weighted median-------------
# ---Step 1----Setting parameters----------
# ---Step 2--- Run ----------------
# ---Step 3--- simple validation --------------
# ---Step 4--- Match with harvested area, mask
# ---Step 5--- Remove outliers, Useful tool,Pixels number should larger than 6, NUE more then 150; set non, 
# ---Step 6--- Check mean value
# ---Step 7--- Extract country mean NUE value--Useful---
# ---Step 8--- Extract county mean NUE value--China and US---
# ---Step 9--- -2matchPFPN--Remove PFPN outliers--Extract county mean PFPN value--
# =====================================================
# ============================================================
# Year-by-Year GeoTIFF Masking - Intersection Only used 2match
#
# Dataset A: Reference mask (e.g. Area_1995.tif)
# Dataset B: Target data    (e.g. NUE_1995.tif)
#
# Rules:
#   A valid, B valid   -> Keep original B value
#   A valid, B NoData  -> NoData
#   A invalid, B valid -> NoData
#   A invalid, B NoData -> NoData
#
# No interpolation, no filling, no resampling.
# ============================================================

from pathlib import Path
import re
import csv
import numpy as np
import rasterio


# ============================================================
# 1. USER CONFIGURATION
# ============================================================

# Dataset A: Reference mask
FOLDER_A = Path(
    r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\GCSN_HarvestedArea_1995_2020\02StandardizedDatasets\02MaizeHarvestedArea"
)

# Dataset B: Target data
FOLDER_B = Path(
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\02Global_NUE_Maize\02MaizeNUE_M2_20261029V1_PFPN"
)

# Output folder
OUTPUT_FOLDER = Path(
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\02Global_NUE_Maize\02MaizeNUE_M2_20261029V1_PFPN_Intersection"
)

# ------------------------------------------------------------
# Mask definition
# ------------------------------------------------------------

# "positive": A > 0 is valid (recommended for harvested area)
# "non_nodata": Any finite A value, including zero, is valid
MASK_MODE = "positive"

MASK_THRESHOLD = 0.0

# Optional additional NoData values
# GeoTIFF metadata NoData is automatically recognized
ADDITIONAL_NODATA_A = [-9999.0]
ADDITIONAL_NODATA_B = [-9999.0]

# Output settings
OUTPUT_NODATA = np.nan
OVERWRITE = False


# ============================================================
# 2. INITIALIZE
# ============================================================

FOLDER_A = FOLDER_A.resolve()
FOLDER_B = FOLDER_B.resolve()
OUTPUT_FOLDER = OUTPUT_FOLDER.resolve()

if not FOLDER_A.is_dir():
    raise FileNotFoundError(f"Folder A not found: {FOLDER_A}")

if not FOLDER_B.is_dir():
    raise FileNotFoundError(f"Folder B not found: {FOLDER_B}")

if MASK_MODE not in ("positive", "non_nodata"):
    raise ValueError("Invalid MASK_MODE")

if OUTPUT_FOLDER in (FOLDER_A, FOLDER_B):
    raise ValueError("Output folder must differ from input folders")

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)


# ============================================================
# 3. COLLECT FILES BY YEAR
# ============================================================

def extract_year(filename):

    stem = Path(filename).stem

    match = re.search(
        r"_(19\d{2}|20\d{2})$",
        stem
    )

    return int(match.group(1)) if match else None


def collect_files(folder):

    files = {}

    for path in sorted(folder.iterdir()):

        if not path.is_file():
            continue

        if path.suffix.lower() not in (".tif", ".tiff"):
            continue

        year = extract_year(path.name)

        if year is None:
            continue

        if year in files:
            raise ValueError(
                f"Duplicate year {year}: "
                f"{files[year].name}, {path.name}"
            )

        files[year] = path

    return files


files_A = collect_files(FOLDER_A)
files_B = collect_files(FOLDER_B)

common_years = sorted(
    set(files_A) & set(files_B)
)

print("=" * 65)
print("GEOTIFF INTERSECTION MASKING")
print("=" * 65)

print(f"A files       : {len(files_A)}")
print(f"B files       : {len(files_B)}")
print(f"Matched years : {len(common_years)}")
print(f"Years         : {common_years}")

print("Missing A:", sorted(set(files_B) - set(files_A)))
print("Missing B:", sorted(set(files_A) - set(files_B)))

if not common_years:
    raise ValueError("No matched years found")


# ============================================================
# 4. CHECK GRID ALIGNMENT
# ============================================================

def check_grid(src_a, src_b):

    if src_a.crs is None or src_b.crs is None:
        raise ValueError("Missing raster CRS")

    if src_a.crs != src_b.crs:
        raise ValueError("CRS mismatch")

    if (
        src_a.width != src_b.width
        or src_a.height != src_b.height
    ):
        raise ValueError("Raster dimensions mismatch")

    if not src_a.transform.almost_equals(
        src_b.transform,
        precision=1e-10
    ):
        raise ValueError("Raster transform mismatch")

    if src_a.count != 1 or src_b.count != 1:
        raise ValueError("Only single-band GeoTIFF supported")


# ============================================================
# 5. READ DATA AND IDENTIFY VALID PIXELS
# ============================================================

def read_valid_data(src, additional_nodata):

    # Automatically mask metadata-defined NoData pixels
    data = src.read(1, masked=True)

    values = data.astype(np.float32).filled(np.nan)

    valid = np.isfinite(values)

    for nd in additional_nodata:
        valid &= (values != nd)

    return values, valid


# ============================================================
# 6. PROCESS ONE YEAR
# ============================================================

def process_year(year, path_a, path_b):

    output_path = OUTPUT_FOLDER / path_b.name

    if output_path.exists() and not OVERWRITE:

        print(f"{year}: SKIPPED - Already exists")

        return {
            "year": year,
            "status": "SKIPPED"
        }

    with rasterio.open(path_a) as src_a, \
         rasterio.open(path_b) as src_b:

        # Ensure identical grids
        check_grid(src_a, src_b)

        # ----------------------------------------------------
        # Read A
        # ----------------------------------------------------

        a_data, a_valid = read_valid_data(
            src_a,
            ADDITIONAL_NODATA_A
        )

        # Define mask
        if MASK_MODE == "positive":

            mask_a = (
                a_valid
                & (a_data > MASK_THRESHOLD)
            )

        else:
            mask_a = a_valid

        # ----------------------------------------------------
        # Read B
        # ----------------------------------------------------

        b_data, b_valid = read_valid_data(
            src_b,
            ADDITIONAL_NODATA_B
        )

        # ----------------------------------------------------
        # Intersection: A valid AND B valid
        # ----------------------------------------------------

        intersection_mask = mask_a & b_valid

        # Initialize all pixels as NoData
        output = np.full(
            b_data.shape,
            np.nan,
            dtype=np.float32
        )

        # Only copy valid B values inside A
        output[intersection_mask] = b_data[intersection_mask]

        # ----------------------------------------------------
        # Statistics
        # ----------------------------------------------------

        total_pixels = b_data.size

        a_pixels = int(np.count_nonzero(mask_a))

        b_pixels = int(np.count_nonzero(b_valid))

        intersection_pixels = int(
            np.count_nonzero(intersection_mask)
        )

        # A valid, B missing
        a_only = int(
            np.count_nonzero(mask_a & (~b_valid))
        )

        # B valid, outside A
        b_removed = int(
            np.count_nonzero((~mask_a) & b_valid)
        )

        # ----------------------------------------------------
        # Export GeoTIFF
        # ----------------------------------------------------

        profile = src_b.profile.copy()

        profile.update(
            driver="GTiff",
            dtype="float32",
            count=1,
            nodata=OUTPUT_NODATA,
            compress="deflate",
            predictor=3,
            tiled=True,
            blockxsize=256,
            blockysize=256,
            BIGTIFF="IF_SAFER"
        )

        with rasterio.open(
            output_path,
            "w",
            **profile
        ) as dst:

            dst.write(output, 1)

        print(
            f"{year} | "
            f"A={a_pixels:,} | "
            f"B={b_pixels:,} | "
            f"Intersection={intersection_pixels:,} | "
            f"Removed_B={b_removed:,} | "
            f"A_without_B={a_only:,}"
        )

        return {
            "year": year,
            "status": "SUCCESS",
            "total_pixels": total_pixels,
            "A_valid": a_pixels,
            "B_valid": b_pixels,
            "intersection": intersection_pixels,
            "B_removed": b_removed,
            "A_without_B": a_only,
            "output_file": str(output_path)
        }


# ============================================================
# 7. BATCH PROCESS
# ============================================================

results = []

for index, year in enumerate(common_years, start=1):

    print(f"\n[{index}/{len(common_years)}] Processing {year}")

    try:

        result = process_year(
            year,
            files_A[year],
            files_B[year]
        )

        results.append(result)

    except Exception as e:

        print(f"ERROR {year}: {e}")

        results.append({
            "year": year,
            "status": "FAILED",
            "error": str(e)
        })


# ============================================================
# 8. EXPORT PROCESSING LOG
# ============================================================

log_path = OUTPUT_FOLDER / "Intersection_Mask_Log.csv"

fieldnames = [
    "year",
    "status",
    "total_pixels",
    "A_valid",
    "B_valid",
    "intersection",
    "B_removed",
    "A_without_B",
    "output_file",
    "error"
]

with open(
    log_path,
    "w",
    newline="",
    encoding="utf-8-sig"
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=fieldnames
    )

    writer.writeheader()
    writer.writerows(results)


# ============================================================
# 9. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("INTERSECTION MASKING COMPLETED")
print("=" * 65)

for status in ("SUCCESS", "SKIPPED", "FAILED"):

    count = sum(
        r["status"] == status
        for r in results
    )

    print(f"{status}: {count}")

print(f"\nOutput folder: {OUTPUT_FOLDER}")
print(f"Log file: {log_path}")

print("=" * 65)

In [ ]:
# =====================================================
# -------Global Maize NUE---Based on the weighted median-------------
# ---Step 1----Setting parameters----------
# ---Step 2--- Run ----------------
# ---Step 3--- simple validation --------------
# ---Step 4--- Match with harvested area, mask
# ---Step 5--- Remove outliers, Useful tool,Pixels number should larger than 6, NUE more then 150; set non, 
# ---Step 6--- Check mean value
# ---Step 7--- Extract country mean NUE value--Useful---
# ---Step 8--- Extract county mean NUE value--China and US---
# ---Step 9--- -2matchPFPN--Remove PFPN outliers--Extract county mean PFPN value--
# =====================================================
# Global NUE - Parallel Outlier Processing Options 4PFPN
# Step 1: N < MIN_COUNT or N missing -> REMOVE
# Step 2:
#   Upper threshold -> REMOVE / NEAREST
#   Lower threshold -> REMOVE / NEAREST
#
# Step 3:
#   Optional country-level mean + STD_FACTOR * STD
#   -> REMOVE / NEAREST
#   Maximum selected proportion per country: 5%
#
# NaN / NoData safe
# Parallel processing by year
# Output: GeoTIFF + Excel (3 original sheets)
#
# JupyterLab single-cell executable
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

from rasterio.features import geometry_mask
from scipy.ndimage import distance_transform_edt
from joblib import Parallel, delayed

from openpyxl import Workbook
from openpyxl.styles import (
    Font, PatternFill, Alignment, Border, Side
)
from openpyxl.utils import get_column_letter

# ============================================================
# 1. USER CONFIGURATION
# ===========================================================

A_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\02Global_NUE_Maize\02MaizeNUE_M2_20261029V1_PFPN_Intersection"
)

MASK_FOLDER = (
   r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\02Global_NUE_Maize\02MaizeNUE_M2_20261029V1"
)

OUT_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\02Global_NUE_Maize\02MaizeNUE_M2_20261029V1PFPN_M6_200R"
)

SHP_PATH = (
    r"Z:\staff\Qiang_S\09_Datasets"
    r"\09OtherDataUsed\GDAM_GlobalCountryLevel_Shp"
    r"\GlobalCountryLevel_Shp.shp"
)

M49_FIELD = "M49_Code"
NAME_FIELD = "COUNTRY"

EXCEL_OUT = os.path.join(
    OUT_FOLDER,
    "outlier_removal_summary_V1_M6_200R.xlsx"
)

# ------------------------------------------------------------
# Years and parallel processing
# ------------------------------------------------------------

START_YEAR = 1995
END_YEAR = 2020

# None = all years
# Example: [1995, 1996, 2000, 2020]
YEARS_TO_RUN = None

PARALLEL_YEARS = 5

# ------------------------------------------------------------
# Common settings
# ------------------------------------------------------------

NODATA = -9999.0

# ------------------------------------------------------------
# Step 1: Minimum valid sample count
# ------------------------------------------------------------
MIN_COUNT = 6

# ------------------------------------------------------------
# Step 2: Hard thresholds
# ------------------------------------------------------------
LOWER_HARD = 1
UPPER_HARD = 200

# Options: "REMOVE" or "NEAREST"
UPPER_OUTLIER_ACTION = "REMOVE"
LOWER_OUTLIER_ACTION = "REMOVE"

# ------------------------------------------------------------
# Step 3: Country standard deviation
# ------------------------------------------------------------

# True = enable country standard deviation processing
# False = skip Step 3

APPLY_STD_REMOVAL = False #True

# Mean + STD_FACTOR * standard deviation
STD_FACTOR = 2.0
# Maximum selected outlier ratio per country
MAX_REMOVE_RATIO = 0.05
# NEW:
# "REMOVE"  = Set selected Step 3 outliers to NoData
# "NEAREST" = Replace with nearest qualified value inside the SAME country

STD_OUTLIER_ACTION = "REMOVE"

# ------------------------------------------------------------
# Nearest-neighbor settings
# ------------------------------------------------------------

# None = No maximum distance
# Example: 12 = maximum 12 raster pixels
MAX_NEAREST_DISTANCE_PIXELS = None

# If no qualified nearest value exists,
# the target becomes NoData.

# ============================================================
# 2. PARAMETER VALIDATION
# ============================================================

VALID_ACTIONS = {"REMOVE", "NEAREST"}

UPPER_OUTLIER_ACTION = UPPER_OUTLIER_ACTION.upper()
LOWER_OUTLIER_ACTION = LOWER_OUTLIER_ACTION.upper()
STD_OUTLIER_ACTION = STD_OUTLIER_ACTION.upper()

for name, action in [
    ("UPPER_OUTLIER_ACTION", UPPER_OUTLIER_ACTION),
    ("LOWER_OUTLIER_ACTION", LOWER_OUTLIER_ACTION),
    ("STD_OUTLIER_ACTION", STD_OUTLIER_ACTION)
]:
    if action not in VALID_ACTIONS:
        raise ValueError(
            f"{name} must be REMOVE or NEAREST"
        )

if LOWER_HARD >= UPPER_HARD:
    raise ValueError(
        "LOWER_HARD must be smaller than UPPER_HARD"
    )

if PARALLEL_YEARS < 1:
    raise ValueError("PARALLEL_YEARS must be >= 1")

if not (0 <= MAX_REMOVE_RATIO <= 1):
    raise ValueError(
        "MAX_REMOVE_RATIO must be between 0 and 1"
    )

if (
    MAX_NEAREST_DISTANCE_PIXELS is not None
    and MAX_NEAREST_DISTANCE_PIXELS < 0
):
    raise ValueError(
        "MAX_NEAREST_DISTANCE_PIXELS must be >= 0"
    )

if not os.path.isfile(SHP_PATH):
    raise FileNotFoundError(SHP_PATH)

os.makedirs(OUT_FOLDER, exist_ok=True)


# ============================================================
# 3. RASTER UTILITY FUNCTIONS
# ============================================================

def read_raster_fixed(src):
    """
    Read raster and standardize:
      Metadata NoData
      NaN / Inf
      -9999

    Valid values remain unchanged.
    """

    data = (
        src.read(1, masked=True)
        .astype(np.float32)
        .filled(np.nan)
    )

    data = np.asarray(
        data,
        dtype=np.float32
    )

    invalid = (
        ~np.isfinite(data)
        | (data == NODATA)
    )

    data[invalid] = NODATA

    return data


def get_valid_mask(data):

    return (
        np.isfinite(data)
        & (data != NODATA)
    )


def empty_country_record(year, m49, name):

    return {
        "Year": year,
        "M49": m49,
        "Country": name,
        "Valid_before_step3": 0,
        "Mean": np.nan,
        "Std": np.nan,
        "Upper_sigma": np.nan,
        "Candidate_removed": 0,
        "Max_allowed_remove": 0,
        "Step3_removed": 0,
        "Step3_removed_%": 0.0
    }


# ============================================================
# 4. NEAREST-NEIGHBOR REPLACEMENT
# ============================================================

def replace_by_nearest(
    raster,
    target_mask,
    donor_mask,
    max_distance=None
):
    """
    Replace selected target pixels with nearest donor values.

    Parameters
    ----------
    raster:
        2D NumPy array, modified in place.

    target_mask:
        True for pixels selected for replacement.

    donor_mask:
        True for qualified source pixels.

    max_distance:
        Maximum distance in pixel units.
        None = unlimited.

    Returns
    -------
    replaced_count:
        Number successfully replaced.

    failed_count:
        Number without an eligible donor.
        These pixels become NoData.

    Notes
    -----
    - Original donor values are copied exactly.
    - No averaging or interpolation.
    - Calculations use row/column Euclidean distance.
    - Processing uses a local bounding window.
    """

    target_rows, target_cols = np.where(target_mask)

    target_count = len(target_rows)

    if target_count == 0:
        return 0, 0

    donor_mask = (
        donor_mask
        & get_valid_mask(raster)
        & ~target_mask
    )

    donor_rows, donor_cols = np.where(donor_mask)

    # No qualified donors
    if len(donor_rows) == 0:

        raster[target_rows, target_cols] = NODATA

        return 0, target_count

    # --------------------------------------------------------
    # Bounding window containing targets and donors
    # --------------------------------------------------------

    row_min = int(min(
        target_rows.min(),
        donor_rows.min()
    ))

    row_max = int(max(
        target_rows.max(),
        donor_rows.max()
    )) + 1

    col_min = int(min(
        target_cols.min(),
        donor_cols.min()
    ))

    col_max = int(max(
        target_cols.max(),
        donor_cols.max()
    )) + 1

    local_donors = donor_mask[
        row_min:row_max,
        col_min:col_max
    ]

    local_rows = target_rows - row_min
    local_cols = target_cols - col_min

    # --------------------------------------------------------
    # Nearest eligible donor
    # --------------------------------------------------------

    if max_distance is None:

        nearest_indices = distance_transform_edt(
            ~local_donors,
            return_distances=False,
            return_indices=True
        )

        nearest_rows = (
            nearest_indices[
                0,
                local_rows,
                local_cols
            ] + row_min
        )

        nearest_cols = (
            nearest_indices[
                1,
                local_rows,
                local_cols
            ] + col_min
        )

        replacement_values = raster[
            nearest_rows,
            nearest_cols
        ].copy()

        raster[
            target_rows,
            target_cols
        ] = replacement_values

        return target_count, 0

    else:

        distances, nearest_indices = distance_transform_edt(
            ~local_donors,
            return_distances=True,
            return_indices=True
        )

        target_distances = distances[
            local_rows,
            local_cols
        ]

        within_distance = (
            target_distances <= max_distance
        )

        # Qualified targets
        valid_rows = target_rows[within_distance]
        valid_cols = target_cols[within_distance]

        nearest_rows = (
            nearest_indices[
                0,
                local_rows[within_distance],
                local_cols[within_distance]
            ] + row_min
        )

        nearest_cols = (
            nearest_indices[
                1,
                local_rows[within_distance],
                local_cols[within_distance]
            ] + col_min
        )

        replacement_values = raster[
            nearest_rows,
            nearest_cols
        ].copy()

        raster[
            valid_rows,
            valid_cols
        ] = replacement_values

        # Targets exceeding maximum distance
        failed_rows = target_rows[~within_distance]
        failed_cols = target_cols[~within_distance]

        raster[
            failed_rows,
            failed_cols
        ] = NODATA

        replaced_count = int(
            np.count_nonzero(within_distance)
        )

        failed_count = target_count - replaced_count

        return replaced_count, failed_count


# ============================================================
# 5. PROCESS ONE YEAR
# ============================================================

def process_one_year(year):

    country_records = []

    A_tif = os.path.join(
        A_FOLDER,
        f"PFPN_M2_median_{year}.tif"
    )

    N_tif = os.path.join(
        MASK_FOLDER,
        f"NUE_M2_n_{year}.tif"
    )

    OUT_tif = os.path.join(
        OUT_FOLDER,
        f"PFPN_median_Opt_{year}.tif"
    )

    try:

        if not os.path.exists(A_tif):
            return {
                "year": year,
                "status": "MISSING_A",
                "message": A_tif,
                "annual": None,
                "countries": []
            }

        if not os.path.exists(N_tif):
            return {
                "year": year,
                "status": "MISSING_N",
                "message": N_tif,
                "annual": None,
                "countries": []
            }

        # ----------------------------------------------------
        # Read country polygons within each process
        # ----------------------------------------------------

        gdf = gpd.read_file(SHP_PATH)

        if gdf.crs is None:
            raise ValueError("Shapefile CRS missing")

        if gdf.crs.to_epsg() != 4326:
            gdf = gdf.to_crs(epsg=4326)

        gdf = gdf.sort_values(
            by=M49_FIELD
        ).reset_index(drop=True)

        # ----------------------------------------------------
        # Read raster A
        # ----------------------------------------------------

        with rasterio.open(A_tif) as src_a:

            A = read_raster_fixed(src_a)

            profile = src_a.profile.copy()
            transform = src_a.transform
            crs = src_a.crs

            height, width = A.shape

        # ----------------------------------------------------
        # Read raster N
        # ----------------------------------------------------

        with rasterio.open(N_tif) as src_n:

            N = read_raster_fixed(src_n)

            if (
                src_n.crs != crs
                or src_n.transform != transform
            ):
                raise ValueError(
                    "A and N raster grids are not aligned"
                )

        if A.shape != N.shape:
            raise ValueError(
                f"Shape mismatch: A={A.shape}, N={N.shape}"
            )

        if crs is None:
            raise ValueError("Raster CRS missing")

        # ====================================================
        # STEP 1 - N < MIN_COUNT / Missing N
        # ====================================================

        valid_mask = get_valid_mask(A)

        initial_count = int(
            np.count_nonzero(valid_mask)
        )

        n_valid_mask = get_valid_mask(N)

        step1_mask = (
            valid_mask
            & (
                ~n_valid_mask
                | (N < MIN_COUNT)
            )
        )

        step1_removed = int(
            np.count_nonzero(step1_mask)
        )

        A[step1_mask] = NODATA

        valid_mask = get_valid_mask(A)

        # ====================================================
        # STEP 2 - HARD THRESHOLDS
        # ====================================================

        step2_upper = (
            (A > UPPER_HARD)
            & valid_mask
        )

        step2_lower = (
            (A < LOWER_HARD)
            & valid_mask
        )

        upper_count = int(
            np.count_nonzero(step2_upper)
        )

        lower_count = int(
            np.count_nonzero(step2_lower)
        )

        # Same reported field as original script:
        # Number of pixels identified outside hard thresholds
        step2_removed = upper_count + lower_count

        # ----------------------------------------------------
        # Determine remove / nearest targets
        # ----------------------------------------------------

        step2_remove_mask = np.zeros_like(
            valid_mask,
            dtype=bool
        )

        step2_nearest_mask = np.zeros_like(
            valid_mask,
            dtype=bool
        )

        if UPPER_OUTLIER_ACTION == "REMOVE":
            step2_remove_mask |= step2_upper
        else:
            step2_nearest_mask |= step2_upper

        if LOWER_OUTLIER_ACTION == "REMOVE":
            step2_remove_mask |= step2_lower
        else:
            step2_nearest_mask |= step2_lower

        # Qualified donor pixels are those inside
        # the original hard thresholds, after Step 1.

        step2_donors = (
            valid_mask
            & (A >= LOWER_HARD)
            & (A <= UPPER_HARD)
        )

        # ----------------------------------------------------
        # REMOVE
        # ----------------------------------------------------

        step2_direct_removed = int(
            np.count_nonzero(step2_remove_mask)
        )

        A[step2_remove_mask] = NODATA

        # ----------------------------------------------------
        # NEAREST
        # ----------------------------------------------------

        step2_replaced, step2_failed = replace_by_nearest(
            A,
            step2_nearest_mask,
            step2_donors,
            MAX_NEAREST_DISTANCE_PIXELS
        )

        step2_actual_removed = (
            step2_direct_removed
            + step2_failed
        )

        valid_mask = get_valid_mask(A)

        # ====================================================
        # STEP 3 - COUNTRY STANDARD DEVIATION
        # ====================================================

        # Reported identified/selected count
        step3_total_removed = 0

        # Actual deletion count
        step3_actual_removed = 0

        # Successful replacement count
        step3_total_replaced = 0

        if APPLY_STD_REMOVAL:

            if gdf.crs != crs:
                gdf_reproj = gdf.to_crs(crs)
            else:
                gdf_reproj = gdf

            # Preserve M49 country processing order
            for _, country_row in gdf_reproj.iterrows():

                m49 = country_row[M49_FIELD]

                name = (
                    country_row[NAME_FIELD]
                    if NAME_FIELD in country_row
                    else str(m49)
                )

                geom = country_row.geometry

                if geom is None or geom.is_empty:

                    country_records.append(
                        empty_country_record(
                            year, m49, name
                        )
                    )

                    continue

                # --------------------------------------------
                # Country raster mask
                # --------------------------------------------

                try:

                    country_mask_arr = ~geometry_mask(
                        [geom],
                        transform=transform,
                        invert=False,
                        out_shape=(height, width)
                    )

                except Exception:

                    country_records.append(
                        empty_country_record(
                            year, m49, name
                        )
                    )

                    continue

                # --------------------------------------------
                # Country valid pixels
                # --------------------------------------------

                in_country_valid = (
                    country_mask_arr
                    & valid_mask
                )

                pixel_indices = np.where(
                    in_country_valid
                )

                n_valid = len(pixel_indices[0])

                if n_valid == 0:

                    country_records.append(
                        empty_country_record(
                            year, m49, name
                        )
                    )

                    continue

                # --------------------------------------------
                # Mean and standard deviation
                # --------------------------------------------

                vals = A[pixel_indices]

                mean_val = float(
                    np.mean(
                        vals,
                        dtype=np.float64
                    )
                )

                std_val = float(
                    np.std(
                        vals,
                        dtype=np.float64
                    )
                )

                upper_sigma = (
                    mean_val
                    + STD_FACTOR * std_val
                )

                # --------------------------------------------
                # Identify Step 3 candidates
                # --------------------------------------------

                candidate_mask = (
                    in_country_valid
                    & (A > upper_sigma)
                )

                n_candidate = int(
                    np.count_nonzero(candidate_mask)
                )

                max_allowed = int(
                    np.floor(
                        n_valid * MAX_REMOVE_RATIO
                    )
                )

                # --------------------------------------------
                # Select outliers
                #
                # Same 5% cap and descending-value rule
                # as the original script
                # --------------------------------------------

                selected_mask = np.zeros_like(
                    valid_mask,
                    dtype=bool
                )

                if n_candidate <= max_allowed:

                    selected_mask = candidate_mask

                    actual_selected = n_candidate

                else:

                    actual_selected = max_allowed

                    if max_allowed > 0:

                        cand_rows, cand_cols = np.where(
                            candidate_mask
                        )

                        cand_vals = A[
                            cand_rows,
                            cand_cols
                        ]

                        sort_idx = np.argsort(
                            -cand_vals
                        )[:max_allowed]

                        selected_mask[
                            cand_rows[sort_idx],
                            cand_cols[sort_idx]
                        ] = True

                # --------------------------------------------
                # NEW: Step 3 action
                # --------------------------------------------

                if STD_OUTLIER_ACTION == "REMOVE":

                    A[selected_mask] = NODATA

                    country_replaced = 0
                    country_actual_removed = actual_selected

                else:

                    # Nearest-neighbor donors:
                    # 1. Inside the same country
                    # 2. Valid after Step 2
                    # 3. Not above the country's sigma limit
                    #
                    # Exclude ALL values above sigma as donors,
                    # not just selected outliers.

                    country_donors = (
                        in_country_valid
                        & (A <= upper_sigma)
                    )

                    (
                        country_replaced,
                        country_actual_removed
                    ) = replace_by_nearest(
                        A,
                        selected_mask,
                        country_donors,
                        MAX_NEAREST_DISTANCE_PIXELS
                    )

                # --------------------------------------------
                # Update global raster valid mask
                # --------------------------------------------

                valid_mask = get_valid_mask(A)

                # Preserve existing reported meaning:
                # Step3_removed = selected outlier count
                # even when action is NEAREST.

                step3_total_removed += actual_selected

                step3_actual_removed += (
                    country_actual_removed
                )

                step3_total_replaced += (
                    country_replaced
                )

                # --------------------------------------------
                # Original country Excel fields
                # --------------------------------------------

                country_records.append({
                    "Year": year,
                    "M49": m49,
                    "Country": name,
                    "Valid_before_step3": n_valid,
                    "Mean": round(mean_val, 6),
                    "Std": round(std_val, 6),
                    "Upper_sigma": round(
                        upper_sigma, 6
                    ),
                    "Candidate_removed": n_candidate,
                    "Max_allowed_remove": max_allowed,
                    "Step3_removed": actual_selected,
                    "Step3_removed_%": (
                        round(
                            actual_selected / n_valid * 100,
                            4
                        )
                        if n_valid > 0 else 0.0
                    )
                })

        # ====================================================
        # 6. ANNUAL SUMMARY
        # ====================================================

        final_count = int(
            np.count_nonzero(
                get_valid_mask(A)
            )
        )

        total_removed = (
            initial_count - final_count
        )

        removed_ratio = (
            total_removed / initial_count * 100
            if initial_count > 0 else 0
        )

        # True number of deleted pixels:
        # replacement does NOT count as actual deletion.

        expected_actual_removed = (
            step1_removed
            + step2_actual_removed
            + step3_actual_removed
        )

        count_check = (
            total_removed
            - expected_actual_removed
        )

        # Original Annual Summary column structure
        annual_record = {
            "Year": year,
            "Initial_Valid": initial_count,
            "Step1_removed": step1_removed,
            "Step1_%": (
                round(
                    step1_removed / initial_count * 100,
                    4
                )
                if initial_count else 0
            ),
            "Step2_removed": step2_removed,
            "Step2_%": (
                round(
                    step2_removed / initial_count * 100,
                    4
                )
                if initial_count else 0
            ),
            "Step3_removed": step3_total_removed,
            "Step3_%": (
                round(
                    step3_total_removed / initial_count * 100,
                    4
                )
                if initial_count else 0
            ),
            "Final_Valid": final_count,
            "Total_Removed": total_removed,
            "Removed_Ratio_%": round(
                removed_ratio, 4
            )
        }

        # ====================================================
        # 7. EXPORT GEOTIFF
        # ====================================================

        profile.update(
            dtype="float32",
            nodata=NODATA,
            compress="lzw"
        )

        with rasterio.open(
            OUT_tif,
            "w",
            **profile
        ) as dst:

            dst.write(
                A.astype(np.float32),
                1
            )

        # ----------------------------------------------------
        # Return results to main process
        # ----------------------------------------------------

        return {
            "year": year,
            "status": "SUCCESS",
            "message": "",
            "annual": annual_record,
            "countries": country_records,
            "check": count_check,
            "step2_replaced": step2_replaced,
            "step3_replaced": step3_total_replaced,
            "actual_removed": total_removed,
            "output": OUT_tif
        }

    except Exception as e:

        return {
            "year": year,
            "status": "FAILED",
            "message": repr(e),
            "annual": None,
            "countries": []
        }


# ============================================================
# 8. PARALLEL PROCESSING BY YEAR
# ============================================================

YEARS = (
    list(range(START_YEAR, END_YEAR + 1))
    if YEARS_TO_RUN is None
    else sorted(set(map(int, YEARS_TO_RUN)))
)

if not YEARS:
    raise ValueError("No years selected")

print("=" * 70)
print("GLOBAL NUE OUTLIER PROCESSING")
print("=" * 70)

print(f"Years: {YEARS}")
print(f"Parallel workers: {PARALLEL_YEARS}")

print(f"Step 1 MIN_COUNT: {MIN_COUNT}")

print(
    f"Step 2 Upper: {UPPER_HARD} "
    f"({UPPER_OUTLIER_ACTION})"
)

print(
    f"Step 2 Lower: {LOWER_HARD} "
    f"({LOWER_OUTLIER_ACTION})"
)

print(
    f"Step 3 Enabled: {APPLY_STD_REMOVAL}"
)

print(
    f"Step 3 Action: {STD_OUTLIER_ACTION}"
)

print(
    f"Step 3 STD_FACTOR: {STD_FACTOR}"
)

print("=" * 70, flush=True)


if PARALLEL_YEARS == 1 or len(YEARS) == 1:

    results = [
        process_one_year(year)
        for year in YEARS
    ]

else:

    results = Parallel(
        n_jobs=min(
            PARALLEL_YEARS,
            len(YEARS)
        ),
        backend="loky",
        batch_size=1,
        pre_dispatch="n_jobs",
        verbose=10
    )(
        delayed(process_one_year)(year)
        for year in YEARS
    )


# ============================================================
# 9. COLLECT PARALLEL RESULTS
# ============================================================

results.sort(
    key=lambda item: item["year"]
)

records_year = []
records_country = []

for result in results:

    year = result["year"]
    status = result["status"]

    if status == "SUCCESS":

        records_year.append(
            result["annual"]
        )

        records_country.extend(
            result["countries"]
        )

        a = result["annual"]

        print(
            f"[{year}] "
            f"Initial={a['Initial_Valid']:,} | "
            f"Step1={a['Step1_removed']:,} | "
            f"Step2={a['Step2_removed']:,} | "
            f"Step3={a['Step3_removed']:,} | "
            f"Step2_Replaced={result['step2_replaced']:,} | "
            f"Step3_Replaced={result['step3_replaced']:,} | "
            f"Final={a['Final_Valid']:,} | "
            f"Check={result['check']}"
        )

    else:

        print(
            f"[{year}] {status}: "
            f"{result['message']}"
        )


# ============================================================
# 10. EXCEL REPORT
# ============================================================

COUNTRY_COLS = [
    "Year",
    "M49",
    "Country",
    "Valid_before_step3",
    "Mean",
    "Std",
    "Upper_sigma",
    "Candidate_removed",
    "Max_allowed_remove",
    "Step3_removed",
    "Step3_removed_%"
]

if not records_year:

    print(
        "\nNo successful years. Excel not generated."
    )

else:

    df_year = (
        pd.DataFrame(records_year)
        .sort_values("Year")
        .reset_index(drop=True)
    )

    df_country = pd.DataFrame(
        records_country,
        columns=COUNTRY_COLS
    )

    if not df_country.empty:

        df_country = (
            df_country
            .sort_values(["Year", "M49"])
            .reset_index(drop=True)
        )

    wb = Workbook()

    thin = Side(
        style="thin",
        color="AAAAAA"
    )

    border = Border(
        left=thin,
        right=thin,
        top=thin,
        bottom=thin
    )

    # --------------------------------------------------------
    # Excel formatting helpers
    # --------------------------------------------------------

    def hdr_cell(
        ws, row, col, value,
        bg="2F75B6",
        fg="FFFFFF"
    ):

        c = ws.cell(
            row=row,
            column=col,
            value=value
        )

        c.font = Font(
            bold=True,
            color=fg,
            name="Arial",
            size=10
        )

        c.fill = PatternFill(
            "solid",
            start_color=bg
        )

        c.alignment = Alignment(
            horizontal="center",
            wrap_text=True
        )

        c.border = border

        return c


    def data_cell(
        ws, row, col, value,
        bg="FFFFFF",
        fmt=None
    ):

        if isinstance(value, np.generic):
            value = value.item()

        if (
            isinstance(value, float)
            and not np.isfinite(value)
        ):
            value = None

        c = ws.cell(
            row=row,
            column=col,
            value=value
        )

        c.font = Font(
            name="Arial",
            size=10
        )

        c.fill = PatternFill(
            "solid",
            start_color=bg
        )

        c.alignment = Alignment(
            horizontal="center"
        )

        c.border = border

        if fmt:
            c.number_format = fmt

        return c


    # ========================================================
    # SHEET 1: ANNUAL SUMMARY
    # ========================================================

    ws1 = wb.active

    ws1.title = "Annual Summary"

    col_colors_year = {
        "Year": "FFFFFF",
        "Initial_Valid": "FFFFFF",
        "Step1_removed": "D6E4F0",
        "Step1_%": "D6E4F0",
        "Step2_removed": "D5E8D4",
        "Step2_%": "D5E8D4",
        "Step3_removed": "FFF2CC",
        "Step3_%": "FFF2CC",
        "Final_Valid": "FFFFFF",
        "Total_Removed": "F4CCCC",
        "Removed_Ratio_%": "F4CCCC"
    }

    cols_y = list(df_year.columns)

    for ci, col in enumerate(cols_y, 1):
        hdr_cell(ws1, 1, ci, col)

    for ri, row in enumerate(
        df_year.itertuples(index=False),
        2
    ):

        for ci, (col, val) in enumerate(
            zip(cols_y, row),
            1
        ):

            fmt = (
                "0.00"
                if "%" in col
                else (
                    "#,##0"
                    if col != "Year"
                    else "0"
                )
            )

            data_cell(
                ws1,
                ri,
                ci,
                val,
                bg=col_colors_year.get(
                    col,
                    "FFFFFF"
                ),
                fmt=fmt
            )

    # Original TOTAL/AVG row
    sr = len(df_year) + 2

    data_cell(
        ws1,
        sr,
        1,
        "TOTAL/AVG",
        bg="D9D9D9"
    ).font = Font(
        bold=True,
        name="Arial",
        size=10
    )

    for ci, col in enumerate(
        cols_y[1:],
        2
    ):

        cl = get_column_letter(ci)

        rng = (
            f"{cl}2:"
            f"{cl}{len(df_year) + 1}"
        )

        formula = (
            f"=AVERAGE({rng})"
            if "%" in col
            else f"=SUM({rng})"
        )

        c = ws1.cell(
            row=sr,
            column=ci,
            value=formula
        )

        c.font = Font(
            bold=True,
            name="Arial",
            size=10
        )

        c.fill = PatternFill(
            "solid",
            start_color="D9D9D9"
        )

        c.alignment = Alignment(
            horizontal="center"
        )

        c.border = border

        c.number_format = (
            "0.00"
            if "%" in col
            else "#,##0"
        )

    for ci, col in enumerate(cols_y, 1):

        ws1.column_dimensions[
            get_column_letter(ci)
        ].width = max(
            len(col) + 4,
            14
        )

    ws1.row_dimensions[1].height = 36
    ws1.freeze_panes = "B2"

    # ========================================================
    # SHEET 2: COUNTRY DETAIL
    # ========================================================

    ws2 = wb.create_sheet(
        "Country Detail"
    )

    cols_c = COUNTRY_COLS

    col_colors_cty = {
        "Year": "FFFFFF",
        "M49": "FFFFFF",
        "Country": "FFFFFF",
        "Valid_before_step3": "FFFFFF",
        "Mean": "EAD1DC",
        "Std": "EAD1DC",
        "Upper_sigma": "EAD1DC",
        "Candidate_removed": "FFF2CC",
        "Max_allowed_remove": "FFF2CC",
        "Step3_removed": "F4CCCC",
        "Step3_removed_%": "F4CCCC"
    }

    for ci, col in enumerate(cols_c, 1):
        hdr_cell(ws2, 1, ci, col)

    for ri, row in enumerate(
        df_country.itertuples(index=False),
        2
    ):

        for ci, (col, val) in enumerate(
            zip(cols_c, row),
            1
        ):

            if "%" in col:
                fmt = "0.00"

            elif col in (
                "Mean",
                "Std",
                "Upper_sigma"
            ):
                fmt = "0.000000"

            elif col not in (
                "Year",
                "M49",
                "Country"
            ):
                fmt = "#,##0"

            else:
                fmt = None

            data_cell(
                ws2,
                ri,
                ci,
                val,
                bg=col_colors_cty.get(
                    col,
                    "FFFFFF"
                ),
                fmt=fmt
            )

    for ci, col in enumerate(cols_c, 1):

        ws2.column_dimensions[
            get_column_letter(ci)
        ].width = max(
            len(col) + 3,
            12
        )

    ws2.column_dimensions["C"].width = 22
    ws2.row_dimensions[1].height = 36
    ws2.freeze_panes = "D2"

    # ========================================================
    # SHEET 3: PARAMETERS
    # ========================================================

    ws3 = wb.create_sheet(
        "Parameters"
    )

    params = [
        ("Parameter", "Value"),
        ("MIN_COUNT (Step1)", MIN_COUNT),
        ("LOWER_HARD (Step2)", LOWER_HARD),
        ("UPPER_HARD (Step2)", UPPER_HARD),
        ("LOWER_OUTLIER_ACTION", LOWER_OUTLIER_ACTION),
        ("UPPER_OUTLIER_ACTION", UPPER_OUTLIER_ACTION),
        ("APPLY_STD_REMOVAL", APPLY_STD_REMOVAL),
        ("STD_FACTOR (Step3)", STD_FACTOR),
        ("STD_OUTLIER_ACTION", STD_OUTLIER_ACTION),
        ("MAX_REMOVE_RATIO (Step3)", MAX_REMOVE_RATIO),
        (
            "MAX_NEAREST_DISTANCE_PIXELS",
            "None" if MAX_NEAREST_DISTANCE_PIXELS is None
            else MAX_NEAREST_DISTANCE_PIXELS
        ),
        ("M49_FIELD", M49_FIELD),
        ("SHP_PATH", SHP_PATH),
        ("START_YEAR", START_YEAR),
        ("END_YEAR", END_YEAR),
        ("NODATA", NODATA),
        ("PARALLEL_YEARS", PARALLEL_YEARS),
        (
            "YEARS_TO_RUN",
            "All" if YEARS_TO_RUN is None
            else str(YEARS)
        )
    ]

    for ri, (k, v) in enumerate(
        params,
        1
    ):

        for ci, val in enumerate(
            (k, v),
            1
        ):

            c = ws3.cell(
                row=ri,
                column=ci,
                value=val
            )

            c.font = Font(
                bold=(ri == 1),
                name="Arial",
                size=10,
                color=(
                    "FFFFFF"
                    if ri == 1
                    else "000000"
                )
            )

            c.fill = PatternFill(
                "solid",
                start_color=(
                    "2F75B6"
                    if ri == 1
                    else "F5F5F5"
                )
            )

            c.border = border

            c.alignment = Alignment(
                horizontal="left"
            )

    ws3.column_dimensions["A"].width = 38
    ws3.column_dimensions["B"].width = 60

    # --------------------------------------------------------
    # Save Excel
    # --------------------------------------------------------

    wb.save(EXCEL_OUT)

    print(
        f"\nExcel saved: {EXCEL_OUT}"
    )

    # ========================================================
    # 11. FINAL VALIDATION
    # ========================================================

    check_rows = []

    for result in results:

        if result["status"] == "SUCCESS":

            check_rows.append({
                "Year": result["year"],
                "Step2_detected": (
                    result["annual"]["Step2_removed"]
                ),
                "Step3_detected": (
                    result["annual"]["Step3_removed"]
                ),
                "Step2_replaced": (
                    result["step2_replaced"]
                ),
                "Step3_replaced": (
                    result["step3_replaced"]
                ),
                "Actual_removed": (
                    result["actual_removed"]
                ),
                "Check": result["check"]
            })

    df_check = pd.DataFrame(
        check_rows
    )

    print("\n" + "=" * 70)
    print("FINAL VALIDATION SUMMARY")
    print("=" * 70)

    print(
        df_check.to_string(
            index=False
        )
    )

    if (df_check["Check"] == 0).all():

        print(
            "\nPASS: All annual pixel counts are consistent."
        )

    else:

        print(
            "\nWARNING: Pixel count inconsistency detected."
        )

print("\nAll requested years finished.")

In [ ]:
# =====================================================
# -------Global Maize NUE---Based on the weighted median-------------
# ---Step 1----Setting parameters----------
# ---Step 2--- Run ----------------
# ---Step 3--- simple validation --------------
# ---Step 4--- Match with harvested area, mask
# ---Step 5--- Remove outliers, Useful tool,Pixels number should larger than 6, NUE more then 150; set non, 
# ---Step 6--- Check mean value
# ---Step 7--- Extract country mean NUE value--Useful---
# ---Step 8--- Extract county mean NUE value--China and US---
# ---Step 9--- Extract county mean PFPN value---
# ===============2026.10.28 ==========PFPN3M===================================
# Survey County-level PFPN Validation
# Method 1:
#   PA_NUE = arithmetic mean of gridded PFPN
# Method 2:
#   PA_NUE_AreaWeighted =
#       SUM(PFPN * HA) / SUM(HA)
# Method 3 (NEW):
#   PA_PFPN_RatioOfMeans =
#       SUM(Yield * HA) / SUM(Nfert * HA)
#
# Common pixel mask:
#   Yield valid and >= 0
#   Nfert valid and >= 0
#   Harvested Area valid and > 0
#   Optional: PFPN raster must also be valid
#
# One JupyterLab Cell
# ============================================================

import os
import glob
import re
import time

from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

from rasterio.mask import mask
from IPython.display import display


# ============================================================
# 1. USER CONFIGURATION
# ============================================================

excel_file = (
    r"D:\1_PrecisionAgriculture\1_Researching\05_Dataset_GlobalNUE\10_FieldSurveyData\SurveyDataForNUEValidation20260325.xlsx"
)

shp_root = (
    r"D:\1_PrecisionAgriculture\1_Researching"
    r"\04_Dataset_Classification\GDAM_Level_0_1_2_3\gadm41_Global"
)

# Existing PFPN/NUE GeoTIFF directory
# Contains wheat / maize / rice subfolders

tif_root = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\09CropResults4PFPNValidation\PFPN"
)

# ------------------------------------------------------------
# Harvested Area root
# ------------------------------------------------------------

area_root = (
    r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\GCSN_HarvestedArea_1995_2020\02StandardizedDatasets"
)

# ------------------------------------------------------------
# NEW: Yield and Nfert GeoTIFF roots
# CHANGE to your actual data folders.
# Crop subfolders are supported.
# ------------------------------------------------------------

yield_root = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\09CropResults4PFPNValidation\Yield"
)

nfert_root = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\09CropResults4PFPNValidation\NFertilizer"
)

# ------------------------------------------------------------
# Yield source
#
# "DIRECT":
#   Read Yield raster directly.
#
# "FROM_PFPN":
#   Derive Yield = PFPN * Nfert.
#   Does not require yield_root.
# ------------------------------------------------------------

YIELD_SOURCE = "DIRECT"

# ------------------------------------------------------------
# File name patterns
#
# Supported examples:
#   Wheat_HarArea_1995.tif
#   Maize_Yield_1995.tif
#   Rice_Nfert_1995.tif
#
# Modify regex for different filenames.
# ------------------------------------------------------------

AREA_FILENAME_REGEX = (
    r"^(?P<crop>wheat|maize|rice)_HarArea_"
    r"(?P<year>(?:19|20)\d{2})\.(?:tif|tiff)$"
)

YIELD_FILENAME_REGEX = (
    r"^(?P<crop>wheat|maize|rice)_"
    r"(?:Yield|Yld)_"
    r"(?P<year>(?:19|20)\d{2})\.(?:tif|tiff)$"
)

NFERT_FILENAME_REGEX = (
    r"^(?P<crop>wheat|maize|rice)_"
    r"(?:Fertilizer|Nrate|NFertilizer)_"
    r"(?P<year>(?:19|20)\d{2})\.(?:tif|tiff)$"
)

# ------------------------------------------------------------
# Output Excel
# ------------------------------------------------------------

output_excel = os.path.join(
    tif_root,
    "PFPN4Validation4_WithRatioOfMeans.xlsx"
)

# ------------------------------------------------------------
# NoData settings
# ------------------------------------------------------------

nodata_value = -9999.0

ADDITIONAL_NODATA = []

# ------------------------------------------------------------
# Zonal mask settings
# ------------------------------------------------------------

ALL_TOUCHED = False

RASTER_BAND = 1

AREA_THRESHOLD = 0.0

# ------------------------------------------------------------
# NEW: Pixel-selection option
#
# True:
#   NEW ratio also requires original PFPN raster
#   to have a valid value in the same pixel.
#
# False:
#   NEW ratio uses only Yield, Nfert and HA.
#
# Recommended True for comparison with existing
# PFPN raster products on a common spatial support.
# ------------------------------------------------------------

REQUIRE_PFPN_VALID_FOR_RATIO = True

# ------------------------------------------------------------
# Yield unit conversion
#
# If Yield raster unit = kg/ha: 1.0
# If Yield raster unit = t/ha: 1000.0
#
# Used only for DIRECT Yield source.
# Nfert should be kg N/ha.
# ------------------------------------------------------------

YIELD_TO_KG_PER_HA = 1.0


# ============================================================
# 2. VALIDATION
# ============================================================

start_time = time.perf_counter()

YIELD_SOURCE = YIELD_SOURCE.upper()

if YIELD_SOURCE not in {
    "DIRECT",
    "FROM_PFPN"
}:
    raise ValueError(
        "YIELD_SOURCE must be DIRECT or FROM_PFPN"
    )

if YIELD_TO_KG_PER_HA <= 0:
    raise ValueError(
        "YIELD_TO_KG_PER_HA must be positive"
    )

if not os.path.isfile(excel_file):
    raise FileNotFoundError(excel_file)

for folder, label in [
    (shp_root, "SHP root"),
    (tif_root, "PFPN root"),
    (area_root, "Harvested Area root"),
    (nfert_root, "Nfert root")
]:
    if not os.path.isdir(folder):
        raise FileNotFoundError(
            f"{label} not found:\n{folder}"
        )

if YIELD_SOURCE == "DIRECT":

    if not os.path.isdir(yield_root):
        raise FileNotFoundError(
            f"Yield root not found:\n{yield_root}"
        )

os.makedirs(
    os.path.dirname(output_excel),
    exist_ok=True
)


# ============================================================
# 3. READ SURVEY DATA
# ============================================================

df = pd.read_excel(excel_file)

required_cols = [
    "m49",
    "Crop",
    "Year"
]

for col in required_cols:
    if col not in df.columns:
        raise ValueError(
            f"Missing Excel column: {col}"
        )

df["Year"] = pd.to_numeric(
    df["Year"],
    errors="coerce"
)

df = df.loc[
    df["Year"].notna()
].copy()

df["Year"] = df["Year"].astype(int)

for col in ["Level_1", "Level_2"]:
    if col not in df.columns:
        df[col] = np.nan

print("=" * 75)
print("SURVEY COUNTY-LEVEL PFPN VALIDATION")
print("=" * 75)

print(f"Survey rows: {len(df):,}")
print(f"Yield source: {YIELD_SOURCE}")
print(
    "Require PFPN valid for ratio:",
    REQUIRE_PFPN_VALID_FOR_RATIO
)


# ============================================================
# 4. INDEX ANNUAL RASTERS
# ============================================================

def build_raster_index(folder, filename_regex):
    """
    Return:
      {(crop, year): GeoTIFF path}
    """

    pattern = re.compile(
        filename_regex,
        re.IGNORECASE
    )

    index = {}

    for root, _, filenames in os.walk(folder):

        for filename in filenames:

            match = pattern.fullmatch(filename)

            if match is None:
                continue

            crop = match.group("crop").lower()
            year = int(match.group("year"))

            key = (crop, year)

            path = os.path.join(
                root,
                filename
            )

            if key in index:
                raise ValueError(
                    f"Duplicate files for {key}:\n"
                    f"{index[key]}\n{path}"
                )

            index[key] = path

    return index


area_index = build_raster_index(
    area_root,
    AREA_FILENAME_REGEX
)

nfert_index = build_raster_index(
    nfert_root,
    NFERT_FILENAME_REGEX
)

if YIELD_SOURCE == "DIRECT":

    yield_index = build_raster_index(
        yield_root,
        YIELD_FILENAME_REGEX
    )

else:

    yield_index = {}

print(f"HA files: {len(area_index)}")
print(f"Nfert files: {len(nfert_index)}")
print(f"Yield files: {len(yield_index)}")


# ============================================================
# 5. UTILITY: GET VALID PIXEL MASK
# ============================================================

def get_valid_values_mask(masked_array):

    ma = np.ma.asarray(masked_array)

    values = np.asarray(
        np.ma.getdata(ma),
        dtype=np.float64
    )

    valid = (
        ~np.ma.getmaskarray(ma)
    ).copy()

    valid &= np.isfinite(values)
    valid &= (values != nodata_value)

    for nd in ADDITIONAL_NODATA:
        valid &= (values != nd)

    return values, valid


# ============================================================
# 6. UTILITY: GRID ALIGNMENT
# ============================================================

class GridAlignmentError(ValueError):
    pass


def verify_raster_alignment(
    reference,
    other,
    label
):

    if reference.crs is None or other.crs is None:
        raise GridAlignmentError(
            f"{label}: CRS missing"
        )

    if reference.crs != other.crs:
        raise GridAlignmentError(
            f"{label}: CRS mismatch"
        )

    if (
        reference.width != other.width
        or reference.height != other.height
    ):
        raise GridAlignmentError(
            f"{label}: Raster size mismatch"
        )

    if not reference.transform.almost_equals(
        other.transform,
        precision=1e-9
    ):
        raise GridAlignmentError(
            f"{label}: Pixel grid mismatch"
        )


def read_region(
    src,
    geometries,
    reference_transform=None,
    reference_shape=None
):
    """
    Read exactly the same polygon region.

    Returns:
      values
      valid mask
      cropped transform
    """

    mini, transform = mask(
        src,
        geometries,
        crop=True,
        filled=False,
        indexes=RASTER_BAND,
        all_touched=ALL_TOUCHED
    )

    if reference_shape is not None:

        if mini.shape != reference_shape:
            raise GridAlignmentError(
                "Cropped raster shapes differ"
            )

    if reference_transform is not None:

        if not transform.almost_equals(
            reference_transform,
            precision=1e-9
        ):
            raise GridAlignmentError(
                "Cropped raster transforms differ"
            )

    values, valid = get_valid_values_mask(
        mini
    )

    return values, valid, transform


# ============================================================
# 7. SHP AND PFPN FILE CACHE
# ============================================================

shp_cache = {}
pfpn_file_cache = {}


def get_country_shapefile(m49, level):

    key = (m49, level)

    if key in shp_cache:
        return shp_cache[key]

    country_dirs = sorted(
        path
        for path in glob.glob(
            os.path.join(
                shp_root,
                f"*_{m49}"
            )
        )
        if os.path.isdir(path)
    )

    if not country_dirs:
        raise FileNotFoundError(
            f"No country SHP for M49={m49}"
        )

    matches = sorted(
        glob.glob(
            os.path.join(
                country_dirs[0],
                f"gadm41_*_{level}.shp"
            )
        )
    )

    if not matches:
        raise FileNotFoundError(
            f"No SHP level {level}, M49={m49}"
        )

    gdf = gpd.read_file(matches[0])

    if gdf.crs is None:
        raise ValueError(
            f"Missing SHP CRS: {matches[0]}"
        )

    shp_cache[key] = gdf

    return gdf


def get_pfpn_file(crop, year):

    key = (crop, year)

    if key in pfpn_file_cache:
        return pfpn_file_cache[key]

    crop_dir = os.path.join(
        tif_root,
        crop
    )

    matches = sorted(
        glob.glob(
            os.path.join(
                crop_dir,
                f"*_{year}.tif"
            )
        )
        +
        glob.glob(
            os.path.join(
                crop_dir,
                f"*_{year}.tiff"
            )
        )
    )

    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one PFPN GeoTIFF "
            f"for {crop}, {year}. "
            f"Found: {len(matches)}"
        )

    pfpn_file_cache[key] = matches[0]

    return matches[0]


# ============================================================
# 8. OUTPUT FIELDS
# ============================================================

output_fields = [
    "PA_NUE",
    "PA_NUE_AreaWeighted",
    "N_Common_Pixels",
    "Harvested_Area_Used",

    # NEW
    "PA_PFPN_RatioOfMeans",
    "N_Ratio_Common_Pixels",
    "Mean_Yield_HA",
    "Mean_Nfert_HA",
    "Sum_Yield_x_HA",
    "Sum_Nfert_x_HA",
    "HA_Used_Ratio"
]

results = []


# ============================================================
# 9. MAIN SURVEY ROW LOOP
# ============================================================

for row_num, (idx, row) in enumerate(
    df.iterrows(),
    start=1
):

    result = {
        field: np.nan
        for field in output_fields
    }

    try:

        # ----------------------------------------------------
        # 9.1 Read survey record
        # ----------------------------------------------------

        m49 = str(
            int(float(row["m49"]))
        )

        crop = str(
            row["Crop"]
        ).strip().lower()

        year = int(
            row["Year"]
        )

        level1 = row["Level_1"]
        level2 = row["Level_2"]

        if crop not in {
            "wheat", "maize", "rice"
        }:
            raise ValueError(
                f"Unsupported crop: {crop}"
            )

        print(
            f"[{row_num}/{len(df)}] "
            f"M49={m49}, {crop}, {year}",
            flush=True
        )

        # ----------------------------------------------------
        # 9.2 Determine SHP administrative level
        # ----------------------------------------------------

        if pd.notna(level2):
            level = 2

        elif pd.notna(level1):
            level = 1

        else:
            level = 0

        # ----------------------------------------------------
        # 9.3 Select polygon
        # ----------------------------------------------------

        country_gdf = get_country_shapefile(
            m49,
            level
        )

        region = country_gdf

        if level >= 1 and pd.notna(level1):

            if "NAME_1" not in region.columns:
                raise ValueError(
                    "NAME_1 missing in SHP"
                )

            region = region.loc[
                region["NAME_1"].astype(str).str.strip()
                == str(level1).strip()
            ]

        if level == 2:

            if "NAME_2" not in region.columns:
                raise ValueError(
                    "NAME_2 missing in SHP"
                )

            region = region.loc[
                region["NAME_2"].astype(str).str.strip()
                == str(level2).strip()
            ]

        region = region.loc[
            region.geometry.notna()
            & ~region.geometry.is_empty
        ]

        if region.empty:
            raise ValueError(
                "Region not found in SHP"
            )

        # ----------------------------------------------------
        # 9.4 Match annual rasters
        # ----------------------------------------------------

        pfpn_file = get_pfpn_file(
            crop,
            year
        )

        area_file = area_index.get(
            (crop, year)
        )

        nfert_file = nfert_index.get(
            (crop, year)
        )

        yield_file = (
            yield_index.get((crop, year))
            if YIELD_SOURCE == "DIRECT"
            else None
        )

        # ----------------------------------------------------
        # 9.5 Read existing PFPN raster
        # ----------------------------------------------------

        with rasterio.open(pfpn_file) as pfpn_src:

            if pfpn_src.crs is None:
                raise ValueError(
                    "PFPN raster CRS missing"
                )

            if region.crs != pfpn_src.crs:

                region_projected = region.to_crs(
                    pfpn_src.crs
                )

            else:

                region_projected = region

            geometries = (
                region_projected.geometry.tolist()
            )

            (
                pfpn_values,
                pfpn_valid,
                crop_transform
            ) = read_region(
                pfpn_src,
                geometries
            )

            crop_shape = pfpn_values.shape

            # ================================================
            # METHOD 1: Original arithmetic PFPN mean
            # ================================================

            if np.any(pfpn_valid):

                result["PA_NUE"] = float(
                    np.mean(
                        pfpn_values[pfpn_valid],
                        dtype=np.float64
                    )
                )

            # ================================================
            # Read Harvested Area
            # ================================================

            if area_file is None:

                print(
                    "  WARNING: Harvested Area missing"
                )

                results.append(result)
                continue

            with rasterio.open(area_file) as area_src:

                verify_raster_alignment(
                    pfpn_src,
                    area_src,
                    "PFPN / HA"
                )

                (
                    area_values,
                    area_valid,
                    _
                ) = read_region(
                    area_src,
                    geometries,
                    reference_transform=crop_transform,
                    reference_shape=crop_shape
                )

            # ================================================
            # METHOD 2: Original HA-weighted PFPN
            # ================================================

            common_area = (
                pfpn_valid
                & area_valid
                & (area_values > AREA_THRESHOLD)
            )

            n_area = int(
                np.count_nonzero(
                    common_area
                )
            )

            result["N_Common_Pixels"] = n_area

            result["Harvested_Area_Used"] = 0.0

            if n_area > 0:

                pfpn_a = pfpn_values[
                    common_area
                ]

                area_a = area_values[
                    common_area
                ]

                ha_sum = float(
                    np.sum(
                        area_a,
                        dtype=np.float64
                    )
                )

                result["Harvested_Area_Used"] = ha_sum

                if ha_sum > 0:

                    result["PA_NUE_AreaWeighted"] = (
                        float(
                            np.sum(
                                pfpn_a * area_a,
                                dtype=np.float64
                            )
                        )
                        / ha_sum
                    )

            # ================================================
            # METHOD 3: NEW Ratio of Means
            # ================================================

            if nfert_file is None:

                print(
                    "  WARNING: Nfert file missing"
                )

                results.append(result)
                continue

            # ------------------------------------------------
            # Read Nfert (kg N/ha)
            # ------------------------------------------------

            with rasterio.open(nfert_file) as nfert_src:

                verify_raster_alignment(
                    pfpn_src,
                    nfert_src,
                    "PFPN / Nfert"
                )

                (
                    nfert_values,
                    nfert_valid,
                    _
                ) = read_region(
                    nfert_src,
                    geometries,
                    reference_transform=crop_transform,
                    reference_shape=crop_shape
                )

            # ------------------------------------------------
            # Read Yield or derive Yield
            # ------------------------------------------------

            if YIELD_SOURCE == "DIRECT":

                if yield_file is None:

                    print(
                        "  WARNING: Yield file missing"
                    )

                    results.append(result)
                    continue

                with rasterio.open(yield_file) as yield_src:

                    verify_raster_alignment(
                        pfpn_src,
                        yield_src,
                        "PFPN / Yield"
                    )

                    (
                        yield_values,
                        yield_valid,
                        _
                    ) = read_region(
                        yield_src,
                        geometries,
                        reference_transform=crop_transform,
                        reference_shape=crop_shape
                    )

                # Convert t/ha to kg/ha if needed
                yield_values = (
                    yield_values
                    * YIELD_TO_KG_PER_HA
                )

            else:

                # Yield = PFPN * Nfert
                # Valid only if gridded PFPN has units
                # kg grain / kg N and Nfert is kg N/ha.

                yield_valid = (
                    pfpn_valid
                    & nfert_valid
                )

                yield_values = np.zeros_like(
                    pfpn_values,
                    dtype=np.float64
                )

                yield_values[yield_valid] = (
                    pfpn_values[yield_valid]
                    * nfert_values[yield_valid]
                )

            # ------------------------------------------------
            # Common valid pixels for new ratio
            # ------------------------------------------------

            ratio_mask = (
                yield_valid
                & np.isfinite(yield_values)
                & (yield_values >= 0)
                & nfert_valid
                & (nfert_values >= 0)
                & area_valid
                & (area_values > AREA_THRESHOLD)
            )

            # Optional PFPN core mask
            if REQUIRE_PFPN_VALID_FOR_RATIO:

                ratio_mask &= pfpn_valid

            # ------------------------------------------------
            # Extract matched pixels
            # ------------------------------------------------

            y = yield_values[ratio_mask]
            n = nfert_values[ratio_mask]
            a = area_values[ratio_mask]

            # Guard against numerical overflow.
            with np.errstate(
                invalid="ignore",
                over="ignore"
            ):

                yield_terms = y * a
                nfert_terms = n * a

            finite_terms = (
                np.isfinite(yield_terms)
                & np.isfinite(nfert_terms)
            )

            yield_terms = yield_terms[finite_terms]
            nfert_terms = nfert_terms[finite_terms]
            a = a[finite_terms]

            n_ratio = int(
                len(a)
            )

            result["N_Ratio_Common_Pixels"] = n_ratio

            result["HA_Used_Ratio"] = 0.0

            if n_ratio > 0:

                area_sum = float(
                    np.sum(
                        a,
                        dtype=np.float64
                    )
                )

                yield_sum = float(
                    np.sum(
                        yield_terms,
                        dtype=np.float64
                    )
                )

                nfert_sum = float(
                    np.sum(
                        nfert_terms,
                        dtype=np.float64
                    )
                )

                if not all(
                    np.isfinite(x)
                    for x in [
                        area_sum,
                        yield_sum,
                        nfert_sum
                    ]
                ):
                    raise ValueError(
                        "Non-finite aggregated sums"
                    )

                result["HA_Used_Ratio"] = area_sum

                result["Sum_Yield_x_HA"] = (
                    yield_sum
                )

                result["Sum_Nfert_x_HA"] = (
                    nfert_sum
                )

                if area_sum > 0:

                    result["Mean_Yield_HA"] = (
                        yield_sum / area_sum
                    )

                    result["Mean_Nfert_HA"] = (
                        nfert_sum / area_sum
                    )

                # Exact requested formula
                if nfert_sum > 0:

                    result["PA_PFPN_RatioOfMeans"] = (
                        yield_sum / nfert_sum
                    )

        print(
            f"  Mean={result['PA_NUE']:.5f} | "
            f"HA-Weighted="
            f"{result['PA_NUE_AreaWeighted']:.5f} | "
            f"RatioOfMeans="
            f"{result['PA_PFPN_RatioOfMeans']:.5f} | "
            f"N={result['N_Ratio_Common_Pixels']}",
            flush=True
        )

    except GridAlignmentError:
        # Do not silently accept spatial mismatch.
        raise

    except Exception as e:

        print(
            f"  WARNING: {e}",
            flush=True
        )

    results.append(result)


# ============================================================
# 10. WRITE RESULTS TO EXCEL
# ============================================================

for col in output_fields:

    df[col] = [
        record[col]
        for record in results
    ]

numeric_columns = [
    "PA_NUE",
    "PA_NUE_AreaWeighted",
    "Harvested_Area_Used",
    "PA_PFPN_RatioOfMeans",
    "Mean_Yield_HA",
    "Mean_Nfert_HA",
    "Sum_Yield_x_HA",
    "Sum_Nfert_x_HA",
    "HA_Used_Ratio"
]

for col in numeric_columns:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

    df[col] = df[col].replace(
        [np.inf, -np.inf],
        np.nan
    )

df.to_excel(
    output_excel,
    index=False,
    engine="openpyxl"
)


# ============================================================
# 11. FINAL REPORT
# ============================================================

elapsed = (
    time.perf_counter()
    - start_time
)

print("\n" + "=" * 75)
print("COUNTY-LEVEL PFPN VALIDATION COMPLETED")
print("=" * 75)

print(f"Survey rows: {len(df):,}")

for col in [
    "PA_NUE",
    "PA_NUE_AreaWeighted",
    "PA_PFPN_RatioOfMeans"
]:

    print(
        f"{col}: "
        f"{df[col].notna().sum():,} "
        "valid records"
    )

print(f"\nProcessing time: {elapsed:.2f}s")

print(f"Excel saved to:\n{output_excel}")

print("=" * 75)

display(
    df[
        [
            "m49",
            "Crop",
            "Year",
            "PA_NUE",
            "PA_NUE_AreaWeighted",
            "PA_PFPN_RatioOfMeans",
            "N_Ratio_Common_Pixels",
            "Mean_Yield_HA",
            "Mean_Nfert_HA"
        ]
    ].head(15)
)

In [ ]:
# =====================================================
# -------Global Maize NUE---Based on the weighted median-------------
# ---Step 1----Setting parameters----------
# ---Step 2--- Run ----------------
# ---Step 3--- simple validation --------------
# ---Step 4--- Match with harvested area, mask
# ---Step 5--- Remove outliers, Useful tool,Pixels number should larger than 6, NUE more then 150; set non, 
# ---Step 6--- Check mean value
# ---Step 7--- Extract country mean value--Useful---
# ---Step 8--- Extract county mean value--China and US---
# ---Step 9--- Extract county mean PFPN value---
# ---Step 10--- Extract GeoTIFF Pixel Values Using Excel Coordinates & Year-- validation field scale-
# ==========================================================
# Extract GeoTIFF Pixel Values Using Excel Coordinates & Year
# Excel required columns:
#   Latitude
#   Longitude
#   Harvest year
#
# GeoTIFF filename examples:
#   NUE_1995.tif
#   NUE_1996.tif
#   NUE_2020.tif
#
# Output:
#   Original Excel columns +
#   Raster_Value
#   Raster_File
#   Extract_Status
#
# No interpolation or resampling.
# NaN / NoData handling included.
#
# Run directly in JupyterLab
# ============================================================

import os
import re
import numpy as np
import pandas as pd
import rasterio

from pathlib import Path
from rasterio.crs import CRS
from rasterio.warp import transform as crs_transform


# ============================================================
# 1. USER CONFIGURATION
# ============================================================

# Input Excel
INPUT_EXCEL = (
    r"D:\1_PrecisionAgriculture\7_My paper\01_Global NUE\03_Revision20261008\09_Validation\FieldLevel"
    r"\DatabaseMaize_With_Local_Fertilization.xlsx"
)

# Excel sheet
# 0 = first sheet
SHEET_NAME = 0

# GeoTIFF folder
TIF_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\09CropResults4Validation1021V1_500_M6\maize"
)

# Output Excel
OUTPUT_EXCEL = (
    r"D:\1_PrecisionAgriculture\7_My paper\01_Global NUE\03_Revision20261008\09_Validation\FieldLevel"
    r"\01Extracted_GeoTIFF_PFPNValues.xlsx"
)

# Excel column names
LAT_COLUMN = "Latitude"
LON_COLUMN = "Longitude"
YEAR_COLUMN = "Harvest year"

# GeoTIFF filename prefix (optional)
# None = accept any xxx_YYYY.tif
# "NUE_" = only accept names starting with NUE_
TIF_PREFIX = None

# Raster band
RASTER_BAND = 1

# Additional NoData values
ADDITIONAL_NODATA = [-9999.0]

# Output column names
VALUE_COLUMN = "Raster_Value"
FILE_COLUMN = "Raster_File"
STATUS_COLUMN = "Extract_Status"


# ============================================================
# 2. VALIDATE PATHS
# ============================================================

if not os.path.isfile(INPUT_EXCEL):
    raise FileNotFoundError(
        f"Excel not found: {INPUT_EXCEL}"
    )

if not os.path.isdir(TIF_FOLDER):
    raise FileNotFoundError(
        f"GeoTIFF folder not found: {TIF_FOLDER}"
    )

if (
    os.path.abspath(INPUT_EXCEL).lower()
    == os.path.abspath(OUTPUT_EXCEL).lower()
):
    raise ValueError(
        "OUTPUT_EXCEL must differ from INPUT_EXCEL."
    )

os.makedirs(
    os.path.dirname(os.path.abspath(OUTPUT_EXCEL)),
    exist_ok=True
)


# ============================================================
# 3. READ EXCEL
# ============================================================

print("=" * 70)
print("EXTRACT GEOTIFF VALUES BY COORDINATES AND YEAR")
print("=" * 70)

df = pd.read_excel(
    INPUT_EXCEL,
    sheet_name=SHEET_NAME
)

# Remove leading/trailing spaces in column names
df.columns = [
    str(col).strip()
    for col in df.columns
]

required_columns = [
    LAT_COLUMN,
    LON_COLUMN,
    YEAR_COLUMN
]

for col in required_columns:

    if col not in df.columns:
        raise ValueError(
            f"Missing Excel column: {col}\n"
            f"Available columns: {df.columns.tolist()}"
        )

# Prevent overwriting existing columns
for col in [
    VALUE_COLUMN,
    FILE_COLUMN,
    STATUS_COLUMN
]:
    if col in df.columns:
        raise ValueError(
            f"Output column already exists: {col}"
        )

print(f"Excel rows: {len(df):,}")
print(f"Excel columns: {len(df.columns)}")


# ============================================================
# 4. PARSE HARVEST YEAR
# ============================================================

def parse_year(value):
    """
    Supported examples:
        1996
        1996.0
        "1996"
        datetime / Excel date
        "1996-07-15"

    Invalid or missing year -> NaN
    """

    if pd.isna(value):
        return np.nan

    if isinstance(value, pd.Timestamp):
        return value.year

    text = str(value).strip()

    # Numeric year
    match = re.fullmatch(
        r"(19\d{2}|20\d{2})(?:\.0+)?",
        text
    )

    if match:
        return int(match.group(1))

    # Date-like value
    try:

        date_value = pd.to_datetime(
            text,
            errors="coerce"
        )

        if pd.notna(date_value):
            return int(date_value.year)

    except Exception:
        pass

    return np.nan


df["_match_year"] = df[YEAR_COLUMN].apply(
    parse_year
)

# Convert coordinates
df["_lat_numeric"] = pd.to_numeric(
    df[LAT_COLUMN],
    errors="coerce"
)

df["_lon_numeric"] = pd.to_numeric(
    df[LON_COLUMN],
    errors="coerce"
)


# ============================================================
# 5. FIND GEOTIFF FILES AND MATCH YEARS
# ============================================================

tif_folder = Path(TIF_FOLDER)

# Recursively search all subfolders
tif_files = sorted(
    p for p in tif_folder.rglob("*")
    if p.is_file()
    and p.suffix.lower() in (".tif", ".tiff")
)

year_to_tif = {}

for tif in tif_files:

    filename = tif.name

    if TIF_PREFIX is not None:

        if not filename.lower().startswith(
            TIF_PREFIX.lower()
        ):
            continue

    # Last four digits are the year
    # Example: NUE_1996.tif -> 1996
    match = re.search(
        r"_(19\d{2}|20\d{2})\.tiff?$",
        filename,
        re.IGNORECASE
    )

    if match is None:
        continue

    year = int(match.group(1))

    # Prevent ambiguous matching
    if year in year_to_tif:

        raise ValueError(
            f"Multiple GeoTIFF files found for {year}:\n"
            f"  {year_to_tif[year]}\n"
            f"  {tif}\n"
            "Please set TIF_PREFIX or use a folder "
            "containing one raster per year."
        )

    year_to_tif[year] = tif


if not year_to_tif:
    raise RuntimeError(
        "No GeoTIFF files matching xxx_YYYY.tif found."
    )

print(f"\nMatched GeoTIFF files: {len(year_to_tif)}")
print(f"Available years: {sorted(year_to_tif)}")


# ============================================================
# 6. INITIALIZE OUTPUT COLUMNS
# ============================================================

df[VALUE_COLUMN] = np.nan
df[FILE_COLUMN] = ""
df[STATUS_COLUMN] = ""

# Validate years
valid_year = df["_match_year"].notna()

# Validate coordinate values and ranges
lat = df["_lat_numeric"]
lon = df["_lon_numeric"]

valid_coord = (
    np.isfinite(lat)
    & np.isfinite(lon)
    & lat.between(-90, 90)
    & lon.between(-180, 180)
)

# Initialize status
df.loc[
    ~valid_year,
    STATUS_COLUMN
] = "INVALID_YEAR"

df.loc[
    valid_year & ~valid_coord,
    STATUS_COLUMN
] = "INVALID_COORDINATES"

ready_mask = valid_year & valid_coord

df.loc[
    ready_mask,
    STATUS_COLUMN
] = "RASTER_YEAR_NOT_FOUND"


# ============================================================
# 7. PROCESS EACH YEAR
# ============================================================

required_years = sorted(
    int(y)
    for y in df.loc[
        ready_mask,
        "_match_year"
    ].unique()
)

for year in required_years:

    # Indices of Excel records for this year
    selected_indices = df.index[
        ready_mask
        & (df["_match_year"] == year)
    ].tolist()

    if year not in year_to_tif:

        print(
            f"[{year}] Raster not found | "
            f"Points: {len(selected_indices)}"
        )

        continue

    tif_path = year_to_tif[year]

    print("\n" + "-" * 70)
    print(f"Processing year: {year}")
    print(f"Raster: {tif_path.name}")
    print(f"Points: {len(selected_indices)}")

    # --------------------------------------------------------
    # 7.1 Open corresponding GeoTIFF
    # --------------------------------------------------------

    with rasterio.open(tif_path) as src:

        if src.crs is None:
            raise ValueError(
                f"GeoTIFF CRS missing: {tif_path}"
            )

        if RASTER_BAND > src.count:
            raise ValueError(
                f"Band {RASTER_BAND} not found in {tif_path.name}"
            )

        # Record filename for matched points
        df.loc[
            selected_indices,
            FILE_COLUMN
        ] = tif_path.name

        # ----------------------------------------------------
        # 7.2 Get coordinates from Excel
        # ----------------------------------------------------

        latitudes = df.loc[
            selected_indices,
            "_lat_numeric"
        ].to_numpy(dtype=float)

        longitudes = df.loc[
            selected_indices,
            "_lon_numeric"
        ].to_numpy(dtype=float)

        # ----------------------------------------------------
        # 7.3 Transform WGS84 coordinates to raster CRS
        # ----------------------------------------------------

        source_crs = CRS.from_epsg(4326)

        xs, ys = crs_transform(
            source_crs,
            src.crs,
            longitudes.tolist(),
            latitudes.tolist()
        )

        # ----------------------------------------------------
        # 7.4 Validate whether point is within raster
        # ----------------------------------------------------

        valid_points = []
        sample_coordinates = []

        for excel_index, x, y in zip(
            selected_indices,
            xs,
            ys
        ):

            if not (
                np.isfinite(x)
                and np.isfinite(y)
            ):

                df.at[
                    excel_index,
                    STATUS_COLUMN
                ] = "INVALID_TRANSFORM"

                continue

            # Get corresponding raster row and column
            row, col = src.index(x, y)

            if (
                row < 0
                or row >= src.height
                or col < 0
                or col >= src.width
            ):

                df.at[
                    excel_index,
                    STATUS_COLUMN
                ] = "OUTSIDE_RASTER"

                continue

            valid_points.append(excel_index)

            sample_coordinates.append(
                (x, y)
            )

        # ----------------------------------------------------
        # 7.5 Sample raster values
        # ----------------------------------------------------

        samples = src.sample(
            sample_coordinates,
            indexes=RASTER_BAND,
            masked=True
        )

        for excel_index, sample in zip(
            valid_points,
            samples
        ):

            # Check raster mask
            if np.any(np.ma.getmaskarray(sample)):

                df.at[
                    excel_index,
                    STATUS_COLUMN
                ] = "NODATA"

                continue

            value = float(sample[0])

            # NaN / Inf
            if not np.isfinite(value):

                df.at[
                    excel_index,
                    STATUS_COLUMN
                ] = "NODATA"

                continue

            # Additional NoData
            if any(
                value == nd
                for nd in ADDITIONAL_NODATA
            ):

                df.at[
                    excel_index,
                    STATUS_COLUMN
                ] = "NODATA"

                continue

            # -----------------------------------------------
            # Valid value: save original pixel value
            # -----------------------------------------------

            df.at[
                excel_index,
                VALUE_COLUMN
            ] = value

            df.at[
                excel_index,
                STATUS_COLUMN
            ] = "SUCCESS"

    success_count = int(
        (
            df.loc[
                selected_indices,
                STATUS_COLUMN
            ] == "SUCCESS"
        ).sum()
    )

    print(
        f"[{year}] Success: "
        f"{success_count}/{len(selected_indices)}"
    )


# ============================================================
# 8. REMOVE TEMPORARY COLUMNS
# ============================================================

df.drop(
    columns=[
        "_match_year",
        "_lat_numeric",
        "_lon_numeric"
    ],
    inplace=True
)


# ============================================================
# 9. EXPORT EXCEL
# ============================================================

df.to_excel(
    OUTPUT_EXCEL,
    index=False,
    sheet_name="Extracted_Values",
    na_rep=""
)


# ============================================================
# 10. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("GEOTIFF EXTRACTION COMPLETED")
print("=" * 70)

print(f"Total Excel rows: {len(df):,}")

print(
    f"Successfully extracted: "
    f"{(df[STATUS_COLUMN] == 'SUCCESS').sum():,}"
)

print("\nExtraction status summary:")

print(
    df[STATUS_COLUMN]
    .value_counts(dropna=False)
    .to_string()
)

print(f"\nOutput Excel:\n{OUTPUT_EXCEL}")

print("=" * 70)

display(df.head(10))

In [ ]:
# =====================================================
# -------Global Maize NUE---Based on the weighted median-------------
# ---Step 1----Setting parameters----------
# ---Step 2--- Run ----------------
# ---Step 3--- simple validation --------------
# ---Step 4--- Match with harvested area, mask
# ---Step 5--- Remove outliers, Useful tool,Pixels number should larger than 6, NUE more then 150; set non, 
# ---Step 6--- Check mean value
# ---Step 7--- Extract country mean value--Useful---
# ---Step 8--- Extract county mean value--China and US---
# ---Step 9--- Extract county mean PFPN value---
# ---Step 10--- Extract GeoTIFF Pixel Values Using Excel Coordinates & Year-- validation field scale-
# ---Step 11--- the normalized ensemble range of NUE---
# ==========================================================

import os
import glob
import numpy as np
import rasterio

# =========================
# Configuration
# =========================
A_FOLDER = r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\02Global_NUE_Maize\02MaizeNUE_V6_20260320"

B_FOLDER = r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\02Global_NUE_Maize\02MaizeNUE_V6_M6_150C"

OUT_FOLDER = r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\02Global_NUE_Maize/02MaizeNUE_V6_M6_150C_Nor"

nodata = -9999

os.makedirs(OUT_FOLDER, exist_ok=True)

# =========================
# Collect A files
# =========================
a_files = sorted(glob.glob(os.path.join(A_FOLDER, "NUE_dif_*.tif")))

if not a_files:
    raise RuntimeError("No NUE_dif_*.tif files found!")

print(f"Found {len(a_files)} year files\n")

# =========================
# Batch processing
# =========================
for a_path in a_files:
    fname = os.path.basename(a_path)

    # Extract year
    year = fname.replace("NUE_dif_", "").replace(".tif", "")

    b_path = os.path.join(B_FOLDER, f"NUE_median_Opt_{year}.tif")
    out_path = os.path.join(OUT_FOLDER, f"NUE_Nor_{year}.tif")

    if not os.path.exists(b_path):
        print(f"⚠ Missing B file for year {year}, skipped")
        continue

    print(f"Processing year {year}")

    with rasterio.open(a_path) as src_a, rasterio.open(b_path) as src_b:
        A = src_a.read(1).astype(np.float32)
        B = src_b.read(1).astype(np.float32)

        # Copy metadata from A
        profile = src_a.profile.copy()

    # =========================
    # Core calculation
    # =========================

    # Initialize output as nodata
    result = np.full(A.shape, nodata, dtype=np.float32)

    # Valid pixels:
    # B is valid AND not zero (avoid division by zero)
    valid = (B != nodata) & (~np.isnan(B)) & (B != 0)

    result[valid] = A[valid] / B[valid]

    # =========================
    # Export GeoTIFF
    # =========================
    profile.update({
        "dtype": "float32",
        "nodata": nodata,
        "compress": "lzw"
    })

    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(result, 1)

    print(f"✔ Saved: {out_path}")

print("\n✅ All years finished")

In [ ]:
# =====================================================
# -------Global Maize NUE---Based on the weighted median-------------
# ---Step 1----Setting parameters----------
# ---Step 2--- Run ----------------
# ---Step 3--- simple validation --------------
# ---Step 4--- Match with harvested area, mask
# ---Step 5--- Remove outliers, Useful tool,Pixels number should larger than 6, NUE more then 150; set non, 
# ---Step 6--- Check mean value
# ---Step 7--- Extract country mean value--Useful---
# ---Step 8--- Extract county mean value--China and US---
# ---Step 9--- Extract county mean PFPN value---
# ---Step 10--- Extract GeoTIFF Pixel Values Using Excel Coordinates & Year-- validation field scale-
# ---Step 11--- the normalized ensemble range of NUE---
# ---Step 12--- the Uncertainty quantification of NUE---
# ==========================================================

In [ ]:
# =====================================================
# -------Global Maize NUE---Based on the weighted median-------------
# ---Step 1----Setting parameters----------
# ---Step 2--- Run ----------------
# ---Step 3--- simple validation --------------
# ---Step 4--- Match with harvested area, mask
# ---Step 5--- Remove outliers, Useful tool,Pixels number should larger than 6, NUE more then 150; set non, 
# ---Step 6--- Check mean value
# ---Step 7--- Extract country mean value--Useful---
# ---Step 8--- Extract county mean value--China and US---
# ---Step 9--- Extract county mean PFPN value---
# ---Step 10--- Extract GeoTIFF Pixel Values Using Excel Coordinates & Year-- validation field scale-
# ---Step 12--- the Uncertainty quantification of NUE---
# ---Step 13--- the sensitivity analysis of NUE---
# ==========================================================

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
import rasterio
import numpy as np
from matplotlib.ticker import FuncFormatter

# ... (您的文件路径定义保持不变) ...

# 年份（6 个）
years = [1995, 2000, 2005, 2010, 2015, 2020]

tif_template = (
    "Z:/staff/Qiang_S/09_Datasets/03GlobalNUE_Results/02Global_NUE_Maize/02MaizeNUE_V6_M6_150C/NUE_median_Opt_{}.tif"  # Maize
    
)

shp_file = (
    "Z:/staff/Qiang_S/09_Datasets/09OtherDataUsed/GDAM_GlobalCountryLevel_Shp/GlobalCountryLevel_Shp.shp"
)

out_png = (
    "Z:/staff/Qiang_S/09_Datasets/03GlobalNUE_Results/02Global_NUE_Maize/02MaizeNUE_V6_M6_150C/MaizeNUE_median_6yearsA.png"
)


nodata_value = -9999                  # <-- 空值标记
cmap_style = "RdYlBu"                   # <-- 颜色映射

# 数值范围控制
vmin, vmax = 0, 90                   # <-- 关心的数值范围
overall_min, overall_max = 0, 150     # <-- 数据总范围（用于 colorbar 显示）

# ===============================
# 定义经纬度格式化函数
# ===============================
def format_lon(x, pos):
    val = int(x)
    if val == 0: return "0°"
    return f"{abs(val)}°{'E' if val > 0 else 'W'}"

def format_lat(y, pos):
    val = int(y)
    if val == 0: return "0°"
    return f"{abs(val)}°{'N' if val > 0 else 'S'}"

# ===============================
# 3. 创建 2×3 子图
# 修改点：调整 figsize 以匹配全球地图的长宽比，减少由于比例不适配产生的留白
# ===============================
fig, axes = plt.subplots(
    nrows=3,
    ncols=2,
    figsize=(12, 9),  # ⭐ 修改：高度从15降为9，更紧凑
    sharex=True,
    sharey=True
)

# ⭐ 修改：手动调整子图间距，wsapce/hspace 设为极小值
plt.subplots_adjust(wspace=0.05, hspace=0.05)

axes = axes.flatten()
img_for_cbar = None 

# ===============================
# 4. 循环绘制每一年
# ===============================
for i, year in enumerate(years):
    ax = axes[i]
    
    # ... (读取数据的代码保持不变) ...
    tif_file = tif_template.format(year)
    with rasterio.open(tif_file) as src:
        raster = src.read(1).astype("float32")
        bounds = src.bounds
        
    raster = np.where(raster == nodata_value, np.nan, raster)

    # 绘图
    img = ax.imshow(
        raster,
        cmap=cmap_style,
        extent=[bounds.left, bounds.right, bounds.bottom, bounds.top],
        vmin=vmin,
        vmax=vmax
    )

    if img_for_cbar is None:
        img_for_cbar = img

    # 叠加国界 (线宽调细适合发表)
    gdf = gpd.read_file(shp_file)
    gdf.boundary.plot(ax=ax, color="black", linewidth=0.15)

    # 年份标注 (移到左下角或左上角，加个简单的框)
    ax.text(
        0.03, 0.95, str(year),
        transform=ax.transAxes,
        fontsize=12, 
        fontweight="bold",
        fontfamily= "Arial",
        ha="left", 
        va="top",
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.7, pad=2)
    )

    # ===============================
    # ⭐ 修改：坐标轴控制 (外边框显示)
    # ===============================
    row, col = divmod(i, 2)

    # 设置刻度格式
    ax.xaxis.set_major_formatter(FuncFormatter(format_lon))
    ax.yaxis.set_major_formatter(FuncFormatter(format_lat))
    
    # 设置刻度间隔 (根据需要调整，这里设为经度60度，纬度30度一档)
    ax.set_xticks(np.arange(-120, 121, 60))
    ax.set_yticks(np.arange(-60, 61, 30))
    
    # 刻度线朝内
    ax.tick_params(direction='in', which='both', labelsize=10, labelcolor = '#333333', width=0.5)

    # ---边框控制 ---
    for edge in ['top', 'bottom', 'left', 'right']:
        ax.spines[edge].set_linewidth(0.5)  # 设置边框粗细
        ax.spines[edge].set_color('#333333')   # 设置边框颜色
    # -----------------------

    # ---强制设置经纬度刻度标签的字体样式---
    for label in ax.get_xticklabels():
        label.set_fontfamily('Arial')
        label.set_fontsize(10)
        label.set_fontweight('normal')  # 可选 'bold'

    for label in ax.get_yticklabels():
        label.set_fontfamily('Arial')
        label.set_fontsize(10)
        label.set_fontweight('normal')

# ===============================
# 5. 自定义 Colorbar (缩小尺寸)
# ===============================

# ⭐ 修改：不再使用 fig.colorbar(ax=axes)，而是添加一个新的轴
# [left, bottom, width, height] -> 坐标系是整个 Figure (0到1)
# 调整 bottom 参数可以将图例上下移动
cbar_ax = fig.add_axes([0.3, 0.05, 0.4, 0.02]) 

cbar = fig.colorbar(
    img_for_cbar,
    cax=cbar_ax,
    orientation="horizontal",
    extend="max"
)

cbar.set_label("Maize NUE (%)", fontsize=12, fontfamily= "Arial", fontweight= "bold")
cbar.ax.tick_params(labelsize=10, direction="in", labelcolor = '#333333', width=0.5)

# 设置显示的刻度
cbar.set_ticks([0, 10, 20, 30, 40, 50, 60, 70, 80, 90])
# 新增：强制设置图例刻度数字的字体样式
for t in cbar.ax.get_xticklabels():
    t.set_fontfamily('Arial')
    t.set_fontsize(10)
    t.set_fontweight('normal') # 如果需要加粗可以改为 'bold'

img_for_cbar.set_clim(0, 90) # 保持您原来的clim逻辑

# ----保存---
plt.savefig(out_png, dpi=400, bbox_inches="tight")

# ⭐ 新增：导出 TIFF
out_tif = out_png.replace(".png", ".tif")
plt.savefig(out_tif, dpi=400, bbox_inches="tight", format="tiff")

# ⭐ 新增：导出 PDF
out_pdf = out_png.replace(".png", ".pdf")
plt.savefig(out_pdf, bbox_inches="tight", format="pdf")  # PDF 为矢量图，无需设置 dpi

plt.show()

In [ ]:
# validation field scale

# ============================================================
# Extract GeoTIFF Pixel Values Using Excel Coordinates & Year
#
# Excel required columns:
#   Latitude
#   Longitude
#   Harvest year
#
# GeoTIFF filename examples:
#   NUE_1995.tif
#   NUE_1996.tif
#   NUE_2020.tif
#
# Output:
#   Original Excel columns +
#   Raster_Value
#   Raster_File
#   Extract_Status
#
# No interpolation or resampling.
# NaN / NoData handling included.
#
# Run directly in JupyterLab
# ============================================================

import os
import re
import numpy as np
import pandas as pd
import rasterio

from pathlib import Path
from rasterio.crs import CRS
from rasterio.warp import transform as crs_transform


# ============================================================
# 1. USER CONFIGURATION
# ============================================================

# Input Excel
INPUT_EXCEL = (
    r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\09_Meta_AnalysisFields"
    r"\02DatabaseMaize_all_rowsCGN29v1.xlsx"
)

# Excel sheet
# 0 = first sheet
SHEET_NAME = 0

# GeoTIFF folder
TIF_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\09CropResults4PFPNValidation\PFPN\maize"
)

# Output Excel
OUTPUT_EXCEL = (
    r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\09_Meta_AnalysisFields"
    r"\02Extracted_GeoTIFF_PFPNM29V1.xlsx"
)

# Excel column names
LAT_COLUMN = "Latitude"
LON_COLUMN = "Longitude"
YEAR_COLUMN = "Harvest year"

# GeoTIFF filename prefix (optional)
# None = accept any xxx_YYYY.tif
# "NUE_" = only accept names starting with NUE_
TIF_PREFIX = None

# Raster band
RASTER_BAND = 1

# Additional NoData values
ADDITIONAL_NODATA = [-9999.0]

# Output column names
VALUE_COLUMN = "Raster_Value"
FILE_COLUMN = "Raster_File"
STATUS_COLUMN = "Extract_Status"


# ============================================================
# 2. VALIDATE PATHS
# ============================================================

if not os.path.isfile(INPUT_EXCEL):
    raise FileNotFoundError(
        f"Excel not found: {INPUT_EXCEL}"
    )

if not os.path.isdir(TIF_FOLDER):
    raise FileNotFoundError(
        f"GeoTIFF folder not found: {TIF_FOLDER}"
    )

if (
    os.path.abspath(INPUT_EXCEL).lower()
    == os.path.abspath(OUTPUT_EXCEL).lower()
):
    raise ValueError(
        "OUTPUT_EXCEL must differ from INPUT_EXCEL."
    )

os.makedirs(
    os.path.dirname(os.path.abspath(OUTPUT_EXCEL)),
    exist_ok=True
)


# ============================================================
# 3. READ EXCEL
# ============================================================

print("=" * 70)
print("EXTRACT GEOTIFF VALUES BY COORDINATES AND YEAR")
print("=" * 70)

df = pd.read_excel(
    INPUT_EXCEL,
    sheet_name=SHEET_NAME
)

# Remove leading/trailing spaces in column names
df.columns = [
    str(col).strip()
    for col in df.columns
]

required_columns = [
    LAT_COLUMN,
    LON_COLUMN,
    YEAR_COLUMN
]

for col in required_columns:

    if col not in df.columns:
        raise ValueError(
            f"Missing Excel column: {col}\n"
            f"Available columns: {df.columns.tolist()}"
        )

# Prevent overwriting existing columns
for col in [
    VALUE_COLUMN,
    FILE_COLUMN,
    STATUS_COLUMN
]:
    if col in df.columns:
        raise ValueError(
            f"Output column already exists: {col}"
        )

print(f"Excel rows: {len(df):,}")
print(f"Excel columns: {len(df.columns)}")


# ============================================================
# 4. PARSE HARVEST YEAR
# ============================================================

def parse_year(value):
    """
    Supported examples:
        1996
        1996.0
        "1996"
        datetime / Excel date
        "1996-07-15"

    Invalid or missing year -> NaN
    """

    if pd.isna(value):
        return np.nan

    if isinstance(value, pd.Timestamp):
        return value.year

    text = str(value).strip()

    # Numeric year
    match = re.fullmatch(
        r"(19\d{2}|20\d{2})(?:\.0+)?",
        text
    )

    if match:
        return int(match.group(1))

    # Date-like value
    try:

        date_value = pd.to_datetime(
            text,
            errors="coerce"
        )

        if pd.notna(date_value):
            return int(date_value.year)

    except Exception:
        pass

    return np.nan


df["_match_year"] = df[YEAR_COLUMN].apply(
    parse_year
)

# Convert coordinates
df["_lat_numeric"] = pd.to_numeric(
    df[LAT_COLUMN],
    errors="coerce"
)

df["_lon_numeric"] = pd.to_numeric(
    df[LON_COLUMN],
    errors="coerce"
)


# ============================================================
# 5. FIND GEOTIFF FILES AND MATCH YEARS
# ============================================================

tif_folder = Path(TIF_FOLDER)

# Recursively search all subfolders
tif_files = sorted(
    p for p in tif_folder.rglob("*")
    if p.is_file()
    and p.suffix.lower() in (".tif", ".tiff")
)

year_to_tif = {}

for tif in tif_files:

    filename = tif.name

    if TIF_PREFIX is not None:

        if not filename.lower().startswith(
            TIF_PREFIX.lower()
        ):
            continue

    # Last four digits are the year
    # Example: NUE_1996.tif -> 1996
    match = re.search(
        r"_(19\d{2}|20\d{2})\.tiff?$",
        filename,
        re.IGNORECASE
    )

    if match is None:
        continue

    year = int(match.group(1))

    # Prevent ambiguous matching
    if year in year_to_tif:

        raise ValueError(
            f"Multiple GeoTIFF files found for {year}:\n"
            f"  {year_to_tif[year]}\n"
            f"  {tif}\n"
            "Please set TIF_PREFIX or use a folder "
            "containing one raster per year."
        )

    year_to_tif[year] = tif


if not year_to_tif:
    raise RuntimeError(
        "No GeoTIFF files matching xxx_YYYY.tif found."
    )

print(f"\nMatched GeoTIFF files: {len(year_to_tif)}")
print(f"Available years: {sorted(year_to_tif)}")


# ============================================================
# 6. INITIALIZE OUTPUT COLUMNS
# ============================================================

df[VALUE_COLUMN] = np.nan
df[FILE_COLUMN] = ""
df[STATUS_COLUMN] = ""

# Validate years
valid_year = df["_match_year"].notna()

# Validate coordinate values and ranges
lat = df["_lat_numeric"]
lon = df["_lon_numeric"]

valid_coord = (
    np.isfinite(lat)
    & np.isfinite(lon)
    & lat.between(-90, 90)
    & lon.between(-180, 180)
)

# Initialize status
df.loc[
    ~valid_year,
    STATUS_COLUMN
] = "INVALID_YEAR"

df.loc[
    valid_year & ~valid_coord,
    STATUS_COLUMN
] = "INVALID_COORDINATES"

ready_mask = valid_year & valid_coord

df.loc[
    ready_mask,
    STATUS_COLUMN
] = "RASTER_YEAR_NOT_FOUND"

# ============================================================
# 7. PROCESS EACH YEAR
# ============================================================

required_years = sorted(
    int(y)
    for y in df.loc[
        ready_mask,
        "_match_year"
    ].unique()
)

for year in required_years:

    # Indices of Excel records for this year
    selected_indices = df.index[
        ready_mask
        & (df["_match_year"] == year)
    ].tolist()

    if year not in year_to_tif:

        print(
            f"[{year}] Raster not found | "
            f"Points: {len(selected_indices)}"
        )

        continue

    tif_path = year_to_tif[year]

    print("\n" + "-" * 70)
    print(f"Processing year: {year}")
    print(f"Raster: {tif_path.name}")
    print(f"Points: {len(selected_indices)}")

    # --------------------------------------------------------
    # 7.1 Open corresponding GeoTIFF
    # --------------------------------------------------------

    with rasterio.open(tif_path) as src:

        if src.crs is None:
            raise ValueError(
                f"GeoTIFF CRS missing: {tif_path}"
            )

        if RASTER_BAND > src.count:
            raise ValueError(
                f"Band {RASTER_BAND} not found in {tif_path.name}"
            )

        # Record filename for matched points
        df.loc[
            selected_indices,
            FILE_COLUMN
        ] = tif_path.name

        # ----------------------------------------------------
        # 7.2 Get coordinates from Excel
        # ----------------------------------------------------

        latitudes = df.loc[
            selected_indices,
            "_lat_numeric"
        ].to_numpy(dtype=float)

        longitudes = df.loc[
            selected_indices,
            "_lon_numeric"
        ].to_numpy(dtype=float)

        # ----------------------------------------------------
        # 7.3 Transform WGS84 coordinates to raster CRS
        # ----------------------------------------------------

        source_crs = CRS.from_epsg(4326)

        xs, ys = crs_transform(
            source_crs,
            src.crs,
            longitudes.tolist(),
            latitudes.tolist()
        )

        # ----------------------------------------------------
        # 7.4 Validate whether point is within raster
        # ----------------------------------------------------

        valid_points = []
        sample_coordinates = []

        for excel_index, x, y in zip(
            selected_indices,
            xs,
            ys
        ):

            if not (
                np.isfinite(x)
                and np.isfinite(y)
            ):

                df.at[
                    excel_index,
                    STATUS_COLUMN
                ] = "INVALID_TRANSFORM"

                continue

            # Get corresponding raster row and column
            row, col = src.index(x, y)

            if (
                row < 0
                or row >= src.height
                or col < 0
                or col >= src.width
            ):

                df.at[
                    excel_index,
                    STATUS_COLUMN
                ] = "OUTSIDE_RASTER"

                continue

            valid_points.append(excel_index)

            sample_coordinates.append(
                (x, y)
            )

        # ----------------------------------------------------
        # 7.5 Sample raster values
        # ----------------------------------------------------

        samples = src.sample(
            sample_coordinates,
            indexes=RASTER_BAND,
            masked=True
        )

        for excel_index, sample in zip(
            valid_points,
            samples
        ):

            # Check raster mask
            if np.any(np.ma.getmaskarray(sample)):

                df.at[
                    excel_index,
                    STATUS_COLUMN
                ] = "NODATA"

                continue

            value = float(sample[0])

            # NaN / Inf
            if not np.isfinite(value):

                df.at[
                    excel_index,
                    STATUS_COLUMN
                ] = "NODATA"

                continue

            # Additional NoData
            if any(
                value == nd
                for nd in ADDITIONAL_NODATA
            ):

                df.at[
                    excel_index,
                    STATUS_COLUMN
                ] = "NODATA"

                continue

            # -----------------------------------------------
            # Valid value: save original pixel value
            # -----------------------------------------------

            df.at[
                excel_index,
                VALUE_COLUMN
            ] = value

            df.at[
                excel_index,
                STATUS_COLUMN
            ] = "SUCCESS"

    success_count = int(
        (
            df.loc[
                selected_indices,
                STATUS_COLUMN
            ] == "SUCCESS"
        ).sum()
    )

    print(
        f"[{year}] Success: "
        f"{success_count}/{len(selected_indices)}"
    )


# ============================================================
# 8. REMOVE TEMPORARY COLUMNS
# ============================================================

df.drop(
    columns=[
        "_match_year",
        "_lat_numeric",
        "_lon_numeric"
    ],
    inplace=True
)


# ============================================================
# 9. EXPORT EXCEL
# ============================================================

df.to_excel(
    OUTPUT_EXCEL,
    index=False,
    sheet_name="Extracted_Values",
    na_rep=""
)


# ============================================================
# 10. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("GEOTIFF EXTRACTION COMPLETED")
print("=" * 70)

print(f"Total Excel rows: {len(df):,}")

print(
    f"Successfully extracted: "
    f"{(df[STATUS_COLUMN] == 'SUCCESS').sum():,}"
)

print("\nExtraction status summary:")

print(
    df[STATUS_COLUMN]
    .value_counts(dropna=False)
    .to_string()
)

print(f"\nOutput Excel:\n{OUTPUT_EXCEL}")

print("=" * 70)

display(df.head(10))


In [ ]:
# ============================================================
# Tool: Annual Harvested Area Zonal SUM - PARALLEL
#
# Input:
#   1. Shapefile with arbitrary polygons
#   2. Annual GeoTIFF: xxx_1995.tif, xxx_1996.tif, ...
#
# Calculation:
#   Regional Harvested Area = SUM(valid raster pixels)
#
# Features:
#   - PARALLEL processing by YEAR using joblib
#   - Preserve ALL original SHP attribute columns
#   - Sum harvested area for each polygon
#   - Count valid pixels for each polygon
#   - Calculate total within union of SHP polygons
#   - Automatic CRS matching
#   - Exclude NoData, NaN, Inf, negative values
#   - No interpolation or resampling
#
# Excel:
#   Sheet 1: By_Region
#   Sheet 2: Annual_Total
#
# JupyterLab - ONE CELL
# ============================================================

import os
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

from rasterstats import gen_zonal_stats
from shapely.ops import unary_union
from joblib import Parallel, delayed
from IPython.display import display


# ============================================================
# 1. USER CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# Shapefile
# ------------------------------------------------------------

SHP_PATH = (
    r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\GDAM_GlobalCountryLevel_Shp\GlobalCountryLevel_Shp.shp"
)

# Other examples:
# SHP_PATH = r"...\gadm41_CHN_3.shp"
# SHP_PATH = r"...\GlobalCountryLevel_Shp.shp"

# ------------------------------------------------------------
# Harvested Area GeoTIFF folder
# ------------------------------------------------------------

RASTER_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets"
    r"\09OtherDataUsed"
    r"\GCSN_HarvestedArea_1995_2020"
    r"\02StandardizedDatasets"
    r"\02MaizeHarvestedArea"
)

# Example filenames:
# Maize_HarArea_1995.tif
# Maize_HarArea_1996.tif
# Maize_HarArea_2020.tif

# ------------------------------------------------------------
# Output Excel
# ------------------------------------------------------------

OUTPUT_EXCEL = os.path.join(
    RASTER_FOLDER,
    "Maize_HarvestedArea_ZonalSUM_Parallel.xlsx"
)

# ------------------------------------------------------------
# Year range
# ------------------------------------------------------------

START_YEAR = 1995
END_YEAR = 2020

# ------------------------------------------------------------
# PARALLEL SETTINGS
#
# Different years are calculated by different processes.
#
# Examples:
#   1 = sequential
#   4 = four parallel workers
#   8 = eight parallel workers
#
# Each worker reads one annual raster into memory.
# ------------------------------------------------------------

N_WORKERS = 6

PARALLEL_BACKEND = "loky"

PARALLEL_VERBOSE = 10

# ------------------------------------------------------------
# NoData and value filtering
# ------------------------------------------------------------

NODATA = -9999.0

ADDITIONAL_NODATA = []

# Harvested area threshold
MIN_VALID_AREA = 0.0

# True: only use values > MIN_VALID_AREA
# False: use values >= MIN_VALID_AREA
STRICTLY_GREATER = True

# ------------------------------------------------------------
# Polygon pixel inclusion
# ------------------------------------------------------------

ALL_TOUCHED = False

RASTER_BAND = 1

# ------------------------------------------------------------
# Annual total for the UNION of all SHP polygons
#
# True: calculate total from the union geometry.
# Overlapping pixels are counted only once.
# ------------------------------------------------------------

CALCULATE_ANNUAL_TOTAL = True


# ============================================================
# 2. INITIAL VALIDATION
# ============================================================

start_time = time.perf_counter()

if not os.path.isfile(SHP_PATH):
    raise FileNotFoundError(
        f"Shapefile not found:\n{SHP_PATH}"
    )

if not os.path.isdir(RASTER_FOLDER):
    raise FileNotFoundError(
        f"Raster folder not found:\n{RASTER_FOLDER}"
    )

if START_YEAR > END_YEAR:
    raise ValueError(
        "START_YEAR must be <= END_YEAR"
    )

if N_WORKERS < 1:
    raise ValueError(
        "N_WORKERS must be >= 1"
    )

if not np.isfinite(MIN_VALID_AREA):
    raise ValueError(
        "MIN_VALID_AREA must be finite"
    )


# ============================================================
# 3. READ SHAPEFILE
# ============================================================

gdf = gpd.read_file(SHP_PATH)

if gdf.empty:
    raise ValueError(
        "Shapefile contains no polygons"
    )

if gdf.crs is None:
    raise ValueError(
        "Shapefile CRS is missing"
    )

gdf = gdf.reset_index(drop=True)

geometry_col = gdf.geometry.name

# Preserve ALL original SHP attribute columns
attribute_columns = [
    col
    for col in gdf.columns
    if col != geometry_col
]

# New output columns
new_columns = [
    "year",
    "harvested_area_sum",
    "valid_pixels"
]

# Avoid overwriting existing attributes
conflicts = [
    col
    for col in attribute_columns
    if col.lower() in {
        c.lower() for c in new_columns
    }
]

if conflicts:
    raise ValueError(
        f"Conflicting SHP fields: {conflicts}"
    )

# Keep original attribute table in main process
attributes_df = (
    gdf[attribute_columns]
    .copy()
    .reset_index(drop=True)
)

N_REGIONS = len(gdf)

print("=" * 75)
print("ANNUAL HARVESTED AREA ZONAL SUM - PARALLEL")
print("=" * 75)

print(f"Shapefile       : {Path(SHP_PATH).name}")
print(f"Total polygons  : {N_REGIONS:,}")
print(f"SHP attributes  : {len(attribute_columns)}")
print(f"Original CRS    : {gdf.crs}")
print(f"Workers         : {N_WORKERS}")
print(f"Parallel backend: {PARALLEL_BACKEND}")

print("=" * 75)

# Workers load the SHP independently.
# The parent retains only its attribute table.
del gdf


# ============================================================
# 4. INDEX ANNUAL GEOTIFF FILES
# ============================================================

def extract_year(filename):
    """
    Accept filenames ending in:
       _1995.tif
       _2020.tiff
    """

    match = re.search(
        r"_(19\d{2}|20\d{2})\.(?:tif|tiff)$",
        Path(filename).name,
        flags=re.IGNORECASE
    )

    if match is None:
        return None

    return int(match.group(1))


raster_year_files = {}

for path in sorted(Path(RASTER_FOLDER).iterdir()):

    if not path.is_file():
        continue

    if path.suffix.lower() not in (
        ".tif",
        ".tiff"
    ):
        continue

    year = extract_year(path)

    if year is None:
        continue

    if not (
        START_YEAR <= year <= END_YEAR
    ):
        continue

    if year in raster_year_files:
        raise ValueError(
            f"Duplicate GeoTIFF for year {year}:\n"
            f"{raster_year_files[year]}\n"
            f"{path}"
        )

    raster_year_files[year] = str(path)


if not raster_year_files:
    raise RuntimeError(
        "No annual GeoTIFF files found"
    )

years = sorted(raster_year_files)

# Do not launch more processes than the number of years
actual_workers = min(
    N_WORKERS,
    len(years)
)

print(f"Years found    : {years}")
print(f"GeoTIFF files  : {len(years)}")
print(f"Actual workers : {actual_workers}")

print("=" * 75)


# ============================================================
# 5. FUNCTION: READ AND CLEAN AREA RASTER
# ============================================================

def read_clean_area_raster(src):
    """
    Read Harvested Area raster as float64.

    Invalid:
      - GeoTIFF NoData / mask
      - NaN / Inf
      - Explicit NODATA
      - ADDITIONAL_NODATA
      - Values below the area threshold
    """

    masked = src.read(
        RASTER_BAND,
        masked=True
    )

    values = np.array(
        masked.data,
        dtype=np.float64,
        copy=True
    )

    invalid = (
        np.ma.getmaskarray(masked)
    ).copy()

    invalid |= ~np.isfinite(values)
    invalid |= (values == NODATA)

    for nd in ADDITIONAL_NODATA:
        invalid |= (values == nd)

    if STRICTLY_GREATER:

        invalid |= (
            values <= MIN_VALID_AREA
        )

    else:

        invalid |= (
            values < MIN_VALID_AREA
        )

    values[invalid] = NODATA

    return values


# ============================================================
# 6. PARALLEL WORKER: PROCESS ONE YEAR
# ============================================================

def process_one_year(year, raster_path):
    """
    Each worker:
      1. Reads the SHP locally
      2. Reads one year's GeoTIFF
      3. Matches SHP CRS to the raster
      4. Calculates SUM and COUNT for each polygon
      5. Calculates union-area total
      6. Returns lightweight arrays

    The parent process creates and exports Excel.
    """

    year_start = time.perf_counter()

    # --------------------------------------------------------
    # 6.1 Load SHP independently in this worker
    #
    # This avoids sending large geometry objects
    # between worker processes.
    # --------------------------------------------------------

    local_gdf = gpd.read_file(
        SHP_PATH
    ).reset_index(drop=True)

    if len(local_gdf) != N_REGIONS:
        raise ValueError(
            f"{year}: Shapefile feature count changed"
        )

    if local_gdf.crs is None:
        raise ValueError(
            f"{year}: SHP CRS missing"
        )

    # --------------------------------------------------------
    # 6.2 Read yearly raster
    # --------------------------------------------------------

    with rasterio.open(raster_path) as src:

        if src.crs is None:
            raise ValueError(
                f"{year}: Raster CRS missing"
            )

        if src.count < RASTER_BAND:
            raise ValueError(
                f"{year}: Raster band missing"
            )

        raster_crs = src.crs
        affine = src.transform

        raster_values = (
            read_clean_area_raster(src)
        )

    # --------------------------------------------------------
    # 6.3 Reproject SHP to raster CRS
    # --------------------------------------------------------

    if local_gdf.crs != raster_crs:

        projected = local_gdf.to_crs(
            raster_crs
        )

    else:

        projected = local_gdf

    # --------------------------------------------------------
    # 6.4 Select valid polygon geometries
    # --------------------------------------------------------

    geometry_valid = (
        projected.geometry.notna()
        & ~projected.geometry.is_empty
    )

    valid_indices = np.flatnonzero(
        geometry_valid.to_numpy()
    )

    valid_geometries = (
        projected.geometry
        .iloc[valid_indices]
        .tolist()
    )

    # --------------------------------------------------------
    # 6.5 Initialize results for ALL SHP polygons
    # --------------------------------------------------------

    sums = np.full(
        N_REGIONS,
        np.nan,
        dtype=np.float64
    )

    counts = np.zeros(
        N_REGIONS,
        dtype=np.int64
    )

    # --------------------------------------------------------
    # 6.6 Zonal SUM and COUNT
    # --------------------------------------------------------

    if len(valid_geometries) > 0:

        stat_iterator = gen_zonal_stats(
            valid_geometries,
            raster_values,
            affine=affine,
            stats=["sum", "count"],
            nodata=NODATA,
            all_touched=ALL_TOUCHED,
            geojson_out=False
        )

        for original_idx, stat in zip(
            valid_indices,
            stat_iterator
        ):

            count = int(
                stat.get("count") or 0
            )

            total = stat.get("sum")

            counts[original_idx] = count

            if (
                count > 0
                and total is not None
                and np.isfinite(total)
            ):

                sums[original_idx] = float(
                    total
                )

    # --------------------------------------------------------
    # 6.7 Calculate total for all SHP polygons
    #
    # Union prevents overlapping polygon areas
    # from being counted multiple times.
    # --------------------------------------------------------

    annual_total = np.nan
    annual_count = 0

    if (
        CALCULATE_ANNUAL_TOTAL
        and len(valid_geometries) > 0
    ):

        union_geometry = unary_union(
            valid_geometries
        )

        union_stats = next(
            gen_zonal_stats(
                [union_geometry],
                raster_values,
                affine=affine,
                stats=["sum", "count"],
                nodata=NODATA,
                all_touched=ALL_TOUCHED,
                geojson_out=False
            )
        )

        annual_count = int(
            union_stats.get("count") or 0
        )

        union_sum = union_stats.get("sum")

        if (
            annual_count > 0
            and union_sum is not None
            and np.isfinite(union_sum)
        ):

            annual_total = float(
                union_sum
            )

    elapsed = (
        time.perf_counter() - year_start
    )

    # --------------------------------------------------------
    # 6.8 Return results to main process
    # --------------------------------------------------------

    return {
        "year": year,
        "sums": sums,
        "counts": counts,
        "annual_total": annual_total,
        "annual_count": annual_count,
        "valid_regions": int(
            np.count_nonzero(np.isfinite(sums))
        ),
        "seconds": elapsed
    }


# ============================================================
# 7. RUN PARALLEL PROCESSING BY YEAR
# ============================================================

print("\n" + "=" * 75)
print("START PARALLEL PROCESSING")
print("=" * 75)

parallel_results = Parallel(
    n_jobs=actual_workers,
    backend=PARALLEL_BACKEND,
    verbose=PARALLEL_VERBOSE,
    batch_size=1
)(
    delayed(process_one_year)(
        year,
        raster_year_files[year]
    )
    for year in years
)

# Return results in chronological order
parallel_results.sort(
    key=lambda item: item["year"]
)

print("\n" + "=" * 75)
print("PARALLEL CALCULATION COMPLETED")
print("=" * 75)


# ============================================================
# 8. BUILD REGIONAL AND ANNUAL DATAFRAMES
# ============================================================

region_frames = []
annual_total_rows = []

for result in parallel_results:

    year = result["year"]

    # Preserve ALL original SHP attributes
    year_df = attributes_df.copy()

    # Add calculated values
    year_df["year"] = year

    year_df["harvested_area_sum"] = (
        result["sums"]
    )

    year_df["valid_pixels"] = (
        result["counts"]
    )

    region_frames.append(
        year_df
    )

    annual_total_rows.append({
        "year": year,
        "harvested_area_sum": (
            result["annual_total"]
        ),
        "valid_pixels": (
            result["annual_count"]
        )
    })

    print(
        f"{year} | "
        f"Valid regions={result['valid_regions']:,} | "
        f"Annual total={result['annual_total']:,.4f} | "
        f"Time={result['seconds']:.2f}s"
    )


# ============================================================
# 9. COMBINE RESULTS
# ============================================================

df_region = pd.concat(
    region_frames,
    ignore_index=True
)

df_annual = pd.DataFrame(
    annual_total_rows
)

# Original SHP attribute columns stay first
df_region = df_region[
    attribute_columns
    + [
        "year",
        "harvested_area_sum",
        "valid_pixels"
    ]
]

# Convert complex attributes for Excel
for col in attribute_columns:

    df_region[col] = df_region[col].map(
        lambda value: (
            str(value)
            if isinstance(
                value,
                (list, tuple, dict, np.ndarray)
            )
            else value
        )
    )


# ============================================================
# 10. EXPORT EXCEL
# ============================================================

if len(df_region) > 1_048_575:
    raise ValueError(
        "Excel row limit exceeded. "
        "Split output by year or region."
    )

with pd.ExcelWriter(
    OUTPUT_EXCEL,
    engine="openpyxl"
) as writer:

    df_region.to_excel(
        writer,
        sheet_name="By_Region",
        index=False,
        na_rep=""
    )

    df_annual.to_excel(
        writer,
        sheet_name="Annual_Total",
        index=False,
        na_rep=""
    )


# ============================================================
# 11. FINAL SUMMARY
# ============================================================

elapsed_total = (
    time.perf_counter() - start_time
)

print("\n" + "=" * 75)
print("HARVESTED AREA ZONAL SUM - FINISHED")
print("=" * 75)

print(f"Shapefile        : {Path(SHP_PATH).name}")
print(f"Total polygons   : {N_REGIONS:,}")
print(f"Years processed  : {len(years)}")
print(f"Parallel workers : {actual_workers}")
print(f"Output records   : {len(df_region):,}")

print(
    f"Total time       : {elapsed_total:.2f}s"
)

print(
    f"\nExcel saved to:\n{OUTPUT_EXCEL}"
)

print("=" * 75)

print("\nAnnual totals:")
display(df_annual.head(10))

print("\nRegional results:")
display(df_region.head(10))


In [ ]:
# ============================================================
# Global Maize NUE - Removed Harvested Area Assessment
#
# Compare annual NUE rasters BEFORE and AFTER outlier removal.
#
# Removed pixel:
#   Valid NUE BEFORE AND Invalid NUE AFTER
#
# Removed harvested area:
#   SUM(Harvested Area of removed pixels)
#
# Output:
#   Excel: Annual_Removed_Area
#
# Features:
#   - Match three rasters by year (1995-2020)
#   - Use BEFORE NUE as reference spatial coverage
#   - Exclude NoData / NaN / Inf
#   - Count removed NUE pixels
#   - Calculate removed harvested area (ha)
#   - Calculate percentage of harvested area removed
#   - Diagnose unexpected new valid pixels
#   - No resampling or modifications to original rasters
#   - Memory-efficient block processing
#
# JupyterLab - ONE CELL
# ============================================================

import os
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio

from rasterio.windows import Window
from IPython.display import display


# ============================================================
# 1. USER CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# NUE BEFORE outlier removal
# ------------------------------------------------------------

NUE_BEFORE_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\02Global_NUE_Maize\02MaizeNUE_M2_20261029V1_Median_Intersection"
)

# ------------------------------------------------------------
# NUE AFTER outlier removal
# ------------------------------------------------------------

NUE_AFTER_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\02Global_NUE_Maize\02MaizeNUE_M2_20261029V1_M6_150R"
)

# ------------------------------------------------------------
# Annual Maize Harvested Area
#
# Expected:
#   Maize_HarArea_1995.tif
#  Maize_HarArea_1996.tif
#   ...
#   Maize_HarArea_2020.tif
# ------------------------------------------------------------

HARVESTED_AREA_FOLDER = (
    r"Z:\staff\Qiang_S\09_Datasets\09OtherDataUsed\GCSN_HarvestedArea_1995_2020\02StandardizedDatasets\02MaizeHarvestedArea"
)

# ------------------------------------------------------------
# Output Excel
# ------------------------------------------------------------

OUTPUT_EXCEL = os.path.join(
    NUE_AFTER_FOLDER,
    "Maize_NUE_Removed_HarvestedArea_1995_2020.xlsx"
)

# ------------------------------------------------------------
# Year selection
# ------------------------------------------------------------

START_YEAR = 1995
END_YEAR = 2020

# ------------------------------------------------------------
# GeoTIFF name matching
#
# BEFORE and AFTER:
#   NUE_median_Opt_2000.tif
#
# BEFORE_PATTERN also allows another prefix,
# provided there is only one file per year.
# ------------------------------------------------------------

BEFORE_PATTERN = (
    r"^.*_(?P<year>(?:19|20)\d{2})"
    r"\.(?:tif|tiff)$"
)

AFTER_PATTERN = (
    r"^NUE_median_Opt_"
    r"(?P<year>(?:19|20)\d{2})"
    r"\.(?:tif|tiff)$"
)

AREA_PATTERN = (
    r"^Maize_HarArea_"
    r"(?P<year>(?:19|20)\d{2})"
    r"\.(?:tif|tiff)$"
)

# ------------------------------------------------------------
# Invalid pixel values
#
# -9999 and -1 are treated as original NoData/sentinel.
#
# NOTE:
# If -1 is a genuine numeric outlier in the BEFORE data
# rather than a missing-value sentinel, remove -1 from
# BEFORE_INVALID_VALUES to count its removal.
# ------------------------------------------------------------

BEFORE_INVALID_VALUES = [-9999.0, -1.0]
AFTER_INVALID_VALUES = [-9999.0, -1.0]

AREA_INVALID_VALUES = [-9999.0, -1.0]

# ------------------------------------------------------------
# Harvested Area condition
#
# Only HA > 0 contributes to area statistics.
# ------------------------------------------------------------

AREA_THRESHOLD = 0.0

# ------------------------------------------------------------
# Other settings
# ------------------------------------------------------------

RASTER_BAND = 1

BLOCK_SIZE = 512

# True: Stop if any BEFORE or AFTER year lacks a match
# False: Only process years matched across all 3 folders
REQUIRE_COMPLETE_MATCH = True


# ============================================================
# 2. VALIDATE FOLDERS
# ============================================================

start_time = time.perf_counter()

for folder, label in [
    (NUE_BEFORE_FOLDER, "NUE BEFORE"),
    (NUE_AFTER_FOLDER, "NUE AFTER"),
    (HARVESTED_AREA_FOLDER, "Harvested Area")
]:

    if not os.path.isdir(folder):
        raise FileNotFoundError(
            f"{label} folder not found:\n{folder}"
        )

if START_YEAR > END_YEAR:
    raise ValueError(
        "START_YEAR must be <= END_YEAR"
    )

if BLOCK_SIZE <= 0:
    raise ValueError(
        "BLOCK_SIZE must be positive"
    )


# ============================================================
# 3. INDEX ANNUAL GEOTIFF FILES
# ============================================================

def index_yearly_rasters(folder, filename_pattern):

    pattern = re.compile(
        filename_pattern,
        flags=re.IGNORECASE
    )

    index = {}

    # Search folder and subfolders
    for path in sorted(Path(folder).rglob("*")):

        if not path.is_file():
            continue

        if path.suffix.lower() not in (
            ".tif",
            ".tiff"
        ):
            continue

        match = pattern.fullmatch(
            path.name
        )

        if match is None:
            continue

        year = int(
            match.group("year")
        )

        if not (
            START_YEAR <= year <= END_YEAR
        ):
            continue

        if year in index:
            raise ValueError(
                f"Duplicate GeoTIFF for {year}:\n"
                f"{index[year]}\n{path}"
            )

        index[year] = str(path)

    return index


before_files = index_yearly_rasters(
    NUE_BEFORE_FOLDER,
    BEFORE_PATTERN
)

after_files = index_yearly_rasters(
    NUE_AFTER_FOLDER,
    AFTER_PATTERN
)

area_files = index_yearly_rasters(
    HARVESTED_AREA_FOLDER,
    AREA_PATTERN
)

if not before_files:
    raise RuntimeError(
        "No BEFORE NUE GeoTIFF files found"
    )

if not after_files:
    raise RuntimeError(
        "No AFTER NUE GeoTIFF files found"
    )

if not area_files:
    raise RuntimeError(
        "No annual Harvested Area files found"
    )

# Check annual coverage
nue_years = (
    set(before_files)
    | set(after_files)
)

missing_before = sorted(
    nue_years - set(before_files)
)

missing_after = sorted(
    nue_years - set(after_files)
)

missing_area = sorted(
    nue_years - set(area_files)
)

matched_years = sorted(
    set(before_files)
    & set(after_files)
    & set(area_files)
)

print("=" * 75)
print("GLOBAL NUE OUTLIER REMOVAL - HARVESTED AREA")
print("=" * 75)

print(f"BEFORE NUE files : {len(before_files)}")
print(f"AFTER NUE files  : {len(after_files)}")
print(f"Harvest Area     : {len(area_files)}")
print(f"Matched years    : {len(matched_years)}")

print(f"Missing BEFORE   : {missing_before}")
print(f"Missing AFTER    : {missing_after}")
print(f"Missing HA       : {missing_area}")

if REQUIRE_COMPLETE_MATCH and (
    missing_before
    or missing_after
    or missing_area
):

    raise FileNotFoundError(
        "Annual raster matching is incomplete. "
        "Check the filenames and folders above."
    )

if not matched_years:
    raise RuntimeError(
        "No common years among three datasets"
    )

print(f"\nYears to process: {matched_years}")

print("=" * 75)


# ============================================================
# 4. VALID PIXEL FUNCTIONS
# ============================================================

def get_valid_mask(masked_array, invalid_values):
    """
    Valid pixel:
      - Not masked by raster metadata
      - Finite
      - Not equal to configured invalid values
    """

    values = np.ma.getdata(
        masked_array
    )

    valid = (
        ~np.ma.getmaskarray(masked_array)
    ).copy()

    valid &= np.isfinite(values)

    for invalid_value in invalid_values:

        valid &= (
            values != invalid_value
        )

    return values, valid


def verify_grid_alignment(src_before, src_other, label):
    """
    Verify existing spatial correspondence.
    No resampling.
    """

    if src_before.crs != src_other.crs:
        raise ValueError(
            f"{label}: CRS mismatch"
        )

    if (
        src_before.width != src_other.width
        or src_before.height != src_other.height
    ):
        raise ValueError(
            f"{label}: Raster dimensions mismatch"
        )

    if not src_before.transform.almost_equals(
        src_other.transform,
        precision=1e-9
    ):
        raise ValueError(
            f"{label}: Raster grids are not aligned"
        )


# ============================================================
# 5. PROCESS ONE YEAR
# ============================================================

def process_one_year(year):

    t0 = time.perf_counter()

    before_path = before_files[year]
    after_path = after_files[year]
    area_path = area_files[year]

    # Counters
    before_valid_pixels = 0
    after_valid_pixels = 0

    removed_pixels = 0
    removed_pixels_with_ha = 0

    retained_pixels = 0
    new_valid_pixels = 0

    # Harvested area sums
    before_ha_sum = 0.0
    removed_ha_sum = 0.0
    retained_ha_sum = 0.0
    new_valid_ha_sum = 0.0

    with (
        rasterio.open(before_path) as src_before,
        rasterio.open(after_path) as src_after,
        rasterio.open(area_path) as src_area
    ):

        # ----------------------------------------------------
        # Verify raster alignment
        # ----------------------------------------------------

        verify_grid_alignment(
            src_before,
            src_after,
            f"{year} BEFORE / AFTER"
        )

        verify_grid_alignment(
            src_before,
            src_area,
            f"{year} NUE / Harvested Area"
        )

        width = src_before.width
        height = src_before.height

        # ----------------------------------------------------
        # Process in blocks
        # ----------------------------------------------------

        for yoff in range(0, height, BLOCK_SIZE):

            ysize = min(
                BLOCK_SIZE,
                height - yoff
            )

            for xoff in range(
                0,
                width,
                BLOCK_SIZE
            ):

                xsize = min(
                    BLOCK_SIZE,
                    width - xoff
                )

                window = Window(
                    xoff,
                    yoff,
                    xsize,
                    ysize
                )

                # ============================================
                # Read matching pixels
                # ============================================

                before_ma = src_before.read(
                    RASTER_BAND,
                    window=window,
                    masked=True
                )

                after_ma = src_after.read(
                    RASTER_BAND,
                    window=window,
                    masked=True
                )

                area_ma = src_area.read(
                    RASTER_BAND,
                    window=window,
                    masked=True
                )

                # ============================================
                # BEFORE NUE validity
                # ============================================

                _, before_valid = get_valid_mask(
                    before_ma,
                    BEFORE_INVALID_VALUES
                )

                # ============================================
                # AFTER NUE validity
                # ============================================

                _, after_valid = get_valid_mask(
                    after_ma,
                    AFTER_INVALID_VALUES
                )

                # ============================================
                # Harvested Area validity
                # ============================================

                area_values, area_valid = get_valid_mask(
                    area_ma,
                    AREA_INVALID_VALUES
                )

                area_valid &= (
                    area_values > AREA_THRESHOLD
                )

                # ============================================
                # Removed NUE pixels
                #
                # Valid BEFORE, invalid AFTER
                # ============================================

                removed_mask = (
                    before_valid
                    & ~after_valid
                )

                # ============================================
                # Retained NUE pixels
                # ============================================

                retained_mask = (
                    before_valid
                    & after_valid
                )

                # ============================================
                # Unexpected newly valid pixels
                # ============================================

                new_mask = (
                    ~before_valid
                    & after_valid
                )

                # ============================================
                # Pixel counters
                # ============================================

                before_valid_pixels += int(
                    np.count_nonzero(before_valid)
                )

                after_valid_pixels += int(
                    np.count_nonzero(after_valid)
                )

                removed_pixels += int(
                    np.count_nonzero(removed_mask)
                )

                retained_pixels += int(
                    np.count_nonzero(retained_mask)
                )

                new_valid_pixels += int(
                    np.count_nonzero(new_mask)
                )

                # ============================================
                # Apply harvested-area validity
                # ============================================

                before_ha_mask = (
                    before_valid
                    & area_valid
                )

                removed_ha_mask = (
                    removed_mask
                    & area_valid
                )

                retained_ha_mask = (
                    retained_mask
                    & area_valid
                )

                new_ha_mask = (
                    new_mask
                    & area_valid
                )

                removed_pixels_with_ha += int(
                    np.count_nonzero(
                        removed_ha_mask
                    )
                )

                # ============================================
                # Area sums
                #
                # Unit = original HA raster unit
                # ============================================

                before_ha_sum += float(
                    np.sum(
                        area_values[before_ha_mask],
                        dtype=np.float64
                    )
                )

                removed_ha_sum += float(
                    np.sum(
                        area_values[removed_ha_mask],
                        dtype=np.float64
                    )
                )

                retained_ha_sum += float(
                    np.sum(
                        area_values[retained_ha_mask],
                        dtype=np.float64
                    )
                )

                new_valid_ha_sum += float(
                    np.sum(
                        area_values[new_ha_mask],
                        dtype=np.float64
                    )
                )

    # ========================================================
    # Annual percentage
    # ========================================================

    if before_ha_sum > 0:

        removed_percent = (
            removed_ha_sum
            / before_ha_sum
            * 100.0
        )

    else:

        removed_percent = np.nan

    elapsed = (
        time.perf_counter() - t0
    )

    # Return annual summary
    return {
        "Year": year,

        "Before_Valid_Pixels": before_valid_pixels,
        "After_Valid_Pixels": after_valid_pixels,

        "Removed_Pixels": removed_pixels,
        "Removed_Pixels_With_HA": removed_pixels_with_ha,

        "Retained_Pixels": retained_pixels,
        "New_Valid_Pixels": new_valid_pixels,

        "Before_HA_ha": before_ha_sum,
        "Removed_HA_ha": removed_ha_sum,
        "Retained_HA_ha": retained_ha_sum,
        "New_Valid_HA_ha": new_valid_ha_sum,

        "Removed_HA_Percent": removed_percent,

        "Processing_Seconds": elapsed
    }


# ============================================================
# 6. LOOP THROUGH ALL MATCHED YEARS
# ============================================================

results = []

for i, year in enumerate(
    matched_years,
    start=1
):

    print("\n" + "-" * 75)

    print(
        f"[{i}/{len(matched_years)}] "
        f"Processing year {year}"
    )

    result = process_one_year(year)

    results.append(result)

    print(
        f"Before NUE pixels : "
        f"{result['Before_Valid_Pixels']:,}"
    )

    print(
        f"Removed NUE pixels: "
        f"{result['Removed_Pixels']:,}"
    )

    print(
        f"Before HA         : "
        f"{result['Before_HA_ha']:,.2f} ha"
    )

    print(
        f"Removed HA        : "
        f"{result['Removed_HA_ha']:,.2f} ha"
    )

    print(
        f"Removed HA percent: "
        f"{result['Removed_HA_Percent']:.4f}%"
    )

    if result["New_Valid_Pixels"] > 0:

        print(
            "WARNING: Some pixels are valid AFTER "
            "but invalid BEFORE."
        )

        print(
            "New valid pixels: "
            f"{result['New_Valid_Pixels']:,}"
        )


# ============================================================
# 7. EXPORT ANNUAL RESULTS
# ============================================================

df_result = pd.DataFrame(
    results
)

df_result = df_result.sort_values(
    "Year"
).reset_index(drop=True)

df_result.to_excel(
    OUTPUT_EXCEL,
    index=False,
    sheet_name="Annual_Removed_Area",
    engine="openpyxl"
)


# ============================================================
# 8. FINAL SUMMARY
# ============================================================

elapsed_total = (
    time.perf_counter()
    - start_time
)

print("\n" + "=" * 75)
print("GLOBAL Maize  NUE REMOVED AREA - COMPLETED")
print("=" * 75)

print(f"Processed years: {len(df_result)}")

print(
    f"Total processing time: "
    f"{elapsed_total:.2f}s"
)

print(f"\nExcel saved to:\n{OUTPUT_EXCEL}")

print("=" * 75)

display(
    df_result[
        [
            "Year",
            "Before_HA_ha",
            "Removed_HA_ha",
            "Retained_HA_ha",
            "Removed_HA_Percent",
            "Removed_Pixels",
            "Removed_Pixels_With_HA"
        ]
    ]
)


In [ ]:
# ============================================================
# Batch GeoTIFF Processing + OPTIONAL Annual NUE Mask
#
# 1. Remove original NoData and specific invalid values
# 2. Multiply valid values by ratio
# 3. Remove values outside [range_min, range_max]
# 4. OPTIONAL: Apply annual NUE spatial mask
# 5. Preserve original GeoTIFF spatial properties
# 6. NEW: Configurable output GeoTIFF filenames
#
# JupyterLab - Single Cell
# ============================================================

from osgeo import gdal
import numpy as np
import os
import glob
import re

gdal.UseExceptions()


# ============================================================
# 1. USER PARAMETERS
# ============================================================

# ------------------------------------------------------------
# Input folder
# ------------------------------------------------------------

input_folder = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\NUE_Upload_Zenodo_V2.0\02MaizeNUE_M2_20261029V1_M6_150R"
)

# ------------------------------------------------------------
# Output folder
# ------------------------------------------------------------

output_folder = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\NUE_Upload_Zenodo_V2.0\02_NUE_Maize_V2.0\NUE_Maize_Median_1995_2020"
)

# ------------------------------------------------------------
# NEW: OUTPUT GEOTIFF FILENAME
#
# Examples:
#
# "New_NUE_{year}.tif"
#     -> New_NUE_1995.tif
#     -> New_NUE_2000.tif
#
# "Wheat_NUE_{year}.tif"
#     -> Wheat_NUE_2000.tif
#
# "NUE_median_Opt_{year}.tif"
#     -> NUE_median_Opt_2000.tif
#
# "{original_name}"
#     -> Keep original input filename
#
# IMPORTANT:
# When using {year}, each input file must
# have a filename ending in _YYYY.tif/.tiff.
# ------------------------------------------------------------

OUTPUT_FILENAME_TEMPLATE = "NUE_Maize_Median_{year}.tif"

# ============================================================
# 2. OPTIONAL NUE MASK SETTINGS
# ============================================================

# ------------------------------------------------------------
# ENABLE / DISABLE NUE MASK
#
# True  = Apply annual NUE mask
# False = Do not use NUE mask
# ------------------------------------------------------------

USE_NUE_MASK = True

# ------------------------------------------------------------
# Annual NUE mask folder
#
# Only used when USE_NUE_MASK = True
# ------------------------------------------------------------

nue_mask_folder = (
    r"Z:\staff\Qiang_S\09_Datasets\03GlobalNUE_Results\NUE_Upload_Zenodo_V2.0\02MaizeNUE_M2_20261029V1_M6_150R"
)

# Annual mask filename:
# NUE_median_Opt_1995.tif

NUE_MASK_PREFIX = "NUE_median_Opt_"

# ------------------------------------------------------------
# Mask validity condition
#
# False:
#   NUE mask pixel must be finite and not NoData
#
# True:
#   NUE mask pixel must also be > 0
#
# Only effective when USE_NUE_MASK = True.
# ------------------------------------------------------------

MASK_REQUIRE_POSITIVE = False


# ============================================================
# 3. NUMERICAL PROCESSING SETTINGS
# ============================================================

# ------------------------------------------------------------
# Multiply original values by this ratio
# ------------------------------------------------------------

ratio = 100.0 #for NUE
#ratio = 1.0  # Noutput


# ------------------------------------------------------------
# Remove specific ORIGINAL values
#
# Checked BEFORE multiplication.
# ------------------------------------------------------------

replace_values = [-1]

# ------------------------------------------------------------
# Valid output range
#
# Keep:
#     range_min <= value <= range_max
#
# Remove:
#     value < range_min OR value > range_max
#
# Applied AFTER multiplication.
# ------------------------------------------------------------

range_min = 0.0
range_max = 150.0 #for NUE
#range_max = 10000.0 #for Noutput

# ------------------------------------------------------------
# Output NoData value
# ------------------------------------------------------------

nodata_value = -9999.0

# ------------------------------------------------------------
# Additional input NoData values
# ------------------------------------------------------------

additional_nodata = [-9999.0]

# ------------------------------------------------------------
# Block processing
# ------------------------------------------------------------

BLOCK_SIZE = 512


# ============================================================
# 4. VALIDATE CONFIGURATION
# ============================================================

if not os.path.isdir(input_folder):
    raise FileNotFoundError(
        f"Input folder not found:\n{input_folder}"
    )

if USE_NUE_MASK:
    if not os.path.isdir(nue_mask_folder):
        raise FileNotFoundError(
            f"NUE mask folder not found:\n{nue_mask_folder}"
        )

if os.path.abspath(input_folder) == os.path.abspath(output_folder):
    raise ValueError(
        "Input and output folders must be different."
    )

if not np.isfinite(ratio):
    raise ValueError(
        "ratio must be finite"
    )

if not (
    np.isfinite(range_min)
    and np.isfinite(range_max)
    and range_min <= range_max
):
    raise ValueError(
        "Invalid output value range"
    )

if range_min <= nodata_value <= range_max:
    raise ValueError(
        "Output NoData must be outside the valid range"
    )

if BLOCK_SIZE <= 0:
    raise ValueError(
        "BLOCK_SIZE must be positive"
    )

if not OUTPUT_FILENAME_TEMPLATE:
    raise ValueError(
        "OUTPUT_FILENAME_TEMPLATE cannot be empty"
    )

os.makedirs(
    output_folder,
    exist_ok=True
)


# ============================================================
# 5. COLLECT INPUT GEOTIFF FILES
# ============================================================

tif_files = sorted(
    glob.glob(
        os.path.join(input_folder, "*.tif")
    )
    +
    glob.glob(
        os.path.join(input_folder, "*.tiff")
    )
)


def extract_year(filepath):
    """
    Extract year from filename ending in:

        xxx_1995.tif
        xxx_2000.tif
        xxx_2020.tiff
    """

    match = re.search(
        r"_(19\d{2}|20\d{2})\.(?:tif|tiff)$",
        os.path.basename(filepath),
        flags=re.IGNORECASE
    )

    if match:
        return int(match.group(1))

    return None


# ============================================================
# 6. MATCH ANNUAL FILES AND PREPARE OUTPUT NAMES
# ============================================================

matched_files = []
skipped_files = []

# Prevent different input files from overwriting
# the same output filename.
planned_output_names = set()

for in_file in tif_files:

    fname = os.path.basename(in_file)

    year = extract_year(in_file)

    # --------------------------------------------------------
    # Match annual NUE mask only when enabled
    # --------------------------------------------------------

    if USE_NUE_MASK:

        if year is None:

            skipped_files.append(
                (
                    fname,
                    "Year not found in filename"
                )
            )

            continue

        mask_file = os.path.join(
            nue_mask_folder,
            f"{NUE_MASK_PREFIX}{year}.tif"
        )

        if not os.path.isfile(mask_file):

            skipped_files.append(
                (
                    fname,
                    f"NUE mask missing for {year}"
                )
            )

            continue

    else:

        mask_file = None

    # --------------------------------------------------------
    # NEW: Generate output filename
    # --------------------------------------------------------

    if (
        year is None
        and "{year}" in OUTPUT_FILENAME_TEMPLATE
    ):
        raise ValueError(
            f"Cannot extract year from {fname}. "
            "The output filename template requires {year}."
        )

    try:

        output_name = OUTPUT_FILENAME_TEMPLATE.format(
            year=year,
            original_name=fname
        )

    except (KeyError, ValueError) as e:

        raise ValueError(
            "Invalid OUTPUT_FILENAME_TEMPLATE. "
            "Use {year} and/or {original_name}."
        ) from e

    # Check filename and extension
    if (
        not output_name
        or os.path.basename(output_name) != output_name
        or "/" in output_name
        or "\\" in output_name
        or not output_name.lower().endswith(
            (".tif", ".tiff")
        )
    ):
        raise ValueError(
            f"Invalid output filename: {output_name}"
        )

    # Avoid output name collisions
    output_key = output_name.lower()

    if output_key in planned_output_names:

        raise ValueError(
            f"Duplicate output filename: {output_name}\n"
            "Different input files would overwrite each other."
        )

    planned_output_names.add(output_key)

    out_file = os.path.join(
        output_folder,
        output_name
    )

    matched_files.append(
        (
            year,
            in_file,
            mask_file,
            out_file
        )
    )


# ============================================================
# 7. INITIAL SUMMARY
# ============================================================

print("=" * 75)
print("BATCH GEOTIFF PROCESSING")
print("=" * 75)

print(f"Input files      : {len(tif_files)}")
print(f"Files to process : {len(matched_files)}")
print(f"Skipped files    : {len(skipped_files)}")

print(
    f"NUE mask         : "
    f"{'ENABLED' if USE_NUE_MASK else 'DISABLED'}"
)

if USE_NUE_MASK:

    print(
        f"Mask > 0 required: {MASK_REQUIRE_POSITIVE}"
    )

print(f"Multiply ratio   : {ratio}")

print(
    f"Valid range      : "
    f"[{range_min}, {range_max}]"
)

print(f"Output NoData    : {nodata_value}")

print(
    f"Output filename  : "
    f"{OUTPUT_FILENAME_TEMPLATE}"
)

print("=" * 75)

if not matched_files:

    raise RuntimeError(
        "No GeoTIFF files available for processing"
    )


# ============================================================
# 8. PROCESS FILES
# ============================================================

for i, (
    year,
    in_file,
    mask_file,
    out_file
) in enumerate(
    matched_files,
    start=1
):

    fname = os.path.basename(in_file)

    output_name = os.path.basename(out_file)

    print("\n" + "-" * 75)

    print(
        f"[{i}/{len(matched_files)}] "
        f"Processing year: {year}"
    )

    print(f"  Input : {fname}")

    print(f"  Output: {output_name}")

    # --------------------------------------------------------
    # Open input raster
    # --------------------------------------------------------

    src = gdal.Open(
        in_file,
        gdal.GA_ReadOnly
    )

    if src is None:

        raise RuntimeError(
            f"Cannot open: {in_file}"
        )

    if src.RasterCount != 1:

        raise ValueError(
            f"Expected single-band raster: {fname}"
        )

    src_band = src.GetRasterBand(1)

    src_nodata = src_band.GetNoDataValue()

    width = src.RasterXSize
    height = src.RasterYSize

    # --------------------------------------------------------
    # Open annual NUE mask if enabled
    # --------------------------------------------------------

    mask_ds = None
    mask_band = None
    mask_nodata = None
    nue_has_mask = False

    if USE_NUE_MASK:

        mask_ds = gdal.Open(
            mask_file,
            gdal.GA_ReadOnly
        )

        if mask_ds is None:

            raise RuntimeError(
                f"Cannot open NUE mask: {mask_file}"
            )

        if (
            width != mask_ds.RasterXSize
            or height != mask_ds.RasterYSize
        ):

            raise ValueError(
                f"Raster dimensions differ: {fname}"
            )

        mask_band = mask_ds.GetRasterBand(1)

        mask_nodata = mask_band.GetNoDataValue()

        nue_has_mask = not (
            mask_band.GetMaskFlags()
            & gdal.GMF_ALL_VALID
        )

        print(
            f"  Annual NUE mask: "
            f"{os.path.basename(mask_file)}"
        )

    else:

        print(
            "  NUE mask: DISABLED"
        )

    # --------------------------------------------------------
    # Create output GeoTIFF
    # --------------------------------------------------------

    driver = gdal.GetDriverByName("GTiff")

    out_ds = driver.Create(
        out_file,
        width,
        height,
        1,
        gdal.GDT_Float32,
        options=[
            "COMPRESS=DEFLATE",
            "PREDICTOR=3",
            "TILED=YES",
            "BIGTIFF=IF_SAFER"
        ]
    )

    if out_ds is None:

        raise RuntimeError(
            f"Cannot create output: {out_file}"
        )

    # --------------------------------------------------------
    # Preserve original GeoTIFF spatial properties
    # --------------------------------------------------------

    out_ds.SetGeoTransform(
        src.GetGeoTransform()
    )

    out_ds.SetProjection(
        src.GetProjection()
    )

    out_ds.SetMetadata(
        src.GetMetadata()
    )

    out_band = out_ds.GetRasterBand(1)

    out_band.SetNoDataValue(
        nodata_value
    )

    out_band.SetDescription(
        src_band.GetDescription()
    )

    out_band.SetUnitType(
        src_band.GetUnitType()
    )

    # Preserve band metadata, excluding obsolete stats
    metadata = src_band.GetMetadata() or {}

    metadata = {
        k: v
        for k, v in metadata.items()
        if not k.upper().startswith("STATISTICS_")
    }

    out_band.SetMetadata(
        metadata
    )

    # --------------------------------------------------------
    # Statistics
    # --------------------------------------------------------

    total_pixels = 0

    valid_pixels = 0

    removed_by_mask = 0

    removed_by_range = 0

    # Respect original GDAL mask
    src_has_mask = not (
        src_band.GetMaskFlags()
        & gdal.GMF_ALL_VALID
    )

    # --------------------------------------------------------
    # Process blocks
    # --------------------------------------------------------

    for yoff in range(
        0,
        height,
        BLOCK_SIZE
    ):

        ysize = min(
            BLOCK_SIZE,
            height - yoff
        )

        for xoff in range(
            0,
            width,
            BLOCK_SIZE
        ):

            xsize = min(
                BLOCK_SIZE,
                width - xoff
            )

            # ================================================
            # STEP 1: Read original values
            # ================================================

            arr = src_band.ReadAsArray(
                xoff,
                yoff,
                xsize,
                ysize
            ).astype(
                np.float64
            )

            total_pixels += arr.size

            # ================================================
            # STEP 2: Original valid pixel mask
            # ================================================

            input_valid = np.isfinite(
                arr
            )

            if src_nodata is not None:

                if np.isfinite(src_nodata):

                    input_valid &= (
                        arr != src_nodata
                    )

            for nd in additional_nodata:

                input_valid &= (
                    arr != nd
                )

            # Remove specific values before multiplication
            for v in replace_values:

                input_valid &= (
                    arr != v
                )

            # Respect source GDAL mask
            if src_has_mask:

                src_mask_arr = (
                    src_band.GetMaskBand().ReadAsArray(
                        xoff,
                        yoff,
                        xsize,
                        ysize
                    )
                )

                input_valid &= (
                    src_mask_arr != 0
                )

            # ================================================
            # STEP 3: Optional annual NUE mask
            # ================================================

            if USE_NUE_MASK:

                nue_arr = mask_band.ReadAsArray(
                    xoff,
                    yoff,
                    xsize,
                    ysize
                )

                mask_valid = np.isfinite(
                    nue_arr
                )

                # Exclude mask metadata NoData
                if mask_nodata is not None:

                    if np.isfinite(mask_nodata):

                        mask_valid &= (
                            nue_arr != mask_nodata
                        )

                # Exclude explicit -9999
                mask_valid &= (
                    nue_arr != nodata_value
                )

                # Respect mask raster GDAL validity mask
                if nue_has_mask:

                    nue_mask_arr = (
                        mask_band.GetMaskBand().ReadAsArray(
                            xoff,
                            yoff,
                            xsize,
                            ysize
                        )
                    )

                    mask_valid &= (
                        nue_mask_arr != 0
                    )

                # Optional positive NUE condition
                if MASK_REQUIRE_POSITIVE:

                    mask_valid &= (
                        nue_arr > 0
                    )

            else:

                # No spatial restriction when disabled
                mask_valid = np.ones(
                    arr.shape,
                    dtype=bool
                )

            removed_by_mask += int(
                np.count_nonzero(
                    input_valid & ~mask_valid
                )
            )

            # ================================================
            # STEP 4: Combine valid masks
            # ================================================

            common_valid = (
                input_valid
                & mask_valid
            )

            # ================================================
            # STEP 5: Multiply by ratio
            # ================================================

            scaled = np.full(
                arr.shape,
                np.nan,
                dtype=np.float64
            )

            scaled[common_valid] = (
                arr[common_valid] * ratio
            )

            # ================================================
            # STEP 6: Keep valid numerical range
            #
            # range_min <= scaled <= range_max
            # ================================================

            range_valid = (
                np.isfinite(scaled)
                & (scaled >= range_min)
                & (scaled <= range_max)
                & (
                    np.abs(scaled)
                    <= np.finfo(np.float32).max
                )
            )

            removed_by_range += int(
                np.count_nonzero(
                    common_valid
                    & ~range_valid
                )
            )

            final_valid = (
                common_valid
                & range_valid
            )

            # ================================================
            # STEP 7: Prepare output pixel values
            # ================================================

            output_arr = np.full(
                arr.shape,
                nodata_value,
                dtype=np.float32
            )

            output_arr[final_valid] = (
                scaled[final_valid].astype(
                    np.float32
                )
            )

            valid_pixels += int(
                np.count_nonzero(
                    final_valid
                )
            )

            # ================================================
            # STEP 8: Write GeoTIFF block
            # ================================================

            out_band.WriteArray(
                output_arr,
                xoff,
                yoff
            )

    # --------------------------------------------------------
    # Finalize current GeoTIFF
    # --------------------------------------------------------

    out_band.FlushCache()
    out_ds.FlushCache()

    out_band = None
    out_ds = None

    src_band = None
    mask_band = None

    src = None
    mask_ds = None

    # --------------------------------------------------------
    # Print annual statistics
    # --------------------------------------------------------

    print(
        f"  Total pixels       : "
        f"{total_pixels:,}"
    )

    print(
        f"  Removed by NUE mask: "
        f"{removed_by_mask:,}"
    )

    print(
        f"  Removed by range   : "
        f"{removed_by_range:,}"
    )

    print(
        f"  Final valid pixels : "
        f"{valid_pixels:,}"
    )

    print(
        f"  Saved as           : "
        f"{output_name}"
    )

    if valid_pixels == 0:

        print(
            "  WARNING: No valid pixels remain!"
        )


# ============================================================
# 9. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 75)

print("ALL GEOTIFF FILES FINISHED")

print("=" * 75)

print(f"Input files : {len(tif_files)}")

print(f"Processed   : {len(matched_files)}")

print(f"Skipped     : {len(skipped_files)}")

print(
    f"NUE mask    : "
    f"{'ENABLED' if USE_NUE_MASK else 'DISABLED'}"
)

print(
    f"Output name : "
    f"{OUTPUT_FILENAME_TEMPLATE}"
)

if skipped_files:

    print("\nSkipped files:")

    for fname, reason in skipped_files:

        print(
            f"  {fname}: {reason}"
        )

print(
    f"\nOutput directory:\n"
    f"{output_folder}"
)

print("=" * 75)